## Load modules and packages

In [1]:
import argparse
import numpy as np
import os.path as osp
import logging


import torch
from torch_geometric import seed_everything
from tqdm import tqdm

import random
import os

def set_seed(seed: int):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed) 
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.enabled = False

In [2]:
import sys
from pathlib import Path

project_root = Path.cwd().parent
sys.path.append(str(project_root))

from mhiscl.dataset import load_dataset
from history import History
from mhiscl.loader import EventLoader
from mhiscl.measure import Measure
from mhiscl.model import GatedTGAT,TGAT,GatedTGNN
from mhiscl.loss import contrastive_loss, cosine_similarity

## Initialize the parameters

In [3]:
from types import SimpleNamespace

def get_args():
    args = SimpleNamespace(
        dataset='wikipedia',
        alpha=1,
        con='all',
        time_enc='sste',
        aggr='gat',
        num_neighbors=[15, 10],
        num_workers=8,
        hidden_channels=64,
        num_timeslots=14,
        heads=8,
        nums_layers=2,
        dropout=0.3,
        lr=5e-4,
        epochs=35,
        history_retrieve='mean',
        recurrent='gru',
        tau=0.02,
        batch_size=256,
        zero_edge=False,
        gpu=1,
        gamma =0.3,
        beta=0.5,
        delta=0.5,
        num_neg=64
    )

    print(args)
    return args


# Call this in your notebook
args = get_args()


namespace(dataset='wikipedia', alpha=1, con='all', time_enc='sste', aggr='gat', num_neighbors=[15, 10], num_workers=8, hidden_channels=64, num_timeslots=14, heads=8, nums_layers=2, dropout=0.3, lr=0.0005, epochs=35, history_retrieve='mean', recurrent='gru', tau=0.02, batch_size=256, zero_edge=False, gpu=1, gamma=0.3, beta=0.5, delta=0.5, num_neg=64)


In [4]:
if not os.path.exists('log'):
    os.makedirs('log')
logging.basicConfig(format="%(asctime)s - %(levelname)s - %(message)s", level=logging.INFO, filename=f'log/test_{args.dataset}.txt')
log = logging.getLogger() 
log.info('########################## START #########################')
log.info(f'\n{args.dataset}-param: {args.__dict__}')

In [5]:
data = load_dataset(args.dataset,zero_edge=args.zero_edge)
device = torch.device(f"cuda:{args.gpu}")
print(f"Using {device}")

set_seed(42)
sample_args = {"num_neighbors": args.num_neighbors,
            "num_workers": args.num_workers}

train_loader = EventLoader(
    data,
    input_events=data.train_mask,
    shuffle=True,
    batch_size=args.batch_size,
    **sample_args,
)

valid_loader = EventLoader(
    data,
    input_events=data.val_mask, 
    shuffle=False,
    batch_size=args.batch_size,
    **sample_args,
)

test_loader = EventLoader(
    data,
    input_events=data.test_mask,
    shuffle=False,
    batch_size=args.batch_size,
    **sample_args,
)

history = History(
    data.num_nodes,
    args.num_timeslots,
    args.hidden_channels,
    device=device,
    history_retrieve=args.history_retrieve,
    recurrent=args.recurrent,
).to(device)


==================== Data statistics ====================
Name: wikipedia
Number of nodes: 9227
Number of edges: 157474
Number of node features: 1
Number of edge features: 172
Number of anomalies: 0.138%
Using cuda:1


/home/wassim/Contrastive_Learning/MHisCL/venv/lib/python3.9/site-packages/torch/cuda/__init__.py:654: UserWarning: Can't initialize NVML
  warnings.warn("Can't initialize NVML")


In [6]:
for batch in train_loader:
    break

In [7]:
batch

TemporalData(src=[3321], dst=[3321], t=[3321], msg=[3321, 172], y=[3321], train_mask=[3321], val_mask=[3321], test_mask=[3321], x=[599, 1], n_id=[599], e_id=[3321], input_id=[256], batch_size=256)

In [6]:
## Training Function 

def train(loader, model, history, optimizer, device, data, args):
    model.train()
    history.train()
    pbar = tqdm(loader)
    total_loss = 0
    for batch in pbar:
        batch = batch.to(device)
        optimizer.zero_grad()
        out = model.encode(batch.x, batch.edge_index, batch.t, batch.msg)
        src = batch.src[: batch.batch_size]
        out = out[src]
        idx = batch.input_id.cpu()
        num_neg = batch.batch_size
        neg_mem = history.get_history(torch.randint(
            0, history.num_nodes, size=(num_neg,)))
        raw_msg = torch.zeros(idx.size(0),1).to(device) 
        cur_mem, prev_mem = history(raw_msg, data.src[idx]) 

        if args.con=='his':
            loss = contrastive_loss(prev_mem, cur_mem, args.tau) 
            loss += torch.exp(cosine_similarity(cur_mem, neg_mem) 
                          ).sum(dim=1).log().mean() 
        elif args.con=='stru':
            loss = contrastive_loss(out, cur_mem, args.tau) 
            loss += torch.exp(cosine_similarity(out, neg_mem)
                            ).sum(dim=1).log().mean()
        elif args.con=='all':
            loss = args.alpha*contrastive_loss(out, cur_mem, args.tau) + \
                contrastive_loss(prev_mem, cur_mem, args.tau)
            loss += torch.exp(cosine_similarity(cur_mem, neg_mem) 
                            ).sum(dim=1).log().mean()
            loss += args.alpha*torch.exp(cosine_similarity(out, neg_mem) 
                            ).sum(dim=1).log().mean()


        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        pbar.set_description(f"Train Loss = {loss.item():.4f}")

    return total_loss / len(loader)


## Test function

@torch.no_grad()
def test(loader, model, history, measure, device, data):
    preds = []
    labels = []
    model.eval()
    history.eval()
    for batch in tqdm(loader):
        torch.cuda.empty_cache()
        batch = batch.to(device)
        out = model.encode(batch.x, batch.edge_index, batch.t, batch.msg)
        src = batch.src[: batch.batch_size]
        out = out[src]
        label = batch.y[: batch.batch_size]
        idx = batch.input_id.cpu()
        raw_msg = torch.zeros(idx.size(0),1).to(device) 
        cur_mem, prev_mem = history(raw_msg, data.src[idx], update=True)
        if args.con=='his':
            pred = 1-torch.diag(cosine_similarity(cur_mem, prev_mem)).view(-1)
        elif args.con=='stru': 
            pred = 1-torch.diag(cosine_similarity(out, cur_mem)).view(-1)
        elif args.con=='all':
            p1 = torch.diag(cosine_similarity(out, cur_mem)).view(-1)
            p2 = torch.diag(cosine_similarity(cur_mem, prev_mem)).view(-1)
            pred = (1 - p1) + (1 - p2) 

        pred = pred.sigmoid() 
        torch.cuda.empty_cache()
        preds.append(pred.detach().cpu())
        labels.append(label.detach().cpu())
    preds = torch.cat(preds)
    preds[torch.isnan(preds)] = 0.0 
    labels = torch.cat(labels)
    return measure(labels[labels!=2], preds[labels!=2]) 


In [8]:
data.x.size(1)

1

In [7]:
## define model 

model = GatedTGAT(
            in_channels=data.x.size(1),
            out_channels=1,
            hidden_channels=args.hidden_channels,
            edge_dim=data.msg.size(1),
            heads=args.heads,
            num_layers=args.nums_layers,
            dropout=args.dropout,
        ).to(device)

print(model)
print(history)

## define the optimizer 

optimizer = torch.optim.Adam(
    list(model.parameters()) + list(history.parameters()), lr=args.lr
)

## Define Evaluation Metric
measure = Measure("auc")


GatedTGAT(
  (node_encoder): Linear(1, 64, bias=True)
  (edge_encoder): Linear(172, 64, bias=True)
  (convs): ModuleList(
    (0-1): 2 x GatedTemporalGraphAttention(64, 64, heads=8)
  )
  (decoder): MLP(
    (act): ReLU()
    (lins): ModuleList(
      (0): Linear(64, 512, bias=True)
      (1): Linear(512, 256, bias=True)
      (2): Linear(256, 128, bias=True)
      (3): Linear(128, 1, bias=True)
    )
    (norms): ModuleList(
      (0-2): 3 x Identity()
    )
    (dropout): Dropout(p=0.3, inplace=False)
  )
)
History(
  (recurrent_network): GRUCell(64, 64)
  (lin): Sequential(
    (0): Linear(-1, 64, bias=True)
    (1): ELU(alpha=1.0)
    (2): Linear(64, 128, bias=True)
    (3): ELU(alpha=1.0)
    (4): Linear(128, 64, bias=True)
  )
)


In [8]:
best = 0
best_val = 0

for epoch in range(1, 1 + args.epochs):
    loss = train(train_loader, model, history, optimizer, device, data, args)
    val_auc = test(valid_loader, model, history, measure, device, data)
    test_auc = test(test_loader, model, history, measure, device, data)

    if val_auc > best_val:
        best_val = val_auc 
        best = test_auc

    print(
        f"Epoch: {epoch:03d}, Loss: {loss:.4f}, "
        f"Val AUC: {val_auc:.2%}, Test AUC: {test_auc:.2%}, Best AUC: {best:.2%}  "
    )
    log.info(
        f"Epoch: {epoch:03d}, Loss: {loss:.4f}, "
        f"Val AUC: {val_auc:.2%}, Test AUC: {test_auc:.2%}, Best AUC: {best:.2%} "
    )


100%|██████████| 93/93 [00:01<00:00, 48.28it/s]


Epoch: 001, Loss: 20.5205, Val AUC: 74.92%, Test AUC: 73.35%, Best AUC: 73.35%  


100%|██████████| 93/93 [00:01<00:00, 47.07it/s]


Epoch: 002, Loss: 18.8401, Val AUC: 86.14%, Test AUC: 88.99%, Best AUC: 88.99%  


Train Loss = 20.2222:  16%|█▌        | 68/431 [00:04<00:23, 15.32it/s]


KeyboardInterrupt: 

## Change Model + History part from (MHisCL)

In [98]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import MessagePassing
from torch_geometric.utils import softmax, add_self_loops
from sklearn.metrics import roc_auc_score
from tqdm import tqdm
import math

class RelativeTimeEncoding(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.freq = nn.Parameter(torch.randn(dim))
        self.decay = nn.Parameter(torch.tensor(0.1))

    def forward(self, t_diff):
        # t_diff: [E]
        t_diff = t_diff.float().unsqueeze(-1)
        fourier = torch.sin(t_diff * self.freq)
        decay = torch.exp(-torch.abs(t_diff) * torch.abs(self.decay))
        return fourier * decay

## Change Model 

In [99]:
class GatedTemporalAttention(MessagePassing):
    """
    Neighborhood-Aware Temporal Graph Attention.
    Computes both Attention Scores (Relevance) and Gating Scores (Trustworthiness).
    """
    def __init__(self, in_dim, out_dim, heads=4, dropout=0.1):
        super().__init__(aggr='add', node_dim=0)
        self.in_dim = in_dim
        self.out_dim = out_dim
        self.heads = heads
        self.head_dim = out_dim // heads
        
        self.lin_q = nn.Linear(in_dim, out_dim)
        self.lin_k = nn.Linear(in_dim, out_dim)
        self.lin_v = nn.Linear(in_dim, out_dim)
        self.lin_time = nn.Linear(in_dim, out_dim) # Encode time into attention
        
        # The Anomaly Gate: Decides if neighbor j is 'safe' for node i
        self.trust_gate = nn.Sequential(
            nn.Linear(2 * out_dim + out_dim, out_dim), # Takes [hi, hj, t_emb]
            nn.ReLU(),
            nn.Linear(out_dim, heads), # One gate per head
            nn.Sigmoid() 
        )

        self.att_dropout = nn.Dropout(dropout)
        self.proj_out = nn.Linear(out_dim, out_dim)
        self.norm = nn.LayerNorm(out_dim)
        self.scale = 1.0 / math.sqrt(self.head_dim)

    def forward(self, x, edge_index, t_encoding):
        # x: [N, in_dim]
        # t_encoding: [E, in_dim] (Time encoding per edge)
        
        # 1. Project Query, Key, Value
        q = self.lin_q(x).view(-1, self.heads, self.head_dim)
        k = self.lin_k(x).view(-1, self.heads, self.head_dim)
        v = self.lin_v(x).view(-1, self.heads, self.head_dim)

        # 2. Propagate
        out = self.propagate(edge_index, q=q, k=k, v=v, t_emb=t_encoding)
        
        out = out.view(-1, self.out_dim)
        out = self.proj_out(out)
        return self.norm(x + out) # Residual connection

    def message(self, q_i, k_j, v_j, t_emb, index):
        # q_i: [E, H, D], k_j: [E, H, D]
        
        # Expand time embedding to heads
        t_emb_heads = self.lin_time(t_emb).view(-1, self.heads, self.head_dim)
        
        # --- A. Structural Attention (Standard GAT) ---
        # (Q * (K + Time))
        attn_score = (q_i * (k_j + t_emb_heads)).sum(dim=-1) * self.scale
        attn_weights = softmax(attn_score, index)
        attn_weights = self.att_dropout(attn_weights)
        
        # --- B. Neighborhood Anomaly Gating (The Contribution) ---
        # We compute a trust score. If neighbor j is too different or anomalous,
        # we suppress its message.
        # Cat (Target, Neighbor, Time)
        q_i_flat = q_i.view(-1, self.heads * self.head_dim)
        k_j_flat = k_j.view(-1, self.heads * self.head_dim)
        
        gate_input = torch.cat([q_i_flat, k_j_flat, self.lin_time(t_emb)], dim=-1)
        trust_score = self.trust_gate(gate_input) # [E, H]
        trust_score = trust_score.unsqueeze(-1)   # [E, H, 1]

        # --- C. Weighted Aggregation ---
        # The value is modulated by the Attention Weight AND the Trust Gate
        # If Trust is low, the anomaly cannot camouflage or pollute the node.
        out = v_j * attn_weights.unsqueeze(-1) * trust_score
        
        return out

In [100]:
class MomentumMemory(nn.Module):
    """
    Stores historical representations of nodes to capture evolution.
    """
    def __init__(self, num_nodes, dim, momentum=0.9):
        super().__init__()
        self.register_buffer("memory", torch.zeros(num_nodes, dim))
        self.momentum = momentum

    def forward(self, node_indices, new_emb):
        # Detach previous memory to stop gradients flowing back endlessly in time
        prev_emb = self.memory[node_indices].detach()
        
        # Update memory (Moving Average)
        with torch.no_grad():
            updated_memory = self.momentum * prev_emb + (1 - self.momentum) * new_emb.detach()
            self.memory[node_indices] = updated_memory
            
        return prev_emb

In [101]:
class ATSCGNN(nn.Module):
    def __init__(self, num_nodes, in_dim, hidden_dim, num_layers=2, heads=4):
        super().__init__()
        self.time_enc = RelativeTimeEncoding(hidden_dim)
        self.input_proj = nn.Linear(in_dim, hidden_dim)
        
        self.layers = nn.ModuleList([
            GatedTemporalAttention(hidden_dim, hidden_dim, heads=heads)
            for _ in range(num_layers)
        ])
        
        #self.memory = MomentumMemory(num_nodes, hidden_dim)
        
        # Projections for Contrastive Learning
        self.proj_head = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim)
        )

    def forward(self, batch):
        x = self.input_proj(batch.x)
        
        # Calculate time difference for edges (assuming batch.edge_time exists)
        # If static edges, use zeros or simple encoding
        if hasattr(batch, 'edge_time_diff'):
             t_emb = self.time_enc(batch.edge_time_diff)
        else:
             t_emb = torch.zeros(batch.edge_index.size(1), x.size(1), device=x.device)

        # 1. Spatial-Temporal Aggregation
        for layer in self.layers:
            x = layer(x, batch.edge_index, t_emb)
            
        
        return  x

In [104]:
del(model) 
del(history)

In [105]:
model = ATSCGNN(
    num_nodes=data.num_nodes,
    in_dim=data.x.size(1),
    hidden_dim=64,
    num_layers=2,
    heads=8
).to(device)

In [118]:
history = History(
    data.num_nodes,
    args.num_timeslots,
    args.hidden_channels,
    device=device,
    history_retrieve=args.history_retrieve,
    recurrent=args.recurrent,
).to(device)

In [119]:
## Training Function 

def train(loader, model, history, optimizer, device, data, args):
    model.train()
    history.train()
    pbar = tqdm(loader)
    total_loss = 0
    for batch in pbar:
        batch = batch.to(device)
        optimizer.zero_grad()
        out = model(batch)
        src = batch.src[: batch.batch_size]
        out = out[src]
        idx = batch.input_id.cpu()
        num_neg = batch.batch_size
        neg_mem = history.get_history(torch.randint(
            0, history.num_nodes, size=(num_neg,)))
        raw_msg = torch.zeros(idx.size(0),1).to(device) 
        cur_mem, prev_mem = history(out, data.src[idx]) 

        if args.con=='his':
            loss = contrastive_loss(prev_mem, cur_mem, args.tau) 
            loss += torch.exp(cosine_similarity(cur_mem, neg_mem) 
                          ).sum(dim=1).log().mean()
        elif args.con=='stru':
            loss = contrastive_loss(out, cur_mem, args.tau) 
            loss += torch.exp(cosine_similarity(out, neg_mem)
                            ).sum(dim=1).log().mean()
        elif args.con=='all':
            loss = args.alpha*contrastive_loss(out, cur_mem, args.tau) + \
                contrastive_loss(prev_mem, cur_mem, args.tau)
            loss += torch.exp(cosine_similarity(cur_mem, neg_mem) 
                            ).sum(dim=1).log().mean()
            loss += args.alpha*torch.exp(cosine_similarity(out, neg_mem) 
                            ).sum(dim=1).log().mean()


        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        pbar.set_description(f"Train Loss = {loss.item():.4f}")

    return total_loss / len(loader)


## Test function

@torch.no_grad()
def test(loader, model, history, measure, device, data):
    preds = []
    labels = []
    model.eval()
    history.eval()
    for batch in tqdm(loader):
        torch.cuda.empty_cache()
        batch = batch.to(device)
        out = model(batch)
        src = batch.src[: batch.batch_size]
        out = out[src]
        label = batch.y[: batch.batch_size]
        idx = batch.input_id.cpu()
        raw_msg = torch.zeros(idx.size(0),1).to(device) 
        cur_mem, prev_mem = history(out, data.src[idx], update=True)
        if args.con=='his':
            pred = 1-torch.diag(cosine_similarity(cur_mem, prev_mem)).view(-1)
        elif args.con=='stru': 
            pred = 1-torch.diag(cosine_similarity(out, cur_mem)).view(-1)
        elif args.con=='all':
            p1 = torch.diag(cosine_similarity(out, cur_mem)).view(-1)
            p2 = torch.diag(cosine_similarity(cur_mem, prev_mem)).view(-1)
            pred = (1 - p1) + (1 - p2) 
            
        pred = pred.sigmoid() 
        torch.cuda.empty_cache()
        preds.append(pred.detach().cpu())
        labels.append(label.detach().cpu())
    preds = torch.cat(preds)
    preds[torch.isnan(preds)] = 0.0 
    labels = torch.cat(labels)
    return measure(labels[labels!=2], preds[labels!=2]) 


In [120]:
print(model)
print(history)

## define the optimizer 

optimizer = torch.optim.Adam(
    list(model.parameters()) + list(history.parameters()), lr=args.lr
)

## Define Evaluation Metric
measure = Measure("auc")

TrustAwareTemporalGNN(
  (input_proj): Linear(in_features=1, out_features=64, bias=True)
  (time_enc): RelativeTimeEncoding()
  (memory): TemporalMemory()
  (layers): ModuleList(
    (0-1): 2 x TrustAwareTemporalAttention()
  )
)
History(
  (recurrent_network): GRUCell(64, 64)
  (lin): Sequential(
    (0): Linear(-1, 64, bias=True)
    (1): ELU(alpha=1.0)
    (2): Linear(64, 64, bias=True)
  )
)


In [109]:
best = 0
best_val = 0



for epoch in range(1, 1 + args.epochs):
    loss = train(train_loader, model, history, optimizer, device, data, args)
    val_auc = test(valid_loader, model, history, measure, device, data)
    test_auc = test(test_loader, model, history, measure, device, data)

    if val_auc > best_val:
        best_val = val_auc 
        best = test_auc

    print(
        f"Epoch: {epoch:03d}, Loss: {loss:.4f}, "
        f"Val AUC: {val_auc:.2%}, Test AUC: {test_auc:.2%}, Best AUC: {best:.2%}  "
    )
    log.info(
        f"Epoch: {epoch:03d}, Loss: {loss:.4f}, "
        f"Val AUC: {val_auc:.2%}, Test AUC: {test_auc:.2%}, Best AUC: {best:.2%} "
    )


100%|██████████| 93/93 [00:01<00:00, 56.39it/s]


Epoch: 001, Loss: 19.5228, Val AUC: 81.57%, Test AUC: 86.53%, Best AUC: 86.53%  


100%|██████████| 93/93 [00:01<00:00, 55.80it/s]


Epoch: 002, Loss: 18.2486, Val AUC: 77.30%, Test AUC: 84.67%, Best AUC: 86.53%  


100%|██████████| 93/93 [00:01<00:00, 57.70it/s]


...

Epoch: 028, Loss: 16.7339, Val AUC: 83.12%, Test AUC: 91.40%, Best AUC: 90.51%  


100%|██████████| 93/93 [00:01<00:00, 56.07it/s]


Epoch: 029, Loss: 16.7370, Val AUC: 80.39%, Test AUC: 90.18%, Best AUC: 90.51%  


100%|██████████| 93/93 [00:01<00:00, 52.42it/s]


Epoch: 030, Loss: 16.7348, Val AUC: 83.22%, Test AUC: 89.55%, Best AUC: 90.51%  


## Trust-Aware Temporal GAT with Memory & Deviation

In [6]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
from torch_geometric.nn import MessagePassing
from torch_geometric.utils import softmax
from torch_scatter import scatter_mean


In [7]:
class RelativeTimeEncoding(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.freq = nn.Parameter(torch.randn(dim))
        self.decay = nn.Parameter(torch.tensor(0.1))

    def forward(self, t_diff):
        t = t_diff.float().unsqueeze(-1)
        return torch.sin(t * self.freq) * torch.exp(-torch.abs(t) * self.decay)


In [8]:
class TemporalMemory(nn.Module):
    def __init__(self, num_nodes, dim, momentum=0.95):
        super().__init__()
        self.register_buffer("memory", torch.zeros(num_nodes, dim))
        self.momentum = momentum

    def read(self, idx):
        return self.memory[idx]

    def write(self, idx, emb):
        with torch.no_grad():
            self.memory[idx] = (
                self.momentum * self.memory[idx] +
                (1 - self.momentum) * emb.detach()
            )


In [9]:
class MessageEvolution(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.gru = nn.GRUCell(dim, dim)

    def forward(self, msg, prev):
        return self.gru(msg, prev)


In [10]:
class TrustAwareTemporalAttention(MessagePassing):
    def __init__(self, dim, heads=4, dropout=0.1):
        super().__init__(aggr='add', node_dim=0)

        self.dim = dim
        self.heads = heads
        self.head_dim = dim // heads
        self.scale = 1.0 / math.sqrt(self.head_dim)

        self.lin_q = nn.Linear(dim, dim)
        self.lin_k = nn.Linear(dim, dim)
        self.lin_v = nn.Linear(dim, dim)
        self.lin_t = nn.Linear(dim, dim)

        # Trust MLP (temporal + neighborhood + similarity)
        self.trust_mlp = nn.Sequential(
            nn.Linear(3, dim),
            nn.ReLU(),
            nn.Linear(dim, heads)
        )

        self.msg_evo = MessageEvolution(dim)
        self.dropout = nn.Dropout(dropout)
        self.norm = nn.LayerNorm(dim)

    def forward(self, x, edge_index, t_emb, memory):
        q = self.lin_q(x).view(-1, self.heads, self.head_dim)
        k = self.lin_k(x).view(-1, self.heads, self.head_dim)
        v = self.lin_v(x).view(-1, self.heads, self.head_dim)

        out = self.propagate(
            edge_index,
            q=q, k=k, v=v,
            t_emb=t_emb,
            memory=memory,
            x=x
        )

        out = out.view(-1, self.dim)
        return self.norm(x + out)

        
    def message(self, q_i, k_j, v_j, t_emb, index, memory, x):
        # time embedding
        t_h = self.lin_t(t_emb).view(-1, self.heads, self.head_dim)

        # attention score
        attn = (q_i * (k_j + t_h)).sum(dim=-1) * self.scale

        # -------- TRUST SIGNALS -------- #

        # (1) temporal inconsistency
        mem_j = memory.read(index)
        temporal_diff = torch.norm(k_j.view(k_j.size(0), -1) - mem_j, dim=-1)

        # (2) neighborhood deviation
        neigh_mean = scatter_mean(
            k_j.view(k_j.size(0), -1),
            index,
            dim=0
        )
        dev = torch.norm(
            k_j.view(k_j.size(0), -1) - neigh_mean[index],
            dim=-1
        )

        # (3) similarity
        sim = (q_i * k_j).sum(dim=-1).mean(dim=-1)

        trust_input = torch.stack([temporal_diff, dev, sim], dim=-1)
        trust = torch.sigmoid(self.trust_mlp(trust_input))
        trust = trust.unsqueeze(-1)

        # trust-aware attention normalization
        attn = softmax(attn * trust.squeeze(-1), index)
        attn = self.dropout(attn)

        # message evolution
        v_flat = v_j.view(v_j.size(0), -1)
        evolved_msg = self.msg_evo(v_flat, mem_j)

        out = evolved_msg.view(-1, self.heads, self.head_dim)
        out = out * attn.unsqueeze(-1) * trust

        return out


In [11]:
class TrustAwareTemporalGNN(nn.Module):
    def __init__(
        self,
        num_nodes,
        in_dim,
        hidden_dim,
        num_layers=2,
        heads=4
    ):
        super().__init__()

        self.input_proj = nn.Linear(in_dim, hidden_dim)
        self.time_enc = RelativeTimeEncoding(hidden_dim)
        self.memory = TemporalMemory(num_nodes, hidden_dim)

        self.layers = nn.ModuleList([
            TrustAwareTemporalAttention(hidden_dim, heads)
            for _ in range(num_layers)
        ])

    def forward(self, batch):
        x = self.input_proj(batch.x)

        if hasattr(batch, 'edge_time_diff'):
            t_emb = self.time_enc(batch.edge_time_diff)
        else:
            t_emb = torch.zeros(
                batch.edge_index.size(1),
                x.size(1),
                device=x.device
            )

        for layer in self.layers:
            x = layer(x, batch.edge_index, t_emb, self.memory)

        # update memory
        self.memory.write(
            torch.arange(x.size(0), device=x.device),
            x
        )

        return x


In [13]:



model = TrustAwareTemporalGNN(
    num_nodes=data.num_nodes,
    in_dim=data.x.size(1),
    hidden_dim=64,
    num_layers=2,
    heads=8
).to(device)

In [14]:
## Training Function 

def train(loader, model, history, optimizer, device, data, args):
    model.train()
    history.train()
    pbar = tqdm(loader)
    total_loss = 0
    for batch in pbar:
        batch = batch.to(device)
        optimizer.zero_grad()
        out = model(batch)
        src = batch.src[: batch.batch_size]
        out = out[src]
        idx = batch.input_id.cpu()
        num_neg = batch.batch_size
        neg_mem = history.get_history(torch.randint(
            0, history.num_nodes, size=(num_neg,)))
        raw_msg = torch.zeros(idx.size(0),1).to(device) 
        cur_mem, prev_mem = history(out, data.src[idx]) 

        if args.con=='his':
            loss = contrastive_loss(prev_mem, cur_mem, args.tau) 
            loss += torch.exp(cosine_similarity(cur_mem, neg_mem) 
                          ).sum(dim=1).log().mean()
        elif args.con=='stru':
            loss = contrastive_loss(out, cur_mem, args.tau) 
            loss += torch.exp(cosine_similarity(out, neg_mem)
                            ).sum(dim=1).log().mean()
        elif args.con=='all':
            loss = args.alpha*contrastive_loss(out, cur_mem, args.tau) + \
                contrastive_loss(prev_mem, cur_mem, args.tau)
            loss += torch.exp(cosine_similarity(cur_mem, neg_mem) 
                            ).sum(dim=1).log().mean()
            loss += args.alpha*torch.exp(cosine_similarity(out, neg_mem) 
                            ).sum(dim=1).log().mean()


        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        pbar.set_description(f"Train Loss = {loss.item():.4f}")

    return total_loss / len(loader)


## Test function

@torch.no_grad()
def test(loader, model, history, measure, device, data):
    preds = []
    labels = []
    model.eval()
    history.eval()
    for batch in tqdm(loader):
        torch.cuda.empty_cache()
        batch = batch.to(device)
        out = model(batch)
        src = batch.src[: batch.batch_size]
        out = out[src]
        label = batch.y[: batch.batch_size]
        idx = batch.input_id.cpu()
        raw_msg = torch.zeros(idx.size(0),1).to(device) 
        cur_mem, prev_mem = history(out, data.src[idx], update=True)
        if args.con=='his':
            pred = 1-torch.diag(cosine_similarity(cur_mem, prev_mem)).view(-1)
        elif args.con=='stru': 
            pred = 1-torch.diag(cosine_similarity(out, cur_mem)).view(-1)
        elif args.con=='all':
            p1 = torch.diag(cosine_similarity(out, cur_mem)).view(-1)
            p2 = torch.diag(cosine_similarity(cur_mem, prev_mem)).view(-1)
            pred = (1 - p1) + (1 - p2) 
            
        pred = pred.sigmoid() 
        torch.cuda.empty_cache()
        preds.append(pred.detach().cpu())
        labels.append(label.detach().cpu())
    preds = torch.cat(preds)
    preds[torch.isnan(preds)] = 0.0 
    labels = torch.cat(labels)
    return measure(labels[labels!=2], preds[labels!=2]) 


In [15]:
from mhiscl.history import History

try :
    del history
except NameError:
    pass
    
history = History(
    data.num_nodes,
    args.num_timeslots,
    args.hidden_channels,
    device=device,
    history_retrieve=args.history_retrieve,
    recurrent=args.recurrent,
).to(device)

In [16]:
print(model)
print(history)

TrustAwareTemporalGNN(
  (input_proj): Linear(in_features=1, out_features=64, bias=True)
  (time_enc): RelativeTimeEncoding()
  (memory): TemporalMemory()
  (layers): ModuleList(
    (0-1): 2 x TrustAwareTemporalAttention()
  )
)
History(
  (recurrent_network): GRUCell(64, 64)
  (lin): Sequential(
    (0): Linear(-1, 64, bias=True)
    (1): ELU(alpha=1.0)
    (2): Linear(64, 64, bias=True)
  )
)


In [17]:
best = 0
best_val = 0

optimizer = torch.optim.Adam(
    list(model.parameters()) + list(history.parameters()), lr=args.lr
)
measure = Measure("auc")

for epoch in range(1, 1 + args.epochs):
    loss = train(train_loader, model, history, optimizer, device, data, args)
    val_auc = test(valid_loader, model, history, measure, device, data)
    test_auc = test(test_loader, model, history, measure, device, data)

    if val_auc > best_val:
        best_val = val_auc 
        best = test_auc

    print(
        f"Epoch: {epoch:03d}, Loss: {loss:.4f}, "
        f"Val AUC: {val_auc:.2%}, Test AUC: {test_auc:.2%}, Best AUC: {best:.2%}  "
    )
    log.info(
        f"Epoch: {epoch:03d}, Loss: {loss:.4f}, "
        f"Val AUC: {val_auc:.2%}, Test AUC: {test_auc:.2%}, Best AUC: {best:.2%} "
    )

100%|██████████| 93/93 [00:01<00:00, 53.79it/s]


Epoch: 001, Loss: 21.0016, Val AUC: 72.96%, Test AUC: 71.91%, Best AUC: 71.91%  


100%|██████████| 93/93 [00:01<00:00, 54.50it/s]


Epoch: 002, Loss: 18.9864, Val AUC: 87.05%, Test AUC: 87.96%, Best AUC: 87.96%  


100%|██████████| 93/93 [00:01<00:00, 55.55it/s]


...

Epoch: 018, Loss: 16.8354, Val AUC: 79.16%, Test AUC: 85.48%, Best AUC: 86.46%  


100%|██████████| 93/93 [00:01<00:00, 54.51it/s]


Epoch: 019, Loss: 16.8343, Val AUC: 85.89%, Test AUC: 88.01%, Best AUC: 86.46%  


Train Loss = 16.8004:  30%|███       | 130/431 [00:05<00:13, 23.12it/s]


KeyboardInterrupt: 

In [ ]:
best = 0
best_val = 0

optimizer = torch.optim.Adam(
    list(model.parameters()) + list(history.parameters()), lr=args.lr
)
measure = Measure("auc")

for epoch in range(1, 1 + args.epochs):
    loss = train(train_loader, model, history, optimizer, device, data, args)
    val_auc = test(valid_loader, model, history, measure, device, data)
    test_auc = test(test_loader, model, history, measure, device, data)

    if val_auc > best_val:
        best_val = val_auc 
        best = test_auc

    print(
        f"Epoch: {epoch:03d}, Loss: {loss:.4f}, "
        f"Val AUC: {val_auc:.2%}, Test AUC: {test_auc:.2%}, Best AUC: {best:.2%}  "
    )
    log.info(
        f"Epoch: {epoch:03d}, Loss: {loss:.4f}, "
        f"Val AUC: {val_auc:.2%}, Test AUC: {test_auc:.2%}, Best AUC: {best:.2%} "
    )


100%|██████████| 93/93 [00:01<00:00, 58.42it/s]


Epoch: 001, Loss: 19.8745, Val AUC: 82.27%, Test AUC: 85.49%, Best AUC: 85.49%  


100%|██████████| 93/93 [00:01<00:00, 56.10it/s]


Epoch: 002, Loss: 18.9627, Val AUC: 86.39%, Test AUC: 88.34%, Best AUC: 88.34%  


100%|██████████| 93/93 [00:01<00:00, 52.96it/s]


...

Epoch: 028, Loss: 16.8613, Val AUC: 87.05%, Test AUC: 89.99%, Best AUC: 91.82%  


100%|██████████| 93/93 [00:01<00:00, 55.20it/s]


Epoch: 029, Loss: 16.8381, Val AUC: 86.40%, Test AUC: 89.48%, Best AUC: 91.82%  


100%|██████████| 93/93 [00:01<00:00, 54.79it/s]

Epoch: 030, Loss: 16.8257, Val AUC: 88.18%, Test AUC: 90.24%, Best AUC: 91.82%  


## Enhanced Trust-Aware Multi-View Temporal Attention

In [17]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
from torch_geometric.nn import MessagePassing
from torch_geometric.utils import softmax
from torch_scatter import scatter_mean

In [18]:
class EdgeEncoder(nn.Module):
    def __init__(self, edge_dim, hidden_dim):
        super().__init__()
        self.lin = nn.Sequential(
            nn.Linear(edge_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim)
        )

    def forward(self, edge_attr):
        return self.lin(edge_attr)

In [19]:
class TemporalMemory(nn.Module):
    def __init__(self, num_nodes, dim, momentum=0.95):
        super().__init__()
        self.register_buffer("memory", torch.zeros(num_nodes, dim))
        self.momentum = momentum

    def read(self, idx):
        return self.memory[idx]

    def write(self, idx, emb):
        with torch.no_grad():
            self.memory[idx] = (
                self.momentum * self.memory[idx] +
                (1 - self.momentum) * emb.detach()
            )

In [20]:
class EdgeAwareQKV(nn.Module):
    def __init__(self, dim, heads):
        super().__init__()
        self.dim = dim
        self.heads = heads
        self.head_dim = dim // heads

        self.q = nn.Linear(dim, dim)
        self.k_node = nn.Linear(dim, dim)
        self.v_node = nn.Linear(dim, dim)

        self.k_edge = nn.Linear(dim, dim)
        self.v_edge = nn.Linear(dim, dim)

    def split(self, x):
        return x.view(-1, self.heads, self.head_dim)

    def forward(self, x, edge_emb):
        Q = self.split(self.q(x))
        K_node = self.split(self.k_node(x))
        V_node = self.split(self.v_node(x))

        K_edge = self.split(self.k_edge(edge_emb))
        V_edge = self.split(self.v_edge(edge_emb))

        return Q, K_node, V_node, K_edge, V_edge

In [21]:
class EdgeAwareTrust(nn.Module):
    def __init__(self, dim, heads):
        super().__init__()
        self.mlp = nn.Sequential(
            nn.Linear(4, dim),
            nn.ReLU(),
            nn.Linear(dim, heads),
            nn.Sigmoid()
        )

    def forward(self, delta_hist, dev_nei, volatility, edge_norm):
        x = torch.stack(
            [delta_hist, dev_nei, volatility, edge_norm],
            dim=-1
        )
        return self.mlp(x).unsqueeze(-1)

In [22]:
class EdgeAwareTemporalAttention(MessagePassing):
    def __init__(self, dim, heads=4, dropout=0.1):
        super().__init__(aggr="add", node_dim=0)

        self.dim = dim
        self.heads = heads
        self.head_dim = dim // heads
        self.scale = 1.0 / math.sqrt(self.head_dim)

        self.qkv = EdgeAwareQKV(dim, heads)
        self.trust_net = EdgeAwareTrust(dim, heads)

        self.dropout = nn.Dropout(dropout)
        self.norm = nn.LayerNorm(dim)

    def forward(self, x, edge_index, edge_emb, memory):
        Q, K_node, V_node, K_edge, V_edge = self.qkv(x, edge_emb)

        out = self.propagate(
            edge_index,
            Q=Q,
            K_node=K_node,
            V_node=V_node,
            K_edge=K_edge,
            V_edge=V_edge,
            memory=memory
        )

        out = out.view(-1, self.dim)
        return self.norm(x + out)    
    
    def message(
        self,
        Q_i,
        K_node_j,
        V_node_j,
        K_edge,
        V_edge,
        memory,
        index
    ):
        # ---------- ATTENTION ----------
        K = K_node_j + K_edge
        attn_raw = (Q_i * K).sum(dim=-1) * self.scale

        # ---------- TRUST SIGNALS ----------
        mem_j = memory.read(index)

        delta_hist = torch.norm(
            K_node_j.reshape(K_node_j.size(0), -1) - mem_j,
            dim=-1
        )

        neigh_mean = scatter_mean(
            K_node_j.reshape(K_node_j.size(0), -1),
            index,
            dim=0
        )[index]

        dev_nei = torch.norm(
            K_node_j.reshape(K_node_j.size(0), -1) - neigh_mean,
            dim=-1
        )

        volatility = torch.abs(attn_raw).mean(dim=-1)
        edge_norm = torch.norm(
            K_edge.reshape(K_edge.size(0), -1),
            dim=-1
        )

        trust = self.trust_net(
            delta_hist,
            dev_nei,
            volatility,
            edge_norm
        )

        # ---------- TRUST-AWARE SOFTMAX ----------
        attn = softmax(attn_raw * trust.squeeze(-1), index)
        attn = self.dropout(attn)

        # ---------- MESSAGE ----------
        V = V_node_j + V_edge
        out = V * attn.unsqueeze(-1) * trust

        return out

In [23]:
class EdgeAwareTemporalGNN(nn.Module):
    def __init__(
        self,
        num_nodes,
        in_dim,
        edge_dim,
        hidden_dim,
        num_layers=2,
        heads=4
    ):
        super().__init__()

        self.input_proj = nn.Linear(in_dim, hidden_dim)
        self.edge_enc = EdgeEncoder(edge_dim, hidden_dim)
        self.memory = TemporalMemory(num_nodes, hidden_dim)

        self.layers = nn.ModuleList([
            EdgeAwareTemporalAttention(hidden_dim, heads)
            for _ in range(num_layers)
        ])

    def forward(self, batch):
        x = self.input_proj(batch.x)
        edge_emb = self.edge_enc(batch.msg)

        for layer in self.layers:
            x = layer(x, batch.edge_index, edge_emb, self.memory)

        self.memory.write(
            torch.arange(x.size(0), device=x.device),
            x
        )

        return x

In [24]:
del(history)
history = History(
    data.num_nodes,
    args.num_timeslots,
    args.hidden_channels,
    device=device,
    history_retrieve=args.history_retrieve,
    recurrent=args.recurrent,
).to(device)

del(model)
model = EdgeAwareTemporalGNN(
    num_nodes=data.num_nodes,
    in_dim=data.x.size(1),
    hidden_dim=64,
    num_layers=2,
    heads=8,
    edge_dim=data.msg.size(1)
).to(device)

In [25]:
## Training Function 

def train(loader, model, history, optimizer, device, data, args):
    model.train()
    history.train()
    pbar = tqdm(loader)
    total_loss = 0
    for batch in pbar:
        batch = batch.to(device)
        optimizer.zero_grad()
        out = model(batch)
        src = batch.src[: batch.batch_size]
        out = out[src]
        idx = batch.input_id.cpu()
        num_neg = batch.batch_size
        neg_mem = history.get_history(torch.randint(
            0, history.num_nodes, size=(num_neg,)))
        raw_msg = torch.zeros(idx.size(0),1).to(device) 
        cur_mem, prev_mem = history(out, data.src[idx]) 

        if args.con=='his':
            loss = contrastive_loss(prev_mem, cur_mem, args.tau) 
            loss += torch.exp(cosine_similarity(cur_mem, neg_mem) 
                          ).sum(dim=1).log().mean()
        elif args.con=='stru':
            loss = contrastive_loss(out, cur_mem, args.tau) 
            loss += torch.exp(cosine_similarity(out, neg_mem)
                            ).sum(dim=1).log().mean()
        elif args.con=='all':
            loss = args.alpha*contrastive_loss(out, cur_mem, args.tau) + \
                contrastive_loss(prev_mem, cur_mem, args.tau)
            loss += torch.exp(cosine_similarity(cur_mem, neg_mem) 
                            ).sum(dim=1).log().mean()
            loss += args.alpha*torch.exp(cosine_similarity(out, neg_mem) 
                            ).sum(dim=1).log().mean()


        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        pbar.set_description(f"Train Loss = {loss.item():.4f}")

    return total_loss / len(loader)


## Test function

@torch.no_grad()
def test(loader, model, history, measure, device, data):
    preds = []
    labels = []
    model.eval()
    history.eval()
    for batch in tqdm(loader):
        torch.cuda.empty_cache()
        batch = batch.to(device)
        out = model(batch)
        src = batch.src[: batch.batch_size]
        out = out[src]
        label = batch.y[: batch.batch_size]
        idx = batch.input_id.cpu()
        raw_msg = torch.zeros(idx.size(0),1).to(device) 
        cur_mem, prev_mem = history(out, data.src[idx], update=True)
        if args.con=='his':
            pred = 1-torch.diag(cosine_similarity(cur_mem, prev_mem)).view(-1)
        elif args.con=='stru': 
            pred = 1-torch.diag(cosine_similarity(out, cur_mem)).view(-1)
        elif args.con=='all':
            p1 = torch.diag(cosine_similarity(out, cur_mem)).view(-1)
            p2 = torch.diag(cosine_similarity(cur_mem, prev_mem)).view(-1)
            pred = (1 - p1) + (1 - p2) 
            
        pred = pred.sigmoid() 
        torch.cuda.empty_cache()
        preds.append(pred.detach().cpu())
        labels.append(label.detach().cpu())
    preds = torch.cat(preds)
    preds[torch.isnan(preds)] = 0.0 
    labels = torch.cat(labels)
    return measure(labels[labels!=2], preds[labels!=2]) 


In [26]:
best = 0
best_val = 0

optimizer = torch.optim.Adam(
    list(model.parameters()) + list(history.parameters()), lr=args.lr
)
measure = Measure("auc")

for epoch in range(1, 1 + args.epochs):
    loss = train(train_loader, model, history, optimizer, device, data, args)
    val_auc = test(valid_loader, model, history, measure, device, data)
    test_auc = test(test_loader, model, history, measure, device, data)

    if val_auc > best_val:
        best_val = val_auc 
        best = test_auc

    print(
        f"Epoch: {epoch:03d}, Loss: {loss:.4f}, "
        f"Val AUC: {val_auc:.2%}, Test AUC: {test_auc:.2%}, Best AUC: {best:.2%}  "
    )
    log.info(
        f"Epoch: {epoch:03d}, Loss: {loss:.4f}, "
        f"Val AUC: {val_auc:.2%}, Test AUC: {test_auc:.2%}, Best AUC: {best:.2%} "
    )


100%|██████████| 93/93 [00:01<00:00, 54.65it/s]


Epoch: 001, Loss: 21.1845, Val AUC: 74.63%, Test AUC: 78.44%, Best AUC: 78.44%  


100%|██████████| 93/93 [00:01<00:00, 55.28it/s]


Epoch: 002, Loss: 19.6818, Val AUC: 87.25%, Test AUC: 88.93%, Best AUC: 88.93%  


100%|██████████| 93/93 [00:01<00:00, 52.88it/s]


...

Epoch: 025, Loss: 16.6585, Val AUC: 86.50%, Test AUC: 89.29%, Best AUC: 90.46%  


100%|██████████| 93/93 [00:01<00:00, 52.16it/s]


Epoch: 026, Loss: 16.6604, Val AUC: 88.23%, Test AUC: 90.12%, Best AUC: 90.46%  


Train Loss = 16.6301:  57%|█████▋    | 246/431 [00:09<00:07, 26.33it/s]


KeyboardInterrupt: 

In [93]:
class AdvancedSimilarityTrust(nn.Module):
    def __init__(self, heads):
        super().__init__()
        self.mlp = nn.Sequential(
            nn.Linear(3, 16),
            nn.GELU(),
            nn.Linear(16, 1)
        )
        self.heads = heads

    def forward(self, q_i, k_j, mem_i, mem_j):
        cos_sim = F.cosine_similarity(q_i, k_j, dim=-1)
        drift = (q_i - mem_i).norm(dim=-1)
        deviation = (k_j - mem_j).norm(dim=-1)

        features = torch.stack([cos_sim, drift, deviation], dim=-1)
        trust = self.mlp(features).squeeze(-1)
        return torch.sigmoid(trust)

In [94]:
class EnhancedTemporalTrustAttention(MessagePassing):
    def __init__(self, dim, heads=4, dropout=0.1):
        super().__init__(aggr="add", node_dim=0)

        self.dim = dim
        self.heads = heads
        self.head_dim = dim // heads
        self.scale = 1.0 / math.sqrt(self.head_dim)

        # Node projections
        self.q_proj = nn.Linear(dim, dim)
        self.k_proj = nn.Linear(dim, dim)
        self.v_proj = nn.Linear(dim, dim)

        # Edge + Time projections
        self.edge_proj = nn.Linear(dim, dim)
        self.time_proj = nn.Linear(dim, dim)

        self.trust_net = AdvancedSimilarityTrust(heads)

        self.dropout = nn.Dropout(dropout)
        self.norm = nn.LayerNorm(dim)
        self.residual_gate = nn.Linear(dim, 1)

    def split(self, x):
        return x.view(-1, self.heads, self.head_dim)

    def forward(self, x, edge_index, edge_emb, time_emb, memory):

        Q = self.split(self.q_proj(x))
        K = self.split(self.k_proj(x))
        V = self.split(self.v_proj(x))

        mem = memory.read(torch.arange(x.size(0), device=x.device))
        mem = self.split(mem)

        edge_emb = self.split(self.edge_proj(edge_emb))
        time_emb = self.split(self.time_proj(time_emb))

        out = self.propagate(
            edge_index,
            Q=Q,
            K=K,
            V=V,
            mem=mem,
            edge_emb=edge_emb,
            time_emb=time_emb
        )

        out = out.view(-1, self.dim)

        beta = torch.sigmoid(self.residual_gate(x))
        return self.norm(beta * x + (1 - beta) * out)
    

    def message(self, Q_i, K_j, V_j,
                mem_i, mem_j,
                edge_emb, time_emb,
                index):

        # Add edge + time to keys
        K_j = K_j + edge_emb + time_emb

        # Attention
        attn = (Q_i * K_j).sum(dim=-1) * self.scale

        # Trust modulation
        trust = self.trust_net(Q_i, K_j, mem_i, mem_j)

        attn = attn * trust

        attn = softmax(attn, index)
        attn = self.dropout(attn)

        return V_j * attn.unsqueeze(-1)

In [95]:
class TemporalMemory(nn.Module):
    def __init__(self, num_nodes, dim, momentum=0.9):
        super().__init__()
        self.momentum = momentum
        self.register_buffer("memory", torch.zeros(num_nodes, dim))

    def read(self, idx):
        return self.memory[idx]

    def write(self, idx, values):
        self.memory[idx] = (
            self.momentum * self.memory[idx]
            + (1 - self.momentum) * values.detach()
        )

In [96]:
class EnhancedTrustTemporalGNN(nn.Module):
    def __init__(
        self,
        num_nodes,
        in_dim,
        edge_dim,
        hidden_dim,
        num_layers=2,
        heads=4,
        memory_momentum=0.9
    ):
        super().__init__()

        self.input_proj = nn.Linear(in_dim, hidden_dim)
        self.edge_enc = nn.Linear(edge_dim, hidden_dim)
        self.time_enc = nn.Linear(1, hidden_dim)

        self.memory = TemporalMemory(
            num_nodes,
            hidden_dim,
            momentum=memory_momentum
        )

        self.layers = nn.ModuleList([
            EnhancedTemporalTrustAttention(
                hidden_dim,
                heads=heads
            )
            for _ in range(num_layers)
        ])

    def forward(self, batch):

        x = self.input_proj(batch.x)

        edge_emb = self.edge_enc(batch.msg)

        # make sure time is shape [num_edges,1]
        t = batch.t.view(-1, 1).float()
        time_emb = self.time_enc(t)

        for layer in self.layers:
            x = layer(
                x,
                batch.edge_index,
                edge_emb,
                time_emb,
                self.memory
            )

        self.memory.write(
            torch.arange(x.size(0), device=x.device),
            x
        )

        return x

In [97]:
args.hidden_channels = 64

In [98]:
del(history)
history = History(
    data.num_nodes,
    args.num_timeslots,
    args.hidden_channels,
    device=device,
    history_retrieve=args.history_retrieve,
    recurrent=args.recurrent,
).to(device)

del(model)
model = EnhancedTrustTemporalGNN(
    num_nodes=data.num_nodes,
    in_dim=data.x.size(1),
    hidden_dim=args.hidden_channels,
    num_layers=2,
    heads=8,
    edge_dim=data.msg.size(1)
).to(device)

In [99]:
## Training Function 

def train(loader, model, history, optimizer, device, data, args):
    model.train()
    history.train()
    pbar = tqdm(loader)
    total_loss = 0
    for batch in pbar:
        batch = batch.to(device)
        optimizer.zero_grad()
        out = model(batch)
        src = batch.src[: batch.batch_size]
        out = out[src]
        idx = batch.input_id.cpu()
        num_neg = batch.batch_size
        neg_mem = history.get_history(torch.randint(
            0, history.num_nodes, size=(num_neg,)))
        raw_msg = torch.zeros(idx.size(0),1).to(device) 
        cur_mem, prev_mem = history(out, data.src[idx]) 

        if args.con=='his':
            loss = contrastive_loss(prev_mem, cur_mem, args.tau) 
            loss += torch.exp(cosine_similarity(cur_mem, neg_mem) 
                          ).sum(dim=1).log().mean()
        elif args.con=='stru':
            loss = contrastive_loss(out, cur_mem, args.tau) 
            loss += torch.exp(cosine_similarity(out, neg_mem)
                            ).sum(dim=1).log().mean()
        elif args.con=='all':
            loss = args.alpha*contrastive_loss(out, cur_mem, args.tau) + \
                contrastive_loss(prev_mem, cur_mem, args.tau)
            loss += torch.exp(cosine_similarity(cur_mem, neg_mem) 
                            ).sum(dim=1).log().mean()
            loss += args.alpha*torch.exp(cosine_similarity(out, neg_mem) 
                            ).sum(dim=1).log().mean()


        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        pbar.set_description(f"Train Loss = {loss.item():.4f}")

    return total_loss / len(loader)


## Test function

@torch.no_grad()
def test(loader, model, history, measure, device, data):
    preds = []
    labels = []
    model.eval()
    history.eval()
    for batch in tqdm(loader):
        torch.cuda.empty_cache()
        batch = batch.to(device)
        out = model(batch)
        src = batch.src[: batch.batch_size]
        out = out[src]
        label = batch.y[: batch.batch_size]
        idx = batch.input_id.cpu()
        raw_msg = torch.zeros(idx.size(0),1).to(device) 
        cur_mem, prev_mem = history(out, data.src[idx], update=True)
        if args.con=='his':
            pred = 1-torch.diag(cosine_similarity(cur_mem, prev_mem)).view(-1)
        elif args.con=='stru': 
            pred = 1-torch.diag(cosine_similarity(out, cur_mem)).view(-1)
        elif args.con=='all':
            p1 = torch.diag(cosine_similarity(out, cur_mem)).view(-1)
            p2 = torch.diag(cosine_similarity(cur_mem, prev_mem)).view(-1)
            pred = (1 - p1) + (1 - p2) 
            
        pred = pred.sigmoid() 
        torch.cuda.empty_cache()
        preds.append(pred.detach().cpu())
        labels.append(label.detach().cpu())
    preds = torch.cat(preds)
    preds[torch.isnan(preds)] = 0.0 
    labels = torch.cat(labels)
    return measure(labels[labels!=2], preds[labels!=2]) 


In [100]:
best = 0
best_val = 0

optimizer = torch.optim.Adam(
    list(model.parameters()) + list(history.parameters()), lr=args.lr
)
measure = Measure("auc")

for epoch in range(1, 1 + args.epochs):
    loss = train(train_loader, model, history, optimizer, device, data, args)
    val_auc = test(valid_loader, model, history, measure, device, data)
    test_auc = test(test_loader, model, history, measure, device, data)

    if val_auc > best_val:
        best_val = val_auc 
        best = test_auc

    print(
        f"Epoch: {epoch:03d}, Loss: {loss:.4f}, "
        f"Val AUC: {val_auc:.2%}, Test AUC: {test_auc:.2%}, Best AUC: {best:.2%}  "
    )
    log.info(
        f"Epoch: {epoch:03d}, Loss: {loss:.4f}, "
        f"Val AUC: {val_auc:.2%}, Test AUC: {test_auc:.2%}, Best AUC: {best:.2%} "
    )


100%|██████████| 93/93 [00:01<00:00, 54.28it/s]


Epoch: 001, Loss: 20.4210, Val AUC: 70.98%, Test AUC: 72.00%, Best AUC: 72.00%  


100%|██████████| 93/93 [00:01<00:00, 54.02it/s]


Epoch: 002, Loss: 19.2086, Val AUC: 81.96%, Test AUC: 86.70%, Best AUC: 86.70%  


100%|██████████| 93/93 [00:01<00:00, 54.32it/s]


...

Epoch: 028, Loss: 16.7088, Val AUC: 88.08%, Test AUC: 91.10%, Best AUC: 90.93%  


100%|██████████| 93/93 [00:01<00:00, 53.65it/s]


Epoch: 029, Loss: 16.6916, Val AUC: 86.10%, Test AUC: 91.59%, Best AUC: 90.93%  


100%|██████████| 93/93 [00:01<00:00, 53.87it/s]

Epoch: 030, Loss: 16.6855, Val AUC: 86.72%, Test AUC: 89.97%, Best AUC: 90.93%  


## Enhanced Version With Grouped Query Attention

In [17]:
"""
Advanced Temporal Trust GNN
============================
Enhancements over baseline:
  - Rotary Positional Embeddings (RoPE) for temporal Q/K modulation
  - Grouped Query Attention (GQA): fewer KV heads, more Q heads
  - Memory with uncertainty (epistemic drift) tracking
  - Transformer-style trust via cross-attention over memory (replaces MLP)
  - Mixture-of-Experts (MoE) message routing
  - Adaptive LayerNorm (AdaLN) conditioned on time embeddings
  - Learnable per-head temperature
  - Optional graph diffusion rewiring (DIGL-style PPR)
"""

import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import MessagePassing
from torch_geometric.utils import softmax, add_self_loops


# ─────────────────────────────────────────────
# 1. Rotary Positional Embedding (RoPE)
# ─────────────────────────────────────────────

class RotaryEmbedding(nn.Module):
    """
    RoPE: encodes relative time offsets into Q/K via rotation in 2D subspaces.
    Works for any sequence / edge timestamp.
    """
    def __init__(self, dim):
        super().__init__()
        assert dim % 2 == 0
        inv_freq = 1.0 / (10000 ** (torch.arange(0, dim, 2).float() / dim))
        self.register_buffer("inv_freq", inv_freq)

    def forward(self, x, t):
        """
        x: (N, heads, head_dim)
        t: (N,) timestamps (float)
        """
        # t: (N,) -> (N, dim/2)
        freqs = torch.outer(t.float(), self.inv_freq)          # (N, d/2)
        cos = freqs.cos().unsqueeze(1)                         # (N, 1, d/2)
        sin = freqs.sin().unsqueeze(1)

        x1, x2 = x[..., ::2], x[..., 1::2]
        x_rot = torch.stack([-x2, x1], dim=-1).flatten(-2)    # (N, h, d)
        # interleave cos/sin
        cos_full = cos.expand_as(x1).repeat_interleave(2, dim=-1)
        sin_full = sin.expand_as(x1).repeat_interleave(2, dim=-1)
        return x * cos_full + x_rot * sin_full


# ─────────────────────────────────────────────
# 2. Memory with Uncertainty
# ─────────────────────────────────────────────

class UncertainTemporalMemory(nn.Module):
    """
    Stores per-node exponential moving average + variance (uncertainty).
    Uncertainty is used as an extra trust signal.
    """
    def __init__(self, num_nodes, dim, momentum=0.9):
        super().__init__()
        self.momentum = momentum
        self.register_buffer("memory", torch.zeros(num_nodes, dim))
        self.register_buffer("variance", torch.ones(num_nodes, dim))

    def read(self, idx):
        return self.memory[idx], self.variance[idx]

    @torch.no_grad()
    def write(self, idx, values):
        delta = values.detach() - self.memory[idx]
        self.variance[idx] = (
            self.momentum * self.variance[idx]
            + (1 - self.momentum) * delta ** 2
        )
        self.memory[idx] = (
            self.momentum * self.memory[idx]
            + (1 - self.momentum) * values.detach()
        )


# ─────────────────────────────────────────────
# 3. Cross-Attention Trust (replaces MLP)
# ─────────────────────────────────────────────

class CrossAttentionTrust(nn.Module):
    """
    Computes trust between (q_i, k_j) by attending over memory tokens.
    This is strictly more expressive than a 3-feature MLP.
    """
    def __init__(self, head_dim, num_mem_tokens=4):
        super().__init__()
        self.q_trust = nn.Linear(head_dim * 2, head_dim)
        self.k_trust = nn.Linear(head_dim * 2, head_dim)
        self.scale   = head_dim ** -0.5
        self.out     = nn.Linear(head_dim, 1)

    def forward(self, q_i, k_j, mem_i, mem_i_var, mem_j, mem_j_var):
        """
        All inputs: (E, heads, head_dim)
        Returns:    (E, heads)
        """
        # Uncertainty-aware memory: scale by 1/(1+std)
        uncertainty_i = (1.0 / (1.0 + mem_i_var.sqrt().clamp(min=1e-6)))
        uncertainty_j = (1.0 / (1.0 + mem_j_var.sqrt().clamp(min=1e-6)))

        ctx_i = torch.cat([mem_i * uncertainty_i, q_i], dim=-1)
        ctx_j = torch.cat([mem_j * uncertainty_j, k_j], dim=-1)

        q_t = self.q_trust(ctx_i)          # (E, h, d)
        k_t = self.k_trust(ctx_j)

        score = (q_t * k_t).sum(-1, keepdim=True) * self.scale   # (E, h, 1)
        trust = torch.sigmoid(self.out(torch.tanh(q_t * k_t)))   # (E, h, 1)
        return trust.squeeze(-1)                                   # (E, h)


# ─────────────────────────────────────────────
# 4. Mixture-of-Experts Message Router
# ─────────────────────────────────────────────

class MoEMessageRouter(nn.Module):
    """
    Routes each edge's message through top-k of E experts.
    Experts are cheap 2-layer MLPs with different inductive biases.
    """
    def __init__(self, dim, num_experts=4, top_k=2, dropout=0.1):
        super().__init__()
        self.top_k = top_k
        self.gate  = nn.Linear(dim, num_experts)
        self.experts = nn.ModuleList([
            nn.Sequential(
                nn.Linear(dim, dim * 2),
                nn.GELU(),
                nn.Dropout(dropout),
                nn.Linear(dim * 2, dim)
            ) for _ in range(num_experts)
        ])

    def forward(self, msg):
        """msg: (E, dim)"""
        logits = self.gate(msg)                                    # (E, n_exp)
        topk_vals, topk_idx = logits.topk(self.top_k, dim=-1)     # (E, k)
        weights = F.softmax(topk_vals, dim=-1)                     # (E, k)

        out = torch.zeros_like(msg)
        for ki in range(self.top_k):
            idx = topk_idx[:, ki]                                  # (E,)
            # scatter through each expert
            for ei, expert in enumerate(self.experts):
                mask = (idx == ei)
                if mask.any():
                    out[mask] += weights[mask, ki].unsqueeze(-1) * expert(msg[mask])
        return out


# ─────────────────────────────────────────────
# 5. Adaptive LayerNorm (AdaLN) conditioned on time
# ─────────────────────────────────────────────

class AdaLayerNorm(nn.Module):
    """
    LayerNorm whose scale/shift are predicted from a conditioning signal (time).
    Used in DiT / many diffusion transformers.
    """
    def __init__(self, dim, cond_dim):
        super().__init__()
        self.norm = nn.LayerNorm(dim, elementwise_affine=False)
        self.proj = nn.Sequential(
            nn.SiLU(),
            nn.Linear(cond_dim, dim * 2)
        )

    def forward(self, x, cond):
        """
        x:    (N, dim)
        cond: (N, cond_dim) -- e.g. mean time embedding over incident edges
        """
        scale, shift = self.proj(cond).chunk(2, dim=-1)
        return self.norm(x) * (1 + scale) + shift


# ─────────────────────────────────────────────
# 6. Grouped Query Attention Layer (GQA)
# ─────────────────────────────────────────────

class GQATemporalTrustAttention(MessagePassing):
    """
    Grouped Query Attention version of the trust attention layer.
    Q has `heads` heads; K/V share `kv_heads` heads (heads >= kv_heads, divisible).
    This reduces KV memory and matches modern LLM efficiency tricks.
    """
    def __init__(self, dim, heads=8, kv_heads=2, dropout=0.1, num_experts=4, moe_top_k=2):
        assert heads % kv_heads == 0
        super().__init__(aggr="add", node_dim=0)

        self.dim      = dim
        self.heads    = heads
        self.kv_heads = kv_heads
        self.groups   = heads // kv_heads
        self.head_dim = dim // heads

        # Projections
        self.q_proj  = nn.Linear(dim, dim)
        self.k_proj  = nn.Linear(dim, self.kv_heads * self.head_dim)
        self.v_proj  = nn.Linear(dim, self.kv_heads * self.head_dim)
        self.o_proj  = nn.Linear(dim, dim)

        self.edge_proj = nn.Linear(dim, self.kv_heads * self.head_dim)

        # RoPE for time
        self.rope = RotaryEmbedding(self.head_dim)

        # Learnable per-head temperature
        self.log_temp = nn.Parameter(torch.zeros(heads))

        # Trust module
        self.trust_net = CrossAttentionTrust(self.head_dim)

        # MoE for message routing
        self.moe = MoEMessageRouter(dim, num_experts=num_experts, top_k=moe_top_k, dropout=dropout)

        self.dropout = nn.Dropout(dropout)

        # AdaLN conditioned on time
        self.ada_norm = AdaLayerNorm(dim, dim)

        # Learned residual gate
        self.gate = nn.Linear(dim, 1)

    def split_q(self, x):
        return x.view(-1, self.heads, self.head_dim)

    def split_kv(self, x):
        return x.view(-1, self.kv_heads, self.head_dim)

    def expand_kv(self, x):
        # (N, kv_heads, d) -> (N, heads, d) by repeating
        return x.repeat_interleave(self.groups, dim=1)

    def forward(self, x, edge_index, edge_emb, time_emb, time_per_node, memory):
        N = x.size(0)
        device = x.device

        Q = self.split_q(self.q_proj(x))                          # (N, h, d)
        K = self.split_kv(self.k_proj(x))                         # (N, kv_h, d)
        V = self.split_kv(self.v_proj(x))                         # (N, kv_h, d)

        # RoPE on Q and K (expand K to full heads for rope, then compress back is tricky;
        # we apply rope per kv group)
        Q = self.rope(Q, time_per_node)
        K = self.rope(K, time_per_node)

        # Expand K, V to match Q heads
        K = self.expand_kv(K)                                      # (N, h, d)
        V = self.expand_kv(V)

        mem_vals, mem_vars = memory.read(torch.arange(N, device=device))
        mem_vals = self.split_q(mem_vals.expand(-1, self.dim) if mem_vals.shape[-1] != self.dim
                                else mem_vals)
        # If dim mismatch, pad — handled gracefully below via trust_net
        mem_vars_split = mem_vars.view(N, self.heads, self.head_dim) \
                         if mem_vars.shape[-1] == self.dim \
                         else mem_vars.unsqueeze(1).expand(N, self.heads, self.head_dim)

        edge_emb_kv = self.split_kv(self.edge_proj(edge_emb))     # (E, kv_h, d)
        edge_emb_full = self.expand_kv(edge_emb_kv)               # (E, h, d)

        out = self.propagate(
            edge_index,
            Q=Q, K=K, V=V,
            mem_val=mem_vals, mem_var=mem_vars_split,
            edge_emb=edge_emb_full,
            time_emb=time_emb,
            size=(N, N)
        )

        out = out.view(N, self.dim)
        out = self.moe(out)                                        # MoE routing
        out = self.o_proj(out)

        # Adaptive LayerNorm: condition on per-node mean time embedding
        out = self.ada_norm(out, time_per_node.unsqueeze(-1).expand(N, self.dim)
                            if time_per_node.dim() == 1 else time_per_node)

        # Gated residual
        beta = torch.sigmoid(self.gate(x))
        return beta * x + (1 - beta) * out

    def message(self, Q_i, K_j, V_j,
                mem_val_i, mem_val_j,
                mem_var_i, mem_var_j,
                edge_emb, time_emb,
                index):

        # Inject edge context into keys
        K_j = K_j + edge_emb

        # Dot-product attention with learnable temperature
        temp = self.log_temp.exp().unsqueeze(0)                    # (1, h)
        attn = (Q_i * K_j).sum(dim=-1) / (self.head_dim ** 0.5)   # (E, h)
        attn = attn / temp

        # Trust modulation via cross-attention over memory
        trust = self.trust_net(Q_i, K_j, mem_val_i, mem_var_i, mem_val_j, mem_var_j)  # (E, h)
        attn  = attn * trust

        attn = softmax(attn, index)                                # (E, h)
        attn = self.dropout(attn)

        return V_j * attn.unsqueeze(-1)                            # (E, h, d)


# ─────────────────────────────────────────────
# 7. Optional: PPR-based Graph Rewiring
# ─────────────────────────────────────────────

def ppr_diffusion_rewire(edge_index, num_nodes, alpha=0.15, k=16):
    """
    Approximate Personalized PageRank diffusion to rewire the graph.
    Returns a new edge_index with top-k PPR neighbors per node.
    Falls back gracefully if torch_geometric.transforms unavailable.
    """
    try:
        from torch_geometric.transforms import GDC
        # GDC handles PPR approximation natively
        # This is a simplified stub; full usage requires Data object
        print("[GNN] PPR rewiring available via torch_geometric.transforms.GDC")
    except ImportError:
        pass
    return edge_index  # return original if unavailable


# ─────────────────────────────────────────────
# 8. Full Model
# ─────────────────────────────────────────────

class AdvancedTrustTemporalGNN(nn.Module):
    """
    Full model stack with all innovations:
      - RoPE temporal encoding
      - GQA multi-head attention
      - Cross-attention trust with uncertainty-aware memory
      - MoE message routing
      - AdaLN temporal conditioning
      - Learnable residual gates
      - Optional PPR graph rewiring
    """
    def __init__(
        self,
        num_nodes,
        in_dim,
        edge_dim,
        hidden_dim,
        num_layers     = 3,
        heads          = 8,
        kv_heads       = 2,
        memory_momentum= 0.9,
        num_experts    = 4,
        moe_top_k      = 2,
        dropout        = 0.1,
        use_ppr_rewire = False,
        ppr_alpha      = 0.15,
        ppr_k          = 16,
    ):
        super().__init__()

        self.use_ppr_rewire = use_ppr_rewire
        self.ppr_alpha = ppr_alpha
        self.ppr_k     = ppr_k
        self.num_nodes = num_nodes

        # Input projections
        self.input_proj = nn.Linear(in_dim, hidden_dim)
        self.edge_enc   = nn.Linear(edge_dim, hidden_dim)

        # Single time encoder (produces per-edge embedding for edge-level ops)
        self.time_enc = nn.Sequential(
            nn.Linear(1, hidden_dim),
            nn.SiLU(),
            nn.Linear(hidden_dim, hidden_dim)
        )

        # Uncertainty-aware temporal memory
        self.memory = UncertainTemporalMemory(
            num_nodes, hidden_dim, momentum=memory_momentum
        )

        # GQA Trust Attention layers
        self.layers = nn.ModuleList([
            GQATemporalTrustAttention(
                hidden_dim,
                heads       = heads,
                kv_heads    = kv_heads,
                dropout     = dropout,
                num_experts = num_experts,
                moe_top_k   = moe_top_k
            )
            for _ in range(num_layers)
        ])

        # Final norm
        self.final_norm = nn.LayerNorm(hidden_dim)

        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)

    def forward(self, batch):
        x         = self.input_proj(batch.x)
        edge_emb  = self.edge_enc(batch.msg)
        edge_index = batch.edge_index

        if self.use_ppr_rewire:
            edge_index = ppr_diffusion_rewire(
                edge_index, self.num_nodes,
                alpha=self.ppr_alpha, k=self.ppr_k
            )

        # Per-edge time embedding
        t          = batch.t.view(-1, 1).float()
        time_emb   = self.time_enc(t)                              # (E, hidden)

        # Per-node time: scatter mean of incident edge times
        src = edge_index[0]
        time_per_node = torch.zeros(x.size(0), device=x.device)
        time_per_node.scatter_add_(0, src, batch.t.float())
        count = torch.zeros(x.size(0), device=x.device)
        count.scatter_add_(0, src, torch.ones(src.size(0), device=x.device))
        time_per_node = time_per_node / (count + 1e-6)             # (N,)

        for layer in self.layers:
            x = layer(
                x, edge_index, edge_emb,
                time_emb, time_per_node, self.memory
            )

        x = self.final_norm(x)

        # Update memory after full forward pass
        self.memory.write(
            torch.arange(x.size(0), device=x.device), x
        )

        return x


# ─────────────────────────────────────────────
# Quick sanity check
# ─────────────────────────────────────────────

if __name__ == "__main__":
    from types import SimpleNamespace

    N, E = 100, 300
    IN_DIM, EDGE_DIM, HIDDEN = 32, 16, 64

    batch = SimpleNamespace(
        x          = torch.randn(N, IN_DIM),
        msg        = torch.randn(E, EDGE_DIM),
        t          = torch.rand(E) * 100,
        edge_index = torch.randint(0, N, (2, E))
    )

    model = AdvancedTrustTemporalGNN(
        num_nodes  = N,
        in_dim     = IN_DIM,
        edge_dim   = EDGE_DIM,
        hidden_dim = HIDDEN,
        num_layers = 2,
        heads      = 8,
        kv_heads   = 2,
    )

    out = model(batch)
    print(f"Output shape: {out.shape}")   # (100, 64)
    params = sum(p.numel() for p in model.parameters())
    print(f"Total parameters: {params:,}")

Output shape: torch.Size([100, 64])
Total parameters: 180,972


In [18]:
batch.x.shape

torch.Size([100, 32])

In [19]:
del(history)
history = History(
    data.num_nodes,
    args.num_timeslots,
    args.hidden_channels,
    device=device,
    history_retrieve=args.history_retrieve,
    recurrent=args.recurrent,
).to(device)

del(model)


In [20]:
del(history)
history = History(
    data.num_nodes,
    args.num_timeslots,
    args.hidden_channels,
    device=device,
    history_retrieve=args.history_retrieve,
    recurrent=args.recurrent,
).to(device)

#del(model)
model = AdvancedTrustTemporalGNN(
        num_nodes=data.num_nodes,
        in_dim=data.x.size(1),
        edge_dim=data.msg.size(1),
        hidden_dim=args.hidden_channels,
        num_layers = 2,
        heads      = 8,
        kv_heads   = 2,
    ).to(device)

In [21]:
print(model)
print(history)

AdvancedTrustTemporalGNN(
  (input_proj): Linear(in_features=1, out_features=64, bias=True)
  (edge_enc): Linear(in_features=172, out_features=64, bias=True)
  (time_enc): Sequential(
    (0): Linear(in_features=1, out_features=64, bias=True)
    (1): SiLU()
    (2): Linear(in_features=64, out_features=64, bias=True)
  )
  (memory): UncertainTemporalMemory()
  (layers): ModuleList(
    (0-1): 2 x GQATemporalTrustAttention()
  )
  (final_norm): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
)
History(
  (recurrent_network): GRUCell(64, 64)
  (lin): Sequential(
    (0): Linear(-1, 64, bias=True)
    (1): ELU(alpha=1.0)
    (2): Linear(64, 64, bias=True)
  )
)


In [22]:
## Training Function 

def train(loader, model, history, optimizer, device, data, args):
    model.train()
    history.train()
    pbar = tqdm(loader)
    total_loss = 0
    for batch in pbar:
        batch = batch.to(device)
        optimizer.zero_grad()
        out = model(batch)
        src = batch.src[: batch.batch_size]
        out = out[src]
        idx = batch.input_id.cpu()
        num_neg = batch.batch_size
        neg_mem = history.get_history(torch.randint(
            0, history.num_nodes, size=(num_neg,)))
        raw_msg = torch.zeros(idx.size(0),1).to(device) 
        cur_mem, prev_mem = history(out, data.src[idx]) 

        if args.con=='his':
            loss = contrastive_loss(prev_mem, cur_mem, args.tau) 
            loss += torch.exp(cosine_similarity(cur_mem, neg_mem) 
                          ).sum(dim=1).log().mean()
        elif args.con=='stru':
            loss = contrastive_loss(out, cur_mem, args.tau) 
            loss += torch.exp(cosine_similarity(out, neg_mem)
                            ).sum(dim=1).log().mean()
        elif args.con=='all':
            loss = args.alpha*contrastive_loss(out, cur_mem, args.tau) + \
                contrastive_loss(prev_mem, cur_mem, args.tau)
            loss += torch.exp(cosine_similarity(cur_mem, neg_mem) 
                            ).sum(dim=1).log().mean()
            loss += args.alpha*torch.exp(cosine_similarity(out, neg_mem) 
                            ).sum(dim=1).log().mean()


        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        pbar.set_description(f"Train Loss = {loss.item():.4f}")

    return total_loss / len(loader)


## Test function

@torch.no_grad()
def test(loader, model, history, measure, device, data):
    preds = []
    labels = []
    model.eval()
    history.eval()
    for batch in tqdm(loader):
        torch.cuda.empty_cache()
        batch = batch.to(device)
        out = model(batch)
        src = batch.src[: batch.batch_size]
        out = out[src]
        label = batch.y[: batch.batch_size]
        idx = batch.input_id.cpu()
        raw_msg = torch.zeros(idx.size(0),1).to(device) 
        cur_mem, prev_mem = history(out, data.src[idx], update=True)
        if args.con=='his':
            pred = 1-torch.diag(cosine_similarity(cur_mem, prev_mem)).view(-1)
        elif args.con=='stru': 
            pred = 1-torch.diag(cosine_similarity(out, cur_mem)).view(-1)
        elif args.con=='all':
            p1 = torch.diag(cosine_similarity(out, cur_mem)).view(-1)
            p2 = torch.diag(cosine_similarity(cur_mem, prev_mem)).view(-1)
            pred = (1 - p1) + (1 - p2) 
            
        pred = pred.sigmoid() 
        torch.cuda.empty_cache()
        preds.append(pred.detach().cpu())
        labels.append(label.detach().cpu())
    preds = torch.cat(preds)
    preds[torch.isnan(preds)] = 0.0 
    labels = torch.cat(labels)
    return measure(labels[labels!=2], preds[labels!=2]) 


In [23]:
best = 0
best_val = 0

optimizer = torch.optim.Adam(
    list(model.parameters()) + list(history.parameters()), lr=args.lr
)
measure = Measure("auc")

for epoch in range(1, 1 + args.epochs):
    loss = train(train_loader, model, history, optimizer, device, data, args)
    val_auc = test(valid_loader, model, history, measure, device, data)
    test_auc = test(test_loader, model, history, measure, device, data)

    if val_auc > best_val:
        best_val = val_auc 
        best = test_auc

    print(
        f"Epoch: {epoch:03d}, Loss: {loss:.4f}, "
        f"Val AUC: {val_auc:.2%}, Test AUC: {test_auc:.2%}, Best AUC: {best:.2%}  "
    )
    log.info(
        f"Epoch: {epoch:03d}, Loss: {loss:.4f}, "
        f"Val AUC: {val_auc:.2%}, Test AUC: {test_auc:.2%}, Best AUC: {best:.2%} "
    )


100%|██████████| 93/93 [00:02<00:00, 38.47it/s]


Epoch: 001, Loss: 21.1749, Val AUC: 73.56%, Test AUC: 75.49%, Best AUC: 75.49%  


100%|██████████| 93/93 [00:02<00:00, 40.92it/s]


Epoch: 002, Loss: 18.9916, Val AUC: 85.71%, Test AUC: 88.40%, Best AUC: 88.40%  


100%|██████████| 93/93 [00:02<00:00, 43.74it/s]


...

Epoch: 033, Loss: 11.5523, Val AUC: 86.05%, Test AUC: 87.28%, Best AUC: 89.55%  


100%|██████████| 93/93 [00:02<00:00, 44.69it/s]


Epoch: 034, Loss: 11.5413, Val AUC: 85.49%, Test AUC: 86.30%, Best AUC: 89.55%  


100%|██████████| 93/93 [00:02<00:00, 40.12it/s]

Epoch: 035, Loss: 11.5319, Val AUC: 84.56%, Test AUC: 87.16%, Best AUC: 89.55%  


### Recurrent Based Improuvemnet 

In [8]:

"""
Enhanced Temporal Trust GNN — Recurrent + Frequency-Aware Edition
==================================================================
Key architectural upgrades over the 87% baseline:

  1.  Learnable Time Encoding  — Bochner / random Fourier features replace
      scalar linear encoding; captures periodicity and multi-scale frequency.

  2.  Recurrent Node Memory (GRU-based)  — each node maintains a hidden state
      that is updated every time it appears, giving a true sequential view of
      its behavioral history.

  3.  Interaction Frequency Tracker  — counts how often each node interacts
      and decays the count with an exponential half-life, producing a
      "burstiness" signal.

  4.  Neighborhood Evolution Bank  — stores a rolling window of neighbor
      embeddings per node; the trust score is computed over HOW a node's
      neighborhood has changed, not just its current state.

  5.  Temporal Cross-Attention Trust  — trust between i and j is now based on
      comparing the EVOLUTION TRAJECTORY of j's neighborhood against node i's
      recurrent history, using cross-attention over time steps.

  6.  Fourier Time-Aware Positional Bias  — attention logits get an additive
      bias from the Fourier similarity between edge timestamps and node's
      last-seen time (recency-sensitive attention).

  7.  Recurrent-Augmented Message Aggregation  — aggregated messages are
      fused with the node's GRU hidden state before the output projection,
      so the GNN output reflects both local structure AND temporal history.
"""



"""
Recurrent Evolution GNN — Shape-Fixed Version
==============================================
Fixes:
  - var_i/var_j shape mismatch in TemporalEvolutionTrust (was heads×head_dim, now flat dim)
  - recency_bias shape mismatch (was indexed by N when size is E)
  - evo_vals/evo_times propagation through MessagePassing (window dim causes issues, flattened)
  - mem_vars_split fallback always produces correct (N, dim) shape
  - time_per_node used correctly as AdaLN condition (projected to hidden_dim)
"""

import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import MessagePassing
from torch_geometric.utils import softmax


# ═══════════════════════════════════════════════════════════════════════════════
# 1. Fourier Time Encoding
# ═══════════════════════════════════════════════════════════════════════════════

class FourierTimeEncoding(nn.Module):
    def __init__(self, out_dim):
        super().__init__()
        assert out_dim % 2 == 0
        self.k  = out_dim // 2
        self.W  = nn.Parameter(torch.randn(1, self.k) * 0.02)
        self.b  = nn.Parameter(torch.zeros(1, self.k))
        self.w0 = nn.Parameter(torch.ones(1, 1))
        self.b0 = nn.Parameter(torch.zeros(1, 1))
        self.out_proj = nn.Linear(self.k + 1, out_dim)

    def forward(self, t):
        t        = t.float().view(-1, 1)
        trend    = t * self.w0 + self.b0
        periodic = torch.sin(t * self.W + self.b)
        return self.out_proj(torch.cat([trend, periodic], dim=-1))


# ═══════════════════════════════════════════════════════════════════════════════
# 2. Interaction Frequency Tracker
# ═══════════════════════════════════════════════════════════════════════════════

class FrequencyTracker(nn.Module):
    def __init__(self, num_nodes, decay=0.95):
        super().__init__()
        self.decay = decay
        self.register_buffer("count",     torch.zeros(num_nodes))
        self.register_buffer("last_time", torch.zeros(num_nodes))

    def read(self, idx):
        return self.count[idx]

    @torch.no_grad()
    def update(self, idx, t):
        elapsed            = (t.float() - self.last_time[idx]).clamp(min=0)
        decay              = self.decay ** elapsed
        self.count[idx]    = decay * self.count[idx] + 1.0
        self.last_time[idx] = t.float()


# ═══════════════════════════════════════════════════════════════════════════════
# 3. Recurrent Node Memory (GRU-based)
# ═══════════════════════════════════════════════════════════════════════════════

class RecurrentNodeMemory(nn.Module):
    def __init__(self, num_nodes, dim, momentum=0.9):
        super().__init__()
        self.momentum = momentum
        self.dim      = dim
        self.gru_cell = nn.GRUCell(dim, dim)
        self.register_buffer("hidden",   torch.zeros(num_nodes, dim))
        self.register_buffer("variance", torch.ones(num_nodes, dim))

    def read(self, idx):
        return self.hidden[idx], self.variance[idx]          # (B,D), (B,D)

    def get_hidden(self, idx):
        return self.hidden[idx]

    @torch.no_grad()
    def write(self, idx, new_repr):
        h_prev = self.hidden[idx]
        with torch.enable_grad():
            h_new = self.gru_cell(new_repr.detach(), h_prev)
        delta              = h_new.detach() - h_prev
        self.variance[idx] = (self.momentum * self.variance[idx]
                              + (1 - self.momentum) * delta ** 2)
        self.hidden[idx]   = h_new.detach()


# ═══════════════════════════════════════════════════════════════════════════════
# 4. Neighborhood Evolution Bank
# ═══════════════════════════════════════════════════════════════════════════════

class NeighborhoodEvolutionBank(nn.Module):
    def __init__(self, num_nodes, dim, window=8):
        super().__init__()
        self.window = window
        self.dim    = dim
        self.register_buffer("bank",       torch.zeros(num_nodes, window, dim))
        self.register_buffer("timestamps", torch.zeros(num_nodes, window))
        self.register_buffer("ptr",        torch.zeros(num_nodes, dtype=torch.long))

    def read(self, idx):
        return self.bank[idx], self.timestamps[idx]          # (B,W,D), (B,W)

    @torch.no_grad()
    def write(self, idx, neighbor_repr, t):
        for bi in range(idx.size(0)):
            p = int(self.ptr[idx[bi]].item()) % self.window
            self.bank[idx[bi], p]       = neighbor_repr[bi].detach()
            self.timestamps[idx[bi], p] = t[bi].float()
            self.ptr[idx[bi]]          += 1


# ═══════════════════════════════════════════════════════════════════════════════
# 5. Temporal Evolution Trust  ← SHAPE BUG FIXED HERE
#    var_i / var_j arrive as (E, dim) flat tensors — NOT split into heads
# ═══════════════════════════════════════════════════════════════════════════════

class TemporalEvolutionTrust(nn.Module):
    def __init__(self, dim, num_heads=4, window=8):
        super().__init__()
        # Use dim directly — no head_dim split inside trust
        self.num_heads = num_heads
        self.head_dim  = dim // num_heads
        self.window    = window

        self.q_proj = nn.Linear(dim, dim)
        self.k_proj = nn.Linear(dim, dim)
        self.v_proj = nn.Linear(dim, dim)

        self.time_decay      = nn.Parameter(torch.ones(1) * 0.1)
        self.uncertainty_gate = nn.Linear(dim, 1)
        self.out             = nn.Linear(dim, 1)
        self.scale           = self.head_dim ** -0.5

    def forward(self, h_i, h_j, evol_j, evol_t_j, var_i, var_j,
                current_t, eps=1e-6):
        """
        h_i:      (E, dim)
        h_j:      (E, dim)
        evol_j:   (E, W, dim)
        evol_t_j: (E, W)
        var_i:    (E, dim)   ← flat, not split into heads
        var_j:    (E, dim)
        current_t:(E,)
        """
        E, W, D = evol_j.shape
        nh, hd  = self.num_heads, self.head_dim

        Q = self.q_proj(h_i).view(E, nh, hd)                        # (E, nh, hd)

        evol_flat = evol_j.reshape(E * W, D)
        K = self.k_proj(evol_flat).view(E, W, nh, hd)               # (E, W, nh, hd)
        V = self.v_proj(evol_flat).view(E, W, nh, hd)

        # Attention: Q over evolution window
        Q_exp = Q.unsqueeze(2)                                       # (E, nh, 1, hd)
        K_t   = K.permute(0, 2, 1, 3)                               # (E, nh, W, hd)
        attn  = (Q_exp * K_t).sum(-1) * self.scale                  # (E, nh, W)

        # Time-decay bias
        dt        = (current_t.unsqueeze(1) - evol_t_j).clamp(min=0)   # (E, W)
        time_bias = -self.time_decay.abs() * dt                         # (E, W)
        attn      = attn + time_bias.unsqueeze(1)

        # Uncertainty gate over evolution window
        unc_gate = torch.sigmoid(
            self.uncertainty_gate(evol_j).squeeze(-1))               # (E, W)
        attn = attn * unc_gate.unsqueeze(1)

        attn   = F.softmax(attn, dim=-1)                             # (E, nh, W)
        V_t    = V.permute(0, 2, 1, 3)                              # (E, nh, W, hd)
        pooled = (attn.unsqueeze(-1) * V_t).sum(2)                  # (E, nh, hd)
        pooled = pooled.reshape(E, D)                                # (E, dim)

        # ── Confidence from flat var ─────────────────────────────────────
        conf_i = 1.0 / (1.0 + var_i.mean(-1).sqrt().clamp(min=eps)) # (E,)
        conf_j = 1.0 / (1.0 + var_j.mean(-1).sqrt().clamp(min=eps)) # (E,)

        trust_logit = self.out(pooled).squeeze(-1)                   # (E,)
        return torch.sigmoid(trust_logit * conf_i * conf_j)          # (E,)


# ═══════════════════════════════════════════════════════════════════════════════
# 6. Rotary Embedding
# ═══════════════════════════════════════════════════════════════════════════════

class RotaryEmbedding(nn.Module):
    def __init__(self, dim):
        super().__init__()
        assert dim % 2 == 0
        inv_freq = 1.0 / (10000 ** (torch.arange(0, dim, 2).float() / dim))
        self.register_buffer("inv_freq", inv_freq)

    def forward(self, x, t):
        freqs    = torch.outer(t.float(), self.inv_freq)
        cos      = freqs.cos().unsqueeze(1)
        sin      = freqs.sin().unsqueeze(1)
        x1, x2  = x[..., ::2], x[..., 1::2]
        x_rot    = torch.stack([-x2, x1], dim=-1).flatten(-2)
        cos_full = cos.expand_as(x1).repeat_interleave(2, dim=-1)
        sin_full = sin.expand_as(x1).repeat_interleave(2, dim=-1)
        return x * cos_full + x_rot * sin_full


# ═══════════════════════════════════════════════════════════════════════════════
# 7. Adaptive LayerNorm
# ═══════════════════════════════════════════════════════════════════════════════

class AdaLayerNorm(nn.Module):
    def __init__(self, dim, cond_dim):
        super().__init__()
        self.norm = nn.LayerNorm(dim, elementwise_affine=False)
        self.proj = nn.Sequential(nn.SiLU(), nn.Linear(cond_dim, dim * 2))

    def forward(self, x, cond):
        scale, shift = self.proj(cond).chunk(2, dim=-1)
        return self.norm(x) * (1 + scale) + shift


# ═══════════════════════════════════════════════════════════════════════════════
# 8. MoE Message Router
# ═══════════════════════════════════════════════════════════════════════════════

class MoEMessageRouter(nn.Module):
    def __init__(self, dim, num_experts=4, top_k=2, dropout=0.1):
        super().__init__()
        self.top_k   = top_k
        self.gate    = nn.Linear(dim, num_experts)
        self.experts = nn.ModuleList([
            nn.Sequential(
                nn.Linear(dim, dim * 2), nn.GELU(),
                nn.Dropout(dropout), nn.Linear(dim * 2, dim)
            ) for _ in range(num_experts)
        ])

    def forward(self, msg):
        logits              = self.gate(msg)
        topk_vals, topk_idx = logits.topk(self.top_k, dim=-1)
        weights             = F.softmax(topk_vals, dim=-1)
        out = torch.zeros_like(msg)
        for ki in range(self.top_k):
            idx = topk_idx[:, ki]
            for ei, expert in enumerate(self.experts):
                mask = (idx == ei)
                if mask.any():
                    out[mask] += weights[mask, ki].unsqueeze(-1) * expert(msg[mask])
        return out


# ═══════════════════════════════════════════════════════════════════════════════
# 9. Core Attention Layer  ← ALL SHAPE BUGS FIXED
# ═══════════════════════════════════════════════════════════════════════════════

class RecurrentEvolutionAttention(MessagePassing):
    def __init__(self, dim, heads=8, kv_heads=2, dropout=0.1,
                 num_experts=4, moe_top_k=2, evo_window=8):
        assert heads % kv_heads == 0
        super().__init__(aggr="add", node_dim=0)

        self.dim        = dim
        self.heads      = heads
        self.kv_heads   = kv_heads
        self.groups     = heads // kv_heads
        self.head_dim   = dim // heads
        self.evo_window = evo_window

        self.q_proj    = nn.Linear(dim, dim)
        self.k_proj    = nn.Linear(dim, kv_heads * self.head_dim)
        self.v_proj    = nn.Linear(dim, kv_heads * self.head_dim)
        self.o_proj    = nn.Linear(dim, dim)
        self.edge_proj = nn.Linear(dim, kv_heads * self.head_dim)

        self.rope = RotaryEmbedding(self.head_dim)

        # Per-head temperature with burstiness modulation
        self.log_temp_base   = nn.Parameter(torch.zeros(heads))
        self.burst_temp_gate = nn.Linear(1, heads)

        # Trust: receives flat (E, dim) vars — no head split
        self.trust_net = TemporalEvolutionTrust(dim, num_heads=4, window=evo_window)

        # Recurrent fusion after aggregation
        self.recurrent_fusion = nn.Sequential(
            nn.Linear(dim * 2, dim), nn.GELU(), nn.Linear(dim, dim))

        # Recency bias: projects time_emb (dim) → heads
        self.recency_bias_proj = nn.Linear(dim, heads)

        self.moe      = MoEMessageRouter(dim, num_experts, moe_top_k, dropout)
        self.dropout  = nn.Dropout(dropout)
        self.ada_norm = AdaLayerNorm(dim, dim)
        self.gate_lin = nn.Linear(dim, 1)

        self.trust_cache = None

    def split_q(self, x):   return x.view(-1, self.heads, self.head_dim)
    def split_kv(self, x):  return x.view(-1, self.kv_heads, self.head_dim)
    def expand_kv(self, x): return x.repeat_interleave(self.groups, dim=1)

    def forward(self, x, edge_index, edge_emb, time_emb,
                node_last_time, node_burst, memory, evo_bank):
        N      = x.size(0)
        E_num  = edge_emb.size(0)
        device = x.device

        Q = self.split_q(self.q_proj(x))
        K = self.expand_kv(self.split_kv(self.k_proj(x)))
        V = self.expand_kv(self.split_kv(self.v_proj(x)))

        Q = self.rope(Q, node_last_time)
        K = self.rope(K, node_last_time)

        # Per-node temperature: (N, heads)
        temp = (self.log_temp_base.unsqueeze(0)
                + self.burst_temp_gate(node_burst)).exp()

        # Memory: flat (N, dim) vars — NOT split into heads
        mem_vals, mem_vars = memory.read(torch.arange(N, device=device))
        # mem_vals: (N, dim), mem_vars: (N, dim)
        mem_vals_split = self.split_q(mem_vals)                      # (N, heads, head_dim)

        # Evolution bank: (N, W, dim) and (N, W)
        evo_vals, evo_times = evo_bank.read(torch.arange(N, device=device))

        edge_emb_kv   = self.split_kv(self.edge_proj(edge_emb))
        edge_emb_full = self.expand_kv(edge_emb_kv)                 # (E, heads, head_dim)

        # Recency bias per EDGE (not per node): index edge src into time_emb
        # time_emb is (E, dim); project to (E, heads)
        recency_bias = self.recency_bias_proj(time_emb)              # (E, heads) ← FIXED

        out = self.propagate(
            edge_index,
            Q=Q, K=K, V=V,
            mem_val=mem_vals_split,
            mem_var=mem_vars,        # flat (N, dim) — will arrive as (E, dim) after gather
            hidden=mem_vals,         # (N, dim) — same gather
            evo_vals=evo_vals,       # (N, W, dim)
            evo_times=evo_times,     # (N, W)
            edge_emb=edge_emb_full,
            node_temp=temp,          # (N, heads)
            recency_bias=recency_bias,  # (E, heads) — passed directly, not gathered
            node_last_time=node_last_time,  # (N,)
            size=(N, N)
        )

        agg = out.view(N, self.dim)

        # Recurrent fusion with GRU hidden state
        h   = memory.get_hidden(torch.arange(N, device=device))     # (N, dim)
        agg = self.recurrent_fusion(torch.cat([agg, h], dim=-1))

        agg = self.moe(agg)
        agg = self.o_proj(agg)

        # AdaLN conditioned on per-node mean time embedding
        # Scatter mean of time_emb over src nodes
        src       = edge_index[0]
        time_cond = torch.zeros(N, self.dim, device=device)
        count_cond = torch.zeros(N, 1, device=device)
        time_cond.scatter_add_(0, src.unsqueeze(1).expand(-1, self.dim), time_emb)
        count_cond.scatter_add_(0, src.unsqueeze(1), torch.ones(E_num, 1, device=device))
        time_cond = time_cond / (count_cond + 1e-6)                  # (N, dim)

        agg  = self.ada_norm(agg, time_cond)
        beta = torch.sigmoid(self.gate_lin(x))
        return beta * x + (1 - beta) * agg

    def message(self, Q_i, K_j, V_j,
                mem_val_i, mem_val_j,
                mem_var_i, mem_var_j,     # (E, dim) flat ← FIXED
                hidden_i, hidden_j,        # (E, dim)
                evo_vals_j, evo_times_j,  # (E, W, dim), (E, W)
                edge_emb,
                node_temp_i,              # (E, heads)
                recency_bias,             # (E, heads) ← passed through, no gather needed
                node_last_time_i,         # (E,)
                index):

        K_j  = K_j + edge_emb
        attn = (Q_i * K_j).sum(dim=-1) / (self.head_dim ** 0.5)     # (E, heads)
        attn = attn / node_temp_i.clamp(min=0.01)
        attn = attn + recency_bias                                    # (E, heads)

        # Trust from evolution — vars are already (E, dim) flat
        trust = self.trust_net(
            hidden_i, hidden_j,
            evo_vals_j, evo_times_j,
            mem_var_i, mem_var_j,        # (E, dim) ← no shape mismatch now
            node_last_time_i
        )                                                             # (E,)

        self.trust_cache = trust.detach()

        attn = attn * trust.unsqueeze(-1)                            # broadcast over heads
        attn = softmax(attn, index)
        attn = self.dropout(attn)
        return V_j * attn.unsqueeze(-1)                              # (E, heads, head_dim)


# ═══════════════════════════════════════════════════════════════════════════════
# 10. Full Model
# ═══════════════════════════════════════════════════════════════════════════════

class RecurrentEvolutionGNN(nn.Module):
    def __init__(
        self,
        num_nodes,
        in_dim,
        edge_dim,
        hidden_dim,
        num_layers      = 2,
        heads           = 8,
        kv_heads        = 2,
        memory_momentum = 0.9,
        num_experts     = 4,
        moe_top_k       = 2,
        dropout         = 0.1,
        evo_window      = 8,
        freq_decay      = 0.95,
    ):
        super().__init__()
        self.num_nodes  = num_nodes
        self.hidden_dim = hidden_dim

        self.input_proj = nn.Linear(in_dim, hidden_dim)
        self.edge_enc   = nn.Linear(edge_dim, hidden_dim)
        self.time_enc   = FourierTimeEncoding(hidden_dim)

        self.memory      = RecurrentNodeMemory(num_nodes, hidden_dim, momentum=memory_momentum)
        self.freq_tracker = FrequencyTracker(num_nodes, decay=freq_decay)
        self.evo_bank    = NeighborhoodEvolutionBank(num_nodes, hidden_dim, window=evo_window)

        self.layers = nn.ModuleList([
            RecurrentEvolutionAttention(
                hidden_dim, heads=heads, kv_heads=kv_heads,
                dropout=dropout, num_experts=num_experts,
                moe_top_k=moe_top_k, evo_window=evo_window
            ) for _ in range(num_layers)
        ])

        self.final_norm = nn.LayerNorm(hidden_dim)
        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)

    def forward(self, batch):
        N          = batch.x.size(0)
        device     = batch.x.device
        edge_index = batch.edge_index
        src        = edge_index[0]
        E_num      = src.size(0)

        x        = self.input_proj(batch.x)
        edge_emb = self.edge_enc(batch.msg)
        time_emb = self.time_enc(batch.t)                            # (E, hidden)

        # Per-node last-seen time via scatter max
        node_last_time = torch.zeros(N, device=device)
        node_last_time.scatter_reduce_(
            0, src, batch.t.float(), reduce='amax', include_self=True)

        # Burstiness: (N, 1)
        node_burst = self.freq_tracker.read(
            torch.arange(N, device=device)).unsqueeze(-1).float()

        for layer in self.layers:
            x = layer(
                x, edge_index, edge_emb, time_emb,
                node_last_time, node_burst,
                self.memory, self.evo_bank
            )

        x = self.final_norm(x)

        # ── Updates ─────────────────────────────────────────────────────
        all_idx = torch.arange(N, device=device)
        self.memory.write(all_idx, x)

        active_src = src.unique()
        self.freq_tracker.update(active_src, node_last_time[active_src])

        # Neighborhood mean for evolution bank
        dst          = edge_index[1]
        neigh_mean   = torch.zeros(N, self.hidden_dim, device=device)
        neigh_count  = torch.zeros(N, 1, device=device)
        neigh_mean.scatter_add_(0, dst.unsqueeze(1).expand(-1, self.hidden_dim), x[src])
        neigh_count.scatter_add_(0, dst.unsqueeze(1), torch.ones(E_num, 1, device=device))
        neigh_mean   = neigh_mean / (neigh_count + 1e-6)

        active_dst = dst.unique()
        self.evo_bank.write(active_dst, neigh_mean[active_dst], node_last_time[active_dst])

        return x





# ═══════════════════════════════════════════════════════════════════════════════
# Sanity check
# ═══════════════════════════════════════════════════════════════════════════════

if __name__ == "__main__":
    from types import SimpleNamespace

    N, E_num = 200, 600
    IN_DIM, EDGE_DIM, HIDDEN = 1, 172, 64

    batch = SimpleNamespace(
        x          = torch.randn(N, IN_DIM),
        msg        = torch.randn(E_num, EDGE_DIM),
        t          = torch.rand(E_num) * 1000,
        edge_index = torch.randint(0, N, (2, E_num))
    )

    model = RecurrentEvolutionGNN(
        num_nodes  = N,
        in_dim     = IN_DIM,
        edge_dim   = EDGE_DIM,
        hidden_dim = HIDDEN,
        num_layers = 2,
        heads      = 8,
        kv_heads   = 2,
        evo_window = 8,
    )

    out = model(batch)
    print(f"Output shape : {out.shape}")        # (200, 64) ✓
    params = sum(p.numel() for p in model.parameters())
    print(f"Total parameters: {params:,}")
    print("Sanity check passed ✓")

Output shape : torch.Size([200, 64])
Total parameters: 262,450
Sanity check passed ✓


In [9]:
print(model)

RecurrentEvolutionGNN(
  (input_proj): Linear(in_features=1, out_features=64, bias=True)
  (edge_enc): Linear(in_features=172, out_features=64, bias=True)
  (time_enc): FourierTimeEncoding(
    (out_proj): Linear(in_features=33, out_features=64, bias=True)
  )
  (memory): RecurrentNodeMemory(
    (gru_cell): GRUCell(64, 64)
  )
  (freq_tracker): FrequencyTracker()
  (evo_bank): NeighborhoodEvolutionBank()
  (layers): ModuleList(
    (0-1): 2 x RecurrentEvolutionAttention()
  )
  (final_norm): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
)


In [116]:
model = RecurrentEvolutionGNN(
        num_nodes  = data.num_nodes,
        in_dim     = data.x.size(1),
        edge_dim   = data.msg.size(1),
        hidden_dim = args.hidden_channels,
        num_layers = 2,
        heads      = 8,
        kv_heads   = 2,
        evo_window = 4,
    )


In [117]:
model = model.to(device)

In [123]:
## Training Function 

def train(loader, model, history, optimizer, device, data, args):
    model.train()
    history.train()
    pbar = tqdm(loader)
    total_loss = 0
    for batch in pbar:
        batch = batch.to(device)
        optimizer.zero_grad()
        out = model(batch)
        src = batch.src[: batch.batch_size]
        out = out[src]
        idx = batch.input_id.cpu()
        num_neg = batch.batch_size
        neg_mem = history.get_history(torch.randint(
            0, history.num_nodes, size=(num_neg,)))
        raw_msg = torch.zeros(idx.size(0),1).to(device) 
        cur_mem, prev_mem = history(out, data.src[idx]) 

        if args.con=='his':
            loss = contrastive_loss(prev_mem, cur_mem, args.tau) 
            loss += torch.exp(cosine_similarity(cur_mem, neg_mem) 
                          ).sum(dim=1).log().mean()
        elif args.con=='stru':
            loss = contrastive_loss(out, cur_mem, args.tau) 
            loss += torch.exp(cosine_similarity(out, neg_mem)
                            ).sum(dim=1).log().mean()
        elif args.con=='all':
            loss = args.alpha*contrastive_loss(out, cur_mem, args.tau) + \
                contrastive_loss(prev_mem, cur_mem, args.tau)
            loss += torch.exp(cosine_similarity(cur_mem, neg_mem) 
                            ).sum(dim=1).log().mean()
            loss += args.alpha*torch.exp(cosine_similarity(out, neg_mem) 
                            ).sum(dim=1).log().mean()


        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        pbar.set_description(f"Train Loss = {loss.item():.4f}")

    return total_loss / len(loader)


## Test function

@torch.no_grad()
def test(loader, model, history, measure, device, data):
    preds = []
    labels = []
    model.eval()
    history.eval()
    for batch in tqdm(loader):
        torch.cuda.empty_cache()
        batch = batch.to(device)
        out = model(batch)
        src = batch.src[: batch.batch_size]
        out = out[src]
        label = batch.y[: batch.batch_size]
        idx = batch.input_id.cpu()
        raw_msg = torch.zeros(idx.size(0),1).to(device) 
        cur_mem, prev_mem = history(out, data.src[idx], update=True)
        if args.con=='his':
            pred = 1-torch.diag(cosine_similarity(cur_mem, prev_mem)).view(-1)
        elif args.con=='stru': 
            pred = 1-torch.diag(cosine_similarity(out, cur_mem)).view(-1)
        elif args.con=='all':
            p1 = torch.diag(cosine_similarity(out, cur_mem)).view(-1)
            p2 = torch.diag(cosine_similarity(cur_mem, prev_mem)).view(-1)
            pred = (1 - p1) + (1 - p2) 
            
        pred = pred.sigmoid() 
        torch.cuda.empty_cache()
        preds.append(pred.detach().cpu())
        labels.append(label.detach().cpu())
    preds = torch.cat(preds)
    preds[torch.isnan(preds)] = 0.0 
    labels = torch.cat(labels)
    return measure(labels[labels!=2], preds[labels!=2]) 


In [124]:
best = 0
best_val = 0

optimizer = torch.optim.Adam(
    list(model.parameters()) + list(history.parameters()), lr=args.lr
)
measure = Measure("auc")

for epoch in range(1, 1 + args.epochs):
    loss = train(train_loader, model, history, optimizer, device, data, args)
    val_auc = test(valid_loader, model, history, measure, device, data)
    test_auc = test(test_loader, model, history, measure, device, data)

    if val_auc > best_val:
        best_val = val_auc 
        best = test_auc

    print(
        f"Epoch: {epoch:03d}, Loss: {loss:.4f}, "
        f"Val AUC: {val_auc:.2%}, Test AUC: {test_auc:.2%}, Best AUC: {best:.2%}  "
    )
    log.info(
        f"Epoch: {epoch:03d}, Loss: {loss:.4f}, "
        f"Val AUC: {val_auc:.2%}, Test AUC: {test_auc:.2%}, Best AUC: {best:.2%} "
    )


100%|██████████| 93/93 [00:06<00:00, 15.00it/s]


Epoch: 001, Loss: 17.9858, Val AUC: 76.66%, Test AUC: 84.36%, Best AUC: 84.36%  


100%|██████████| 93/93 [00:06<00:00, 14.77it/s]


Epoch: 002, Loss: 13.1112, Val AUC: 81.81%, Test AUC: 89.45%, Best AUC: 89.45%  


100%|██████████| 93/93 [00:06<00:00, 14.82it/s]


...

Epoch: 024, Loss: 11.5798, Val AUC: 61.81%, Test AUC: 67.44%, Best AUC: 87.41%  


100%|██████████| 93/93 [00:06<00:00, 15.03it/s]


Epoch: 025, Loss: 11.6072, Val AUC: 75.66%, Test AUC: 66.65%, Best AUC: 87.41%  


Train Loss = 11.6107:  39%|███▉      | 169/431 [00:23<00:37,  7.07it/s]


KeyboardInterrupt: 

In [29]:
"""
Enhanced Temporal Anomaly GNN
==============================
Targeted improvements over the 87% baseline:

  1. TEMPORAL MODELING
     - Time2Vec encoding: learns periodic + trend components
     - Relative time encoding between node's last-seen and current edge
     - Per-node time gap features fed into attention

  2. TRUST (core contribution)
     - Trust is computed from NEIGHBORHOOD HISTORY EVOLUTION
     - Cross-attention between node i's recurrent state and
       node j's recent neighbor embeddings (window of K steps)
     - Uncertainty from memory variance gates the trust score
     - Trust is per-edge scalar modulating attention weights

  3. ATTENTION
     - Standard multi-head attention (no GQA — simpler = faster + more stable)
     - Trust-modulated attention scores
     - Edge features injected into keys
     - Time-relative positional bias per edge

  4. CONTRASTIVE LEARNING
     - Two views: structural (GNN now) vs temporal (history recall)
     - Hard negatives: nodes structurally similar but historically different
     - In-batch InfoNCE — no complex multi-term loss
     - Single temperature, stable training

  5. ANOMALY SCORE
     - Primary: structural vs temporal view disagreement
     - Secondary: current vs previous memory consistency
     - Confidence-weighted by memory variance
"""

import math
import copy
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import MessagePassing
from torch_geometric.utils import softmax
from tqdm import tqdm


# ═══════════════════════════════════════════════════════════════════════════════
# 1. Time2Vec — learns periodic + trend components from raw timestamps
# ═══════════════════════════════════════════════════════════════════════════════

class Time2Vec(nn.Module):
    """
    t → [w0*t + b0,  sin(w1*t + b1), ..., sin(wk*t + bk)]
    Linear term captures trend, sinusoids capture periodicity.
    Frequencies w are LEARNED — model discovers daily/weekly patterns.
    """
    def __init__(self, out_dim):
        super().__init__()
        # k periodic components + 1 linear
        k = out_dim - 1
        self.w0 = nn.Parameter(torch.randn(1) * 0.01)
        self.b0 = nn.Parameter(torch.zeros(1))
        self.W  = nn.Parameter(torch.randn(k) * 0.01)
        self.B  = nn.Parameter(torch.zeros(k))
        self.out_dim = out_dim

    def forward(self, t):
        """t: (N,) → (N, out_dim)"""
        t = t.float().unsqueeze(-1)                                    # (N, 1)
        linear   = t * self.w0 + self.b0                               # (N, 1)
        periodic = torch.sin(t * self.W + self.B)                      # (N, k)
        return torch.cat([linear, periodic], dim=-1)                   # (N, out_dim)


class TimeEncoder(nn.Module):
    """
    Full time encoding: Time2Vec → MLP projection → hidden_dim
    Also computes RELATIVE time gap between edge time and node's last seen time.
    """
    def __init__(self, hidden_dim):
        super().__init__()
        self.t2v     = Time2Vec(hidden_dim // 2)
        self.rel_t2v = Time2Vec(hidden_dim // 2)   # for relative time gap
        self.proj = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, hidden_dim))

    def forward(self, t_abs, t_rel=None):
        """
        t_abs: (E,) absolute edge timestamps
        t_rel: (E,) relative gap = edge_time - node_last_seen_time (optional)
        Returns: (E, hidden_dim)
        """
        abs_enc = self.t2v(t_abs)                                      # (E, h/2)
        if t_rel is not None:
            rel_enc = self.rel_t2v(t_rel)                              # (E, h/2)
        else:
            rel_enc = torch.zeros_like(abs_enc)
        combined = torch.cat([abs_enc, rel_enc], dim=-1)               # (E, h)
        return self.proj(combined)                                     # (E, h)


# ═══════════════════════════════════════════════════════════════════════════════
# 2. Recurrent Node Memory with Uncertainty
# ═══════════════════════════════════════════════════════════════════════════════

class RecurrentMemory(nn.Module):
    """
    Per-node GRU hidden state updated every time a node is seen.
    Tracks variance (uncertainty) as EMA of squared delta.
    Low variance = stable node = trustworthy memory.
    """
    def __init__(self, num_nodes, dim, momentum=0.9):
        super().__init__()
        self.gru      = nn.GRUCell(dim, dim)
        self.momentum = momentum
        self.register_buffer("hidden",   torch.zeros(num_nodes, dim))
        self.register_buffer("variance", torch.ones(num_nodes, dim) * 0.1)

    def read(self, idx):
        return self.hidden[idx], self.variance[idx]                    # (B,D), (B,D)

    def get_hidden(self, idx):
        return self.hidden[idx]

    @torch.no_grad()
    def write(self, idx, x):
        h_old = self.hidden[idx]
        h_new = self.gru(x.detach(), h_old)
        delta = h_new - h_old
        self.variance[idx] = (self.momentum * self.variance[idx]
                              + (1 - self.momentum) * delta.pow(2))
        self.hidden[idx] = h_new


# ═══════════════════════════════════════════════════════════════════════════════
# 3. Neighborhood Evolution Bank
# ═══════════════════════════════════════════════════════════════════════════════

class EvolutionBank(nn.Module):
    """
    Rolling window of K most recent neighbor mean embeddings per node.
    Used by TrustModule to see HOW a node's neighborhood has evolved.
    """
    def __init__(self, num_nodes, dim, window=6):
        super().__init__()
        self.window = window
        self.register_buffer("bank", torch.zeros(num_nodes, window, dim))
        self.register_buffer("ptr",  torch.zeros(num_nodes, dtype=torch.long))

    def read(self, idx):
        return self.bank[idx]                                          # (B, W, D)

    @torch.no_grad()
    def write(self, idx, emb):
        """emb: (B, D)"""
        for b in range(idx.size(0)):
            p = int(self.ptr[idx[b]]) % self.window
            self.bank[idx[b], p] = emb[b].detach()
            self.ptr[idx[b]] += 1


# ═══════════════════════════════════════════════════════════════════════════════
# 4. Trust Module — CORE CONTRIBUTION
#
# Trust(i→j) answers: "Given how i has behaved historically and how j's
# neighborhood has evolved, should i trust a message from j?"
#
# Mechanism:
#   Query  = node i's GRU hidden state (its history)
#   Keys   = node j's neighborhood evolution window (j's recent neighbors)
#   Values = same as keys
#   Cross-attention pools the evolution window → single trust scalar
#   Uncertainty from memory variance gates the final score
# ═══════════════════════════════════════════════════════════════════════════════

class TrustModule(nn.Module):
    def __init__(self, dim, num_heads=4, window=6):
        super().__init__()
        assert dim % num_heads == 0
        self.num_heads = num_heads
        self.head_dim  = dim // num_heads
        self.scale     = self.head_dim ** -0.5
        self.window    = window

        # Project node i's history → query
        self.q_proj = nn.Linear(dim, dim)
        # Project j's evolution window → keys and values
        self.k_proj = nn.Linear(dim, dim)
        self.v_proj = nn.Linear(dim, dim)

        # Learned time-decay: older evolution steps matter less
        self.log_decay = nn.Parameter(torch.tensor(-2.0))  # starts at e^-2 ≈ 0.13

        # Final trust scalar from pooled evolution context
        self.trust_out = nn.Sequential(
            nn.Linear(dim, dim // 2),
            nn.GELU(),
            nn.Linear(dim // 2, 1)
        )

    def forward(self, h_i, evo_j, var_i, var_j, step_weights=None, eps=1e-6):
        """
        h_i:    (E, dim)    — node i GRU hidden state
        evo_j:  (E, W, dim) — node j neighborhood evolution window
        var_i:  (E, dim)    — memory variance of i (uncertainty)
        var_j:  (E, dim)    — memory variance of j
        Returns: (E,) trust scores in [0, 1]
        """
        E, W, D = evo_j.shape
        nh, hd  = self.num_heads, self.head_dim

        # Query from i's recurrent history
        Q = self.q_proj(h_i).view(E, nh, hd)                         # (E, nh, hd)

        # Keys and values from j's evolution window
        evo_flat = evo_j.reshape(E * W, D)
        K = self.k_proj(evo_flat).view(E, W, nh, hd)                 # (E, W, nh, hd)
        V = self.v_proj(evo_flat).view(E, W, nh, hd)

        # Attention scores: i's history attends over j's evolution
        Q_exp = Q.unsqueeze(2)                                        # (E, nh, 1, hd)
        K_t   = K.permute(0, 2, 1, 3)                                # (E, nh, W, hd)
        scores = (Q_exp * K_t).sum(-1) * self.scale                  # (E, nh, W)

        # Time-decay bias: step 0 is oldest, step W-1 is newest
        # Newer steps should get higher weight
        steps     = torch.arange(W, device=h_i.device).float()
        decay_w   = torch.exp(self.log_decay.exp() * steps)          # (W,) increasing
        decay_w   = decay_w / decay_w.sum()
        scores    = scores + decay_w.view(1, 1, W)                   # broadcast

        attn   = F.softmax(scores, dim=-1)                           # (E, nh, W)
        V_t    = V.permute(0, 2, 1, 3)                               # (E, nh, W, hd)
        pooled = (attn.unsqueeze(-1) * V_t).sum(2)                   # (E, nh, hd)
        pooled = pooled.reshape(E, D)                                 # (E, dim)

        # Raw trust logit
        trust_logit = self.trust_out(pooled).squeeze(-1)             # (E,)

        # Uncertainty gating:
        # High variance in i → i's history is unreliable → lower trust weight
        # High variance in j → j's neighborhood is erratic → lower trust
        conf_i = 1.0 / (1.0 + var_i.mean(-1).sqrt().clamp(min=eps)) # (E,)
        conf_j = 1.0 / (1.0 + var_j.mean(-1).sqrt().clamp(min=eps)) # (E,)
        conf   = (conf_i * conf_j).sqrt()                            # geometric mean

        return torch.sigmoid(trust_logit + conf.log().clamp(min=-5)) # (E,) in [0,1]


# ═══════════════════════════════════════════════════════════════════════════════
# 5. Temporal Trust Attention Layer
# ═══════════════════════════════════════════════════════════════════════════════

class TemporalTrustAttention(MessagePassing):
    def __init__(self, dim, heads=8, dropout=0.1, window=6):
        super().__init__(aggr="add", node_dim=0)
        assert dim % heads == 0
        self.dim      = dim
        self.heads    = heads
        self.head_dim = dim // heads
        self.scale    = self.head_dim ** -0.5
        self.window   = window

        self.q_proj    = nn.Linear(dim, dim)
        self.k_proj    = nn.Linear(dim, dim)
        self.v_proj    = nn.Linear(dim, dim)
        self.o_proj    = nn.Linear(dim, dim)
        self.edge_proj = nn.Linear(dim, dim)   # edge features → key bias
        self.time_proj = nn.Linear(dim, dim)   # time encoding → key bias

        self.trust  = TrustModule(dim, num_heads=4, window=window)
        self.dropout = nn.Dropout(dropout)

        # Gated residual
        self.gate = nn.Linear(dim, 1)
        self.norm = nn.LayerNorm(dim)

        # Fusion with recurrent hidden state
        self.fusion = nn.Sequential(
            nn.Linear(dim * 2, dim),
            nn.GELU(),
            nn.Linear(dim, dim)
        )

        self.trust_cache = None

    def split(self, x):
        return x.view(-1, self.heads, self.head_dim)

    def forward(self, x, edge_index, edge_emb, time_emb, memory, evo_bank):
        N      = x.size(0)
        device = x.device

        Q = self.split(self.q_proj(x))                                # (N, h, hd)
        K = self.split(self.k_proj(x))
        V = self.split(self.v_proj(x))

        mem_vals, mem_vars = memory.read(torch.arange(N, device=device))
        evo_vals = evo_bank.read(torch.arange(N, device=device))      # (N, W, D)

        edge_emb_proj = self.split(self.edge_proj(edge_emb))          # (E, h, hd)
        time_emb_proj = self.split(self.time_proj(time_emb))          # (E, h, hd)

        out = self.propagate(
            edge_index,
            Q=Q, K=K, V=V,
            mem_val=mem_vals,
            mem_var=mem_vars,
            hidden=mem_vals,
            evo_vals=evo_vals,
            edge_emb=edge_emb_proj,
            time_emb=time_emb_proj,
            size=(N, N)
        )

        out = out.view(N, self.dim)

        # Fuse with recurrent hidden state
        h   = memory.get_hidden(torch.arange(N, device=device))
        out = self.fusion(torch.cat([out, h], dim=-1))
        out = self.o_proj(out)

        # Gated residual
        g = torch.sigmoid(self.gate(x))
        return self.norm(g * x + (1 - g) * out)

    def message(self, Q_i, K_j, V_j,
                mem_val_i, mem_val_j,
                mem_var_i, mem_var_j,
                hidden_i, hidden_j,
                evo_vals_j,
                edge_emb, time_emb,
                index):

        # Enrich keys with edge and time features
        K_j = K_j + edge_emb + time_emb                               # (E, h, hd)

        # Attention scores
        attn = (Q_i * K_j).sum(-1) * self.scale                      # (E, h)

        # Trust score from neighborhood evolution
        # Uses i's recurrent history vs j's evolution window
        trust = self.trust(
            hidden_i, evo_vals_j,
            mem_var_i, mem_var_j
        )                                                              # (E,)
        self.trust_cache = trust.detach()

        # Modulate attention by trust
        attn = attn * trust.unsqueeze(-1)                             # (E, h)
        attn = softmax(attn, index)
        attn = self.dropout(attn)

        return V_j * attn.unsqueeze(-1)                               # (E, h, hd)


# ═══════════════════════════════════════════════════════════════════════════════
# 6. Full Model
# ═══════════════════════════════════════════════════════════════════════════════

class EnhancedTemporalGNN(nn.Module):
    def __init__(self, num_nodes, in_dim, edge_dim, hidden_dim,
                 num_layers=2, heads=8, dropout=0.1,
                 window=6, memory_momentum=0.9):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.num_nodes  = num_nodes

        self.input_proj = nn.Linear(in_dim, hidden_dim)
        self.edge_enc   = nn.Linear(edge_dim, hidden_dim)
        self.time_enc   = TimeEncoder(hidden_dim)

        self.memory   = RecurrentMemory(num_nodes, hidden_dim, momentum=memory_momentum)
        self.evo_bank = EvolutionBank(num_nodes, hidden_dim, window=window)

        self.layers = nn.ModuleList([
            TemporalTrustAttention(hidden_dim, heads=heads,
                                   dropout=dropout, window=window)
            for _ in range(num_layers)
        ])

        self.final_norm = nn.LayerNorm(hidden_dim)
        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)

    def forward(self, batch):
        N          = batch.x.size(0)
        device     = batch.x.device
        edge_index = batch.edge_index
        src, dst   = edge_index

        x        = self.input_proj(batch.x)
        edge_emb = self.edge_enc(batch.msg)

        # Per-node last-seen time for relative encoding
        node_last_t = torch.zeros(N, device=device)
        node_last_t.scatter_reduce_(
            0, src, batch.t.float(), reduce='amax', include_self=True)

        # Relative time gap per edge: how long since src was last seen
        t_rel    = batch.t.float() - node_last_t[src]                 # (E,)
        time_emb = self.time_enc(batch.t, t_rel)                      # (E, hidden)

        for layer in self.layers:
            x = layer(x, edge_index, edge_emb, time_emb,
                      self.memory, self.evo_bank)

        x = self.final_norm(x)

        # Update memory and evolution bank
        self.memory.write(torch.arange(N, device=device), x)

        # Neighborhood mean for evolution bank
        neigh = torch.zeros(N, self.hidden_dim, device=device)
        cnt   = torch.zeros(N, 1, device=device)
        neigh.scatter_add_(0, dst.unsqueeze(1).expand(-1, self.hidden_dim), x[src])
        cnt.scatter_add_(0, dst.unsqueeze(1), torch.ones(src.size(0), 1, device=device))
        neigh = neigh / (cnt + 1e-6)
        self.evo_bank.write(dst.unique(), neigh[dst.unique()])

        return x


# ═══════════════════════════════════════════════════════════════════════════════
# 7. Contrastive Learning
#    Two views: structural (GNN output) vs temporal (history recall)
#    Hard negatives: structurally similar but historically different nodes
#    Single InfoNCE loss — clean and stable
# ═══════════════════════════════════════════════════════════════════════════════

def info_nce(anchor, positive, negatives, tau=0.07):
    """
    anchor:    (B, D)
    positive:  (B, D)
    negatives: (B, K, D)
    """
    a = F.normalize(anchor,    dim=-1)
    p = F.normalize(positive,  dim=-1)
    n = F.normalize(negatives, dim=-1)

    pos_sim = (a * p).sum(-1) / tau                                    # (B,)
    neg_sim = torch.einsum('bd,bkd->bk', a, n) / tau                  # (B, K)

    logits = torch.cat([pos_sim.unsqueeze(1), neg_sim], dim=1)        # (B, K+1)
    labels = torch.zeros(a.size(0), dtype=torch.long, device=a.device)
    return F.cross_entropy(logits, labels)


def hard_negatives(struct_out, mem_out, K=64):
    """
    Mine hard negatives: structurally similar but historically different.
    score = struct_sim - mem_sim  →  high = confusing = hard negative
    Returns: (B, K) indices
    """
    B  = struct_out.size(0)
    K  = min(K, B - 1)
    s  = F.normalize(struct_out.detach(), dim=-1)
    m  = F.normalize(mem_out.detach(),    dim=-1)

    # Structural similarity matrix
    ss = s @ s.T                                                       # (B, B)
    # Temporal similarity matrix
    ms = m @ m.T                                                       # (B, B)
    # Hard score: looks similar now but different history
    score = ss - ms
    score.fill_diagonal_(-1e9)
    return score.topk(K, dim=-1).indices                               # (B, K)


# ═══════════════════════════════════════════════════════════════════════════════
# 8. Training Function
# ═══════════════════════════════════════════════════════════════════════════════

def train(loader, model, history, optimizer, device, data, args):
    model.train()
    history.train()
    pbar       = tqdm(loader)
    total_loss = 0.0
    tau        = getattr(args, 'tau',     0.07)
    num_neg    = getattr(args, 'num_neg', 64)

    for batch in pbar:
        batch = batch.to(device)
        optimizer.zero_grad()

        # Forward
        out = model(batch)
        src = batch.src[: batch.batch_size]
        out = out[src]                                                 # (B, D)
        idx = batch.input_id.cpu()

        # Two views
        cur_mem, prev_mem = history(out, data.src[idx])               # (B, D) each

        # Hard negative mining from in-batch representations
        with torch.no_grad():
            neg_idx = hard_negatives(out, cur_mem, K=num_neg)         # (B, K)
        negatives = cur_mem[neg_idx]                                   # (B, K, D)

        # InfoNCE: structural view vs temporal view
        loss = info_nce(out, cur_mem, negatives, tau=tau)

        # Light temporal consistency: prevent memory from drifting catastrophically
        # Weight 0.1 keeps it from competing with the main loss
        consist = (1.0 - F.cosine_similarity(
            cur_mem, prev_mem.detach(), dim=-1)).mean()
        loss = loss + 0.1 * consist

        loss.backward()
        torch.nn.utils.clip_grad_norm_(
            list(model.parameters()) + list(history.parameters()), 1.0)
        optimizer.step()

        total_loss += loss.item()
        pbar.set_description(f"Loss={loss.item():.4f}")

    return total_loss / len(loader)


# ═══════════════════════════════════════════════════════════════════════════════
# 9. Test Function + Anomaly Score
# ═══════════════════════════════════════════════════════════════════════════════

@torch.no_grad()
def test(loader, model, history, measure, device, data, args):
    """
    Anomaly score = disagreement between structural and temporal views.
    Normal nodes: GNN output ≈ memory recall (two views agree).
    Anomalous nodes: GNN output ≠ memory recall (views diverge).
    """
    preds, labels = [], []
    model.eval()
    history.eval()
    eps = 1e-6

    for batch in tqdm(loader):
        torch.cuda.empty_cache()
        batch = batch.to(device)

        out   = model(batch)
        src   = batch.src[: batch.batch_size]
        out   = out[src]                                               # (B, D)
        label = batch.y[: batch.batch_size]
        idx   = batch.input_id.cpu()

        cur_mem, prev_mem = history(out, data.src[idx], update=True)

        node_ids         = data.src[idx].to(device)
        _, cur_var       = model.memory.read(node_ids)                # (B, D)

        # Confidence: low variance = stable node = score is more reliable
        conf = 1.0 / (1.0 + cur_var.mean(-1).sqrt().clamp(min=eps))  # (B,)

        # Primary: structural vs temporal disagreement
        sim_sv = F.cosine_similarity(out, cur_mem, dim=-1)            # (B,)
        score  = conf * (1.0 - sim_sv)                                # (B,)

        # Secondary: temporal self-consistency
        sim_tv  = F.cosine_similarity(cur_mem, prev_mem, dim=-1)      # (B,)
        score_t = conf * (1.0 - sim_tv)                               # (B,)

        # Optional: neighborhood evolution divergence
        evo = model.evo_bank.read(node_ids)                           # (B, W, D)
        evo_div  = 1.0 - F.cosine_similarity(evo[:, -1], evo[:, 0], dim=-1)
        score_e  = evo_div.clamp(0, 2)

        # Final score: primary dominates
        pred = (score + 0.4 * score_t + 0.2 * score_e).sigmoid()

        torch.cuda.empty_cache()
        preds.append(pred.detach().cpu())
        labels.append(label.detach().cpu())

    preds  = torch.cat(preds)
    preds[torch.isnan(preds)] = 0.0
    labels = torch.cat(labels)
    return measure(labels[labels != 2], preds[labels != 2])

In [30]:
"""
Drop-in replacement for your notebook.
Keeps: data, loaders, History, args, measure
Replaces: model, train(), test()

Just run these cells in order after your existing setup cells.
"""

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import MessagePassing
from torch_geometric.utils import softmax
from tqdm import tqdm


# ── Paste the full enhanced_gnn_v2.py here, or import it ──────────────────────
# from enhanced_gnn_v2 import (EnhancedTemporalGNN, train, test)
# ──────────────────────────────────────────────────────────────────────────────
# If running directly in notebook, paste the class definitions above this cell.


# ═══════════════════════════════════════════
# CELL: Build model + history (unchanged)
# ═══════════════════════════════════════════

try:
    del model
except NameError:
    pass

model = EnhancedTemporalGNN(
    num_nodes  = data.num_nodes,
    in_dim     = data.x.size(1),          # 1 for wikipedia
    edge_dim   = data.msg.size(1),         # 172 for wikipedia
    hidden_dim = args.hidden_channels,     # 64
    num_layers = args.nums_layers,         # 2
    heads      = args.heads,               # 8
    dropout    = args.dropout,             # 0.3
    window     = 8,
    memory_momentum = 0.9,
).to(device)

# History is UNCHANGED — reuse your existing History class
try:
    del history
except NameError:
    pass

from mhiscl.history import History

history = History(
    data.num_nodes,
    args.num_timeslots,
    args.hidden_channels,
    device=device,
    history_retrieve=args.history_retrieve,
    recurrent=args.recurrent,
).to(device)

print(model)
print(history)
print(f"Model params: {sum(p.numel() for p in model.parameters()):,}")




EnhancedTemporalGNN(
  (input_proj): Linear(in_features=1, out_features=64, bias=True)
  (edge_enc): Linear(in_features=172, out_features=64, bias=True)
  (time_enc): TimeEncoder(
    (t2v): Time2Vec()
    (rel_t2v): Time2Vec()
    (proj): Sequential(
      (0): Linear(in_features=64, out_features=64, bias=True)
      (1): GELU(approximate='none')
      (2): Linear(in_features=64, out_features=64, bias=True)
    )
  )
  (memory): RecurrentMemory(
    (gru): GRUCell(64, 64)
  )
  (evo_bank): EvolutionBank()
  (layers): ModuleList(
    (0-1): 2 x TemporalTrustAttention()
  )
  (final_norm): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
)
History(
  (recurrent_network): GRUCell(64, 64)
  (lin): Sequential(
    (0): Linear(-1, 64, bias=True)
    (1): ELU(alpha=1.0)
    (2): Linear(64, 64, bias=True)
  )
)
Model params: 149,062


In [31]:
## Training Function 

def train(loader, model, history, optimizer, device, data, args):
    model.train()
    history.train()
    pbar = tqdm(loader)
    total_loss = 0
    for batch in pbar:
        batch = batch.to(device)
        optimizer.zero_grad()
        out = model(batch)
        src = batch.src[: batch.batch_size]
        out = out[src]
        idx = batch.input_id.cpu()
        num_neg = batch.batch_size
        neg_mem = history.get_history(torch.randint(
            0, history.num_nodes, size=(num_neg,)))
        raw_msg = torch.zeros(idx.size(0),1).to(device) 
        cur_mem, prev_mem = history(raw_msg, data.src[idx]) 

        if args.con=='his':
            loss = contrastive_loss(prev_mem, cur_mem, args.tau) 
            loss += torch.exp(cosine_similarity(cur_mem, neg_mem) 
                          ).sum(dim=1).log().mean()
        elif args.con=='stru':
            loss = contrastive_loss(out, cur_mem, args.tau) 
            loss += torch.exp(cosine_similarity(out, neg_mem)
                            ).sum(dim=1).log().mean()
        elif args.con=='all':
            loss = args.alpha*contrastive_loss(out, cur_mem, args.tau) + \
                contrastive_loss(prev_mem, cur_mem, args.tau)
            loss += torch.exp(cosine_similarity(cur_mem, neg_mem) 
                            ).sum(dim=1).log().mean()
            loss += args.alpha*torch.exp(cosine_similarity(out, neg_mem) 
                            ).sum(dim=1).log().mean()


        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        pbar.set_description(f"Train Loss = {loss.item():.4f}")

    return total_loss / len(loader)


## Test function

@torch.no_grad()
def test(loader, model, history, measure, device, data):
    preds = []
    labels = []
    model.eval()
    history.eval()
    for batch in tqdm(loader):
        torch.cuda.empty_cache()
        batch = batch.to(device)
        out = model(batch)
        src = batch.src[: batch.batch_size]
        out = out[src]
        label = batch.y[: batch.batch_size]
        idx = batch.input_id.cpu()
        raw_msg = torch.zeros(idx.size(0),1).to(device) 
        cur_mem, prev_mem = history(raw_msg, data.src[idx], update=True)
        if args.con=='his':
            pred = 1-torch.diag(cosine_similarity(cur_mem, prev_mem)).view(-1)
        elif args.con=='stru': 
            pred = 1-torch.diag(cosine_similarity(out, cur_mem)).view(-1)
        elif args.con=='all':
            p1 = torch.diag(cosine_similarity(out, cur_mem)).view(-1)
            p2 = torch.diag(cosine_similarity(cur_mem, prev_mem)).view(-1)
            pred = (1 - p1) + (1 - p2) 
            
        pred = pred.sigmoid() 
        torch.cuda.empty_cache()
        preds.append(pred.detach().cpu())
        labels.append(label.detach().cpu())
    preds = torch.cat(preds)
    preds[torch.isnan(preds)] = 0.0 
    labels = torch.cat(labels)
    return measure(labels[labels!=2], preds[labels!=2]) 




In [32]:
# ═══════════════════════════════════════════
# CELL: Training loop (drop-in replacement)
# ═══════════════════════════════════════════

best     = 0
best_val = 0

optimizer = torch.optim.Adam(
    list(model.parameters()) + list(history.parameters()),
    lr=args.lr,
    weight_decay=1e-5       # small L2 reg helps generalization
)

# Cosine LR schedule: smoothly decays over training
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=args.epochs, eta_min=args.lr * 0.1)

measure = Measure("auc")

for epoch in range(1, 1 + args.epochs):
    loss    = train(train_loader, model, history, optimizer, device, data, args)
    val_auc = test(valid_loader,  model, history, measure,   device, data)
    test_auc = test(test_loader,  model, history, measure,   device, data)
    scheduler.step()

    if val_auc > best_val:
        best_val = val_auc
        best     = test_auc

    print(
        f"Epoch: {epoch:03d}, Loss: {loss:.4f}, "
        f"Val AUC: {val_auc:.2%}, Test AUC: {test_auc:.2%}, Best AUC: {best:.2%}"
    )

100%|██████████| 93/93 [00:11<00:00,  7.95it/s]


Epoch: 001, Loss: 20.9436, Val AUC: 68.81%, Test AUC: 72.15%, Best AUC: 72.15%


100%|██████████| 93/93 [00:10<00:00,  8.74it/s]


Epoch: 002, Loss: 19.7840, Val AUC: 83.94%, Test AUC: 85.21%, Best AUC: 85.21%


100%|██████████| 93/93 [00:10<00:00,  8.83it/s]


...

Epoch: 033, Loss: 16.9008, Val AUC: 88.21%, Test AUC: 87.86%, Best AUC: 90.90%


100%|██████████| 93/93 [00:04<00:00, 19.57it/s]


Epoch: 034, Loss: 16.9052, Val AUC: 88.39%, Test AUC: 89.15%, Best AUC: 90.90%


100%|██████████| 93/93 [00:04<00:00, 19.80it/s]

Epoch: 035, Loss: 16.9171, Val AUC: 85.43%, Test AUC: 88.62%, Best AUC: 90.90%


In [ ]:
# ═══════════════════════════════════════════
# CELL: Training loop (drop-in replacement)
# ═══════════════════════════════════════════

best     = 0
best_val = 0

optimizer = torch.optim.Adam(
    list(model.parameters()) + list(history.parameters()),
    lr=args.lr,
    weight_decay=1e-5       # small L2 reg helps generalization
)

# Cosine LR schedule: smoothly decays over training
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=args.epochs, eta_min=args.lr * 0.1)

measure = Measure("auc")

for epoch in range(1, 1 + args.epochs):
    loss    = train(train_loader, model, history, optimizer, device, data, args)
    val_auc = test(valid_loader,  model, history, measure,   device, data)
    test_auc = test(test_loader,  model, history, measure,   device, data)
    scheduler.step()

    if val_auc > best_val:
        best_val = val_auc
        best     = test_auc

    print(
        f"Epoch: {epoch:03d}, Loss: {loss:.4f}, "
        f"Val AUC: {val_auc:.2%}, Test AUC: {test_auc:.2%}, Best AUC: {best:.2%}"
    )

100%|██████████| 93/93 [00:04<00:00, 20.80it/s]


Epoch: 001, Loss: 20.8903, Val AUC: 77.20%, Test AUC: 78.54%, Best AUC: 78.54%


100%|██████████| 93/93 [00:04<00:00, 20.87it/s]


Epoch: 002, Loss: 18.5117, Val AUC: 86.94%, Test AUC: 89.55%, Best AUC: 89.55%


100%|██████████| 93/93 [00:04<00:00, 20.84it/s]


...

Epoch: 033, Loss: 17.0610, Val AUC: 81.36%, Test AUC: 88.51%, Best AUC: 91.49%


100%|██████████| 93/93 [00:04<00:00, 20.99it/s]


Epoch: 034, Loss: 16.9388, Val AUC: 86.57%, Test AUC: 88.55%, Best AUC: 91.49%


100%|██████████| 93/93 [00:04<00:00, 20.51it/s]

Epoch: 035, Loss: 17.0319, Val AUC: 85.55%, Test AUC: 88.11%, Best AUC: 91.49%


In [80]:
# ═══════════════════════════════════════════
# CELL: Training loop (drop-in replacement)
# ═══════════════════════════════════════════

best     = 0
best_val = 0

optimizer = torch.optim.Adam(
    list(model.parameters()) + list(history.parameters()),
    lr=args.lr,
    weight_decay=1e-5       # small L2 reg helps generalization
)

# Cosine LR schedule: smoothly decays over training
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=args.epochs, eta_min=args.lr * 0.1)

measure = Measure("auc")

for epoch in range(1, 1 + args.epochs):
    loss    = train(train_loader, model, history, optimizer, device, data, args)
    val_auc = test(valid_loader,  model, history, measure,   device, data,args)
    test_auc = test(test_loader,  model, history, measure,   device, data,args)
    scheduler.step()

    if val_auc > best_val:
        best_val = val_auc
        best     = test_auc

    print(
        f"Epoch: {epoch:03d}, Loss: {loss:.4f}, "
        f"Val AUC: {val_auc:.2%}, Test AUC: {test_auc:.2%}, Best AUC: {best:.2%}"
    )

100%|██████████| 93/93 [00:04<00:00, 20.39it/s]


Epoch: 001, Loss: 5.8465, Val AUC: 82.68%, Test AUC: 84.18%, Best AUC: 84.18%


100%|██████████| 93/93 [00:04<00:00, 20.96it/s]


Epoch: 002, Loss: 1.3736, Val AUC: 63.06%, Test AUC: 69.03%, Best AUC: 84.18%


100%|██████████| 93/93 [00:04<00:00, 20.64it/s]


...

Epoch: 011, Loss: 0.4901, Val AUC: 70.65%, Test AUC: 77.20%, Best AUC: 84.18%


100%|██████████| 93/93 [00:04<00:00, 20.00it/s]


Epoch: 012, Loss: 0.7586, Val AUC: 55.67%, Test AUC: 71.58%, Best AUC: 84.18%


Loss=0.8345:  52%|█████▏    | 226/431 [00:18<00:16, 12.14it/s]


KeyboardInterrupt: 

In [83]:
"""
Prototypical Contrastive Learning for Temporal Anomaly Detection
================================================================
Completely different from history-based view contrastive learning.

CORE IDEA:
  Instead of comparing two views of the same node, we learn PROTOTYPES
  — cluster centers that represent "normal behavior patterns".
  
  A node is anomalous if its representation is FAR from ALL prototypes.
  
  During training:
    - Maintain K prototypes (learned cluster centers of normal behavior)
    - Pull normal node representations toward their nearest prototype
    - The loss has NO positive/negative pairs — just prototype alignment
    - Prototypes are updated via exponential moving average (stable)
  
  During testing:
    - Anomaly score = distance to nearest prototype
    - Normal nodes cluster tightly around prototypes
    - Anomalous nodes are outliers with high distance to all prototypes

WHY THIS IS BETTER:
  1. No collapse: prototypes are EMA-updated, not trained by gradients
     so there is no gradient conflict between push/pull forces
  2. No history module dependency: works purely from model representations  
  3. Naturally handles class imbalance: prototypes are biased toward
     normal nodes (99.86% of wikipedia) — anomalies don't corrupt them
  4. Anomaly score is geometrically meaningful: distance in embedding space
  5. Temporal component: prototypes are TIME-AWARE — each prototype has
     a time embedding so the model learns "normal at time T" vs
     "normal at time T+k" as different prototypes

REFERENCE IDEAS:
  - ProtoNCE (Li et al., 2021): prototypical contrastive learning
  - MoCo (He et al., 2020): momentum queue for stable negatives  
  - DSVDD (Ruff et al., 2018): deep one-class classification
  Adapted and combined for temporal graph anomaly detection.
"""

import torch
import torch.nn as nn
import torch.nn.functional as F
from tqdm import tqdm


# ═══════════════════════════════════════════════════════════════════════════════
# Temporal Prototype Bank
# K prototypes, each associated with a time slot.
# Prototypes updated via EMA — never touched by backprop directly.
# ═══════════════════════════════════════════════════════════════════════════════

class TemporalPrototypeBank(nn.Module):
    """
    Maintains K learnable prototype vectors representing normal behavior.
    
    Key design choices:
      - Prototypes initialized from Xavier uniform (spread out in space)
      - Updated via EMA of assigned node representations (momentum=0.99)
      - Time-aware: each prototype also stores a mean timestamp
        so similar-time nodes are compared to time-appropriate prototypes
      - A memory queue stores recent representations for stable assignment
    """
    def __init__(self, num_prototypes, dim, queue_size=4096, momentum=0.99):
        super().__init__()
        self.K        = num_prototypes
        self.dim      = dim
        self.momentum = momentum
        self.queue_size = queue_size

        # Prototype vectors — NOT nn.Parameter, updated via EMA
        prototypes = torch.empty(num_prototypes, dim)
        nn.init.xavier_uniform_(prototypes)
        self.register_buffer("prototypes", F.normalize(prototypes, dim=-1))

        # Prototype time centroids (mean timestamp of assigned nodes)
        self.register_buffer("proto_time", torch.zeros(num_prototypes))
        self.register_buffer("proto_count", torch.ones(num_prototypes))

        # Memory queue: circular buffer of recent node representations
        # Used for stable prototype assignment and negative sampling
        queue = torch.randn(queue_size, dim)
        self.register_buffer("queue", F.normalize(queue, dim=-1))
        self.register_buffer("queue_ptr", torch.zeros(1, dtype=torch.long))
        self.register_buffer("queue_time", torch.zeros(queue_size))

    @torch.no_grad()
    def enqueue(self, x, t):
        """Add batch of representations to the circular queue."""
        B      = x.size(0)
        ptr    = int(self.queue_ptr)
        # Wrap around if needed
        if ptr + B > self.queue_size:
            # Fill to end, then wrap
            end  = self.queue_size - ptr
            self.queue[ptr:]  = F.normalize(x[:end].detach(), dim=-1)
            self.queue_time[ptr:] = t[:end].float()
            remainder = B - end
            self.queue[:remainder]  = F.normalize(x[end:].detach(), dim=-1)
            self.queue_time[:remainder] = t[end:].float()
            self.queue_ptr[0] = remainder
        else:
            self.queue[ptr: ptr + B] = F.normalize(x.detach(), dim=-1)
            self.queue_time[ptr: ptr + B] = t.float()
            self.queue_ptr[0] = (ptr + B) % self.queue_size

    @torch.no_grad()
    def update_prototypes(self, x, assignments, t):
        """
        EMA update prototypes from current batch assignments.
        x:           (B, D)
        assignments: (B,) prototype index for each node
        t:           (B,) timestamps
        """
        x_norm = F.normalize(x.detach(), dim=-1)
        for k in range(self.K):
            mask = (assignments == k)
            if mask.sum() == 0:
                continue
            # EMA update prototype vector
            batch_mean = x_norm[mask].mean(0)
            self.prototypes[k] = (
                self.momentum       * self.prototypes[k]
                + (1 - self.momentum) * batch_mean
            )
            self.prototypes[k] = F.normalize(self.prototypes[k], dim=-1)
            # Track time centroid
            self.proto_time[k] = (
                self.momentum       * self.proto_time[k]
                + (1 - self.momentum) * t[mask].float().mean()
            )

    def assign(self, x, t=None, time_weight=0.1):
        """
        Assign each node to its nearest prototype.
        Optional: weight assignment by temporal proximity.
        Returns: (B,) prototype indices, (B,) distances
        """
        x_norm  = F.normalize(x, dim=-1)                              # (B, D)
        # Cosine similarity to all prototypes
        sim     = x_norm @ self.prototypes.T                          # (B, K)

        if t is not None:
            # Temporal proximity bonus: assign to time-appropriate prototype
            t_norm   = t.float().unsqueeze(1)                         # (B, 1)
            pt_norm  = self.proto_time.unsqueeze(0)                   # (1, K)
            t_diff   = (t_norm - pt_norm).abs()
            t_max    = t_diff.max().clamp(min=1.0)
            t_sim    = 1.0 - t_diff / t_max                           # (B, K) in [0,1]
            sim      = sim + time_weight * t_sim

        assignments = sim.argmax(dim=-1)                              # (B,)
        # Distance = 1 - max_similarity (cosine distance to nearest prototype)
        distances   = 1.0 - sim.max(dim=-1).values                    # (B,)
        return assignments, distances, sim


# ═══════════════════════════════════════════════════════════════════════════════
# Prototypical Contrastive Loss
# ═══════════════════════════════════════════════════════════════════════════════

def proto_contrastive_loss(x, proto_bank, t, tau=0.1, queue_negatives=True):
    """
    ProtoNCE loss: pull node representations toward their assigned prototype,
    push away from all other prototypes.
    
    Unlike pair-based InfoNCE:
      - Positive = nearest prototype (a stable cluster center, NOT another node)
      - Negatives = all other K-1 prototypes (fixed, no gradient conflict)
      - Optional: also push away from queue (recent representations of other nodes)
    
    This is STABLE because prototypes are EMA-updated buffers —
    gradients only flow through x, not through the "positive" target.
    """
    B = x.size(0)
    x_norm = F.normalize(x, dim=-1)                                   # (B, D)

    # Get prototype assignments
    with torch.no_grad():
        assignments, _, sim_all = proto_bank.assign(x, t)            # (B,), (B,), (B,K)

    # ProtoNCE: for each node, positive = assigned prototype
    # sim_all is already (B, K) — cosine sim to all K prototypes
    proto_logits = sim_all / tau                                       # (B, K)

    # Cross-entropy: push toward assigned prototype, away from others
    proto_loss = F.cross_entropy(proto_logits, assignments)

    # Optional: additionally push away from queue (recent other-node reprs)
    if queue_negatives and proto_bank.queue_size > 0:
        # Sample from queue
        queue = proto_bank.queue.detach()                              # (Q, D)
        # (B, D) x (D, Q) → (B, Q)
        queue_sim    = (x_norm @ queue.T) / tau
        # These are ALL negatives (no positive in queue)
        # Maximize distance from queue = minimize mean similarity
        queue_loss   = torch.logsumexp(queue_sim, dim=-1).mean()
        return proto_loss + 0.1 * queue_loss

    return proto_loss


def temporal_smoothness_loss(x, t, proto_bank, tau_time=0.2):
    """
    Temporal smoothness: nodes at similar timestamps should have
    similar distances to prototypes. This prevents the model from
    learning time-trivial representations.
    
    For pairs of nodes (i, j) with similar timestamps:
      their prototype assignment distributions should be similar.
    """
    B = x.size(0)
    if B < 4:
        return torch.tensor(0.0, device=x.device)

    x_norm = F.normalize(x, dim=-1)
    with torch.no_grad():
        _, _, sim_all = proto_bank.assign(x, t)                      # (B, K)

    # Soft assignment distribution per node
    soft_assign = F.softmax(sim_all / 0.5, dim=-1)                   # (B, K)

    # Time similarity between nodes in batch
    t_f    = t.float()
    t_diff = (t_f.unsqueeze(1) - t_f.unsqueeze(0)).abs()             # (B, B)
    t_max  = t_diff.max().clamp(min=1.0)
    t_sim  = torch.exp(-t_diff / (t_max * tau_time))                 # (B, B)

    # KL divergence between assignment distributions of time-similar pairs
    log_sa = soft_assign.log().unsqueeze(1).expand(B, B, -1)         # (B, B, K)
    sa_exp = soft_assign.unsqueeze(0).expand(B, B, -1)               # (B, B, K)
    kl     = (sa_exp * (sa_exp.log() - log_sa)).sum(-1)              # (B, B)

    # Weight KL by time similarity: time-close pairs should have low KL
    loss   = (t_sim * kl).sum() / (t_sim.sum() + 1e-6)
    return loss.clamp(max=5.0)                                        # prevent explosion


# ═══════════════════════════════════════════════════════════════════════════════
# Training Function
# ═══════════════════════════════════════════════════════════════════════════════

def train(loader, model, proto_bank, optimizer, device, data, args):
    """
    Training with prototypical contrastive learning.
    No history module needed — prototypes replace it entirely.
    """
    model.train()
    pbar       = tqdm(loader)
    total_loss = 0.0
    tau        = getattr(args, 'tau', 0.1)

    for batch in pbar:
        batch = batch.to(device)
        optimizer.zero_grad()

        # Forward
        out = model(batch)
        src = batch.src[: batch.batch_size]
        out = out[src]                                                 # (B, D)
        t   = batch.t[: batch.batch_size]                             # (B,)

        # ── Main loss: ProtoNCE ───────────────────────────────────────────
        loss = proto_contrastive_loss(out, proto_bank, t, tau=tau)

        # ── Temporal smoothness regularizer ───────────────────────────────
        # Weight 0.05: small enough not to dominate, enough to help structure
        t_smooth = temporal_smoothness_loss(out, t, proto_bank)
        loss = loss + 0.05 * t_smooth

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        # ── Update prototype bank (no gradient) ───────────────────────────
        with torch.no_grad():
            assignments, _, _ = proto_bank.assign(out.detach(), t)
            proto_bank.update_prototypes(out, assignments, t)
            proto_bank.enqueue(out, t)

        total_loss += loss.item()
        pbar.set_description(f"Loss={loss.item():.4f}")

    return total_loss / len(loader)


# ═══════════════════════════════════════════════════════════════════════════════
# Test Function + Anomaly Score
# ═══════════════════════════════════════════════════════════════════════════════

@torch.no_grad()
def test(loader, model, proto_bank, measure, device, data, args):
    """
    Anomaly score = distance to nearest prototype.
    
    Normal nodes cluster tightly around prototypes → low distance.
    Anomalous nodes are outliers → high distance to ALL prototypes.
    
    Additionally: anomalous nodes often have HIGH prototype assignment
    uncertainty (similar distance to multiple prototypes) — we capture
    this via the entropy of the soft assignment distribution.
    """
    preds, labels = [], []
    model.eval()
    eps = 1e-6

    for batch in tqdm(loader):
        torch.cuda.empty_cache()
        batch = batch.to(device)

        out   = model(batch)
        src   = batch.src[: batch.batch_size]
        out   = out[src]                                               # (B, D)
        label = batch.y[: batch.batch_size]
        t     = batch.t[: batch.batch_size]                           # (B,)

        # ── Prototype distance score ──────────────────────────────────────
        _, distances, sim_all = proto_bank.assign(out, t)             # (B,), (B,K)
        # distances = 1 - max_similarity ∈ [0, 2] but typically [0, 1]
        score_dist = distances.clamp(0, 1)                            # (B,)

        # ── Prototype assignment entropy ──────────────────────────────────
        # Normal nodes: confidently assigned to ONE prototype → low entropy
        # Anomalous nodes: confused across prototypes → high entropy
        soft   = F.softmax(sim_all / 0.5, dim=-1)                    # (B, K)
        entropy = -(soft * (soft + eps).log()).sum(-1)                # (B,)
        # Normalize entropy to [0, 1] by max possible entropy
        max_entropy = math.log(proto_bank.K)
        score_ent   = (entropy / max_entropy).clamp(0, 1)            # (B,)

        # ── Memory variance component (if model has it) ───────────────────
        score_var = torch.zeros(out.size(0), device=device)
        if hasattr(model, 'memory'):
            node_ids   = data.src[batch.input_id.cpu()].to(device)
            _, cur_var = model.memory.read(node_ids)                  # (B, D)
            # High variance = erratic node = more anomalous
            batch_var  = cur_var.mean(-1)
            z_score    = (batch_var - batch_var.mean()) / (batch_var.std() + eps)
            score_var  = z_score.relu().clamp(0, 3) / 3.0            # normalize to [0,1]

        # ── Combined anomaly score ────────────────────────────────────────
        # Distance is the primary signal, entropy is secondary
        pred = (0.6 * score_dist
              + 0.3 * score_ent
              + 0.1 * score_var).sigmoid()

        torch.cuda.empty_cache()
        preds.append(pred.detach().cpu())
        labels.append(label.detach().cpu())

    preds  = torch.cat(preds)
    preds[torch.isnan(preds)] = 0.0
    labels = torch.cat(labels)
    return measure(labels[labels != 2], preds[labels != 2])


# ═══════════════════════════════════════════════════════════════════════════════
# Notebook Integration
# ═══════════════════════════════════════════════════════════════════════════════


if True :
    
    import math  # needed for log in test()

    proto_bank = TemporalPrototypeBank(
        num_prototypes = 32,          # K: try 16, 32, 64
        dim            = args.hidden_channels,
        queue_size     = 4096,
        momentum       = 0.99
    ).to(device)

    optimizer = torch.optim.Adam(
        model.parameters(),           # NOTE: no history module needed
        lr=args.lr,
        weight_decay=1e-5
    )

    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=args.epochs, eta_min=args.lr * 0.1)

    best = 0
    best_val = 0

    for epoch in range(1, 1 + args.epochs):
        loss     = train(train_loader, model, proto_bank, optimizer, device, data, args)
        val_auc  = test(valid_loader,  model, proto_bank, measure,   device, data, args)
        test_auc = test(test_loader,   model, proto_bank, measure,   device, data, args)
        scheduler.step()

        if val_auc > best_val:
            best_val = val_auc
            best     = test_auc

        print(
            f"Epoch: {epoch:03d}, Loss: {loss:.4f}, "
            f"Val AUC: {val_auc:.2%}, Test AUC: {test_auc:.2%}, Best AUC: {best:.2%}"
        )



100%|██████████| 93/93 [00:04<00:00, 21.20it/s]


Epoch: 001, Loss: 4.3873, Val AUC: 46.48%, Test AUC: 57.77%, Best AUC: 57.77%


100%|██████████| 93/93 [00:04<00:00, 20.63it/s]


Epoch: 002, Loss: 4.5592, Val AUC: 60.65%, Test AUC: 57.08%, Best AUC: 57.08%


100%|██████████| 93/93 [00:04<00:00, 20.97it/s]


...

Epoch: 004, Loss: 3.9746, Val AUC: 44.26%, Test AUC: 55.99%, Best AUC: 57.08%


100%|██████████| 93/93 [00:04<00:00, 20.76it/s]


Epoch: 005, Loss: 3.8280, Val AUC: 63.22%, Test AUC: 50.83%, Best AUC: 50.83%


Loss=3.7168:  22%|██▏       | 94/431 [00:08<00:31, 10.63it/s]


KeyboardInterrupt: 

In [86]:
"""
Final Version — Back to What Worked, Surgically Improved
=========================================================
What reached 91.82%:
  - TrustAwareTemporalGNN (your original model)
  - History-based contrastive: stable_contrastive_loss with frozen buffer negatives
  - con='all': structural + temporal loss terms
  - Standard cosine anomaly score

What we keep:
  - Your original History module (it works, don't touch it)
  - Your original contrastive structure (his/stru/all modes)
  - Your original train/test loop structure

What we fix precisely:
  1. tau=0.02 is too small → use 0.1 (prevents loss explosion)
  2. Negatives from frozen buffer (history.get_history) — keeps it stable
  3. Anomaly score adds memory variance weighting (confidence)
  4. Gradient clipping on BOTH model and history parameters
  5. Nothing else changes
"""

import torch
import torch.nn as nn
import torch.nn.functional as F
from tqdm import tqdm


# ═══════════════════════════════════════════════════════════════════════════════
# The ONE loss function that is stable and works
# ═══════════════════════════════════════════════════════════════════════════════

def contrastive_loss(anchor, positive, tau=0.1):
    """
    Your original contrastive loss — this is what worked.
    Kept exactly as-is. tau=0.1 instead of 0.02.
    """
    anchor   = F.normalize(anchor,   dim=-1)
    positive = F.normalize(positive, dim=-1)
    sim      = (anchor * positive).sum(-1) / tau
    return -sim.mean()


def cosine_similarity(a, b):
    """Your original cosine similarity — kept exactly as-is."""
    a = F.normalize(a, dim=-1)
    b = F.normalize(b, dim=-1)
    return (a.unsqueeze(1) * b.unsqueeze(0)).sum(-1)


# ═══════════════════════════════════════════════════════════════════════════════
# Training — identical structure to your 91.82% run, tau fixed
# ═══════════════════════════════════════════════════════════════════════════════

def train(loader, model, history, optimizer, device, data, args):
    model.train()
    history.train()
    pbar       = tqdm(loader)
    total_loss = 0.0

    # KEY FIX: tau=0.1 not 0.02. At tau=0.02 the loss explodes to 17+
    # because exp(sim/0.02) overflows for any non-trivial similarity.
    tau = 0.1

    for batch in pbar:
        batch = batch.to(device)
        optimizer.zero_grad()

        out = model(batch)
        src = batch.src[: batch.batch_size]
        out = out[src]
        idx = batch.input_id.cpu()

        # Frozen buffer negatives — stable, no gradient conflict
        num_neg = batch.batch_size
        neg_mem = history.get_history(
            torch.randint(0, history.num_nodes, (num_neg,))
        ).detach()

        cur_mem, prev_mem = history(out, data.src[idx])

        if args.con == 'his':
            loss  = contrastive_loss(prev_mem, cur_mem, tau)
            loss += torch.exp(
                cosine_similarity(cur_mem, neg_mem)
            ).sum(dim=1).log().mean()

        elif args.con == 'stru':
            loss  = contrastive_loss(out, cur_mem, tau)
            loss += torch.exp(
                cosine_similarity(out, neg_mem)
            ).sum(dim=1).log().mean()

        elif args.con == 'all':
            loss  = args.alpha * contrastive_loss(out, cur_mem, tau)
            loss += contrastive_loss(prev_mem, cur_mem, tau)
            loss += torch.exp(
                cosine_similarity(cur_mem, neg_mem)
            ).sum(dim=1).log().mean()
            loss += args.alpha * torch.exp(
                cosine_similarity(out, neg_mem)
            ).sum(dim=1).log().mean()

        loss.backward()

        # Clip BOTH model and history — missing this caused instability
        torch.nn.utils.clip_grad_norm_(
            list(model.parameters()) + list(history.parameters()), 1.0
        )

        optimizer.step()
        total_loss += loss.item()
        pbar.set_description(f"Loss={loss.item():.4f}")

    return total_loss / len(loader)


# ═══════════════════════════════════════════════════════════════════════════════
# Test — your original structure + confidence weighting
# ═══════════════════════════════════════════════════════════════════════════════

@torch.no_grad()
def test(loader, model, history, measure, device, data, args):
    preds, labels = [], []
    model.eval()
    history.eval()
    eps = 1e-6

    for batch in tqdm(loader):
        torch.cuda.empty_cache()
        batch = batch.to(device)

        out   = model(batch)
        src   = batch.src[: batch.batch_size]
        out   = out[src]
        label = batch.y[: batch.batch_size]
        idx   = batch.input_id.cpu()

        cur_mem, prev_mem = history(out, data.src[idx], update=True)

        # Confidence weight from memory variance (only if model has it)
        if hasattr(model, 'memory'):
            node_ids = data.src[idx].to(device)
            _, var   = model.memory.read(node_ids)
            conf     = 1.0 / (1.0 + var.mean(-1).sqrt().clamp(min=eps))
        else:
            conf = torch.ones(out.size(0), device=device)

        if args.con == 'his':
            pred = conf * (1 - torch.diag(
                cosine_similarity(cur_mem, prev_mem)).view(-1))

        elif args.con == 'stru':
            pred = conf * (1 - torch.diag(
                cosine_similarity(out, cur_mem)).view(-1))

        elif args.con == 'all':
            p1   = torch.diag(cosine_similarity(out,     cur_mem)).view(-1)
            p2   = torch.diag(cosine_similarity(cur_mem, prev_mem)).view(-1)
            pred = conf * ((1 - p1) + (1 - p2))

        pred = pred.sigmoid()
        torch.cuda.empty_cache()
        preds.append(pred.detach().cpu())
        labels.append(label.detach().cpu())

    preds  = torch.cat(preds)
    preds[torch.isnan(preds)] = 0.0
    labels = torch.cat(labels)
    return measure(labels[labels != 2], preds[labels != 2])


# ═══════════════════════════════════════════════════════════════════════════════
# Training loop — use your BEST model (TrustAwareTemporalGNN, 91.82%)
# ═══════════════════════════════════════════════════════════════════════════════
if True :

    args.tau = 0.1   # override the 0.02 — this is the critical fix

    optimizer = torch.optim.Adam(
        list(model.parameters()) + list(history.parameters()),
        lr=args.lr,           # 5e-4
        weight_decay=1e-5
    )

    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=args.epochs, eta_min=args.lr * 0.1
    )

    best = 0
    best_val = 0

    for epoch in range(1, 1 + args.epochs):
        loss     = train(train_loader, model, history, optimizer, device, data, args)
        val_auc  = test(valid_loader,  model, history, measure,   device, data, args)
        test_auc = test(test_loader,   model, history, measure,   device, data, args)
        scheduler.step()

        if val_auc > best_val:
            best_val = val_auc
            best     = test_auc

        print(
            f"Epoch: {epoch:03d}, Loss: {loss:.4f}, "
            f"Val AUC: {val_auc:.2%}, Test AUC: {test_auc:.2%}, Best AUC: {best:.2%}"
        )



100%|██████████| 93/93 [00:04<00:00, 20.65it/s]


Epoch: 001, Loss: -6.3799, Val AUC: 77.97%, Test AUC: 80.55%, Best AUC: 80.55%


100%|██████████| 93/93 [00:04<00:00, 20.90it/s]


Epoch: 002, Loss: -7.0600, Val AUC: 83.98%, Test AUC: 84.31%, Best AUC: 84.31%


100%|██████████| 93/93 [00:04<00:00, 21.07it/s]


...

Epoch: 017, Loss: -7.8958, Val AUC: 86.64%, Test AUC: 88.36%, Best AUC: 87.52%


100%|██████████| 93/93 [00:04<00:00, 20.62it/s]


Epoch: 018, Loss: -7.8884, Val AUC: 83.76%, Test AUC: 88.47%, Best AUC: 87.52%


Loss=-7.9451:  55%|█████▌    | 238/431 [00:19<00:15, 12.21it/s]


KeyboardInterrupt: 

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import MessagePassing
from torch_geometric.utils import softmax


# ═══════════════════════════════════════════════════════════════════════
# Time2Vec
# ═══════════════════════════════════════════════════════════════════════

class Time2Vec(nn.Module):
    def __init__(self, out_dim):
        super().__init__()
        k = out_dim - 1
        self.w0 = nn.Parameter(torch.randn(1) * 0.01)
        self.b0 = nn.Parameter(torch.zeros(1))
        self.W  = nn.Parameter(torch.randn(k) * 0.01)
        self.B  = nn.Parameter(torch.zeros(k))

    def forward(self, t):
        t = t.float().unsqueeze(-1)
        linear = t * self.w0 + self.b0
        periodic = torch.sin(t * self.W + self.B)
        return torch.cat([linear, periodic], dim=-1)


class TimeEncoder(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.abs_t = Time2Vec(dim // 2)
        self.rel_t = Time2Vec(dim // 2)
        self.proj = nn.Sequential(
            nn.Linear(dim, dim),
            nn.GELU(),
            nn.Linear(dim, dim)
        )

    def forward(self, t_abs, t_rel):
        x = torch.cat([self.abs_t(t_abs), self.rel_t(t_rel)], dim=-1)
        return self.proj(x)


# ═══════════════════════════════════════════════════════════════════════
# Multi-Scale Event-Driven Memory
# ═══════════════════════════════════════════════════════════════════════

class MultiScaleRecurrentMemory(nn.Module):
    def __init__(self, num_nodes, dim, tau_fast=1.0, tau_slow=10.0, momentum=0.9):
        super().__init__()
        self.fast = nn.GRUCell(dim, dim)
        self.slow = nn.GRUCell(dim, dim)
        self.tau_fast = tau_fast
        self.tau_slow = tau_slow
        self.momentum = momentum

        self.register_buffer("h_fast", torch.zeros(num_nodes, dim))
        self.register_buffer("h_slow", torch.zeros(num_nodes, dim))
        self.register_buffer("var", torch.ones(num_nodes, dim) * 0.1)
        self.register_buffer("last_t", torch.zeros(num_nodes))

    def read(self, idx):
        h = 0.5 * (self.h_fast[idx] + self.h_slow[idx])
        return h, self.var[idx]

    def hidden(self, idx):
        return 0.5 * (self.h_fast[idx] + self.h_slow[idx])

    @torch.no_grad()
    def update(self, idx, msg, t):
        dt = (t - self.last_t[idx]).clamp(min=0).unsqueeze(-1)

        g_f = torch.exp(-dt / self.tau_fast)
        g_s = torch.exp(-dt / self.tau_slow)

        h_f = self.fast(msg, self.h_fast[idx])
        h_s = self.slow(msg, self.h_slow[idx])

        h = g_f * h_f + (1 - g_f) * self.h_fast[idx] \
          + g_s * h_s + (1 - g_s) * self.h_slow[idx]

        delta = h - self.hidden(idx)
        self.var[idx] = self.momentum * self.var[idx] + (1 - self.momentum) * delta.pow(2)

        self.h_fast[idx] = h_f
        self.h_slow[idx] = h_s
        self.last_t[idx] = t


# ═══════════════════════════════════════════════════════════════════════
# Δ-Neighborhood Evolution Bank
# ═══════════════════════════════════════════════════════════════════════

class DeltaEvolutionBank(nn.Module):
    def __init__(self, num_nodes, dim, window=6):
        super().__init__()
        self.window = window
        self.register_buffer("bank", torch.zeros(num_nodes, window, dim))
        self.register_buffer("prev", torch.zeros(num_nodes, dim))
        self.register_buffer("ptr", torch.zeros(num_nodes, dtype=torch.long))

    def read(self, idx):
        return self.bank[idx]

    @torch.no_grad()
    def write(self, idx, emb):
        for i, node in enumerate(idx):
            delta = emb[i] - self.prev[node]
            p = int(self.ptr[node] % self.window)
            self.bank[node, p] = delta
            self.ptr[node] += 1
            self.prev[node] = emb[i]


# ═══════════════════════════════════════════════════════════════════════
# Trust Module (Δ-Evolution Aware)
# ═══════════════════════════════════════════════════════════════════════

class TrustModule(nn.Module):
    def __init__(self, dim, heads=4):
        super().__init__()
        self.heads = heads
        self.dh = dim // heads
        self.scale = self.dh ** -0.5

        self.q = nn.Linear(dim, dim)
        self.k = nn.Linear(dim, dim)
        self.v = nn.Linear(dim, dim)

        self.log_decay = nn.Parameter(torch.tensor(-2.0))

        self.out = nn.Sequential(
            nn.Linear(dim, dim // 2),
            nn.GELU(),
            nn.Linear(dim // 2, 1)
        )

    def forward(self, h_i, evo_j, var_i, var_j):
        E, W, D = evo_j.shape
        Q = self.q(h_i).view(E, self.heads, self.dh)
        K = self.k(evo_j.view(E * W, D)).view(E, W, self.heads, self.dh)
        V = self.v(evo_j.view(E * W, D)).view(E, W, self.heads, self.dh)

        Q = Q.unsqueeze(2)
        K = K.permute(0, 2, 1, 3)
        scores = (Q * K).sum(-1) * self.scale

        decay = torch.exp(self.log_decay.exp() * torch.arange(W, device=h_i.device))
        decay = decay / decay.sum()
        scores = scores + decay.view(1, 1, W)

        attn = F.softmax(scores, dim=-1)
        V = V.permute(0, 2, 1, 3)
        pooled = (attn.unsqueeze(-1) * V).sum(2).reshape(E, D)

        trust = self.out(pooled).squeeze(-1)

        conf_i = 1 / (1 + var_i.mean(-1).sqrt())
        conf_j = 1 / (1 + var_j.mean(-1).sqrt())
        conf = (conf_i * conf_j).sqrt()

        return torch.sigmoid(trust + conf.log().clamp(min=-5))


# ═══════════════════════════════════════════════════════════════════════
# Temporal Trust Attention Layer
# ═══════════════════════════════════════════════════════════════════════

class TemporalTrustAttention(MessagePassing):
    def __init__(self, dim, heads=8, dropout=0.1):
        super().__init__(aggr="add", node_dim=0)  
        self.h = heads
        self.dh = dim // heads
        self.scale = self.dh ** -0.5

        self.q = nn.Linear(dim, dim)
        self.k = nn.Linear(dim, dim)
        self.v = nn.Linear(dim, dim)

        self.edge = nn.Linear(dim, dim)
        self.time = nn.Linear(dim, dim)

        self.trust = TrustModule(dim)
        self.drop = nn.Dropout(dropout)
        self.out = nn.Linear(dim, dim)
        self.norm = nn.LayerNorm(dim)

    def forward(self, x, edge_index, edge_emb, time_emb, memory, evo):
        Q = self.q(x).view(-1, self.h, self.dh)
        K = self.k(x).view(-1, self.h, self.dh)
        V = self.v(x).view(-1, self.h, self.dh)

        mem_h, mem_var = memory.read(torch.arange(x.size(0), device=x.device))
        evo_vals = evo.read(torch.arange(x.size(0), device=x.device))  # (N,W,D)

        out = self.propagate(
            edge_index,
            Q=Q, K=K, V=V,
            mem_h=mem_h,
            mem_var=mem_var,
            evo=evo_vals,
            edge=self.edge(edge_emb).view(-1, self.h, self.dh),
            time=self.time(time_emb).view(-1, self.h, self.dh)
        )

        out = out.view(x.size(0), -1)
        return self.norm(x + self.out(out))

    def message(self, Q_i, K_j, V_j,
                mem_h_i, mem_var_i, mem_var_j,
                evo_j, edge, time, index):

        K_j = K_j + edge + time
        attn = (Q_i * K_j).sum(-1) * self.scale

        trust = self.trust(mem_h_i, evo_j, mem_var_i, mem_var_j)
        attn = softmax(attn * trust.unsqueeze(-1), index)
        attn = self.drop(attn)

        return V_j * attn.unsqueeze(-1)



# ═══════════════════════════════════════════════════════════════════════
# Full Model
# ═══════════════════════════════════════════════════════════════════════

class EnhancedTemporalGNN(nn.Module):
    def __init__(self, num_nodes, in_dim, edge_dim, hidden_dim,
                 num_layers=2, heads=8, window=6):
        super().__init__()

        self.x = nn.Linear(in_dim, hidden_dim)
        self.e = nn.Linear(edge_dim, hidden_dim)
        self.t = TimeEncoder(hidden_dim)

        self.memory = MultiScaleRecurrentMemory(num_nodes, hidden_dim)
        self.evo = DeltaEvolutionBank(num_nodes, hidden_dim, window)

        self.layers = nn.ModuleList([
            TemporalTrustAttention(hidden_dim, heads)
            for _ in range(num_layers)
        ])

        self.norm = nn.LayerNorm(hidden_dim)

    def forward(self, batch):
        x = self.x(batch.x)
        edge_emb = self.e(batch.msg)

        src, dst = batch.edge_index
        last_t = torch.zeros(x.size(0), device=x.device)
        last_t.scatter_reduce_(0, src, batch.t.float(), reduce="amax", include_self=True)
        t_rel = batch.t - last_t[src]

        time_emb = self.t(batch.t, t_rel)

        for layer in self.layers:
            x = layer(x, batch.edge_index, edge_emb, time_emb,
                      self.memory, self.evo)

        x = self.norm(x)

        self.memory.update(dst, x[dst], batch.t)
        self.evo.write(dst.unique(), x[dst.unique()])

        return x

In [13]:
"""
Drop-in replacement for your notebook.
Keeps: data, loaders, History, args, measure
Replaces: model, train(), test()

Just run these cells in order after your existing setup cells.
"""

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import MessagePassing
from torch_geometric.utils import softmax
from tqdm import tqdm


# ── Paste the full enhanced_gnn_v2.py here, or import it ──────────────────────
# from enhanced_gnn_v2 import (EnhancedTemporalGNN, train, test)
# ──────────────────────────────────────────────────────────────────────────────
# If running directly in notebook, paste the class definitions above this cell.


# ═══════════════════════════════════════════
# CELL: Build model + history (unchanged)
# ═══════════════════════════════════════════

try:
    del model
except NameError:
    pass

from models.model_5 import EnhancedTemporalGNN

model = EnhancedTemporalGNN(
    num_nodes  = data.num_nodes,
    in_dim     = data.x.size(1),          # 1 for wikipedia
    edge_dim   = data.msg.size(1),         # 172 for wikipedia
    hidden_dim = args.hidden_channels,     # 64
    num_layers = args.nums_layers,         # 2
    heads      = args.heads,               # 8
    dropout    = args.dropout,             # 0.3
    window     = 6,
    memory_momentum = 0.9,
).to(device)

# History is UNCHANGED — reuse your existing History class
try:
    del history
except NameError:
    pass

from mhiscl.history import History

history = History(
    data.num_nodes,
    args.num_timeslots,
    args.hidden_channels,
    device=device,
    history_retrieve=args.history_retrieve,
    recurrent=args.recurrent,
).to(device)

print(model)
print(history)
print(f"Model params: {sum(p.numel() for p in model.parameters()):,}")




EnhancedTemporalGNN(
  (input_proj): Linear(in_features=1, out_features=64, bias=True)
  (edge_enc): Linear(in_features=172, out_features=64, bias=True)
  (time_enc): TimeEncoder(
    (t2v): Time2Vec()
    (rel_t2v): Time2Vec()
    (proj): Sequential(
      (0): Linear(in_features=64, out_features=64, bias=True)
      (1): GELU(approximate='none')
      (2): Linear(in_features=64, out_features=64, bias=True)
    )
  )
  (memory): RecurrentMemory(
    (gru): GRUCell(64, 64)
  )
  (evo_bank): EvolutionBank()
  (layers): ModuleList(
    (0-1): 2 x TemporalTrustAttention()
  )
  (final_norm): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
)
History(
  (recurrent_network): GRUCell(64, 64)
  (lin): Sequential(
    (0): Linear(-1, 64, bias=True)
    (1): ELU(alpha=1.0)
    (2): Linear(64, 64, bias=True)
  )
)
Model params: 149,062


In [14]:
## Training Function 

def train(loader, model, history, optimizer, device, data, args):
    model.train()
    history.train()
    pbar = tqdm(loader)
    total_loss = 0
    for batch in pbar:
        batch = batch.to(device)
        optimizer.zero_grad()
        out = model(batch)
        src = batch.src[: batch.batch_size]
        out = out[src]
        idx = batch.input_id.cpu()
        num_neg = batch.batch_size
        neg_mem = history.get_history(torch.randint(
            0, history.num_nodes, size=(num_neg,)))
        raw_msg = torch.zeros(idx.size(0),1).to(device) 
        cur_mem, prev_mem = history(out, data.src[idx]) 

        if args.con=='his':
            loss = contrastive_loss(prev_mem, cur_mem, args.tau) 
            loss += torch.exp(cosine_similarity(cur_mem, neg_mem) 
                          ).sum(dim=1).log().mean()
        elif args.con=='stru':
            loss = contrastive_loss(out, cur_mem, args.tau) 
            loss += torch.exp(cosine_similarity(out, neg_mem)
                            ).sum(dim=1).log().mean()
        elif args.con=='all':
            loss = args.alpha*contrastive_loss(out, cur_mem, args.tau) + \
                contrastive_loss(prev_mem, cur_mem, args.tau)
            loss += torch.exp(cosine_similarity(cur_mem, neg_mem) 
                            ).sum(dim=1).log().mean()
            loss += args.alpha*torch.exp(cosine_similarity(out, neg_mem) 
                            ).sum(dim=1).log().mean()


        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        pbar.set_description(f"Train Loss = {loss.item():.4f}")

    return total_loss / len(loader)


## Test function

@torch.no_grad()
def test(loader, model, history, measure, device, data):
    preds = []
    labels = []
    model.eval()
    history.eval()
    for batch in tqdm(loader):
        torch.cuda.empty_cache()
        batch = batch.to(device)
        out = model(batch)
        src = batch.src[: batch.batch_size]
        out = out[src]
        label = batch.y[: batch.batch_size]
        idx = batch.input_id.cpu()
        raw_msg = torch.zeros(idx.size(0),1).to(device) 
        cur_mem, prev_mem = history(out, data.src[idx], update=True)
        if args.con=='his':
            pred = 1-torch.diag(cosine_similarity(cur_mem, prev_mem)).view(-1)
        elif args.con=='stru': 
            pred = 1-torch.diag(cosine_similarity(out, cur_mem)).view(-1)
        elif args.con=='all':
            p1 = torch.diag(cosine_similarity(out, cur_mem)).view(-1)
            p2 = torch.diag(cosine_similarity(cur_mem, prev_mem)).view(-1)
            pred = (1 - p1) + (1 - p2) 
            
        pred = pred.sigmoid() 
        torch.cuda.empty_cache()
        preds.append(pred.detach().cpu())
        labels.append(label.detach().cpu())
    preds = torch.cat(preds)
    preds[torch.isnan(preds)] = 0.0 
    labels = torch.cat(labels)
    return measure(labels[labels!=2], preds[labels!=2]) 




In [15]:
# ═══════════════════════════════════════════
# CELL: Training loop (drop-in replacement)
# ═══════════════════════════════════════════

best     = 0
best_val = 0

optimizer = torch.optim.Adam(
    list(model.parameters()) + list(history.parameters()),
    lr=args.lr,
    weight_decay=1e-5       # small L2 reg helps generalization
)

# Cosine LR schedule: smoothly decays over training
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=args.epochs, eta_min=args.lr * 0.1)

measure = Measure("auc")

for epoch in range(1, 1 + args.epochs):
    loss    = train(train_loader, model, history, optimizer, device, data, args)
    val_auc = test(valid_loader,  model, history, measure,   device, data)
    test_auc = test(test_loader,  model, history, measure,   device, data)
    scheduler.step()

    if val_auc > best_val:
        best_val = val_auc
        best     = test_auc

    print(
        f"Epoch: {epoch:03d}, Loss: {loss:.4f}, "
        f"Val AUC: {val_auc:.2%}, Test AUC: {test_auc:.2%}, Best AUC: {best:.2%}"
    )

100%|██████████| 93/93 [00:04<00:00, 21.53it/s]


Epoch: 001, Loss: 20.9926, Val AUC: 72.60%, Test AUC: 70.74%, Best AUC: 70.74%


100%|██████████| 93/93 [00:04<00:00, 21.48it/s]


Epoch: 002, Loss: 18.9934, Val AUC: 86.40%, Test AUC: 89.88%, Best AUC: 89.88%


100%|██████████| 93/93 [00:04<00:00, 21.38it/s]


...

Epoch: 025, Loss: 16.9654, Val AUC: 88.86%, Test AUC: 89.72%, Best AUC: 90.51%


100%|██████████| 93/93 [00:04<00:00, 21.39it/s]


Epoch: 026, Loss: 16.9805, Val AUC: 87.92%, Test AUC: 88.83%, Best AUC: 90.51%


Train Loss = 17.0075:   1%|▏         | 6/431 [00:00<01:03,  6.72it/s]


KeyboardInterrupt: 

In [17]:
"""
Neighborhood-Aware Temporal Graph Attention Network  (NATGAT)
=============================================================
Representation-learning encoder only.

Returns a single unified node embedding  z ∈ ℝ^hidden_dim  that is
jointly aware of trust-filtered neighborhood structure and the node's
own temporal evolution — feed  z  directly into your contrastive objective.

Design goals
------------
1. TEMPORAL MODELING via K-step recurrent view buffer
   Each node keeps a rolling window of its K most recent embedding
   snapshots together with their timestamps.  DeltaTimeRNN processes
   the full K-step sequence through a GRU whose input gate is explicitly
   modulated by the inter-view time gap Δt, so the model natively
   captures high-frequency bursts (small Δt) and periodic rhythms
   (recurring Δt patterns).  FreqAwareEmbed encodes Δt with a learnable
   log-spaced sinusoidal bank (both cos and sin) that can adapt to the
   dataset's actual temporal scales.

2. TRUST-GATED AGGREGATION via three-signal TrustScorer
   For each directed edge i←j a scalar trust τ(i,j) ∈ (0,1) is computed
   from:
     (A) Attribute alignment  — cosine similarity of current embeddings
     (B) Activity divergence  — cross-attention between i's and j's
                                 K-step hidden-state sequences
     (C) Edge plausibility    — MLP over the edge attribute embedding
   Trust is applied BEFORE softmax so suspicious neighbors are
   down-weighted both in their share of the attention distribution
   and in the information they carry.

3. EDGE ATTRIBUTES as first-class signals
   Edge features enrich both attention keys (routing) and values
   (content), ensuring edge-level anomalies affect both where the
   model looks and what it receives.

Components
----------
FreqAwareEmbed   — Δt → learnable sinusoidal embedding
ViewBuffer       — rolling K-snapshot circular buffer per node
DeltaTimeRNN     — Δt-gated GRU over K views → temporal summary
TrustScorer      — per-edge trust from attribute + activity + edge
NATGATConv       — trust-gated multi-head message-passing layer
NATGAT           — full encoder; returns unified z  (N, hidden_dim)
"""

import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import MessagePassing
from torch_geometric.utils import softmax


# ═══════════════════════════════════════════════════════════════════════════════
# 1.  FreqAwareEmbed  —  scalar Δt → learnable multi-frequency embedding
# ═══════════════════════════════════════════════════════════════════════════════

class FreqAwareEmbed(nn.Module):
    """
    Δt  →  [linear,  cos(w₁Δt+φ₁), sin(w₁Δt+φ₁), …,
                     cos(wₙΔt+φₙ), sin(wₙΔt+φₙ)]  →  MLP  →  ℝᵈ

    Frequencies w are initialised on a log-scale spanning five decades
    (0.01 … 100 rad/unit) and are *learned*, so the model discovers the
    dataset's characteristic temporal scales automatically.
    Both cos and sin are used so any phase can be represented without
    cancellation artefacts.
    """

    def __init__(self, out_dim: int, num_freqs: int = None):
        super().__init__()
        if num_freqs is None:
            num_freqs = (out_dim - 1) // 2          # 1 linear + 2·F sinusoidal

        # Base log-spaced frequencies (fixed initialisation, then learned)
        base = torch.logspace(-2, 2, num_freqs)     # (F,)
        self.register_buffer("base_freq", base)
        self.log_freq_scale = nn.Parameter(torch.zeros(num_freqs))  # additive log-scale
        self.phase          = nn.Parameter(torch.zeros(num_freqs))  # phase offset

        raw_dim = 1 + 2 * num_freqs
        self.proj = nn.Sequential(
            nn.Linear(raw_dim, out_dim),
            nn.GELU(),
            nn.Linear(out_dim, out_dim),
        )
        self.num_freqs = num_freqs
        self.out_dim   = out_dim

    def forward(self, dt: torch.Tensor) -> torch.Tensor:
        """dt : (...,)  →  (..., out_dim)"""
        shape   = dt.shape
        dt_flat = dt.float().reshape(-1, 1)                      # (N, 1)

        freqs  = self.base_freq * torch.exp(self.log_freq_scale) # (F,)
        angles = dt_flat * freqs + self.phase                    # (N, F)
        raw    = torch.cat([dt_flat,
                            torch.cos(angles),
                            torch.sin(angles)], dim=-1)          # (N, 1+2F)
        return self.proj(raw).reshape(*shape, self.out_dim)


# ═══════════════════════════════════════════════════════════════════════════════
# 2.  ViewBuffer  —  rolling K-snapshot circular buffer per node
# ═══════════════════════════════════════════════════════════════════════════════

class ViewBuffer(nn.Module):
    """
    Stores the K most recent (embedding, timestamp) snapshots for every node.
    Snapshots are written after each forward pass; reads return them ordered
    oldest → newest together with inter-view time deltas Δt.

    Buffers
    -------
    emb  : (num_nodes, K, dim)
    time : (num_nodes, K)
    ptr  : (num_nodes,)   circular write pointer
    """

    def __init__(self, num_nodes: int, dim: int, K: int = 8):
        super().__init__()
        self.K   = K
        self.dim = dim
        self.register_buffer("emb",  torch.zeros(num_nodes, K, dim))
        self.register_buffer("time", torch.zeros(num_nodes, K))
        self.register_buffer("ptr",  torch.zeros(num_nodes, dtype=torch.long))

    def read(self, idx: torch.Tensor):
        """
        idx : (B,)
        Returns
        -------
        views : (B, K, dim)   oldest-first
        times : (B, K)        absolute timestamps
        dts   : (B, K)        Δt[k] = time[k] − time[k−1], 0 at k=0
        """
        B     = idx.size(0)
        views = self.emb[idx]   # (B, K, D)
        times = self.time[idx]  # (B, K)

        # Reorder so slot 0 = oldest
        rolls = (-self.ptr[idx]) % self.K
        views = torch.stack([torch.roll(views[b], int(rolls[b]), dims=0)
                             for b in range(B)])
        times = torch.stack([torch.roll(times[b], int(rolls[b]), dims=0)
                             for b in range(B)])

        dts = torch.zeros_like(times)
        dts[:, 1:] = (times[:, 1:] - times[:, :-1]).clamp(min=0)
        return views, times, dts

    @torch.no_grad()
    def update(self, idx: torch.Tensor, emb: torch.Tensor, t: torch.Tensor):
        """
        idx : (B,)  |  emb : (B, dim)  |  t : (B,)
        Write new snapshot into the circular buffer.
        """
        for b in range(idx.size(0)):
            p = int(self.ptr[idx[b]]) % self.K
            self.emb[idx[b],  p] = emb[b].detach()
            self.time[idx[b], p] = t[b].detach()
            self.ptr[idx[b]]    += 1


# ═══════════════════════════════════════════════════════════════════════════════
# 3.  DeltaTimeRNN  —  Δt-gated GRU over K historical views
# ═══════════════════════════════════════════════════════════════════════════════

class DeltaTimeRNN(nn.Module):
    """
    Processes a node's K-step view sequence through a GRU where the input
    is explicitly modulated by the inter-view time gap Δt.

    Intuition
    ---------
    Large Δt  → node may have drifted; gate opens wide to absorb the new snapshot.
    Small Δt  → burst activity; gate is more conservative (consecutive observations
                are correlated, so marginal information is lower).

    The gate modulation is *learned*: Δt is encoded by FreqAwareEmbed and
    projected to a multiplicative mask over the snapshot embedding before the GRU
    sees it.

    Outputs
    -------
    h_final : (B, dim)    — temporal summary of the node's trajectory
    h_seq   : (B, K, dim) — all GRU hidden states (used by TrustScorer)
    """

    def __init__(self, dim: int, K: int = 8, num_freqs: int = None):
        super().__init__()
        self.dim = dim
        self.K   = K

        self.dt_embed    = FreqAwareEmbed(dim, num_freqs=num_freqs)
        self.dt_gate_proj = nn.Sequential(
            nn.Linear(dim, dim),
            nn.Sigmoid(),
        )

        # Input = gated snapshot ‖ Δt embedding → 2·dim
        self.gru      = nn.GRU(input_size=dim * 2, hidden_size=dim, batch_first=True)
        self.out_norm = nn.LayerNorm(dim)

    def forward(self, views: torch.Tensor, dts: torch.Tensor):
        """
        views : (B, K, dim)
        dts   : (B, K)        inter-view Δt (0 at position 0)
        """
        dt_emb  = self.dt_embed(dts)             # (B, K, dim)
        dt_gate = self.dt_gate_proj(dt_emb)      # (B, K, dim) ∈ (0,1)

        inp      = torch.cat([views * dt_gate, dt_emb], dim=-1)  # (B, K, 2·dim)
        h_seq, _ = self.gru(inp)                                  # (B, K, dim)
        h_final  = self.out_norm(h_seq[:, -1, :])                 # (B, dim)
        return h_final, h_seq


# ═══════════════════════════════════════════════════════════════════════════════
# 4.  TrustScorer  —  per-edge trust from attribute + activity + edge signal
# ═══════════════════════════════════════════════════════════════════════════════

class TrustScorer(nn.Module):
    """
    Computes τ(i→j) ∈ (0,1) for each directed edge from three signals:

    (A) Attribute alignment
        cosine_similarity(xᵢ, xⱼ) mapped to (0,1).
        Structurally similar nodes should trust each other more.

    (B) Activity alignment
        Cross-attention diagonal of i's GRU sequence vs j's GRU sequence.
        High diagonal alignment ≈ compatible temporal histories.
        Anomalous nodes often have alien behavioral patterns.

    (C) Edge plausibility
        MLP(edge_emb) → (0,1).
        Unusual edge types / magnitudes reduce trust independently of node
        similarity.

    A small MLP combines the three scalars into the final trust score.
    """

    def __init__(self, dim: int, K: int = 8, num_heads: int = 4):
        super().__init__()
        assert dim % num_heads == 0
        self.num_heads = num_heads
        self.head_dim  = dim // num_heads
        self.scale     = self.head_dim ** -0.5

        # (B) cross-attention projections
        self.q_proj = nn.Linear(dim, dim)
        self.k_proj = nn.Linear(dim, dim)

        # (C) edge plausibility
        self.edge_plaus = nn.Sequential(
            nn.Linear(dim, dim // 2),
            nn.GELU(),
            nn.Linear(dim // 2, 1),
            nn.Sigmoid(),
        )

        # Final combination
        self.out_proj = nn.Sequential(
            nn.Linear(3, 16),
            nn.GELU(),
            nn.Linear(16, 1),
        )

    def forward(self,
                x_i:      torch.Tensor,   # (E, dim)
                x_j:      torch.Tensor,   # (E, dim)
                h_seq_i:  torch.Tensor,   # (E, K, dim)
                h_seq_j:  torch.Tensor,   # (E, K, dim)
                edge_emb: torch.Tensor,   # (E, dim)
                ) -> torch.Tensor:        # (E,) ∈ (0,1)

        # ── (A) Attribute alignment ──────────────────────────────────────────
        attr_score = (F.cosine_similarity(x_i, x_j, dim=-1) + 1.0) / 2.0   # (E,)

        # ── (B) Activity alignment via cross-attention diagonal ──────────────
        E, K, D = h_seq_i.shape
        nh, hd  = self.num_heads, self.head_dim

        Q  = self.q_proj(h_seq_i.reshape(E * K, D)).view(E, K, nh, hd)
        Kv = self.k_proj(h_seq_j.reshape(E * K, D)).view(E, K, nh, hd)

        Q_t  = Q.permute(0, 2, 1, 3)    # (E, nh, K, hd)
        Kv_t = Kv.permute(0, 2, 3, 1)   # (E, nh, hd, K)
        attn = F.softmax(
            torch.matmul(Q_t, Kv_t) * self.scale, dim=-1)         # (E, nh, K, K)

        # Mean diagonal = fraction of attention that stays "on-step" → alignment
        activity_score = torch.diagonal(attn, dim1=-2, dim2=-1).mean(dim=(-1, -2))  # (E,)

        # ── (C) Edge plausibility ─────────────────────────────────────────────
        edge_score = self.edge_plaus(edge_emb).squeeze(-1)         # (E,)

        # ── Combine ───────────────────────────────────────────────────────────
        signals = torch.stack([attr_score, activity_score, edge_score], dim=-1)  # (E,3)
        return torch.sigmoid(self.out_proj(signals).squeeze(-1))                 # (E,)


# ═══════════════════════════════════════════════════════════════════════════════
# 5.  NATGATConv  —  trust-gated message-passing layer
# ═══════════════════════════════════════════════════════════════════════════════

class NATGATConv(MessagePassing):
    """
    Single trust-gated attention layer.

    For target node i aggregating from neighbors j:

        msg(j→i) = (V_j + ev_j) · τ(i,j) · softmax_α(i,j)

    Trust τ multiplies attention BEFORE softmax so anomalous neighbors are
    reduced both in their attention weight and relative share.

    Edge attributes contribute to:
      - Keys  : via ek_j → routing (which neighbors are attended to)
      - Values: via ev_j → content (what information is carried)
    """

    def __init__(self, dim: int, heads: int = 8, K: int = 8,
                 dropout: float = 0.1, num_freqs: int = None):
        super().__init__(aggr="add", node_dim=0)
        assert dim % heads == 0
        self.dim      = dim
        self.heads    = heads
        self.head_dim = dim // heads
        self.scale    = self.head_dim ** -0.5

        self.q_proj   = nn.Linear(dim, dim)
        self.k_proj   = nn.Linear(dim, dim)
        self.v_proj   = nn.Linear(dim, dim)
        self.o_proj   = nn.Linear(dim, dim)

        self.edge_k_proj = nn.Linear(dim, dim)   # edge → key bias
        self.edge_v_proj = nn.Linear(dim, dim)   # edge → value bias
        self.time_k_proj = nn.Linear(dim, dim)   # time enc → key bias

        self.trust_scorer = TrustScorer(dim, K=K, num_heads=max(1, heads // 2))

        self.dropout = nn.Dropout(dropout)
        self.norm    = nn.LayerNorm(dim)

        # Simple residual after aggregation — temporal fusion is done once at
        # the model level after all layers, not per-layer
        self.gate = nn.Linear(dim, 1)

        self._trust_cache: torch.Tensor = None

    def split(self, x):
        return x.view(-1, self.heads, self.head_dim)

    def forward(self, x, edge_index, edge_emb, time_emb, h_seq):
        """
        x          : (N, dim)
        edge_index : (2, E)
        edge_emb   : (E, dim)   edge attribute embedding
        time_emb   : (E, dim)   timestamp encoding
        h_seq      : (N, K, dim) DeltaTimeRNN sequence (for TrustScorer)
        """
        N   = x.size(0)
        src = edge_index[0]
        dst = edge_index[1]

        Q      = self.split(self.q_proj(x))       # (N, heads, hd)
        K_base = self.split(self.k_proj(x))
        V_base = self.split(self.v_proj(x))

        ek = self.split(self.edge_k_proj(edge_emb))   # (E, heads, hd)
        ev = self.split(self.edge_v_proj(edge_emb))
        tk = self.split(self.time_k_proj(time_emb))

        # Per-edge trust — computed from node attributes, K-step activity
        # histories, and edge features; modulates aggregation before softmax
        trust = self.trust_scorer(
            x_i      = x[src],
            x_j      = x[dst],
            h_seq_i  = h_seq[src],
            h_seq_j  = h_seq[dst],
            edge_emb = edge_emb,
        )                                              # (E,)
        self._trust_cache = trust.detach()

        out = self.propagate(
            edge_index,
            Q=Q, K_base=K_base, V_base=V_base,
            ek=ek, ev=ev, tk=tk,
            trust=trust,
            size=(N, N),
        )                                              # (N, heads, hd)

        out = self.o_proj(out.view(N, self.dim))

        # Gated residual — keeps the node's own signal stable
        g = torch.sigmoid(self.gate(x))
        return self.norm(g * x + (1 - g) * out)

    def message(self, Q_i, K_base_j, V_base_j,
                ek, ev, tk, trust, index):
        # Enrich keys with edge type and temporal bias
        K_j = K_base_j + ek + tk                          # (E, heads, hd)
        # Enrich values with edge content signal
        V_j = V_base_j + ev                               # (E, heads, hd)

        # Attention score, trust-gated before softmax
        attn = (Q_i * K_j).sum(-1) * self.scale           # (E, heads)
        attn = attn * trust.unsqueeze(-1)                  # trust modulation
        attn = softmax(attn, index)                        # (E, heads)
        attn = self.dropout(attn)

        return V_j * attn.unsqueeze(-1)                    # (E, heads, hd)


# ═══════════════════════════════════════════════════════════════════════════════
# 6.  NATGAT  —  full representation-learning encoder
# ═══════════════════════════════════════════════════════════════════════════════

class NATGAT(nn.Module):
    """
    Neighborhood-Aware Temporal Graph Attention Network — representation encoder.

    Produces a single unified node embedding  z ∈ ℝ^hidden_dim  that jointly encodes:

      • Trust-filtered neighborhood structure  (trust-gated GNN layers)
        Per-edge trust is computed from attribute alignment, K-step activity
        history divergence, and edge plausibility — and multiplies attention
        weights BEFORE softmax so anomalous neighbors are suppressed in both
        routing and message content.

      • Node temporal evolution  (DeltaTimeRNN over K historical views)
        The node's K most recent snapshots are processed with explicit Δt
        gating so the model distinguishes bursts, drifts and periodic patterns.

    Both signals are fused at the final readout via a learned cross-gate.
    Feed  z  directly into your contrastive / self-supervised objective.

    Parameters
    ----------
    num_nodes  : total nodes in the graph
    in_dim     : raw node feature size
    edge_dim   : edge attribute size
    hidden_dim : internal embedding size  (must be divisible by heads)
    num_layers : number of NATGATConv layers
    heads      : attention heads per layer
    K          : history window — number of past views kept per node
    dropout    : attention dropout
    num_freqs  : sinusoidal frequency count (None = auto from hidden_dim)
    """

    def __init__(
        self,
        num_nodes:  int,
        in_dim:     int,
        edge_dim:   int,
        hidden_dim: int,
        num_layers: int   = 2,
        heads:      int   = 8,
        K:          int   = 8,
        dropout:    float = 0.1,
        num_freqs:  int   = None,
    ):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.num_nodes  = num_nodes
        self.K          = K

        # Input projections
        self.node_proj = nn.Linear(in_dim,   hidden_dim)
        self.edge_proj = nn.Linear(edge_dim, hidden_dim)

        # Timestamp encoding (absolute + relative)
        self.abs_t_embed = FreqAwareEmbed(hidden_dim // 2, num_freqs=num_freqs)
        self.rel_t_embed = FreqAwareEmbed(hidden_dim // 2, num_freqs=num_freqs)
        self.time_proj   = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, hidden_dim),
        )

        # Per-node K-step history
        self.view_buffer = ViewBuffer(num_nodes, hidden_dim, K=K)

        # Recurrent temporal summarizer
        self.delta_rnn = DeltaTimeRNN(hidden_dim, K=K, num_freqs=num_freqs)

        # Message-passing stack
        self.layers = nn.ModuleList([
            NATGATConv(dim=hidden_dim, heads=heads, K=K,
                       dropout=dropout, num_freqs=num_freqs)
            for _ in range(num_layers)
        ])

        # ── Final readout: fuse structural (GNN) + temporal (DeltaTimeRNN) views
        # into a single unified representation that is simultaneously aware of:
        #   • trust-filtered neighborhood structure  (from the GNN layers)
        #   • the node's own temporal evolution      (from DeltaTimeRNN)
        # A cross-gating mechanism lets the model decide per-node and per-dim
        # how much to rely on each signal.
        self.readout_gate = nn.Sequential(
            nn.Linear(hidden_dim * 2, hidden_dim),
            nn.Sigmoid(),
        )
        self.readout_proj = nn.Sequential(
            nn.Linear(hidden_dim * 2, hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, hidden_dim),
        )
        self.final_norm = nn.LayerNorm(hidden_dim)
        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)

    # ------------------------------------------------------------------
    def _encode_time(self, t_abs, t_rel):
        """Combine absolute + relative timestamp encodings → (E, hidden_dim)"""
        return self.time_proj(
            torch.cat([self.abs_t_embed(t_abs),
                       self.rel_t_embed(t_rel)], dim=-1))

    # ------------------------------------------------------------------
    def get_temporal_repr(self, node_ids: torch.Tensor):
        """
        Read K-step history from ViewBuffer and run DeltaTimeRNN.

        Returns
        -------
        h_temporal : (B, hidden_dim)   final GRU state (temporal view)
        h_seq      : (B, K, hidden_dim) full GRU sequence (for TrustScorer)
        """
        views, _, dts      = self.view_buffer.read(node_ids)
        h_temporal, h_seq  = self.delta_rnn(views, dts)
        return h_temporal, h_seq

    # ------------------------------------------------------------------
    def forward(self, batch):
        """
        batch attributes required
        -------------------------
        x          (N, in_dim)    node features
        edge_index (2, E)
        msg        (E, edge_dim)  edge attributes / message features
        t          (E,)           edge timestamps

        Returns
        -------
        z : (N, hidden_dim)
            Unified node embedding that jointly encodes:
              • trust-filtered neighborhood structure  (trust-gated GNN layers)
              • the node's own temporal trajectory     (DeltaTimeRNN over K views)
            Feed directly into your contrastive / self-supervised objective.
        """
        device   = batch.x.device
        N        = batch.x.size(0)
        src      = batch.edge_index[0]
        all_ids  = torch.arange(N, device=device)

        # ── Project inputs ────────────────────────────────────────────────────
        x        = self.node_proj(batch.x)          # (N, D)
        edge_emb = self.edge_proj(batch.msg)         # (E, D)

        # ── Temporal encoding per edge (absolute + relative to node's last view)
        _, times_all, _ = self.view_buffer.read(all_ids)
        node_last_t     = times_all[:, -1]           # (N,) most recent snapshot time

        t_abs    = batch.t.float()
        t_rel    = (t_abs - node_last_t[src]).clamp(min=0.0)    # (E,)
        time_emb = self._encode_time(t_abs, t_rel)              # (E, D)

        # ── Temporal summaries from K-step history ────────────────────────────
        # h_temporal : node's own trajectory summary  (N, D)
        # h_seq      : full K-step sequence           (N, K, D)  → TrustScorer
        h_temporal, h_seq = self.get_temporal_repr(all_ids)

        # ── Trust-gated message passing ───────────────────────────────────────
        # Each layer uses h_seq to compute per-edge trust, so suspicious
        # neighbors are down-weighted based on activity divergence, attribute
        # misalignment and edge plausibility — all before softmax normalisation.
        for layer in self.layers:
            x = layer(
                x          = x,
                edge_index  = batch.edge_index,
                edge_emb    = edge_emb,
                time_emb    = time_emb,
                h_seq       = h_seq,
            )
        # x here is the trust-filtered structural view  (N, D)

        # ── Final readout: fuse structural + temporal into one representation ─
        # Cross-gate learns per-node, per-dim how much to rely on each signal:
        #   • x          carries neighborhood context filtered by trust
        #   • h_temporal carries the node's own behavioral evolution over time
        # Together they make the representation aware of BOTH suspicious
        # neighbors AND temporal dynamics without leaking either as a
        # separate output to the downstream objective.
        concat = torch.cat([x, h_temporal], dim=-1)              # (N, 2D)
        gate   = self.readout_gate(concat)                       # (N, D) ∈ (0,1)
        proj   = self.readout_proj(concat)                       # (N, D)
        z      = self.final_norm(gate * x + (1 - gate) * proj)   # (N, D)

        # ── Update ViewBuffer with the unified embedding for the next step ────
        node_cur_t = torch.zeros(N, device=device)
        node_cur_t.scatter_reduce_(0, src, t_abs, reduce='amax', include_self=True)
        self.view_buffer.update(all_ids, z, node_cur_t)

        return z


# ═══════════════════════════════════════════════════════════════════════════════
# Smoke test
# ═══════════════════════════════════════════════════════════════════════════════

if __name__ == "__main__":
    import types, random

    torch.manual_seed(42)
    random.seed(42)

    N, E      = 64, 256
    in_dim    = 32
    edge_dim  = 16
    hidden    = 64
    K         = 8
    num_nodes = 128

    batch = types.SimpleNamespace()
    batch.x          = torch.randn(N, in_dim)
    batch.edge_index = torch.randint(0, N, (2, E))
    batch.msg        = torch.randn(E, edge_dim)
    batch.t          = torch.rand(E) * 1000

    model = NATGAT(
        num_nodes  = num_nodes,
        in_dim     = in_dim,
        edge_dim   = edge_dim,
        hidden_dim = hidden,
        num_layers = 2,
        heads      = 4,
        K          = K,
        dropout    = 0.1,
    )

    z = model(batch)

    print(f"unified node embedding z : {z.shape}")   # (64, 64)
    print(f"value range              : [{z.min():.3f}, {z.max():.3f}]")
    print("Smoke test passed.")

unified node embedding z : torch.Size([64, 64])
value range              : [-4.206, 3.779]
Smoke test passed.


In [19]:
"""
Drop-in replacement for your notebook.
Keeps: data, loaders, History, args, measure
Replaces: model, train(), test()

Just run these cells in order after your existing setup cells.
"""

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import MessagePassing
from torch_geometric.utils import softmax
from tqdm import tqdm


# ── Paste the full enhanced_gnn_v2.py here, or import it ──────────────────────
# from enhanced_gnn_v2 import (EnhancedTemporalGNN, train, test)
# ──────────────────────────────────────────────────────────────────────────────
# If running directly in notebook, paste the class definitions above this cell.


# ═══════════════════════════════════════════
# CELL: Build model + history (unchanged)
# ═══════════════════════════════════════════

try:
    del model
except NameError:
    pass



model = NATGAT(
    num_nodes  = data.num_nodes,
    in_dim     = data.x.size(1),          # 1 for wikipedia
    edge_dim   = data.msg.size(1),         # 172 for wikipedia
    hidden_dim = args.hidden_channels,     # 64
    num_layers = args.nums_layers,         # 2
    heads      = args.heads,               # 8
    dropout    = args.dropout,             # 0.3
    K     = 8
).to(device)

# History is UNCHANGED — reuse your existing History class
try:
    del history
except NameError:
    pass

from mhiscl.history import History

history = History(
    data.num_nodes,
    args.num_timeslots,
    args.hidden_channels,
    device=device,
    history_retrieve=args.history_retrieve,
    recurrent=args.recurrent,
).to(device)

print(model)
print(history)
print(f"Model params: {sum(p.numel() for p in model.parameters()):,}")




NATGAT(
  (node_proj): Linear(in_features=1, out_features=64, bias=True)
  (edge_proj): Linear(in_features=172, out_features=64, bias=True)
  (abs_t_embed): FreqAwareEmbed(
    (proj): Sequential(
      (0): Linear(in_features=31, out_features=32, bias=True)
      (1): GELU(approximate='none')
      (2): Linear(in_features=32, out_features=32, bias=True)
    )
  )
  (rel_t_embed): FreqAwareEmbed(
    (proj): Sequential(
      (0): Linear(in_features=31, out_features=32, bias=True)
      (1): GELU(approximate='none')
      (2): Linear(in_features=32, out_features=32, bias=True)
    )
  )
  (time_proj): Sequential(
    (0): Linear(in_features=64, out_features=64, bias=True)
    (1): GELU(approximate='none')
    (2): Linear(in_features=64, out_features=64, bias=True)
  )
  (view_buffer): ViewBuffer()
  (delta_rnn): DeltaTimeRNN(
    (dt_embed): FreqAwareEmbed(
      (proj): Sequential(
        (0): Linear(in_features=63, out_features=64, bias=True)
        (1): GELU(approximate='none')
  

In [ ]:
# ═══════════════════════════════════════════
# CELL: Training loop (drop-in replacement)
# ═══════════════════════════════════════════

best     = 0
best_val = 0

optimizer = torch.optim.Adam(
    list(model.parameters()) + list(history.parameters()),
    lr=args.lr,
    weight_decay=1e-5       # small L2 reg helps generalization
)

# Cosine LR schedule: smoothly decays over training
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=args.epochs, eta_min=args.lr * 0.1)

measure = Measure("auc")

for epoch in range(1, 1 + args.epochs):
    loss    = train(train_loader, model, history, optimizer, device, data, args)
    val_auc = test(valid_loader,  model, history, measure,   device, data)
    test_auc = test(test_loader,  model, history, measure,   device, data)
    scheduler.step()

    if val_auc > best_val:
        best_val = val_auc
        best     = test_auc

    print(
        f"Epoch: {epoch:03d}, Loss: {loss:.4f}, "
        f"Val AUC: {val_auc:.2%}, Test AUC: {test_auc:.2%}, Best AUC: {best:.2%}"
    )

100%|██████████| 93/93 [00:10<00:00,  8.71it/s]


Epoch: 001, Loss: 17.7805, Val AUC: 78.76%, Test AUC: 86.62%, Best AUC: 86.62%


100%|██████████| 93/93 [00:10<00:00,  8.61it/s]


Epoch: 002, Loss: 14.1672, Val AUC: 76.28%, Test AUC: 84.91%, Best AUC: 86.62%


100%|██████████| 93/93 [00:10<00:00,  8.70it/s]


...

100%|██████████| 93/93 [00:10<00:00,  8.63it/s]


Epoch: 004, Loss: 13.6245, Val AUC: 82.22%, Test AUC: 84.25%, Best AUC: 84.25%


100%|██████████| 93/93 [00:40<00:00,  2.30it/s]


Epoch: 005, Loss: 13.2502, Val AUC: 69.06%, Test AUC: 76.98%, Best AUC: 84.25%


Train Loss = 13.8493:   4%|▍         | 19/431 [00:14<05:17,  1.30it/s]

### Change model from logits to similarity

In [22]:
"""
Enhanced Temporal Anomaly GNN — v5
====================================
Based on the 91% baseline, with one conceptual fix to the Trust module.

═══════════════════════════════════════════════════════════════════
THE TRUST CONCEPTUAL FIX
═══════════════════════════════════════════════════════════════════

OLD BEHAVIOR (wrong for unsupervised learning):
  trust(i→j) was used as a GATE — low trust suppressed j's message
  toward zero. The implicit assumption was "j is anomalous, ignore it."
  This is supervised thinking: you need labels to know who to ignore.

  Problems:
    1. An anomalous node i would have its neighbors suppressed →
       it learns from nobody → its representation collapses → 
       no useful anomaly signal.
    2. A normal node surrounded by anomalies gets all messages
       suppressed → also collapses.
    3. Asymmetric: normal nodes ignore anomalous neighbors, but
       anomalous nodes also ignore normal neighbors → both groups
       become isolated → contrastive signal disappears.

NEW BEHAVIOR (correct for unsupervised):
  trust(i→j) is a SOFT SIMILARITY WEIGHT — not a gate, but a
  reweighting inside the softmax normalization.

  The principle: every node learns proportionally more from
  neighbors that resemble itself, whether those neighbors are
  normal OR anomalous.

    • Normal node i, normal neighbor j:    high similarity → learns a lot from j
    • Normal node i, anomalous neighbor j: low similarity  → learns less from j
    • Anomalous node i, anomalous neighbor j: high similarity → learns a lot from j  ✓
    • Anomalous node i, normal neighbor j: low similarity  → learns less from j     ✓

  This means:
    - Anomalous nodes naturally cluster with other anomalous nodes
      in representation space, because they learn most from each other.
    - Normal nodes cluster with normal nodes.
    - The contrastive loss then has clean signal: the two groups
      diverge in representation space without any labels.
    - Nobody is "deactivated" — every node still receives messages,
      just weighted by how similar the sender is.

HOW IT WORKS MECHANICALLY:
  Instead of:
    attn = attn * trust          # hard multiply → can zero out messages
    attn = softmax(attn, index)  # normalize → distrusted neighbors get ~0 mass

  We do:
    sim_weight = similarity_weight(i, j)    # soft [0,1], never forced to 0
    attn = attn + log(sim_weight + eps)     # ADD in log-space before softmax
    attn = softmax(attn, index)             # normalize → similar neighbors get MORE mass

  Adding in log-space is equivalent to multiplying the softmax argument
  by sim_weight, which is a proper probabilistic reweighting. The key
  difference from the old approach: sim_weight never kills a message
  completely — it just gives it proportionally less weight relative to
  more similar neighbors.

  The similarity weight itself is:
    sim(i,j) = softplus( cosine(h_i, h_j) )    where h = GRU memory
    + neighborhood_evolution_consistency(j)      whether j has been stable
    + uncertainty_confidence(i, j)               stable memory → more weight

  This is symmetric in intent: if i and j look alike, they learn from
  each other. If they look different, they learn less from each other —
  but never zero.

EVERYTHING ELSE is identical to the 91% baseline:
  - Time2Vec temporal encoding (absolute + relative gap)
  - RecurrentMemory GRU per node with variance tracking
  - EvolutionBank rolling window of neighborhood means
  - InfoNCE contrastive loss with hard negatives
  - Anomaly score = structural vs temporal view disagreement
"""

import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import MessagePassing
from torch_geometric.utils import softmax
from tqdm import tqdm


# ═══════════════════════════════════════════════════════════════════════════════
# 1. Time2Vec
# ═══════════════════════════════════════════════════════════════════════════════

class Time2Vec(nn.Module):
    """
    t → [w0*t + b0,  sin(w1*t+b1), ..., sin(wk*t+bk)]
    Linear = trend, sinusoids = periodicity. Frequencies are learned.
    """
    def __init__(self, out_dim):
        super().__init__()
        k = out_dim - 1
        self.w0 = nn.Parameter(torch.randn(1) * 0.01)
        self.b0 = nn.Parameter(torch.zeros(1))
        self.W  = nn.Parameter(torch.randn(k) * 0.01)
        self.B  = nn.Parameter(torch.zeros(k))

    def forward(self, t):
        t = t.float().unsqueeze(-1)
        return torch.cat([t * self.w0 + self.b0,
                          torch.sin(t * self.W + self.B)], dim=-1)


class TimeEncoder(nn.Module):
    def __init__(self, hidden_dim):
        super().__init__()
        self.t2v     = Time2Vec(hidden_dim // 2)
        self.rel_t2v = Time2Vec(hidden_dim // 2)
        self.proj    = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, hidden_dim),
        )

    def forward(self, t_abs, t_rel=None):
        abs_enc = self.t2v(t_abs)
        rel_enc = self.rel_t2v(t_rel) if t_rel is not None else torch.zeros_like(abs_enc)
        return self.proj(torch.cat([abs_enc, rel_enc], dim=-1))


# ═══════════════════════════════════════════════════════════════════════════════
# 2. RecurrentMemory
# ═══════════════════════════════════════════════════════════════════════════════

class RecurrentMemory(nn.Module):
    """
    Per-node GRU hidden state + EMA variance.
    Low variance = stable node = more confident memory.
    """
    def __init__(self, num_nodes, dim, momentum=0.9):
        super().__init__()
        self.gru      = nn.GRUCell(dim, dim)
        self.momentum = momentum
        self.register_buffer("hidden",   torch.zeros(num_nodes, dim))
        self.register_buffer("variance", torch.ones(num_nodes, dim) * 0.1)

    def read(self, idx):
        return self.hidden[idx], self.variance[idx]

    def get_hidden(self, idx):
        return self.hidden[idx]

    @torch.no_grad()
    def write(self, idx, x):
        h_old = self.hidden[idx]
        h_new = self.gru(x.detach(), h_old)
        delta = h_new - h_old
        self.variance[idx] = (self.momentum * self.variance[idx]
                              + (1 - self.momentum) * delta.pow(2))
        self.hidden[idx] = h_new


# ═══════════════════════════════════════════════════════════════════════════════
# 3. EvolutionBank
# ═══════════════════════════════════════════════════════════════════════════════

class EvolutionBank(nn.Module):
    """
    Rolling window of W most recent neighborhood mean embeddings per node.
    Tells the trust module how j's neighborhood has evolved over time.
    """
    def __init__(self, num_nodes, dim, window=6):
        super().__init__()
        self.window = window
        self.register_buffer("bank", torch.zeros(num_nodes, window, dim))
        self.register_buffer("ptr",  torch.zeros(num_nodes, dtype=torch.long))

    def read(self, idx):
        return self.bank[idx]                                          # (B, W, D)

    @torch.no_grad()
    def write(self, idx, emb):
        for b in range(idx.size(0)):
            p = int(self.ptr[idx[b]]) % self.window
            self.bank[idx[b], p] = emb[b].detach()
            self.ptr[idx[b]] += 1


# ═══════════════════════════════════════════════════════════════════════════════
# 4. SimilarityWeight Module  ← THE FIX
#
# Replaces the old TrustModule gate with a soft similarity reweighting.
#
# OLD: trust → gate (can zero out messages from "suspicious" neighbors)
# NEW: sim_weight → log-space bias (reweights proportionally, never zeros)
#
# Produces a scalar w(i,j) ∈ (0, 1] for each edge, representing how
# similar i and j are. This is added in log-space to attention logits:
#
#   attn_logit(i,j) += log(w(i,j) + ε)
#
# After softmax this means similar neighbors get proportionally more
# attention mass. But every neighbor still gets SOME mass — no deactivation.
#
# w(i,j) is computed from three components:
#
#   (A) HISTORICAL SIMILARITY
#       cosine(GRU_hidden_i, GRU_hidden_j)
#       Nodes with similar behavioral histories attract each other.
#       A normal node has high similarity with other normal nodes.
#       An anomalous node has high similarity with other anomalous nodes.
#       → Each group naturally learns from its own kind.
#
#   (B) NEIGHBORHOOD EVOLUTION COHERENCE
#       How coherent is j's neighborhood evolution window?
#       j's evolution window = [neigh_mean_{t-W}, ..., neigh_mean_{t-1}]
#       If consecutive entries are similar → j is stable → consistent partner.
#       This is not "j is normal" — it's "j has been consistent."
#       An anomalous node that has been consistently anomalous is still
#       coherent and will have high coherence with other similar anomalies.
#
#   (C) UNCERTAINTY CONFIDENCE
#       Nodes with stable GRU states (low variance) produce more reliable
#       similarity estimates. We weight the combination by confidence.
#       Low confidence → w(i,j) pulled toward uniform (0.5) rather than
#       pushed to extremes, avoiding overconfident reweighting early in training.
#
# Final combination:
#   raw = MLP([hist_sim, coherence_j, conf])
#   w   = sigmoid(raw)   ∈ (0, 1)
#
# Note sigmoid never reaches exactly 0 or 1 — the floor ensures no
# message is ever completely silenced.
# ═══════════════════════════════════════════════════════════════════════════════

class SimilarityWeight(nn.Module):
    def __init__(self, dim, num_heads=4, window=6):
        super().__init__()
        assert dim % num_heads == 0
        self.num_heads = num_heads
        self.head_dim  = dim // num_heads
        self.scale     = self.head_dim ** -0.5
        self.window    = window

        # (A) Historical similarity projection
        # We project before cosine so the model can learn a better
        # similarity metric than raw cosine in the original space.
        self.proj_i = nn.Linear(dim, dim)
        self.proj_j = nn.Linear(dim, dim)

        # (B) Neighborhood evolution coherence
        # Reads j's W evolution steps and scores their consistency.
        # Cross-attention: i's history queries j's evolution window.
        # This lets i assess whether j has been a coherent partner,
        # not just whether j looks similar at the current moment.
        self.q_proj = nn.Linear(dim, dim)
        self.k_proj = nn.Linear(dim, dim)
        self.v_proj = nn.Linear(dim, dim)

        # Recency decay: newer evolution steps weighted more
        self.log_decay = nn.Parameter(torch.tensor(-2.0))

        # Pool evolution context → coherence scalar
        self.coherence_out = nn.Sequential(
            nn.Linear(dim, dim // 2),
            nn.GELU(),
            nn.Linear(dim // 2, 1),
        )

        # (C) Final combination: [hist_sim, coherence, conf] → weight
        self.combine = nn.Sequential(
            nn.Linear(3, 32),
            nn.GELU(),
            nn.Linear(32, 1),
        )

    def forward(self, h_i, h_j, evo_j, var_i, var_j, eps=1e-6):
        """
        h_i:   (E, D)    GRU hidden state of source node i
        h_j:   (E, D)    GRU hidden state of destination node j
        evo_j: (E, W, D) evolution window of j's neighborhood
        var_i: (E, D)    GRU variance of i
        var_j: (E, D)    GRU variance of j

        Returns: (E,) similarity weights in (0, 1)
                 — to be added in log-space to attention logits
        """
        E, W, D = evo_j.shape

        # ── (A) Historical similarity ────────────────────────────────────────
        # Project i and j's histories to a learned similarity space,
        # then compute cosine similarity.
        pi = F.normalize(self.proj_i(h_i), dim=-1)                    # (E, D)
        pj = F.normalize(self.proj_j(h_j), dim=-1)                    # (E, D)
        hist_sim = (pi * pj).sum(-1)                                   # (E,) ∈ [-1, 1]
        # Shift to [0, 1] — we want a weight, not a signed score
        hist_sim = (hist_sim + 1.0) / 2.0                             # (E,) ∈ [0, 1]

        # ── (B) Neighborhood evolution coherence ─────────────────────────────
        # Cross-attention: i's history (query) attends over j's evolution window
        nh, hd = self.num_heads, self.head_dim

        Q = self.q_proj(h_i).view(E, nh, hd)                         # (E, nh, hd)
        evo_flat = evo_j.reshape(E * W, D)
        K = self.k_proj(evo_flat).view(E, W, nh, hd)
        V = self.v_proj(evo_flat).view(E, W, nh, hd)

        Q_exp  = Q.unsqueeze(2)                                        # (E, nh, 1, hd)
        K_t    = K.permute(0, 2, 1, 3)                                # (E, nh, W, hd)
        scores = (Q_exp * K_t).sum(-1) * self.scale                   # (E, nh, W)

        # Recency decay: newer steps get higher base weight
        steps   = torch.arange(W, device=h_i.device).float()
        decay_w = torch.exp(self.log_decay.exp() * steps)
        decay_w = decay_w / decay_w.sum()
        scores  = scores + decay_w.view(1, 1, W)

        attn   = F.softmax(scores, dim=-1)                            # (E, nh, W)
        V_t    = V.permute(0, 2, 1, 3)                                # (E, nh, W, hd)
        pooled = (attn.unsqueeze(-1) * V_t).sum(2).reshape(E, D)     # (E, D)
        coherence = torch.sigmoid(
            self.coherence_out(pooled).squeeze(-1))                    # (E,) ∈ (0,1)

        # ── (C) Uncertainty confidence ───────────────────────────────────────
        # Nodes with stable memories → confident similarity estimate.
        # Pull toward 0.5 (uniform weight) when uncertain,
        # rather than pushing to extremes and over-suppressing.
        conf_i = 1.0 / (1.0 + var_i.mean(-1).sqrt().clamp(min=eps))  # (E,) ∈ (0,1]
        conf_j = 1.0 / (1.0 + var_j.mean(-1).sqrt().clamp(min=eps))
        conf   = (conf_i * conf_j).sqrt()                             # (E,) geometric mean

        # ── Combine all three signals ─────────────────────────────────────────
        signals = torch.stack([hist_sim, coherence, conf], dim=-1)    # (E, 3)
        w = torch.sigmoid(self.combine(signals).squeeze(-1))          # (E,) ∈ (0, 1)

        # w is the soft similarity weight.
        # sigmoid floor ≈ 0.007, ceiling ≈ 0.993 — never exactly 0 or 1.
        # Every neighbor still contributes; dissimilar ones just contribute less.
        return w


# ═══════════════════════════════════════════════════════════════════════════════
# 5. Temporal Attention Layer with Similarity Reweighting
# ═══════════════════════════════════════════════════════════════════════════════

class TemporalTrustAttention(MessagePassing):
    def __init__(self, dim, heads=8, dropout=0.1, window=6):
        super().__init__(aggr="add", node_dim=0)
        assert dim % heads == 0
        self.dim      = dim
        self.heads    = heads
        self.head_dim = dim // heads
        self.scale    = self.head_dim ** -0.5
        self.window   = window

        self.q_proj    = nn.Linear(dim, dim)
        self.k_proj    = nn.Linear(dim, dim)
        self.v_proj    = nn.Linear(dim, dim)
        self.o_proj    = nn.Linear(dim, dim)
        self.edge_proj = nn.Linear(dim, dim)
        self.time_proj = nn.Linear(dim, dim)

        # Renamed from TrustModule → SimilarityWeight to reflect new semantics
        self.sim_weight = SimilarityWeight(dim, num_heads=4, window=window)
        self.dropout    = nn.Dropout(dropout)

        self.gate    = nn.Linear(dim, 1)
        self.norm    = nn.LayerNorm(dim)
        self.fusion  = nn.Sequential(
            nn.Linear(dim * 2, dim),
            nn.GELU(),
            nn.Linear(dim, dim),
        )

        # Cache for inspection / anomaly scoring
        self.sim_cache = None

    def split(self, x):
        return x.view(-1, self.heads, self.head_dim)

    def forward(self, x, edge_index, edge_emb, time_emb, memory, evo_bank):
        N      = x.size(0)
        device = x.device
        idx    = torch.arange(N, device=device)

        Q = self.split(self.q_proj(x))
        K = self.split(self.k_proj(x))
        V = self.split(self.v_proj(x))

        mem_vals, mem_vars = memory.read(idx)                          # (N, D) each
        evo_vals           = evo_bank.read(idx)                        # (N, W, D)

        out = self.propagate(
            edge_index,
            Q=Q, K=K, V=V,
            mem_val=mem_vals,
            mem_var=mem_vars,
            hidden=mem_vals,
            evo_vals=evo_vals,
            edge_emb=self.split(self.edge_proj(edge_emb)),
            time_emb=self.split(self.time_proj(time_emb)),
            size=(N, N)
        )
        out = out.view(N, self.dim)

        h   = memory.get_hidden(idx)
        out = self.fusion(torch.cat([out, h], dim=-1))
        out = self.o_proj(out)

        g = torch.sigmoid(self.gate(x))
        return self.norm(g * x + (1 - g) * out)

    def message(self, Q_i, K_j, V_j,
                mem_val_i, mem_val_j,
                mem_var_i, mem_var_j,
                hidden_i, hidden_j,
                evo_vals_j,
                edge_emb, time_emb,
                index):

        # Enrich keys with edge and time context (unchanged from baseline)
        K_j = K_j + edge_emb + time_emb                               # (E, h, hd)

        # Raw dot-product attention logits
        attn = (Q_i * K_j).sum(-1) * self.scale                      # (E, h)

        # ── SIMILARITY REWEIGHTING (the fix) ────────────────────────────────
        # Compute soft similarity weight w(i,j) ∈ (0, 1)
        w = self.sim_weight(
            hidden_i, hidden_j,
            evo_vals_j,
            mem_var_i, mem_var_j
        )                                                               # (E,)
        self.sim_cache = w.detach()

        # Add log(w) to the attention logit IN LOG-SPACE before softmax.
        #
        # Why log-space?
        #   softmax(a + log(w)) = softmax(a) * w / Z
        # This is equivalent to multiplying the unnormalized attention
        # probability by w, then renormalizing. It is a clean probabilistic
        # upweighting of similar neighbors and downweighting of dissimilar ones.
        #
        # Why NOT multiply after softmax?
        #   attn = softmax(a); attn = attn * w   ← this breaks normalization.
        #   The weights no longer sum to 1, so the aggregation is biased.
        #
        # Why NOT multiply before softmax (old gate approach)?
        #   attn = a * w; softmax(attn)
        #   When w ≈ 0, the logit is zeroed → after softmax that neighbor
        #   gets nearly zero mass. This is the deactivation problem.
        #   With log(w), even w=0.01 → log(w)≈-4.6, not -∞.
        #   The neighbor still gets a small positive mass after softmax.
        #
        # Clamp log(w) to [-6, 0]: floor prevents -inf, ceiling (0) ensures
        # the weight never INCREASES beyond the base attention score.
        log_w = torch.log(w.clamp(min=1e-6)).clamp(min=-6.0, max=0.0) # (E,)
        attn  = attn + log_w.unsqueeze(-1)                             # (E, h)

        attn = softmax(attn, index)                                    # normalize over neighbors
        attn = self.dropout(attn)

        return V_j * attn.unsqueeze(-1)                                # (E, h, hd)


# ═══════════════════════════════════════════════════════════════════════════════
# 6. Full Model (identical to 91% baseline except SimilarityWeight)
# ═══════════════════════════════════════════════════════════════════════════════

class EnhancedTemporalGNN(nn.Module):
    def __init__(self, num_nodes, in_dim, edge_dim, hidden_dim,
                 num_layers=2, heads=8, dropout=0.1,
                 window=6, memory_momentum=0.9):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.num_nodes  = num_nodes

        self.input_proj = nn.Linear(in_dim, hidden_dim)
        self.edge_enc   = nn.Linear(edge_dim, hidden_dim)
        self.time_enc   = TimeEncoder(hidden_dim)

        self.memory   = RecurrentMemory(num_nodes, hidden_dim, momentum=memory_momentum)
        self.evo_bank = EvolutionBank(num_nodes, hidden_dim, window=window)

        self.layers = nn.ModuleList([
            TemporalTrustAttention(hidden_dim, heads=heads,
                                   dropout=dropout, window=window)
            for _ in range(num_layers)
        ])

        self.final_norm = nn.LayerNorm(hidden_dim)
        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)

    def forward(self, batch):
        N          = batch.x.size(0)
        device     = batch.x.device
        edge_index = batch.edge_index
        src, dst   = edge_index

        x        = self.input_proj(batch.x)
        edge_emb = self.edge_enc(batch.msg)

        # Relative time gap: how long since src last appeared in this batch
        node_last_t = torch.zeros(N, device=device)
        node_last_t.scatter_reduce_(
            0, src, batch.t.float(), reduce='amax', include_self=True)
        t_rel    = batch.t.float() - node_last_t[src]
        time_emb = self.time_enc(batch.t, t_rel)

        for layer in self.layers:
            x = layer(x, edge_index, edge_emb, time_emb,
                      self.memory, self.evo_bank)

        x = self.final_norm(x)

        # Update memory
        self.memory.write(torch.arange(N, device=device), x)

        # Update evolution bank with neighborhood means
        neigh = torch.zeros(N, self.hidden_dim, device=device)
        cnt   = torch.zeros(N, 1, device=device)
        neigh.scatter_add_(0, dst.unsqueeze(1).expand(-1, self.hidden_dim), x[src])
        cnt.scatter_add_(0, dst.unsqueeze(1),
                         torch.ones(src.size(0), 1, device=device))
        neigh = neigh / (cnt + 1e-6)
        unique_dst = dst.unique()
        self.evo_bank.write(unique_dst, neigh[unique_dst])

        return x




In [23]:
"""
Drop-in replacement for your notebook.
Keeps: data, loaders, History, args, measure
Replaces: model, train(), test()

Just run these cells in order after your existing setup cells.
"""

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import MessagePassing
from torch_geometric.utils import softmax
from tqdm import tqdm


# ── Paste the full enhanced_gnn_v2.py here, or import it ──────────────────────
# from enhanced_gnn_v2 import (EnhancedTemporalGNN, train, test)
# ──────────────────────────────────────────────────────────────────────────────
# If running directly in notebook, paste the class definitions above this cell.


# ═══════════════════════════════════════════
# CELL: Build model + history (unchanged)
# ═══════════════════════════════════════════

try:
    del model
except NameError:
    pass

model = EnhancedTemporalGNN(
    num_nodes  = data.num_nodes,
    in_dim     = data.x.size(1),          # 1 for wikipedia
    edge_dim   = data.msg.size(1),         # 172 for wikipedia
    hidden_dim = args.hidden_channels,     # 64
    num_layers = args.nums_layers,         # 2
    heads      = args.heads,               # 8
    dropout    = args.dropout,             # 0.3
    window     = 8,
    memory_momentum = 0.9,
).to(device)

# History is UNCHANGED — reuse your existing History class
try:
    del history
except NameError:
    pass

from mhiscl.history import History

history = History(
    data.num_nodes,
    args.num_timeslots,
    args.hidden_channels,
    device=device,
    history_retrieve=args.history_retrieve,
    recurrent=args.recurrent,
).to(device)

print(model)
print(history)
print(f"Model params: {sum(p.numel() for p in model.parameters()):,}")




EnhancedTemporalGNN(
  (input_proj): Linear(in_features=1, out_features=64, bias=True)
  (edge_enc): Linear(in_features=172, out_features=64, bias=True)
  (time_enc): TimeEncoder(
    (t2v): Time2Vec()
    (rel_t2v): Time2Vec()
    (proj): Sequential(
      (0): Linear(in_features=64, out_features=64, bias=True)
      (1): GELU(approximate='none')
      (2): Linear(in_features=64, out_features=64, bias=True)
    )
  )
  (memory): RecurrentMemory(
    (gru): GRUCell(64, 64)
  )
  (evo_bank): EvolutionBank()
  (layers): ModuleList(
    (0-1): 2 x TemporalTrustAttention()
  )
  (final_norm): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
)
History(
  (recurrent_network): GRUCell(64, 64)
  (lin): Sequential(
    (0): Linear(-1, 64, bias=True)
    (1): ELU(alpha=1.0)
    (2): Linear(64, 64, bias=True)
  )
)
Model params: 166,024


In [24]:
## Training Function 

def train(loader, model, history, optimizer, device, data, args):
    model.train()
    history.train()
    pbar = tqdm(loader)
    total_loss = 0
    for batch in pbar:
        batch = batch.to(device)
        optimizer.zero_grad()
        out = model(batch)
        src = batch.src[: batch.batch_size]
        out = out[src]
        idx = batch.input_id.cpu()
        num_neg = batch.batch_size
        neg_mem = history.get_history(torch.randint(
            0, history.num_nodes, size=(num_neg,)))
        raw_msg = torch.zeros(idx.size(0),1).to(device) 
        cur_mem, prev_mem = history(out, data.src[idx]) 

        if args.con=='his':
            loss = contrastive_loss(prev_mem, cur_mem, args.tau) 
            loss += torch.exp(cosine_similarity(cur_mem, neg_mem) 
                          ).sum(dim=1).log().mean()
        elif args.con=='stru':
            loss = contrastive_loss(out, cur_mem, args.tau) 
            loss += torch.exp(cosine_similarity(out, neg_mem)
                            ).sum(dim=1).log().mean()
        elif args.con=='all':
            loss = args.alpha*contrastive_loss(out, cur_mem, args.tau) + \
                contrastive_loss(prev_mem, cur_mem, args.tau)
            loss += torch.exp(cosine_similarity(cur_mem, neg_mem) 
                            ).sum(dim=1).log().mean()
            loss += args.alpha*torch.exp(cosine_similarity(out, neg_mem) 
                            ).sum(dim=1).log().mean()


        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        pbar.set_description(f"Train Loss = {loss.item():.4f}")

    return total_loss / len(loader)


## Test function

@torch.no_grad()
def test(loader, model, history, measure, device, data):
    preds = []
    labels = []
    model.eval()
    history.eval()
    for batch in tqdm(loader):
        torch.cuda.empty_cache()
        batch = batch.to(device)
        out = model(batch)
        src = batch.src[: batch.batch_size]
        out = out[src]
        label = batch.y[: batch.batch_size]
        idx = batch.input_id.cpu()
        raw_msg = torch.zeros(idx.size(0),1).to(device) 
        cur_mem, prev_mem = history(out, data.src[idx], update=True)
        if args.con=='his':
            pred = 1-torch.diag(cosine_similarity(cur_mem, prev_mem)).view(-1)
        elif args.con=='stru': 
            pred = 1-torch.diag(cosine_similarity(out, cur_mem)).view(-1)
        elif args.con=='all':
            p1 = torch.diag(cosine_similarity(out, cur_mem)).view(-1)
            p2 = torch.diag(cosine_similarity(cur_mem, prev_mem)).view(-1)
            pred = (1 - p1) + (1 - p2) 
            
        pred = pred.sigmoid() 
        torch.cuda.empty_cache()
        preds.append(pred.detach().cpu())
        labels.append(label.detach().cpu())
    preds = torch.cat(preds)
    preds[torch.isnan(preds)] = 0.0 
    labels = torch.cat(labels)
    return measure(labels[labels!=2], preds[labels!=2]) 




In [25]:
# ═══════════════════════════════════════════
# CELL: Training loop (drop-in replacement)
# ═══════════════════════════════════════════

best     = 0
best_val = 0

optimizer = torch.optim.Adam(
    list(model.parameters()) + list(history.parameters()),
    lr=args.lr,
    weight_decay=1e-5       # small L2 reg helps generalization
)

# Cosine LR schedule: smoothly decays over training
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=args.epochs, eta_min=args.lr * 0.1)

measure = Measure("auc")

for epoch in range(1, 1 + args.epochs):
    loss    = train(train_loader, model, history, optimizer, device, data, args)
    val_auc = test(valid_loader,  model, history, measure,   device, data)
    test_auc = test(test_loader,  model, history, measure,   device, data)
    scheduler.step()

    if val_auc > best_val:
        best_val = val_auc
        best     = test_auc

    print(
        f"Epoch: {epoch:03d}, Loss: {loss:.4f}, "
        f"Val AUC: {val_auc:.2%}, Test AUC: {test_auc:.2%}, Best AUC: {best:.2%}"
    )

100%|██████████| 93/93 [00:04<00:00, 20.19it/s]


Epoch: 001, Loss: 21.0469, Val AUC: 73.97%, Test AUC: 75.27%, Best AUC: 75.27%


100%|██████████| 93/93 [00:04<00:00, 19.42it/s]


Epoch: 002, Loss: 19.6161, Val AUC: 86.77%, Test AUC: 87.89%, Best AUC: 87.89%


100%|██████████| 93/93 [00:04<00:00, 19.68it/s]


...

Epoch: 033, Loss: 16.7921, Val AUC: 78.26%, Test AUC: 82.38%, Best AUC: 87.61%


100%|██████████| 93/93 [00:04<00:00, 19.21it/s]


Epoch: 034, Loss: 16.8103, Val AUC: 84.76%, Test AUC: 89.85%, Best AUC: 87.61%


100%|██████████| 93/93 [00:04<00:00, 20.33it/s]

Epoch: 035, Loss: 16.8073, Val AUC: 80.26%, Test AUC: 86.94%, Best AUC: 87.61%


In [31]:
"""
Enhanced Temporal Anomaly GNN — v6
====================================
Based on the 91% baseline. The only change is how we control how much
each node learns from each neighbor — replacing the external similarity
gate/weight with MEMORY-ROUTED ATTENTION.

═══════════════════════════════════════════════════════════════════
THE PROBLEM WITH EXTERNAL SIMILARITY WEIGHTING (v5)
═══════════════════════════════════════════════════════════════════

In v5 we computed a separate similarity score w(i,j) and injected it
as log(w) into the attention logits. This had two issues:

  1. The similarity module and the attention module are DECOUPLED.
     They don't co-adapt during training. The attention learns one
     thing; the similarity learns another. The log-space injection
     is a heuristic bridge between two independent mechanisms.

  2. The similarity is based on comparing current representations,
     which are themselves still being learned. Early in training,
     all representations are close to random → similarity scores
     are meaningless → the reweighting provides no useful signal
     precisely when it's needed most.

═══════════════════════════════════════════════════════════════════
THE NEW APPROACH: MEMORY-ROUTED ATTENTION
═══════════════════════════════════════════════════════════════════

The key insight: node i's GRU hidden state h_i IS a compressed
summary of everything i has done so far. It already encodes "what
kind of node i is." We should let h_i directly CONTROL how much i
attends to each neighbor — without any separate similarity module.

Concretely, we replace the standard Q/K/V structure:

  OLD:  Q = linear(x_i)           ← query from current representation
        K = linear(x_j) + edge + time
        attn = softmax(Q·K / √d)

  NEW:  Q_content = linear(x_i)   ← what i looks like now (structural)
        Q_memory  = linear(h_i)   ← what i has been historically (temporal)
        Q = Q_content + Q_memory  ← fused query: now + history

        K_content = linear(x_j)   ← what j looks like now
        K_memory  = linear(h_j)   ← what j has been historically
        K = K_content + K_memory + edge + time

        attn = softmax((Q_content + Q_memory) · (K_content + K_memory) / √d)

WHY THIS WORKS:
  The dot product Q·K now decomposes into four terms:

    Q_content · K_content  →  do i and j look similar NOW?
    Q_content · K_memory   →  does j's history match i's current behavior?
    Q_memory  · K_content  →  does j's current behavior match i's history?
    Q_memory  · K_memory   →  do i and j have similar behavioral histories?

  The last term is exactly what we want: nodes with similar histories
  will have high Q_memory · K_memory, meaning they attend to each other
  more strongly — regardless of whether they are normal or anomalous.

  This is NOT a separate module injecting a bias. It is INSIDE the
  attention computation itself. The attention heads, the memory
  projections, and the representation learning all co-adapt together
  through backpropagation via the contrastive loss.

WHAT THIS GIVES:
  - Normal node i, normal neighbor j:
      h_i ≈ h_j (both have normal histories) → Q_mem·K_mem high → high attn
  - Anomalous node i, anomalous neighbor j:
      h_i ≈ h_j (both have anomalous histories) → Q_mem·K_mem high → high attn ✓
  - Normal node i, anomalous neighbor j:
      h_i ≉ h_j (different histories) → Q_mem·K_mem low → low attn ✓
  - Anomalous node i, normal neighbor j:
      h_i ≉ h_j → Q_mem·K_mem low → low attn ✓

  Each group (normal / anomalous) naturally learns from its own kind.
  No external similarity module. No heuristic log-space injection.
  No deactivation. Just attention doing what attention does, but now
  informed by temporal memory on both the query and key side.

ADDITIONAL: MEMORY-GATED RESIDUAL
  After aggregation, we control HOW MUCH the aggregated neighborhood
  context updates i's representation using a memory-derived gate:

    update_amount = sigmoid( linear(h_i) )   ← how open is i to change?

  A node with a stable, confident history (low GRU variance) should
  be harder to shift — the gate will learn to be more closed.
  A node encountering genuinely new neighbors should be more open.
  This replaces the fixed scalar gate from the baseline.

EVERYTHING ELSE is identical to the 91% baseline.
"""

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import MessagePassing
from torch_geometric.utils import softmax
from tqdm import tqdm


# ═══════════════════════════════════════════════════════════════════════════════
# 1. Time2Vec
# ═══════════════════════════════════════════════════════════════════════════════

class Time2Vec(nn.Module):
    def __init__(self, out_dim):
        super().__init__()
        k = out_dim - 1
        self.w0 = nn.Parameter(torch.randn(1) * 0.01)
        self.b0 = nn.Parameter(torch.zeros(1))
        self.W  = nn.Parameter(torch.randn(k) * 0.01)
        self.B  = nn.Parameter(torch.zeros(k))

    def forward(self, t):
        t = t.float().unsqueeze(-1)
        return torch.cat([t * self.w0 + self.b0,
                          torch.sin(t * self.W + self.B)], dim=-1)


class TimeEncoder(nn.Module):
    def __init__(self, hidden_dim):
        super().__init__()
        self.t2v     = Time2Vec(hidden_dim // 2)
        self.rel_t2v = Time2Vec(hidden_dim // 2)
        self.proj    = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, hidden_dim),
        )

    def forward(self, t_abs, t_rel=None):
        abs_enc = self.t2v(t_abs)
        rel_enc = self.rel_t2v(t_rel) if t_rel is not None else torch.zeros_like(abs_enc)
        return self.proj(torch.cat([abs_enc, rel_enc], dim=-1))


# ═══════════════════════════════════════════════════════════════════════════════
# 2. RecurrentMemory
# ═══════════════════════════════════════════════════════════════════════════════

class RecurrentMemory(nn.Module):
    """
    Per-node GRU hidden state + EMA variance.
    Low variance = stable node = confident memory.
    """
    def __init__(self, num_nodes, dim, momentum=0.9):
        super().__init__()
        self.gru      = nn.GRUCell(dim, dim)
        self.momentum = momentum
        self.register_buffer("hidden",   torch.zeros(num_nodes, dim))
        self.register_buffer("variance", torch.ones(num_nodes, dim) * 0.1)

    def read(self, idx):
        return self.hidden[idx], self.variance[idx]

    def get_hidden(self, idx):
        return self.hidden[idx]

    @torch.no_grad()
    def write(self, idx, x):
        h_old = self.hidden[idx]
        h_new = self.gru(x.detach(), h_old)
        delta = h_new - h_old
        self.variance[idx] = (self.momentum * self.variance[idx]
                              + (1 - self.momentum) * delta.pow(2))
        self.hidden[idx] = h_new


# ═══════════════════════════════════════════════════════════════════════════════
# 3. EvolutionBank
# ═══════════════════════════════════════════════════════════════════════════════

class EvolutionBank(nn.Module):
    def __init__(self, num_nodes, dim, window=6):
        super().__init__()
        self.window = window
        self.register_buffer("bank", torch.zeros(num_nodes, window, dim))
        self.register_buffer("ptr",  torch.zeros(num_nodes, dtype=torch.long))

    def read(self, idx):
        return self.bank[idx]                                          # (B, W, D)

    @torch.no_grad()
    def write(self, idx, emb):
        for b in range(idx.size(0)):
            p = int(self.ptr[idx[b]]) % self.window
            self.bank[idx[b], p] = emb[b].detach()
            self.ptr[idx[b]] += 1


# ═══════════════════════════════════════════════════════════════════════════════
# 4. Memory-Routed Temporal Attention  ← THE NEW MECHANISM
#
# How much node i learns from neighbor j is controlled by the dot product
# of their FUSED queries and keys, where each is the sum of:
#   - a content projection  (what the node looks like now)
#   - a memory projection   (what the node has been historically)
#
# The memory projections use separate learned linear layers so that
# the model can learn DIFFERENT subspaces for "current behavior" and
# "historical pattern" — they don't interfere with each other.
#
# Additionally, the residual gate is memory-driven:
#   gate = sigmoid(W_gate · h_i)
# so a node's openness to updating is a learned function of its history,
# not a fixed scalar or a function of the current (noisy) representation.
# ═══════════════════════════════════════════════════════════════════════════════

class MemoryRoutedAttention(MessagePassing):
    def __init__(self, dim, heads=8, dropout=0.1, window=6):
        super().__init__(aggr="add", node_dim=0)
        assert dim % heads == 0
        self.dim      = dim
        self.heads    = heads
        self.head_dim = dim // heads
        self.scale    = self.head_dim ** -0.5
        self.window   = window

        # ── Content projections (current representation x) ────────────────
        self.q_content = nn.Linear(dim, dim)
        self.k_content = nn.Linear(dim, dim)
        self.v_proj    = nn.Linear(dim, dim)    # values from current repr
        self.o_proj    = nn.Linear(dim, dim)

        # ── Memory projections (GRU hidden state h) ───────────────────────
        # Separate weights so content and memory learn different subspaces.
        self.q_memory  = nn.Linear(dim, dim)
        self.k_memory  = nn.Linear(dim, dim)

        # ── Edge and time enrichment (unchanged from baseline) ─────────────
        self.edge_proj = nn.Linear(dim, dim)
        self.time_proj = nn.Linear(dim, dim)

        self.dropout = nn.Dropout(dropout)

        # ── Memory-driven residual gate ────────────────────────────────────
        # gate = sigmoid(linear(h_i)) — openness to update from neighbors
        # Produces per-dimension gate so different dimensions can update
        # at different rates (more expressive than scalar gate).
        self.memory_gate = nn.Linear(dim, dim)

        # ── Fusion: aggregated context + own memory → output ──────────────
        self.norm   = nn.LayerNorm(dim)
        self.fusion = nn.Sequential(
            nn.Linear(dim * 2, dim),
            nn.GELU(),
            nn.Linear(dim, dim),
        )

    def split(self, x):
        return x.view(-1, self.heads, self.head_dim)

    def forward(self, x, edge_index, edge_emb, time_emb, memory):
        N      = x.size(0)
        device = x.device
        idx    = torch.arange(N, device=device)

        # Read GRU memory for all nodes in this batch
        mem_h, _ = memory.read(idx)                                    # (N, D)

        # Fused query: content + memory (per node)
        Q = self.split(self.q_content(x) + self.q_memory(mem_h))      # (N, h, hd)

        # Content key and value (per node — will be lifted to per-edge by PyG)
        K_c = self.split(self.k_content(x))                            # (N, h, hd)
        K_m = self.split(self.k_memory(mem_h))                         # (N, h, hd)
        V   = self.split(self.v_proj(x))                               # (N, h, hd)

        out = self.propagate(
            edge_index,
            Q=Q,
            K_c=K_c, K_m=K_m, V=V,
            mem_h=mem_h,
            edge_emb=self.split(self.edge_proj(edge_emb)),
            time_emb=self.split(self.time_proj(time_emb)),
            size=(N, N)
        )
        out = out.view(N, self.dim)

        # Fuse aggregated context with node's own memory
        out = self.fusion(torch.cat([out, mem_h], dim=-1))
        out = self.o_proj(out)

        # Memory-driven gate: how open is node i to incorporating neighbor info?
        # sigmoid(W·h_i) — a function of i's own history, not the noisy current x.
        gate = torch.sigmoid(self.memory_gate(mem_h))                  # (N, D)
        return self.norm(gate * x + (1.0 - gate) * out)

    def message(self, Q_i,
                K_c_j, K_m_j,
                V_j,
                mem_h_i, mem_h_j,
                edge_emb, time_emb,
                index):
        # ── Fused key: content + memory of neighbor j ─────────────────────
        K_j = K_c_j + K_m_j + edge_emb + time_emb                    # (E, h, hd)

        # ── Attention: Q_i · K_j decomposes into 4 terms ─────────────────
        # Q_i = Q_content_i + Q_memory_i  (already fused before split+propagate)
        # K_j = K_content_j + K_memory_j + edge + time
        #
        # The Q_memory_i · K_memory_j term ensures nodes with similar
        # behavioral histories attend to each other strongly, creating
        # the "learn from your own kind" effect without any external module.
        attn = (Q_i * K_j).sum(-1) * self.scale                       # (E, h)
        attn = softmax(attn, index)
        attn = self.dropout(attn)

        return V_j * attn.unsqueeze(-1)                                # (E, h, hd)


# ═══════════════════════════════════════════════════════════════════════════════
# 5. Full Model
# ═══════════════════════════════════════════════════════════════════════════════

class EnhancedTemporalGNN(nn.Module):
    def __init__(self, num_nodes, in_dim, edge_dim, hidden_dim,
                 num_layers=2, heads=8, dropout=0.1,
                 window=6, memory_momentum=0.9):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.num_nodes  = num_nodes

        self.input_proj = nn.Linear(in_dim, hidden_dim)
        self.edge_enc   = nn.Linear(edge_dim, hidden_dim)
        self.time_enc   = TimeEncoder(hidden_dim)

        self.memory   = RecurrentMemory(num_nodes, hidden_dim, momentum=memory_momentum)
        self.evo_bank = EvolutionBank(num_nodes, hidden_dim, window=window)

        self.layers = nn.ModuleList([
            MemoryRoutedAttention(hidden_dim, heads=heads,
                                  dropout=dropout, window=window)
            for _ in range(num_layers)
        ])

        self.final_norm = nn.LayerNorm(hidden_dim)
        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)

    def forward(self, batch):
        N          = batch.x.size(0)
        device     = batch.x.device
        edge_index = batch.edge_index
        src, dst   = edge_index

        x        = self.input_proj(batch.x)
        edge_emb = self.edge_enc(batch.msg)

        # Relative time gap (same as 91% baseline)
        node_last_t = torch.zeros(N, device=device)
        node_last_t.scatter_reduce_(
            0, src, batch.t.float(), reduce='amax', include_self=True)
        t_rel    = batch.t.float() - node_last_t[src]
        time_emb = self.time_enc(batch.t, t_rel)

        for layer in self.layers:
            x = layer(x, edge_index, edge_emb, time_emb, self.memory)

        x = self.final_norm(x)

        # Update memory
        self.memory.write(torch.arange(N, device=device), x)

        # Update evolution bank
        neigh = torch.zeros(N, self.hidden_dim, device=device)
        cnt   = torch.zeros(N, 1, device=device)
        neigh.scatter_add_(0, dst.unsqueeze(1).expand(-1, self.hidden_dim), x[src])
        cnt.scatter_add_(0, dst.unsqueeze(1),
                         torch.ones(src.size(0), 1, device=device))
        neigh = neigh / (cnt + 1e-6)
        unique_dst = dst.unique()
        self.evo_bank.write(unique_dst, neigh[unique_dst])

        return x


# ═══════════════════════════════════════════════════════════════════════════════
# 6. Contrastive Learning (identical to 91% baseline)
# ═══════════════════════════════════════════════════════════════════════════════

def info_nce(anchor, positive, negatives, tau=0.07):
    a = F.normalize(anchor,    dim=-1)
    p = F.normalize(positive,  dim=-1)
    n = F.normalize(negatives, dim=-1)
    pos_sim = (a * p).sum(-1) / tau
    neg_sim = torch.einsum('bd,bkd->bk', a, n) / tau
    logits  = torch.cat([pos_sim.unsqueeze(1), neg_sim], dim=1)
    labels  = torch.zeros(a.size(0), dtype=torch.long, device=a.device)
    return F.cross_entropy(logits, labels)


def hard_negatives(struct_out, mem_out, K=64):
    B = struct_out.size(0)
    K = min(K, B - 1)
    s = F.normalize(struct_out.detach(), dim=-1)
    m = F.normalize(mem_out.detach(),    dim=-1)
    score = s @ s.T - m @ m.T
    score.fill_diagonal_(-1e9)
    return score.topk(K, dim=-1).indices


# ═══════════════════════════════════════════════════════════════════════════════
# 7. Training (identical to 91% baseline)
# ═══════════════════════════════════════════════════════════════════════════════

def train(loader, model, history, optimizer, device, data, args):
    model.train()
    history.train()
    pbar       = tqdm(loader)
    total_loss = 0.0
    tau        = getattr(args, 'tau',     0.07)
    num_neg    = getattr(args, 'num_neg', 64)

    for batch in pbar:
        batch = batch.to(device)
        optimizer.zero_grad()

        out = model(batch)
        src = batch.src[: batch.batch_size]
        out = out[src]
        idx = batch.input_id.cpu()

        cur_mem, prev_mem = history(out, data.src[idx])

        with torch.no_grad():
            neg_idx = hard_negatives(out, cur_mem, K=num_neg)
        negatives = cur_mem[neg_idx]

        loss = info_nce(out, cur_mem, negatives, tau=tau)

        consist = (1.0 - F.cosine_similarity(
            cur_mem, prev_mem.detach(), dim=-1)).mean()
        loss = loss + 0.1 * consist

        loss.backward()
        torch.nn.utils.clip_grad_norm_(
            list(model.parameters()) + list(history.parameters()), 1.0)
        optimizer.step()

        total_loss += loss.item()
        pbar.set_description(f"Loss={loss.item():.4f}")

    return total_loss / len(loader)


# ═══════════════════════════════════════════════════════════════════════════════
# 8. Test (identical to 91% baseline)
# ═══════════════════════════════════════════════════════════════════════════════

@torch.no_grad()
def test(loader, model, history, measure, device, data):
    preds, labels = [], []
    model.eval()
    history.eval()
    eps = 1e-6

    for batch in tqdm(loader):
        torch.cuda.empty_cache()
        batch = batch.to(device)

        out   = model(batch)
        src   = batch.src[: batch.batch_size]
        out   = out[src]
        label = batch.y[: batch.batch_size]
        idx   = batch.input_id.cpu()

        cur_mem, prev_mem = history(out, data.src[idx], update=True)

        node_ids   = data.src[idx].to(device)
        _, cur_var = model.memory.read(node_ids)
        conf = 1.0 / (1.0 + cur_var.mean(-1).sqrt().clamp(min=eps))

        sim_sv  = F.cosine_similarity(out, cur_mem, dim=-1)
        score   = conf * (1.0 - sim_sv)

        sim_tv  = F.cosine_similarity(cur_mem, prev_mem, dim=-1)
        score_t = conf * (1.0 - sim_tv)

        evo     = model.evo_bank.read(node_ids)
        evo_div = 1.0 - F.cosine_similarity(evo[:, -1], evo[:, 0], dim=-1)
        score_e = evo_div.clamp(0, 2)

        pred = (score + 0.4 * score_t + 0.2 * score_e).sigmoid()

        torch.cuda.empty_cache()
        preds.append(pred.detach().cpu())
        labels.append(label.detach().cpu())

    preds  = torch.cat(preds)
    preds[torch.isnan(preds)] = 0.0
    labels = torch.cat(labels)
    return measure(labels[labels != 2], preds[labels != 2])

In [32]:
"""
Drop-in replacement for your notebook.
Keeps: data, loaders, History, args, measure
Replaces: model, train(), test()

Just run these cells in order after your existing setup cells.
"""

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import MessagePassing
from torch_geometric.utils import softmax
from tqdm import tqdm


# ── Paste the full enhanced_gnn_v2.py here, or import it ──────────────────────
# from enhanced_gnn_v2 import (EnhancedTemporalGNN, train, test)
# ──────────────────────────────────────────────────────────────────────────────
# If running directly in notebook, paste the class definitions above this cell.


# ═══════════════════════════════════════════
# CELL: Build model + history (unchanged)
# ═══════════════════════════════════════════

try:
    del model
except NameError:
    pass

model = EnhancedTemporalGNN(
    num_nodes  = data.num_nodes,
    in_dim     = data.x.size(1),          # 1 for wikipedia
    edge_dim   = data.msg.size(1),         # 172 for wikipedia
    hidden_dim = args.hidden_channels,     # 64
    num_layers = args.nums_layers,         # 2
    heads      = args.heads,               # 8
    dropout    = args.dropout,             # 0.3
    window     = 8,
    memory_momentum = 0.9,
).to(device)

# History is UNCHANGED — reuse your existing History class
try:
    del history
except NameError:
    pass

from mhiscl.history import History

history = History(
    data.num_nodes,
    args.num_timeslots,
    args.hidden_channels,
    device=device,
    history_retrieve=args.history_retrieve,
    recurrent=args.recurrent,
).to(device)

print(model)
print(history)
print(f"Model params: {sum(p.numel() for p in model.parameters()):,}")




EnhancedTemporalGNN(
  (input_proj): Linear(in_features=1, out_features=64, bias=True)
  (edge_enc): Linear(in_features=172, out_features=64, bias=True)
  (time_enc): TimeEncoder(
    (t2v): Time2Vec()
    (rel_t2v): Time2Vec()
    (proj): Sequential(
      (0): Linear(in_features=64, out_features=64, bias=True)
      (1): GELU(approximate='none')
      (2): Linear(in_features=64, out_features=64, bias=True)
    )
  )
  (memory): RecurrentMemory(
    (gru): GRUCell(64, 64)
  )
  (evo_bank): EvolutionBank()
  (layers): ModuleList(
    (0-1): 2 x MemoryRoutedAttention()
  )
  (final_norm): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
)
History(
  (recurrent_network): GRUCell(64, 64)
  (lin): Sequential(
    (0): Linear(-1, 64, bias=True)
    (1): ELU(alpha=1.0)
    (2): Linear(64, 64, bias=True)
  )
)
Model params: 144,704


In [52]:
## Training Function 

def train(loader, model, history, optimizer, device, data, args):
    model.train()
    history.train()
    pbar = tqdm(loader)
    total_loss = 0
    for batch in pbar:
        batch = batch.to(device)
        optimizer.zero_grad()
        out = model(batch)
        src = batch.src[: batch.batch_size]
        out = out[src]
        idx = batch.input_id.cpu()
        num_neg = batch.batch_size
        neg_mem = history.get_history(torch.randint(
            0, history.num_nodes, size=(num_neg,)))
        raw_msg = torch.zeros(idx.size(0),1).to(device) 
        cur_mem, prev_mem = history(out, data.src[idx]) 

        if args.con=='his':
            loss = contrastive_loss(prev_mem, cur_mem, args.tau) 
            loss += torch.exp(cosine_similarity(cur_mem, neg_mem) 
                          ).sum(dim=1).log().mean()
        elif args.con=='stru':
            loss = contrastive_loss(out, cur_mem, args.tau) 
            loss += torch.exp(cosine_similarity(out, neg_mem)
                            ).sum(dim=1).log().mean()
        elif args.con=='all':
            loss = args.alpha*contrastive_loss(out, cur_mem, args.tau) + \
                contrastive_loss(prev_mem, cur_mem, args.tau)
            loss += torch.exp(cosine_similarity(cur_mem, neg_mem) 
                            ).sum(dim=1).log().mean()
            loss += args.alpha*torch.exp(cosine_similarity(out, neg_mem) 
                            ).sum(dim=1).log().mean()


        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        pbar.set_description(f"Train Loss = {loss.item():.4f}")

    return total_loss / len(loader)


## Test function

@torch.no_grad()
def test(loader, model, history, measure, device, data):
    preds = []
    labels = []
    model.eval()
    history.eval()
    for batch in tqdm(loader):
        torch.cuda.empty_cache()
        batch = batch.to(device)
        out = model(batch)
        src = batch.src[: batch.batch_size]
        out = out[src]
        label = batch.y[: batch.batch_size]
        idx = batch.input_id.cpu()
        raw_msg = torch.zeros(idx.size(0),1).to(device) 
        cur_mem, prev_mem = history(out, data.src[idx], update=True)
        if args.con=='his':
            pred = 1-torch.diag(cosine_similarity(cur_mem, prev_mem)).view(-1)
        elif args.con=='stru': 
            pred = 1-torch.diag(cosine_similarity(out, cur_mem)).view(-1)
        elif args.con=='all':
            p1 = torch.diag(cosine_similarity(out, cur_mem)).view(-1)
            p2 = torch.diag(cosine_similarity(cur_mem, prev_mem)).view(-1)
            pred = (1 - p1) + (1 - p2) 
            
        pred = pred.sigmoid() 
        torch.cuda.empty_cache()
        preds.append(pred.detach().cpu())
        labels.append(label.detach().cpu())
    preds = torch.cat(preds)
    preds[torch.isnan(preds)] = 0.0 
    labels = torch.cat(labels)
    return measure(labels[labels!=2], preds[labels!=2]) 




In [33]:
# ═══════════════════════════════════════════
# CELL: Training loop (drop-in replacement)
# ═══════════════════════════════════════════

best     = 0
best_val = 0

optimizer = torch.optim.Adam(
    list(model.parameters()) + list(history.parameters()),
    lr=args.lr,
    weight_decay=1e-5       # small L2 reg helps generalization
)

# Cosine LR schedule: smoothly decays over training
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=args.epochs, eta_min=args.lr * 0.1)

measure = Measure("auc")

for epoch in range(1, 1 + args.epochs):
    loss    = train(train_loader, model, history, optimizer, device, data, args)
    val_auc = test(valid_loader,  model, history, measure,   device, data)
    test_auc = test(test_loader,  model, history, measure,   device, data)
    scheduler.step()

    if val_auc > best_val:
        best_val = val_auc
        best     = test_auc

    print(
        f"Epoch: {epoch:03d}, Loss: {loss:.4f}, "
        f"Val AUC: {val_auc:.2%}, Test AUC: {test_auc:.2%}, Best AUC: {best:.2%}"
    )

100%|██████████| 93/93 [00:04<00:00, 22.03it/s]


Epoch: 001, Loss: 3.5641, Val AUC: 75.91%, Test AUC: 76.92%, Best AUC: 76.92%


100%|██████████| 93/93 [00:04<00:00, 22.71it/s]


Epoch: 002, Loss: 3.5486, Val AUC: 76.01%, Test AUC: 74.19%, Best AUC: 74.19%


100%|██████████| 93/93 [00:04<00:00, 22.26it/s]


Epoch: 003, Loss: 2.9148, Val AUC: 16.79%, Test AUC: 19.77%, Best AUC: 74.19%


100%|██████████| 93/93 [00:04<00:00, 21.57it/s]


Epoch: 004, Loss: 2.7958, Val AUC: 22.00%, Test AUC: 30.05%, Best AUC: 74.19%


Loss=2.5644:  26%|██▌       | 110/431 [00:08<00:25, 12.65it/s]


KeyboardInterrupt: 

In [ ]:
"""
Temporal Snapshot GNN — v8  (clean, fully vectorized)
=======================================================

Pipeline for every node n at time t
─────────────────────────────────────────────────────

  ┌──────────────────────────────────────────────────────────────────┐
  │  STEP 1 — K-EVENT SNAPSHOT BANK                                  │
  │                                                                  │
  │  A circular buffer of K slots per node. Each slot stores:        │
  │    • neigh_emb  (D,)  — projected embedding of the neighbor      │
  │    • neigh_mem  (D,)  — GRU memory of the neighbor at that time  │
  │    • edge_emb   (D,)  — projected edge features                  │
  │    • timestamp  scalar — when the event happened                 │
  │                                                                  │
  │  Written after every forward pass (new events → new slots).      │
  │  Read before forward pass → gives n its personal history.        │
  └──────────────────────────────────────────────────────────────────┘
                           ↓  (B, K, D) × 3  +  (B, K) timestamps
  ┌──────────────────────────────────────────────────────────────────┐
  │  STEP 2 — PER-EVENT TRUST-WEIGHTED SNAPSHOT EMBEDDING            │
  │                                                                  │
  │  For each of n's K stored events, compute a snapshot vector s_k: │
  │                                                                  │
  │    trust_k = σ( MLP( [cos(mem_n, neigh_mem_k),                  │
  │                        conf_n, conf_neighbor_k] ) )              │
  │                                                                  │
  │    s_k = trust_k · neigh_emb_k  +  (1 - trust_k) · edge_emb_k  │
  │                                                                  │
  │  trust uses MEMORY SPACE (stable GRU states), not current repr.  │
  │  High trust → learn from neighbor content.                       │
  │  Low trust  → fall back to edge content. Nothing is zeroed.      │
  │                                                                  │
  │  Anomalous n: high trust with anomalous neighbors → absorbs their │
  │  signal. Normal n: high trust with normal neighbors → absorbs    │
  │  theirs. Each group learns from its own kind automatically.      │
  └──────────────────────────────────────────────────────────────────┘
                           ↓  snapshots (B, K, D)
  ┌──────────────────────────────────────────────────────────────────┐
  │  STEP 3 — TIME-MODULATED GRU  (hierarchical recurrence)         │
  │                                                                  │
  │  Process [s_1, s_2, ..., s_K] in chronological order.           │
  │  Between step k-1 and k, the time gap Δt_k is encoded by        │
  │  Time2Vec → a learned D-dimensional DECAY GATE:                  │
  │                                                                  │
  │    φ(Δt_k)  = Time2Vec(normalize(Δt_k))      (D_t,)             │
  │    decay_k  = sigmoid( Linear(φ(Δt_k)) )     (D,) per dimension │
  │    h_k      = GRU( s_k,  decay_k ⊙ h_{k-1} )                    │
  │                                                                  │
  │  Small Δt → decay ≈ 1 → strong continuity from previous step     │
  │  Large Δt → decay ≈ 0 → reset toward current event               │
  │                                                                  │
  │  The decay is LEARNED: the model discovers which dimensions of   │
  │  the hidden state matter across long gaps vs short gaps.         │
  │                                                                  │
  │  Initial hidden state h_0 = n's global long-term memory.         │
  │  → LOCAL GRU (K steps) + GLOBAL MEMORY = two-level hierarchy.    │
  └──────────────────────────────────────────────────────────────────┘
                           ↓  all_h (B, K, D),  h_K (B, D)
  ┌──────────────────────────────────────────────────────────────────┐
  │  STEP 4 — RECENCY-CONTENT ATTENTION POOLING                      │
  │                                                                  │
  │    α_k = softmax_k( MLP(h_k) + recency_bias_k )                 │
  │    z_n = Σ_k  α_k · h_k                                         │
  │                                                                  │
  │  z_n is the final node representation. It encodes:               │
  │    what n has done × who it trusted × when it acted              │
  └──────────────────────────────────────────────────────────────────┘
                           ↓  z_n  (N, D)  — returned
  ┌──────────────────────────────────────────────────────────────────┐
  │  CONTRASTIVE LEARNING                                            │
  │                                                                  │
  │  Anchor   = z_n  (K-snapshot view, current behavior)            │
  │  Positive = global memory h_n (full historical view)            │
  │  Normal n:    z_n ≈ h_n  → low anomaly score                    │
  │  Anomalous n: z_n ≉ h_n  → high anomaly score                   │
  └──────────────────────────────────────────────────────────────────┘

NaN safety
───────────
  • Running normalization of timestamps before Time2Vec
  • safe_norm / safe_cos for all cosine operations
  • All scatter ops vectorized — no Python loops over nodes
  • Logit clamping in InfoNCE
"""

import torch
import torch.nn as nn
import torch.nn.functional as F
from tqdm import tqdm

EPS = 1e-8

# ── Safe math ────────────────────────────────────────────────────────────────

def safe_norm(x):
    return x / x.norm(dim=-1, keepdim=True).clamp(min=EPS)

def safe_cos(a, b):
    return (safe_norm(a) * safe_norm(b)).sum(-1).clamp(-1., 1.)


# ═══════════════════════════════════════════════════════════════════════════════
# 1.  Running normalizer  —  keeps Time2Vec inputs in a sane range
#     Uses online EMA so it works across batches without any extra data pass.
# ═══════════════════════════════════════════════════════════════════════════════

class RunningNorm(nn.Module):
    def __init__(self, momentum=0.01):
        super().__init__()
        self.momentum = momentum
        self.register_buffer("mean", torch.tensor(0.0))
        self.register_buffer("std",  torch.tensor(1.0))
        self.register_buffer("init", torch.tensor(False))

    @torch.no_grad()
    def update(self, x):
        x = x.float().reshape(-1)
        if not self.init.item():
            self.mean.copy_(x.mean())
            self.std.copy_(x.std().clamp(min=1.0))
            self.init.fill_(True)
        else:
            m = self.momentum
            self.mean = (1-m)*self.mean + m*x.mean()
            self.std  = (1-m)*self.std  + m*x.std().clamp(min=1.0)

    def normalize(self, x):
        return (x.float() - self.mean) / self.std.clamp(min=1.0)


# ═══════════════════════════════════════════════════════════════════════════════
# 2.  Time2Vec
#     φ(t) = [w0·t+b0, sin(w1·t+b1), …, sin(wk·t+bk)]
#     Input must be normalized (handled by RunningNorm above).
# ═══════════════════════════════════════════════════════════════════════════════

class Time2Vec(nn.Module):
    def __init__(self, out_dim):
        super().__init__()
        k = out_dim - 1
        self.w0 = nn.Parameter(torch.ones(1) * 0.1)
        self.b0 = nn.Parameter(torch.zeros(1))
        self.W  = nn.Parameter(torch.randn(k) * 0.1)
        self.B  = nn.Parameter(torch.zeros(k))

    def forward(self, t):
        """t: (N,) normalized scalar  →  (N, out_dim)"""
        t = t.float().unsqueeze(-1)
        return torch.cat([t * self.w0 + self.b0,
                          torch.sin(t * self.W + self.B)], dim=-1)


# ═══════════════════════════════════════════════════════════════════════════════
# 3.  K-Event Snapshot Bank  (fully vectorized, no Python loops over nodes)
#
#     Circular buffer: for each node keeps K slots, each containing
#       neigh_emb  (D,)   neighbor projected embedding
#       neigh_mem  (D,)   neighbor GRU memory state at that moment
#       edge_emb   (D,)   edge projected embedding
#       timestamp  scalar
#
#     write(): scatter-based, handles one edge per call or many at once.
#     read():  returns ordered (oldest→newest) tensors for the GRU.
# ═══════════════════════════════════════════════════════════════════════════════

class SnapshotBank(nn.Module):
    def __init__(self, num_nodes, dim, K):
        super().__init__()
        self.K   = K
        self.dim = dim
        N        = num_nodes

        # Circular buffer contents
        self.register_buffer("neigh_emb", torch.zeros(N, K, dim))
        self.register_buffer("neigh_mem", torch.zeros(N, K, dim))
        self.register_buffer("edge_emb",  torch.zeros(N, K, dim))
        self.register_buffer("ts",        torch.zeros(N, K))

        # Circular pointer and fill count per node
        self.register_buffer("ptr",    torch.zeros(N, dtype=torch.long))
        self.register_buffer("filled", torch.zeros(N, dtype=torch.long))

    # ── Write ────────────────────────────────────────────────────────────────
    @torch.no_grad()
    def write(self, node_idx, neigh_emb, neigh_mem, edge_emb, timestamps):
        """
        Write one new event per entry in node_idx.
        Handles duplicate node_idx (multiple edges for the same node in a
        batch) by processing them sequentially using unique ordering.

        node_idx:   (E,)  — may contain duplicates
        neigh_emb:  (E, D)
        neigh_mem:  (E, D)
        edge_emb:   (E, D)
        timestamps: (E,)
        """
        # Process each unique node; for duplicates keep the LAST occurrence
        # (highest timestamp within the batch — most recent event)
        _, last_occ = torch.unique(node_idx, return_inverse=True)
        # Build a mapping: unique node → its last edge index in this batch
        u_nodes, inv = torch.unique(node_idx, return_inverse=True)
        # last occurrence: for each unique node, the highest index in E
        last_idx = torch.zeros(u_nodes.size(0), dtype=torch.long,
                               device=node_idx.device)
        src_pos  = torch.arange(node_idx.size(0), device=node_idx.device)
        last_idx.scatter_(0, inv, src_pos)   # keeps last (highest) index

        # Gather the data for the unique nodes
        n   = u_nodes                                     # (U,)
        ne  = neigh_emb[last_idx]                         # (U, D)
        nm  = neigh_mem[last_idx]                         # (U, D)
        ee  = edge_emb[last_idx]                          # (U, D)
        ts  = timestamps[last_idx].float()                # (U,)

        # Write slot = ptr[n] % K  (vectorized)
        p   = self.ptr[n] % self.K                        # (U,)

        # Scatter into circular buffer
        self.neigh_emb[n, p] = ne
        self.neigh_mem[n, p] = nm
        self.edge_emb[n, p]  = ee
        self.ts[n, p]        = ts

        self.ptr[n]    += 1
        self.filled[n]  = (self.filled[n] + 1).clamp(max=self.K)

    # ── Read (chronological order) ────────────────────────────────────────────
    def read(self, node_idx):
        """
        Returns K slots ordered oldest → newest for each node.

        Returns:
          ne:   (B, K, D)  neighbor embeddings
          nm:   (B, K, D)  neighbor memory states
          ee:   (B, K, D)  edge embeddings
          ts:   (B, K)     timestamps
          mask: (B, K)     True = valid slot
        """
        B   = node_idx.size(0)
        K   = self.K
        dev = node_idx.device

        ne_raw  = self.neigh_emb[node_idx]                # (B, K, D)
        nm_raw  = self.neigh_mem[node_idx]                # (B, K, D)
        ee_raw  = self.edge_emb[node_idx]                 # (B, K, D)
        ts_raw  = self.ts[node_idx]                       # (B, K)
        fill    = self.filled[node_idx]                   # (B,)
        ptr     = self.ptr[node_idx] % K                  # (B,)

        # ── Chronological reorder (vectorized) ──────────────────────────────
        # For a full buffer (fill==K), oldest slot = ptr (wrap-around).
        # For a partial buffer (fill<K), oldest slot = 0.
        # We build a permutation index (B, K) and use gather.
        slots = torch.arange(K, device=dev).unsqueeze(0).expand(B, -1)  # (B, K)
        full  = fill >= K                                                 # (B,)

        # Shift: for full nodes, rotate left by ptr so oldest = index 0
        shift = torch.where(full, ptr, torch.zeros_like(ptr))            # (B,)
        order = (slots + shift.unsqueeze(1)) % K                         # (B, K)

        expand = order.unsqueeze(-1).expand(-1, -1, self.dim)
        ne   = ne_raw.gather(1, expand)
        nm   = nm_raw.gather(1, expand)
        ee   = ee_raw.gather(1, expand)
        ts   = ts_raw.gather(1, order)

        # Validity mask: first fill[b] slots are valid
        mask = slots < fill.unsqueeze(1)                                  # (B, K)

        return ne, nm, ee, ts, mask


# ═══════════════════════════════════════════════════════════════════════════════
# 4.  Per-Event Trust Module
#
#     For each of the K stored events, weights the neighbor's contribution
#     by how similar the neighbor was to n in MEMORY SPACE at that time.
#
#     trust_k  = σ( MLP( [cos(mem_n, neigh_mem_k), conf_n, conf_nbr_k] ) )
#     s_k      = trust_k · neigh_emb_k  +  (1−trust_k) · edge_emb_k
#
#     Fully batched over (B, K) without any Python loops.
# ═══════════════════════════════════════════════════════════════════════════════

class PerEventTrust(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.neigh_proj = nn.Linear(dim, dim)
        self.edge_proj  = nn.Linear(dim, dim)

        # 3 scalars → trust logit
        self.trust_mlp  = nn.Sequential(
            nn.Linear(3, 32),
            nn.GELU(),
            nn.Linear(32, 1),
        )
        self.out_norm   = nn.LayerNorm(dim)

    def forward(self, ne, nm, ee, mem_n, var_n):
        """
        ne:    (B, K, D)  neighbor embeddings
        nm:    (B, K, D)  neighbor memory states at each stored event
        ee:    (B, K, D)  edge embeddings
        mem_n: (B, D)     n's current global memory state
        var_n: (B, D)     n's memory variance (stability signal)

        Returns:
          s:     (B, K, D)  trust-weighted snapshot embeddings
          trust: (B, K)     per-event trust values (for inspection)
        """
        B, K, D = ne.shape

        # ── (a) Historical similarity in memory space ─────────────────────
        # cos( mem_n, neigh_mem_k ) — how similar was the neighbor to n
        # in terms of long-term behavior at the time of each event
        mem_n_exp = mem_n.unsqueeze(1).expand(-1, K, -1)               # (B, K, D)
        cos_sim   = safe_cos(
            mem_n_exp.reshape(B*K, D),
            nm.reshape(B*K, D)
        ).view(B, K)                                                    # (B, K) ∈ [-1,1]
        cos_01    = (cos_sim + 1.0) / 2.0                              # (B, K) ∈ [0,1]

        # ── (b) Confidence of n and of each stored neighbor ───────────────
        # Low variance = stable memory = confident similarity estimate
        conf_n   = (1.0 / (1.0 + var_n.mean(-1, keepdim=True)
                            .sqrt().clamp(min=EPS))).expand(-1, K)     # (B, K)
        # Approximate neighbor variance with a scalar 0.1 (not stored per-slot)
        # This is sufficient: trust is dominated by cos_sim anyway
        conf_nbr = torch.full_like(cos_01, 0.5)                        # (B, K)

        # ── (c) Trust score ───────────────────────────────────────────────
        signals = torch.stack([cos_01, conf_n, conf_nbr], dim=-1)     # (B, K, 3)
        trust   = torch.sigmoid(
            self.trust_mlp(signals.reshape(B*K, 3)).view(B, K))       # (B, K)

        # ── (d) Weighted snapshot ─────────────────────────────────────────
        w       = trust.unsqueeze(-1)                                  # (B, K, 1)
        s       = w * self.neigh_proj(ne) + (1.0-w) * self.edge_proj(ee)
        s       = self.out_norm(s)                                     # (B, K, D)

        return s, trust


# ═══════════════════════════════════════════════════════════════════════════════
# 5.  Time-Modulated GRU
#
#     Processes K snapshot embeddings chronologically.
#     Between step k-1 and step k, the gap Δt_k is encoded as a
#     learned D-dimensional DECAY gate that scales the previous hidden state:
#
#       φ(Δt_k) = Time2Vec( normalize(Δt_k) )   →  (D_t,)
#       decay_k = σ( Linear(φ(Δt_k)) )          →  (D,)   ∈ (0,1)
#       h_k     = GRUCell( s_k,  decay_k ⊙ h_{k-1} )
#
#     The GRUCell still operates normally — the decay acts on the
#     RECURRENT INPUT h_{k-1}, making the forgetting time-dependent.
#
#     Fully vectorized: processes all B nodes in parallel.
#     Invalid slots (mask=False) leave h unchanged.
# ═══════════════════════════════════════════════════════════════════════════════

class TimeModulatedGRU(nn.Module):
    def __init__(self, dim, time_dim=32):
        super().__init__()
        self.gru  = nn.GRUCell(dim, dim)
        self.t2v  = Time2Vec(time_dim)
        self.tnorm = RunningNorm()

        # φ(Δt) → per-dimension decay gate
        self.decay_gate = nn.Sequential(
            nn.Linear(time_dim, dim),
            nn.LayerNorm(dim),
        )

    def forward(self, s, ts, mask, h0):
        """
        s:    (B, K, D)  snapshot embeddings (oldest → newest)
        ts:   (B, K)     timestamps (oldest → newest)
        mask: (B, K)     True = valid slot
        h0:   (B, D)     initial hidden state (global long-term memory)

        Returns:
          all_h: (B, K, D)  hidden state after each step
          h_K:   (B, D)     final hidden state
        """
        B, K, D = s.shape
        h       = h0.clone()
        all_h   = []

        for k in range(K):
            s_k   = s[:, k, :]                                         # (B, D)
            valid = mask[:, k]                                          # (B,) bool

            if k == 0:
                # No previous timestamp → no decay at first step
                decay = torch.ones(B, D, device=s.device)
            else:
                dt   = (ts[:, k] - ts[:, k-1]).clamp(min=0.0)         # (B,)
                if self.training:
                    self.tnorm.update(dt)
                dt_n  = self.tnorm.normalize(dt)                       # (B,) normalized
                phi   = self.t2v(dt_n)                                 # (B, time_dim)
                decay = torch.sigmoid(self.decay_gate(phi))            # (B, D)
                # Large Δt → decay small → h_{k-1} fades → current event dominates
                # Small Δt → decay large → strong continuity

            h_in  = decay * h                                          # (B, D) decayed prev state
            h_new = self.gru(s_k, h_in)                                # (B, D)

            # Only update h for nodes with a valid snapshot at step k
            h = torch.where(valid.unsqueeze(-1), h_new, h)
            all_h.append(h)

        all_h = torch.stack(all_h, dim=1)                              # (B, K, D)
        return all_h, h


# ═══════════════════════════════════════════════════════════════════════════════
# 6.  Snapshot Attention Pooling
#
#     Combines K hidden states into one vector z_n using learned attention
#     that balances content informativeness and recency.
#
#       score_k = MLP(h_k)  +  recency_k       (recency is a learned scalar)
#       α       = softmax over valid slots
#       z_n     = Σ_k  α_k · h_k
# ═══════════════════════════════════════════════════════════════════════════════

class SnapshotPooling(nn.Module):
    def __init__(self, dim, K):
        super().__init__()
        self.score = nn.Sequential(
            nn.Linear(dim, dim // 2),
            nn.GELU(),
            nn.Linear(dim // 2, 1),
        )
        # One learned recency weight per snapshot position
        # Initialized so newer slots (higher index after reorder) get more weight
        self.recency = nn.Parameter(torch.linspace(-1.0, 1.0, K))     # (K,) learned

    def forward(self, all_h, mask):
        """
        all_h: (B, K, D)
        mask:  (B, K) bool

        Returns: (B, D)
        """
        content = self.score(all_h).squeeze(-1)                        # (B, K)
        scores  = content + self.recency.unsqueeze(0)                  # (B, K)
        scores  = scores.masked_fill(~mask, -1e9)
        alpha   = torch.softmax(scores, dim=-1)                        # (B, K)
        return (alpha.unsqueeze(-1) * all_h).sum(1)                   # (B, D)


# ═══════════════════════════════════════════════════════════════════════════════
# 7.  Global Node Memory (long-term, persistent across all batches)
#
#     One GRU hidden state per node, updated each time the node is seen.
#     Provides:
#       (a) initial hidden state h0 for the local Time-Modulated GRU
#           → creates the two-level hierarchy
#       (b) temporal view for contrastive learning
#       (c) trust reference (n's stable long-term embedding)
# ═══════════════════════════════════════════════════════════════════════════════

class NodeMemory(nn.Module):
    def __init__(self, num_nodes, dim, momentum=0.9):
        super().__init__()
        self.gru      = nn.GRUCell(dim, dim)
        self.momentum = momentum
        self.register_buffer("hidden",   torch.zeros(num_nodes, dim))
        self.register_buffer("variance", torch.ones(num_nodes, dim) * 0.1)

    def read(self, idx):
        return self.hidden[idx], self.variance[idx]                    # (B,D), (B,D)

    @torch.no_grad()
    def write(self, idx, x):
        h_old = self.hidden[idx]
        h_new = self.gru(x.detach(), h_old)
        delta = h_new - h_old
        self.variance[idx] = (self.momentum * self.variance[idx]
                              + (1-self.momentum) * delta.pow(2))
        self.hidden[idx] = h_new


# ═══════════════════════════════════════════════════════════════════════════════
# 8.  Full Model
# ═══════════════════════════════════════════════════════════════════════════════

class TemporalSnapshotGNN(nn.Module):
    def __init__(self, num_nodes, in_dim, edge_dim, hidden_dim,
                 K=8, time_dim=32, dropout=0.1, memory_momentum=0.9):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.K          = K

        # Input projections
        self.node_proj = nn.Sequential(
            nn.Linear(in_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
        )
        self.edge_proj = nn.Sequential(
            nn.Linear(edge_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
        )

        # Components (in pipeline order)
        self.bank     = SnapshotBank(num_nodes, hidden_dim, K=K)
        self.trust    = PerEventTrust(hidden_dim)
        self.tm_gru   = TimeModulatedGRU(hidden_dim, time_dim=time_dim)
        self.pooling  = SnapshotPooling(hidden_dim, K=K)
        self.memory   = NodeMemory(num_nodes, hidden_dim, momentum=memory_momentum)

        self.final_norm = nn.LayerNorm(hidden_dim)
        self.drop       = nn.Dropout(dropout)
        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)

    def forward(self, batch):
        """
        batch.x:          (N, in_dim)
        batch.msg:        (E, edge_dim)
        batch.t:          (E,) timestamps
        batch.edge_index: (2, E)

        Returns: (N, hidden_dim)  — one representation per node, nothing else.
        """
        N      = batch.x.size(0)
        device = batch.x.device
        src, dst = batch.edge_index
        all_idx  = torch.arange(N, device=device)

        # ── Project raw features ──────────────────────────────────────────
        x_proj   = self.node_proj(batch.x)                            # (N, D)
        e_proj   = self.edge_proj(batch.msg)                          # (E, D)

        # ── Identify active source nodes ──────────────────────────────────
        # unique_src: nodes that took an action in this batch
        unique_src = src.unique()                                      # (U,)
        B = unique_src.size(0)

        # ── STEP 1: Read K stored snapshots for active nodes ──────────────
        ne, nm, ee, ts, mask = self.bank.read(unique_src)
        # ne, nm, ee: (B, K, D)   ts: (B, K)   mask: (B, K)

        # ── STEP 2: Per-event trust → snapshot embeddings ─────────────────
        mem_n, var_n = self.memory.read(unique_src)                    # (B, D) each
        s, trust     = self.trust(ne, nm, ee, mem_n, var_n)           # (B, K, D), (B, K)

        # ── STEP 3: Time-modulated GRU over K snapshots ───────────────────
        # h0 = global long-term memory — seeds local GRU (hierarchy Level 1→2)
        all_h, h_K = self.tm_gru(s, ts, mask, h0=mem_n)              # (B,K,D), (B,D)

        # ── STEP 4: Attention pooling → z_n ──────────────────────────────
        z_active = self.pooling(all_h, mask)                           # (B, D)
        z_active = self.drop(self.final_norm(z_active))

        # ── Assemble full (N, D) output ───────────────────────────────────
        # Nodes not in unique_src (dst-only) fall back to their memory state
        z = self.memory.hidden[all_idx].clone()                        # (N, D) default
        z[unique_src] = z_active                                       # (U, D) overwrite

        # ── STEP 5: Write new snapshots into bank ─────────────────────────
        # For each edge src→dst, store dst's current embedding and memory
        # as a new event for src.  Uses vectorized write (handles duplicates).
        dst_emb = x_proj[dst]                                          # (E, D)
        dst_mem = self.memory.hidden[dst]                              # (E, D)
        self.bank.write(src, dst_emb, dst_mem, e_proj, batch.t.float())

        # ── STEP 6: Update global memory with h_K ─────────────────────────
        # h_K is the local GRU summary of the K most recent events.
        # Feeding it into the global GRU creates the hierarchy:
        #   Level 1 (local):  Time-Modulated GRU over K snapshots
        #   Level 2 (global): NodeMemory GRU across all batches
        self.memory.write(unique_src, h_K)

        return z                                                        # (N, hidden_dim)


# ═══════════════════════════════════════════════════════════════════════════════
# 9.  Contrastive Learning
# ═══════════════════════════════════════════════════════════════════════════════

def info_nce(anchor, positive, negatives, tau=0.07):
    """
    anchor:    (B, D)    z_n  — K-snapshot view
    positive:  (B, D)    h_n  — global memory view
    negatives: (B, K, D) other nodes' snapshot views
    """
    a = safe_norm(anchor)
    p = safe_norm(positive)
    n = safe_norm(negatives)
    pos_sim = (a * p).sum(-1) / tau
    neg_sim = torch.einsum('bd,bkd->bk', a, n) / tau
    logits  = torch.cat([pos_sim.unsqueeze(1), neg_sim], dim=1).clamp(-50, 50)
    labels  = torch.zeros(a.size(0), dtype=torch.long, device=a.device)
    return F.cross_entropy(logits, labels)


def hard_negatives(z, h, K=64):
    """
    Hard negatives: nodes similar NOW but different historically.
    Returns (B, K) index tensor.
    """
    B  = z.size(0)
    K  = min(K, B-1)
    sz = safe_norm(z.detach())
    sh = safe_norm(h.detach())
    score = sz @ sz.T - sh @ sh.T
    score.fill_diagonal_(-1e9)
    return score.topk(K, dim=-1).indices


# ═══════════════════════════════════════════════════════════════════════════════
# 10.  Training
# ═══════════════════════════════════════════════════════════════════════════════

def train(loader, model, history, optimizer, device, data, args):
    model.train()
    history.train()
    pbar    = tqdm(loader)
    total   = 0.0
    tau     = getattr(args, 'tau',     0.07)
    num_neg = getattr(args, 'num_neg', 64)

    for batch in pbar:
        batch = batch.to(device)
        optimizer.zero_grad()

        z   = model(batch)                                             # (N, D)
        src = batch.src[:batch.batch_size]
        z   = z[src]                                                   # (B, D)
        idx = batch.input_id.cpu()

        cur_mem, prev_mem = history(z, data.src[idx])

        with torch.no_grad():
            neg_idx = hard_negatives(z, cur_mem, K=num_neg)
        negatives = z[neg_idx]                                         # (B, K, D)

        loss = info_nce(z, cur_mem, negatives, tau=tau)
        consist = (1.0 - F.cosine_similarity(
            cur_mem, prev_mem.detach(), dim=-1)).mean()
        loss = loss + 0.1 * consist

        if not torch.isfinite(loss):
            optimizer.zero_grad()
            continue

        loss.backward()
        nn.utils.clip_grad_norm_(
            list(model.parameters()) + list(history.parameters()), 1.0)
        optimizer.step()

        total += loss.item()
        pbar.set_description(f"Loss={loss.item():.4f}")

    return total / max(len(loader), 1)


# ═══════════════════════════════════════════════════════════════════════════════
# 11.  Test / Anomaly Scoring
# ═══════════════════════════════════════════════════════════════════════════════

@torch.no_grad()
def test(loader, model, history, measure, device, data):
    preds, labels_all = [], []
    model.eval()
    history.eval()
    eps = 1e-6

    for batch in tqdm(loader):
        torch.cuda.empty_cache()
        batch = batch.to(device)

        z     = model(batch)
        src   = batch.src[:batch.batch_size]
        z     = z[src]
        label = batch.y[:batch.batch_size]
        idx   = batch.input_id.cpu()

        cur_mem, prev_mem = history(z, data.src[idx], update=True)

        node_ids  = data.src[idx].to(device)
        _, cur_var = model.memory.read(node_ids)
        conf = 1.0 / (1.0 + cur_var.mean(-1).sqrt().clamp(min=eps))

        score   = conf * (1.0 - F.cosine_similarity(z, cur_mem, dim=-1))
        score_t = conf * (1.0 - F.cosine_similarity(cur_mem, prev_mem, dim=-1))
        pred    = (score + 0.4 * score_t).sigmoid()

        preds.append(pred.detach().cpu())
        labels_all.append(label.detach().cpu())
        torch.cuda.empty_cache()

    preds      = torch.cat(preds)
    preds[torch.isnan(preds)] = 0.0
    labels_all = torch.cat(labels_all)
    return measure(labels_all[labels_all != 2], preds[labels_all != 2])

In [44]:
"""
Drop-in replacement for your notebook.
Keeps: data, loaders, History, args, measure
Replaces: model, train(), test()

Just run these cells in order after your existing setup cells.
"""

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import MessagePassing
from torch_geometric.utils import softmax
from tqdm import tqdm


# ── Paste the full enhanced_gnn_v2.py here, or import it ──────────────────────
# from enhanced_gnn_v2 import (EnhancedTemporalGNN, train, test)
# ──────────────────────────────────────────────────────────────────────────────
# If running directly in notebook, paste the class definitions above this cell.


# ═══════════════════════════════════════════
# CELL: Build model + history (unchanged)
# ═══════════════════════════════════════════

try:
    del model
except NameError:
    pass

model = TemporalSnapshotGNN(
    num_nodes  = data.num_nodes,
    in_dim     = data.x.size(1),          # 1 for wikipedia
    edge_dim   = data.msg.size(1),         # 172 for wikipedia
    hidden_dim = args.hidden_channels,     # 64        # 2            # 8
    dropout    = args.dropout,             # 0.3
    K     = 8,
    memory_momentum = 0.9,
).to(device)

# History is UNCHANGED — reuse your existing History class
try:
    del history
except NameError:
    pass

from mhiscl.history import History

history = History(
    data.num_nodes,
    args.num_timeslots,
    args.hidden_channels,
    device=device,
    history_retrieve=args.history_retrieve,
    recurrent=args.recurrent,
).to(device)

print(model)
print(history)
print(f"Model params: {sum(p.numel() for p in model.parameters()):,}")




TemporalSnapshotGNN(
  (node_proj): Sequential(
    (0): Linear(in_features=1, out_features=64, bias=True)
    (1): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
  )
  (edge_proj): Sequential(
    (0): Linear(in_features=172, out_features=64, bias=True)
    (1): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
  )
  (bank): SnapshotBank()
  (trust): PerEventTrust(
    (neigh_proj): Linear(in_features=64, out_features=64, bias=True)
    (edge_proj): Linear(in_features=64, out_features=64, bias=True)
    (trust_mlp): Sequential(
      (0): Linear(in_features=3, out_features=32, bias=True)
      (1): GELU(approximate='none')
      (2): Linear(in_features=32, out_features=1, bias=True)
    )
    (out_norm): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
  )
  (tm_gru): TimeModulatedGRU(
    (gru): GRUCell(64, 64)
    (t2v): Time2Vec()
    (tnorm): RunningNorm()
    (decay_gate): Sequential(
      (0): Linear(in_features=32, out_features=64, bias=True)
      (1): LayerNorm(

In [45]:
# ═══════════════════════════════════════════
# CELL: Training loop (drop-in replacement)
# ═══════════════════════════════════════════

best     = 0
best_val = 0

optimizer = torch.optim.Adam(
    list(model.parameters()) + list(history.parameters()),
    lr=args.lr,
    weight_decay=1e-5       # small L2 reg helps generalization
)

# Cosine LR schedule: smoothly decays over training
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=args.epochs, eta_min=args.lr * 0.1)

measure = Measure("auc")

for epoch in range(1, 1 + args.epochs):
    loss    = train(train_loader, model, history, optimizer, device, data, args)
    val_auc = test(valid_loader,  model, history, measure,   device, data)
    test_auc = test(test_loader,  model, history, measure,   device, data)
    scheduler.step()

    if val_auc > best_val:
        best_val = val_auc
        best     = test_auc

    print(
        f"Epoch: {epoch:03d}, Loss: {loss:.4f}, "
        f"Val AUC: {val_auc:.2%}, Test AUC: {test_auc:.2%}, Best AUC: {best:.2%}"
    )

100%|██████████| 93/93 [00:01<00:00, 52.07it/s]


Epoch: 001, Loss: 16.5262, Val AUC: 79.68%, Test AUC: 82.75%, Best AUC: 82.75%


100%|██████████| 93/93 [00:01<00:00, 51.41it/s]


Epoch: 002, Loss: 12.2291, Val AUC: 77.81%, Test AUC: 86.88%, Best AUC: 82.75%


100%|██████████| 93/93 [00:01<00:00, 51.21it/s]


...

Epoch: 018, Loss: 11.6307, Val AUC: 42.78%, Test AUC: 50.29%, Best AUC: 87.61%


100%|██████████| 93/93 [00:01<00:00, 49.38it/s]


Epoch: 019, Loss: 11.6152, Val AUC: 41.22%, Test AUC: 48.98%, Best AUC: 87.61%


Train Loss = 11.5307:   7%|▋         | 31/431 [00:02<00:26, 15.35it/s]


KeyboardInterrupt: 

In [46]:
"""
Temporal Snapshot GNN — v9
===========================

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
STAGE A  —  K-SNAPSHOT PIPELINE  (personal event history)
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

  For every node n, maintain a circular buffer of its K most
  recent events.  Each slot stores:
    neigh_emb  — projected embedding of the neighbour
    neigh_mem  — GRU memory state of the neighbour at that time
    edge_emb   — projected edge attributes of that interaction
    timestamp  — when the event occurred

  Step 1 · PerEventTrust
    For each stored event k, weight neighbour vs edge signal:
      trust_k = σ( MLP( [cos(mem_n, neigh_mem_k), conf_n, 0.5] ) )
      s_k     = trust_k · neigh_emb_k  +  (1−trust_k) · edge_emb_k
    Similarity in MEMORY space (stable) drives the weight.
    Normal n trusts normal neighbours more; anomalous n trusts
    anomalous neighbours more.  Nobody is zeroed.

  Step 2 · TimeModulatedGRU
    Process [s_1, …, s_K] chronologically with per-gap decay:
      φ(Δt_k)  = Time2Vec( RunningNorm(Δt_k) )
      decay_k  = σ( Linear(φ) )             D-dimensional gate
      h_k      = GRU( s_k,  decay_k ⊙ h_{k-1} )
    Initial h_0 = NodeMemory(n)  →  two-level hierarchy:
      Level 1 (local) : GRU over K snapshots
      Level 2 (global): NodeMemory GRU across all batches

  Step 3 · SnapshotPooling
    z_local = Σ_k  softmax( MLP(h_k) + recency_k ) · h_k
    z_local encodes: what n did × who it trusted × when it acted.

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
STAGE B  —  L × TRUST-TEMPORAL GRAPH ATTENTION  (multi-hop)
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

  z_local feeds into L stacked TrustTemporalAttention layers.
  Layer ℓ expands the receptive field to ℓ-hop neighbours.

  Each layer splits its H heads into THREE DEDICATED GROUPS
  (H/3 heads each), each attending to a different information view:

  ┌─────────────────────────────────────────────────────────────┐
  │  Group A — STRUCTURAL view                                  │
  │    Q ← z_i          what node i looks like NOW              │
  │    K ← z_j + φ(t)   what neighbour j looks like + WHEN      │
  │    V ← z_j          absorb j's current structural signal    │
  └─────────────────────────────────────────────────────────────┘
  ┌─────────────────────────────────────────────────────────────┐
  │  Group B — TEMPORAL / MEMORY view                           │
  │    Q ← h_i          what node i has been historically       │
  │    K ← h_j + φ(t)   what neighbour j has been + WHEN        │
  │    V ← h_j          absorb j's stable long-term behaviour   │
  └─────────────────────────────────────────────────────────────┘
  ┌─────────────────────────────────────────────────────────────┐
  │  Group C — INTERACTION / EDGE view                          │
  │    Q ← z_i          node i's current state as context       │
  │    K ← edge + φ(t)  what was exchanged and WHEN             │
  │    V ← edge         absorb the interaction content itself   │
  └─────────────────────────────────────────────────────────────┘

  TRUST (shared across all three groups, per edge):
    trust(i,j) = ( cos(h_i, h_j) + 1 ) / 2       ∈ [0, 1]
    logit      += log(trust).clamp(−6, 0)          log-space bias

    Similar neighbours  → trust ≈ 1 → log ≈ 0 → boosted attention
    Dissimilar          → trust ≈ 0 → log ≈ −6 → reduced attention
    Nobody zeroed: every neighbour contributes ≥ exp(−6) relative mass.
    Normal nodes pull from normal neighbours; anomalous from anomalous.
    All co-trained end-to-end with the contrastive loss.

  After each layer: LayerNorm + FF sublayer + LayerNorm (Transformer-style).

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
FINAL FUSION
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  z_final = MLP( concat(z_local, z_graph) )   (N, D)
  z_local  — snapshot view  (personal history + trust)
  z_graph  — graph view     (multi-hop neighbourhood)

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
CONTRASTIVE LEARNING + ANOMALY SCORING
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  Anchor   = z_final  (what n looks like now across all views)
  Positive = h_n      (what n has been across all history)
  InfoNCE  pushes anchor close to positive, far from other nodes.

  Anomaly score = 1 − cos(z_final, h_n)
  Normal n:     z_final ≈ h_n  →  score ≈ 0
  Anomalous n:  z_final ≉ h_n  →  score ≈ 1
"""

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import MessagePassing
from torch_geometric.utils import softmax as pyg_softmax
from tqdm import tqdm

EPS = 1e-8

def safe_norm(x):
    return x / x.norm(dim=-1, keepdim=True).clamp(min=EPS)

def safe_cos(a, b):
    return (safe_norm(a) * safe_norm(b)).sum(-1).clamp(-1., 1.)


# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 1.  Running Normalizer  —  keeps Time2Vec inputs bounded
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

class RunningNorm(nn.Module):
    def __init__(self, momentum=0.01):
        super().__init__()
        self.momentum = momentum
        self.register_buffer("mean", torch.tensor(0.0))
        self.register_buffer("std",  torch.tensor(1.0))
        self.register_buffer("init", torch.tensor(False))

    @torch.no_grad()
    def update(self, x):
        x = x.float().reshape(-1)
        if not self.init.item():
            self.mean.copy_(x.mean())
            self.std.copy_(x.std().clamp(min=1.0))
            self.init.fill_(True)
        else:
            m = self.momentum
            self.mean = (1-m)*self.mean + m*x.mean()
            self.std  = (1-m)*self.std  + m*x.std().clamp(min=1.0)

    def normalize(self, x):
        return (x.float() - self.mean) / self.std.clamp(min=1.0)


# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 2.  Time2Vec
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

class Time2Vec(nn.Module):
    def __init__(self, out_dim):
        super().__init__()
        k = out_dim - 1
        self.w0 = nn.Parameter(torch.ones(1) * 0.1)
        self.b0 = nn.Parameter(torch.zeros(1))
        self.W  = nn.Parameter(torch.randn(k) * 0.1)
        self.B  = nn.Parameter(torch.zeros(k))

    def forward(self, t):
        t = t.float().unsqueeze(-1)
        return torch.cat([t * self.w0 + self.b0,
                          torch.sin(t * self.W + self.B)], dim=-1)


# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 3.  K-Event Snapshot Bank  (vectorized circular buffer)
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

class SnapshotBank(nn.Module):
    def __init__(self, num_nodes, dim, K):
        super().__init__()
        self.K, self.dim = K, dim
        self.register_buffer("neigh_emb", torch.zeros(num_nodes, K, dim))
        self.register_buffer("neigh_mem", torch.zeros(num_nodes, K, dim))
        self.register_buffer("edge_emb",  torch.zeros(num_nodes, K, dim))
        self.register_buffer("ts",        torch.zeros(num_nodes, K))
        self.register_buffer("ptr",    torch.zeros(num_nodes, dtype=torch.long))
        self.register_buffer("filled", torch.zeros(num_nodes, dtype=torch.long))

    @torch.no_grad()
    def write(self, node_idx, neigh_emb, neigh_mem, edge_emb, timestamps):
        # For duplicate node_idx, keep last occurrence (most recent event)
        u_nodes, inv = torch.unique(node_idx, return_inverse=True)
        last_idx = torch.zeros(u_nodes.size(0), dtype=torch.long, device=node_idx.device)
        last_idx.scatter_(0, inv, torch.arange(node_idx.size(0), device=node_idx.device))

        n  = u_nodes
        p  = self.ptr[n] % self.K
        self.neigh_emb[n, p] = neigh_emb[last_idx]
        self.neigh_mem[n, p] = neigh_mem[last_idx]
        self.edge_emb[n, p]  = edge_emb[last_idx]
        self.ts[n, p]        = timestamps[last_idx].float()
        self.ptr[n]         += 1
        self.filled[n]       = (self.filled[n] + 1).clamp(max=self.K)

    def read(self, node_idx):
        B, K, D = node_idx.size(0), self.K, self.dim
        dev     = node_idx.device

        ne_r = self.neigh_emb[node_idx]   # (B, K, D)
        nm_r = self.neigh_mem[node_idx]
        ee_r = self.edge_emb[node_idx]
        ts_r = self.ts[node_idx]           # (B, K)
        fill = self.filled[node_idx]       # (B,)
        ptr  = self.ptr[node_idx] % K      # (B,)

        # Chronological reorder: oldest slot first
        slots = torch.arange(K, device=dev).unsqueeze(0).expand(B, -1)
        shift = torch.where(fill >= K, ptr, torch.zeros_like(ptr))
        order = (slots + shift.unsqueeze(1)) % K                      # (B, K)

        ex    = order.unsqueeze(-1).expand(-1, -1, D)
        ne    = ne_r.gather(1, ex)
        nm    = nm_r.gather(1, ex)
        ee    = ee_r.gather(1, ex)
        ts    = ts_r.gather(1, order)
        mask  = slots < fill.unsqueeze(1)                              # (B, K)
        return ne, nm, ee, ts, mask


# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 4.  Per-Event Trust Aggregation
#     trust_k = σ(MLP([cos(mem_n, neigh_mem_k), conf_n, conf_k]))
#     s_k     = trust_k · neigh_emb_k + (1-trust_k) · edge_emb_k
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

class PerEventTrust(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.neigh_proj = nn.Linear(dim, dim)
        self.edge_proj  = nn.Linear(dim, dim)
        self.trust_mlp  = nn.Sequential(
            nn.Linear(3, 32), nn.GELU(), nn.Linear(32, 1))
        self.out_norm   = nn.LayerNorm(dim)

    def forward(self, ne, nm, ee, mem_n, var_n):
        B, K, D = ne.shape
        mem_exp = mem_n.unsqueeze(1).expand(-1, K, -1)
        cos_01  = (safe_cos(mem_exp.reshape(B*K, D),
                            nm.reshape(B*K, D)).view(B, K) + 1.0) / 2.0
        conf_n  = (1.0 / (1.0 + var_n.mean(-1, keepdim=True)
                           .sqrt().clamp(min=EPS))).expand(-1, K)
        conf_nb = torch.full_like(cos_01, 0.5)
        signals = torch.stack([cos_01, conf_n, conf_nb], dim=-1)      # (B, K, 3)
        trust   = torch.sigmoid(
            self.trust_mlp(signals.reshape(B*K, 3)).view(B, K))       # (B, K)
        w = trust.unsqueeze(-1)
        s = self.out_norm(w * self.neigh_proj(ne) + (1-w) * self.edge_proj(ee))
        return s, trust


# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 5.  Time-Modulated GRU
#     decay_k = σ(Linear(Time2Vec(Δt_k)))  →  (B, D)
#     h_k     = GRU(s_k,  decay_k ⊙ h_{k-1})
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

class TimeModulatedGRU(nn.Module):
    def __init__(self, dim, time_dim=32):
        super().__init__()
        self.gru        = nn.GRUCell(dim, dim)
        self.t2v        = Time2Vec(time_dim)
        self.tnorm      = RunningNorm()
        self.decay_gate = nn.Sequential(
            nn.Linear(time_dim, dim), nn.LayerNorm(dim))

    def forward(self, s, ts, mask, h0):
        B, K, D = s.shape
        h, all_h = h0.clone(), []
        for k in range(K):
            s_k, valid = s[:, k], mask[:, k]
            if k == 0:
                decay = torch.ones(B, D, device=s.device)
            else:
                dt = (ts[:, k] - ts[:, k-1]).clamp(min=0.0)
                if self.training:
                    self.tnorm.update(dt)
                phi   = self.t2v(self.tnorm.normalize(dt))
                decay = torch.sigmoid(self.decay_gate(phi))
            h_new = self.gru(s_k, decay * h)
            h     = torch.where(valid.unsqueeze(-1), h_new, h)
            all_h.append(h)
        return torch.stack(all_h, 1), h               # (B,K,D), (B,D)


# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 6.  Snapshot Attention Pooling  →  z_local  (B, D)
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

class SnapshotPooling(nn.Module):
    def __init__(self, dim, K):
        super().__init__()
        self.score   = nn.Sequential(
            nn.Linear(dim, dim//2), nn.GELU(), nn.Linear(dim//2, 1))
        self.recency = nn.Parameter(torch.linspace(-1.0, 1.0, K))

    def forward(self, all_h, mask):
        scores = self.score(all_h).squeeze(-1) + self.recency          # (B, K)
        scores = scores.masked_fill(~mask, -1e9)
        alpha  = torch.softmax(scores, dim=-1).unsqueeze(-1)           # (B, K, 1)
        return (alpha * all_h).sum(1)                                  # (B, D)


# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 7.  Global Node Memory
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

class NodeMemory(nn.Module):
    def __init__(self, num_nodes, dim, momentum=0.9):
        super().__init__()
        self.gru      = nn.GRUCell(dim, dim)
        self.momentum = momentum
        self.register_buffer("hidden",   torch.zeros(num_nodes, dim))
        self.register_buffer("variance", torch.ones(num_nodes, dim) * 0.1)

    def read(self, idx):
        return self.hidden[idx], self.variance[idx]

    @torch.no_grad()
    def write(self, idx, x):
        h_old = self.hidden[idx]
        h_new = self.gru(x.detach(), h_old)
        self.variance[idx] = (self.momentum * self.variance[idx]
                              + (1-self.momentum) * (h_new - h_old).pow(2))
        self.hidden[idx]   = h_new


# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 8.  Trust-Temporal Graph Attention Layer
#
#  HETEROGENEOUS MULTI-SOURCE Q / K / V
#  ──────────────────────────────────────
#  The total heads H are split into three DEDICATED GROUPS.
#  Each group specialises on one information source:
#
#    Group A  (H_a heads)  — NODE STRUCTURAL view
#      Q_a  ← z_i          (node i's current representation)
#      K_a  ← z_j          (neighbor j's current representation)
#      V_a  ← z_j          (absorb j's structural signal)
#
#    Group B  (H_b heads)  — TEMPORAL / MEMORY view
#      Q_b  ← h_i          (node i's GRU memory  — stable history)
#      K_b  ← h_j          (neighbor j's GRU memory)
#      V_b  ← h_j          (absorb j's historical signal)
#
#    Group C  (H_c heads)  — INTERACTION / EDGE view
#      Q_c  ← z_i          (node i's current repr as context)
#      K_c  ← e_ij + φ(t)  (edge attributes + time encoding)
#      V_c  ← e_ij         (absorb interaction content)
#
#  Why three groups?
#    Group A captures structural neighbourhood similarity
#      ("which current neighbours look like me?")
#    Group B captures historical-behavioural alignment
#      ("which neighbours have acted like me over time?")
#    Group C captures interaction-level signals
#      ("which specific interactions at specific times matter?")
#
#  The outputs of all three groups are concatenated and projected:
#    agg = concat( out_A, out_B, out_C )  →  Linear  →  (N, D)
#
#  TRUST (inside attention, before softmax, applied to ALL groups):
#    trust(i,j) = cos(h_i, h_j) shifted to [0,1]
#               = how similar i and j are in MEMORY space
#    Added as log(trust) to attention logits in log-space:
#      attn_logit += log(trust).clamp(-6, 0)
#    Effect: similar neighbours get more attention mass across all
#    head groups; dissimilar get less — but never zero.
#    Both normal and anomalous nodes pull from their own kind.
#
#  TIME ENCODING:
#    φ(t_ij) = Time2Vec(RunningNorm(t_ij))  →  (E, time_dim)
#    Used in Group C keys and also added to Group A and B keys
#    as a lightweight per-head bias — every head knows WHEN.
#
#  PER-LAYER: LayerNorm(residual) → Feed-forward → LayerNorm(residual)
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

class TrustTemporalAttention(MessagePassing):
    """
    One layer.  Stack L of these to reach L-hop neighbourhood context.
    heads must be divisible by 3 (one third per source group).
    """
    def __init__(self, dim, heads=9, dropout=0.1, time_dim=32):
        super().__init__(aggr="add", node_dim=0)
        assert heads % 3 == 0, "heads must be divisible by 3 (one group per source)"
        assert dim   % heads == 0
        self.dim      = dim
        self.heads    = heads
        self.head_dim = dim // heads
        self.scale    = self.head_dim ** -0.5
        self.h_grp    = heads // 3          # heads per group
        self.d_grp    = self.h_grp * self.head_dim  # dim per group

        # ── Group A: structural (node repr → node repr) ───────────────────
        self.qa = nn.Linear(dim, self.d_grp)   # Q from z_i
        self.ka = nn.Linear(dim, self.d_grp)   # K from z_j
        self.va = nn.Linear(dim, self.d_grp)   # V from z_j

        # ── Group B: temporal (memory → memory) ──────────────────────────
        self.qb = nn.Linear(dim, self.d_grp)   # Q from h_i
        self.kb = nn.Linear(dim, self.d_grp)   # K from h_j
        self.vb = nn.Linear(dim, self.d_grp)   # V from h_j

        # ── Group C: interaction (node repr × edge+time) ──────────────────
        self.qc      = nn.Linear(dim,      self.d_grp)  # Q from z_i
        self.kc_edge = nn.Linear(dim,      self.d_grp)  # K from edge attr
        self.kc_time = nn.Linear(time_dim, self.d_grp)  # K from time enc
        self.vc_edge = nn.Linear(dim,      self.d_grp)  # V from edge attr

        # ── Lightweight time bias added to ALL groups' keys ───────────────
        # Lets every head know WHEN the interaction happened
        self.time_bias_a = nn.Linear(time_dim, self.d_grp, bias=False)
        self.time_bias_b = nn.Linear(time_dim, self.d_grp, bias=False)

        # ── Time encoding shared across groups ────────────────────────────
        self.t2v   = Time2Vec(time_dim)
        self.tnorm = RunningNorm()

        # ── Output projection: concat all groups → dim ────────────────────
        self.o_proj = nn.Linear(dim, dim)

        # ── Per-layer transformer blocks ──────────────────────────────────
        self.norm1 = nn.LayerNorm(dim)
        self.norm2 = nn.LayerNorm(dim)
        self.ff    = nn.Sequential(
            nn.Linear(dim, dim * 2),
            nn.GELU(),
            nn.Linear(dim * 2, dim),
        )
        self.drop  = nn.Dropout(dropout)

    def _sp(self, x, h):
        """Split (N_or_E, d_grp) → (N_or_E, h, head_dim)"""
        return x.view(-1, h, self.head_dim)

    def _time_enc(self, edge_t):
        """Raw edge timestamps → (E, time_dim) normalised Time2Vec."""
        if self.training:
            self.tnorm.update(edge_t)
        return self.t2v(self.tnorm.normalize(edge_t))                  # (E, time_dim)

    def forward(self, z, mem_h, mem_var, edge_index, edge_emb, edge_t):
        """
        z:          (N, D)
        mem_h:      (N, D)  GRU hidden states
        mem_var:    (N, D)  GRU variance (used for trust confidence)
        edge_index: (2, E)
        edge_emb:   (E, D)
        edge_t:     (E,)
        Returns:    (N, D)
        """
        N   = z.size(0)
        h_g = self.h_grp
        hd  = self.head_dim

        # Compute time encoding once — reused by all groups
        phi = self._time_enc(edge_t)                                   # (E, time_dim)

        # ── Group A queries and keys (node-level, lifted to edges by PyG) ─
        QA = self._sp(self.qa(z),     h_g)                             # (N, h_g, hd)
        KA = self._sp(self.ka(z),     h_g)                             # (N, h_g, hd)
        VA = self._sp(self.va(z),     h_g)

        # ── Group B queries and keys ──────────────────────────────────────
        QB = self._sp(self.qb(mem_h), h_g)
        KB = self._sp(self.kb(mem_h), h_g)
        VB = self._sp(self.vb(mem_h), h_g)

        # ── Group C queries (node-level); keys/values from edges ──────────
        QC = self._sp(self.qc(z),     h_g)                             # (N, h_g, hd)
        # Group C K and V are edge-level — computed per-edge in message()
        KC_edge = self._sp(self.kc_edge(edge_emb), h_g)               # (E, h_g, hd)
        KC_time = self._sp(self.kc_time(phi),      h_g)               # (E, h_g, hd)
        VC_edge = self._sp(self.vc_edge(edge_emb), h_g)               # (E, h_g, hd)

        # ── Per-edge time bias for Group A and B keys ─────────────────────
        tba = self._sp(self.time_bias_a(phi), h_g)                    # (E, h_g, hd)
        tbb = self._sp(self.time_bias_b(phi), h_g)

        agg = self.propagate(
            edge_index,
            # Group A
            QA=QA, KA=KA, VA=VA,
            # Group B
            QB=QB, KB=KB, VB=VB,
            # Group C
            QC=QC,
            KC_edge=KC_edge, KC_time=KC_time, VC_edge=VC_edge,
            # Time bias (per-edge)
            tba=tba, tbb=tbb,
            # Memory (for trust)
            mem_h=mem_h,
            size=(N, N),
        )                                                               # (N, heads, hd)
        agg = self.o_proj(agg.view(N, self.dim))

        # Residual + norm → FF → residual + norm
        z = self.norm1(z + self.drop(agg))
        z = self.norm2(z + self.drop(self.ff(z)))
        return z

    def message(self,
                QA_i, KA_j, VA_j,
                QB_i, KB_j, VB_j,
                QC_i, KC_edge, KC_time, VC_edge,
                tba, tbb,
                mem_h_i, mem_h_j,
                index):
        """
        All _i / _j tensors are (E, h_g, hd) — PyG lifts node tensors to edges.
        Edge tensors (KC_edge, KC_time, VC_edge, tba, tbb) are (E, h_g, hd).
        mem_h_i, mem_h_j are (E, D).
        """
        hd = self.head_dim
        s  = self.scale

        # ── Trust score (shared across all groups) ────────────────────────
        # Computed in memory space — stable, not noisy current repr.
        # Added in log-space so no neighbour is ever fully zeroed.
        cos_ij    = safe_cos(mem_h_i, mem_h_j)                        # (E,)
        trust     = (cos_ij + 1.0) / 2.0                              # (E,) ∈ [0,1]
        log_trust = trust.clamp(min=1e-6).log().clamp(-6., 0.)        # (E,)
        lt        = log_trust.unsqueeze(-1)                            # (E, 1)

        # ── Group A: structural attention ─────────────────────────────────
        # K_A = neighbour repr + time bias (every head knows when)
        KA_full = KA_j + tba                                           # (E, h_g, hd)
        attn_a  = (QA_i * KA_full).sum(-1) * s + lt                   # (E, h_g)
        attn_a  = pyg_softmax(attn_a.clamp(-20, 20), index)           # (E, h_g)
        out_a   = (attn_a.unsqueeze(-1) * VA_j)                       # (E, h_g, hd)

        # ── Group B: temporal / memory attention ──────────────────────────
        # K_B = neighbour memory + time bias
        KB_full = KB_j + tbb                                           # (E, h_g, hd)
        attn_b  = (QB_i * KB_full).sum(-1) * s + lt                   # (E, h_g)
        attn_b  = pyg_softmax(attn_b.clamp(-20, 20), index)
        out_b   = (attn_b.unsqueeze(-1) * VB_j)                       # (E, h_g, hd)

        # ── Group C: interaction / edge attention ─────────────────────────
        # K_C = edge attributes + time encoding (no node-repr component)
        KC_full = KC_edge + KC_time                                    # (E, h_g, hd)
        attn_c  = (QC_i * KC_full).sum(-1) * s + lt                   # (E, h_g)
        attn_c  = pyg_softmax(attn_c.clamp(-20, 20), index)
        out_c   = (attn_c.unsqueeze(-1) * VC_edge)                    # (E, h_g, hd)

        # ── Concatenate all groups → (E, heads, hd) ──────────────────────
        return torch.cat([out_a, out_b, out_c], dim=1)                 # (E, heads, hd)


# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 9.  Full Model
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

class TemporalSnapshotGNN(nn.Module):
    """
    Full pipeline:

      Stage A — K-Snapshot (per-node personal history)
        SnapshotBank → PerEventTrust → TimeModulatedGRU → SnapshotPooling
        Produces z_local: what n did × who it trusted × when it acted.

      Stage B — L × TrustTemporalAttention (multi-hop graph attention)
        Each layer expands receptive field by one hop.
        Heterogeneous Q/K/V split across 3 dedicated head groups:
          Group A (structural):   Q←z_i,   K←z_j+φ(t),      V←z_j
          Group B (temporal):     Q←h_i,   K←h_j+φ(t),      V←h_j
          Group C (interaction):  Q←z_i,   K←edge+φ(t),     V←edge
        Trust(i,j) = cos(h_i, h_j) added as log-bias to all groups.

      Final: concat(z_local, z_graph) → MLP → z_final (N, D)

    NOTE: heads must be divisible by 3 (one third per source group).
    """
    def __init__(self, num_nodes, in_dim, edge_dim, hidden_dim,
                 K=8, num_layers=3, heads=9,
                 time_dim=32, dropout=0.1, memory_momentum=0.9):
        assert heads % 3 == 0,        "heads must be divisible by 3"
        assert hidden_dim % heads == 0, "hidden_dim must be divisible by heads"
        super().__init__()
        self.hidden_dim = hidden_dim
        self.K          = K

        # Input projections
        self.node_proj = nn.Sequential(
            nn.Linear(in_dim, hidden_dim), nn.LayerNorm(hidden_dim))
        self.edge_proj = nn.Sequential(
            nn.Linear(edge_dim, hidden_dim), nn.LayerNorm(hidden_dim))

        # Stage A: K-snapshot pipeline
        self.bank     = SnapshotBank(num_nodes, hidden_dim, K=K)
        self.trust    = PerEventTrust(hidden_dim)
        self.tm_gru   = TimeModulatedGRU(hidden_dim, time_dim=time_dim)
        self.pooling  = SnapshotPooling(hidden_dim, K=K)

        # Global long-term memory (shared across both stages)
        self.memory   = NodeMemory(num_nodes, hidden_dim, momentum=memory_momentum)

        # Stage B: L-layer temporal graph attention
        self.gnn_layers = nn.ModuleList([
            TrustTemporalAttention(hidden_dim, heads=heads,
                                   dropout=dropout, time_dim=time_dim)
            for _ in range(num_layers)
        ])

        # Final fusion: z_local (from snapshot) + z_graph (from GNN layers)
        self.final_fuse = nn.Sequential(
            nn.Linear(hidden_dim * 2, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
        )

        self.drop = nn.Dropout(dropout)
        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)

    def forward(self, batch):
        """
        Returns: (N, hidden_dim)  — one representation per node.
        """
        N       = batch.x.size(0)
        device  = batch.x.device
        src, dst = batch.edge_index
        all_idx  = torch.arange(N, device=device)

        # Project raw features
        x_proj  = self.node_proj(batch.x)                             # (N, D)
        e_proj  = self.edge_proj(batch.msg)                           # (E, D)

        # ── STAGE A: K-snapshot → trust → time-modulated GRU → pool ──────

        unique_src = src.unique()                                      # (U,)

        # Read K stored snapshots for active (src) nodes
        ne, nm, ee, ts, mask = self.bank.read(unique_src)

        # Per-event trust → snapshot embeddings
        mem_n, var_n = self.memory.read(unique_src)
        s, _         = self.trust(ne, nm, ee, mem_n, var_n)           # (U, K, D)

        # Time-modulated GRU (seeded with long-term memory → hierarchy)
        all_h, h_K   = self.tm_gru(s, ts, mask, h0=mem_n)            # (U,K,D), (U,D)

        # Attention pooling → z_local per active node
        z_local_u    = self.pooling(all_h, mask)                      # (U, D)

        # Scatter to full N: non-active nodes get their memory state
        z_local      = self.memory.hidden[all_idx].clone()            # (N, D)
        z_local[unique_src] = z_local_u                               # (N, D)

        # ── STAGE B: Multi-layer trust-temporal graph attention ────────────

        # Read memory states and variances for ALL N nodes (needed per-layer)
        mem_h_all, mem_var_all = self.memory.read(all_idx)            # (N,D), (N,D)

        # z starts from z_local; each layer enriches it by one more hop
        z = z_local
        for layer in self.gnn_layers:
            z = layer(
                z,                    # current node repr  (N, D)
                mem_h_all,            # node memory        (N, D)
                mem_var_all,          # node variance      (N, D)
                batch.edge_index,
                e_proj,               # edge attributes    (E, D)
                batch.t.float(),      # edge timestamps    (E,)
            )

        # ── Final fusion: z_local (snapshot view) + z (graph view) ────────
        z_final = self.final_fuse(torch.cat([z_local, z], dim=-1))   # (N, D)
        z_final = self.drop(z_final)

        # ── Write new snapshots (current event) into bank ─────────────────
        dst_emb = x_proj[dst]
        dst_mem = self.memory.hidden[dst]
        self.bank.write(src, dst_emb, dst_mem, e_proj, batch.t.float())

        # ── Update global memory (Level 2 of hierarchy) ───────────────────
        self.memory.write(unique_src, h_K)

        return z_final                                                 # (N, D)


# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 10.  Contrastive Learning
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

def info_nce(anchor, positive, negatives, tau=0.07):
    a = safe_norm(anchor)
    p = safe_norm(positive)
    n = safe_norm(negatives)
    pos = (a * p).sum(-1) / tau
    neg = torch.einsum('bd,bkd->bk', a, n) / tau
    logits = torch.cat([pos.unsqueeze(1), neg], dim=1).clamp(-50, 50)
    labels = torch.zeros(a.size(0), dtype=torch.long, device=a.device)
    return F.cross_entropy(logits, labels)


def hard_negatives(z, h, K=64):
    B  = z.size(0)
    K  = min(K, B-1)
    sz = safe_norm(z.detach())
    sh = safe_norm(h.detach())
    score = sz @ sz.T - sh @ sh.T
    score.fill_diagonal_(-1e9)
    return score.topk(K, dim=-1).indices


# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 11.  Training
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

def train(loader, model, history, optimizer, device, data, args):
    model.train()
    history.train()
    pbar    = tqdm(loader)
    total   = 0.0
    tau     = getattr(args, 'tau',     0.07)
    num_neg = getattr(args, 'num_neg', 64)

    for batch in pbar:
        batch = batch.to(device)
        optimizer.zero_grad()

        z   = model(batch)
        src = batch.src[:batch.batch_size]
        z   = z[src]
        idx = batch.input_id.cpu()

        cur_mem, prev_mem = history(z, data.src[idx])

        with torch.no_grad():
            neg_idx = hard_negatives(z, cur_mem, K=num_neg)
        negatives = z[neg_idx]

        loss = info_nce(z, cur_mem, negatives, tau=tau)
        consist = (1.0 - F.cosine_similarity(
            cur_mem, prev_mem.detach(), dim=-1)).mean()
        loss = loss + 0.1 * consist

        if not torch.isfinite(loss):
            optimizer.zero_grad()
            continue

        loss.backward()
        nn.utils.clip_grad_norm_(
            list(model.parameters()) + list(history.parameters()), 1.0)
        optimizer.step()

        total += loss.item()
        pbar.set_description(f"Loss={loss.item():.4f}")

    return total / max(len(loader), 1)


# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 12.  Test / Anomaly Scoring
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

@torch.no_grad()
def test(loader, model, history, measure, device, data, args):
    preds, labels_all = [], []
    model.eval()
    history.eval()
    eps = 1e-6

    for batch in tqdm(loader):
        torch.cuda.empty_cache()
        batch = batch.to(device)

        z     = model(batch)
        src   = batch.src[:batch.batch_size]
        z     = z[src]
        label = batch.y[:batch.batch_size]
        idx   = batch.input_id.cpu()

        cur_mem, prev_mem = history(z, data.src[idx], update=True)

        node_ids   = data.src[idx].to(device)
        _, cur_var = model.memory.read(node_ids)
        conf = 1.0 / (1.0 + cur_var.mean(-1).sqrt().clamp(min=eps))

        score   = conf * (1.0 - F.cosine_similarity(z, cur_mem, dim=-1))
        score_t = conf * (1.0 - F.cosine_similarity(cur_mem, prev_mem, dim=-1))
        pred    = (score + 0.4 * score_t).sigmoid()

        preds.append(pred.detach().cpu())
        labels_all.append(label.detach().cpu())
        torch.cuda.empty_cache()

    preds      = torch.cat(preds)
    preds[torch.isnan(preds)] = 0.0
    labels_all = torch.cat(labels_all)
    return measure(labels_all[labels_all != 2], preds[labels_all != 2])

In [54]:
"""
Drop-in replacement for your notebook.
Keeps: data, loaders, History, args, measure
Replaces: model, train(), test()

Just run these cells in order after your existing setup cells.
"""

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import MessagePassing
from torch_geometric.utils import softmax
from tqdm import tqdm


# ── Paste the full enhanced_gnn_v2.py here, or import it ──────────────────────
# from enhanced_gnn_v2 import (EnhancedTemporalGNN, train, test)
# ──────────────────────────────────────────────────────────────────────────────
# If running directly in notebook, paste the class definitions above this cell.


# ═══════════════════════════════════════════
# CELL: Build model + history (unchanged)
# ═══════════════════════════════════════════

try:
    del model
except NameError:
    pass

model = TemporalSnapshotGNN(
    num_nodes  = data.num_nodes,
    in_dim     = data.x.size(1),          # 1 for wikipedia
    edge_dim   = data.msg.size(1),         # 172 for wikipedia
    hidden_dim = 225,     # 64
    heads=3,
    num_layers = args.nums_layers,         # 2              # 8
    dropout    = args.dropout,             # 0.3
    K     = 6,
    memory_momentum = 0.9,
).to(device)

# History is UNCHANGED — reuse your existing History class
try:
    del history
except NameError:
    pass

from mhiscl.history import History

history = History(
    data.num_nodes,
    args.num_timeslots,
    225,
    device=device,
    history_retrieve=args.history_retrieve,
    recurrent=args.recurrent,
).to(device)

print(model)
print(history)
print(f"Model params: {sum(p.numel() for p in model.parameters()):,}")




TemporalSnapshotGNN(
  (node_proj): Sequential(
    (0): Linear(in_features=1, out_features=225, bias=True)
    (1): LayerNorm((225,), eps=1e-05, elementwise_affine=True)
  )
  (edge_proj): Sequential(
    (0): Linear(in_features=172, out_features=225, bias=True)
    (1): LayerNorm((225,), eps=1e-05, elementwise_affine=True)
  )
  (bank): SnapshotBank()
  (trust): PerEventTrust(
    (neigh_proj): Linear(in_features=225, out_features=225, bias=True)
    (edge_proj): Linear(in_features=225, out_features=225, bias=True)
    (trust_mlp): Sequential(
      (0): Linear(in_features=3, out_features=32, bias=True)
      (1): GELU(approximate='none')
      (2): Linear(in_features=32, out_features=1, bias=True)
    )
    (out_norm): LayerNorm((225,), eps=1e-05, elementwise_affine=True)
  )
  (tm_gru): TimeModulatedGRU(
    (gru): GRUCell(225, 225)
    (t2v): Time2Vec()
    (tnorm): RunningNorm()
    (decay_gate): Sequential(
      (0): Linear(in_features=32, out_features=225, bias=True)
      (1)

In [55]:
# ═══════════════════════════════════════════
# CELL: Training loop (drop-in replacement)
# ═══════════════════════════════════════════

best     = 0
best_val = 0

optimizer = torch.optim.Adam(
    list(model.parameters()) + list(history.parameters()),
    lr=args.lr,
    weight_decay=1e-5       # small L2 reg helps generalization
)

# Cosine LR schedule: smoothly decays over training
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=args.epochs, eta_min=args.lr * 0.1)

measure = Measure("auc")

for epoch in range(1, 1 + args.epochs):
    loss    = train(train_loader, model, history, optimizer, device, data, args)
    val_auc = test(valid_loader,  model, history, measure,   device, data)
    test_auc = test(test_loader,  model, history, measure,   device, data)
    scheduler.step()

    if val_auc > best_val:
        best_val = val_auc
        best     = test_auc

    print(
        f"Epoch: {epoch:03d}, Loss: {loss:.4f}, "
        f"Val AUC: {val_auc:.2%}, Test AUC: {test_auc:.2%}, Best AUC: {best:.2%}"
    )

  0%|          | 0/431 [00:00<?, ?it/s]

100%|██████████| 93/93 [00:02<00:00, 40.40it/s]


Epoch: 001, Loss: 13.6257, Val AUC: 77.55%, Test AUC: 84.56%, Best AUC: 84.56%


100%|██████████| 93/93 [00:02<00:00, 44.59it/s]


Epoch: 002, Loss: 11.5462, Val AUC: 83.59%, Test AUC: 90.70%, Best AUC: 90.70%


...

Epoch: 033, Loss: 11.5617, Val AUC: 81.23%, Test AUC: 88.73%, Best AUC: 89.71%


100%|██████████| 93/93 [00:02<00:00, 43.39it/s]


Epoch: 034, Loss: 11.5603, Val AUC: 82.98%, Test AUC: 88.01%, Best AUC: 89.71%


100%|██████████| 93/93 [00:02<00:00, 43.97it/s]

Epoch: 035, Loss: 11.5793, Val AUC: 80.01%, Test AUC: 89.10%, Best AUC: 89.71%


## Model snapshots seperatly

In [19]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import MessagePassing
from torch_geometric.utils import softmax

In [20]:
class SimilarityAwareConv(MessagePassing):
    """
    Neighborhood-aware message passing:
    - Similar neighbors contribute more
    - Dissimilar neighbors are suppressed
    """
    def __init__(self, dim, heads=4, dropout=0.1, tau=0.2):
        super().__init__(aggr="add", node_dim=0)
        self.dim = dim
        self.heads = heads
        self.head_dim = dim // heads
        self.scale = self.head_dim ** -0.5
        self.tau = tau

        self.q_proj = nn.Linear(dim, dim)
        self.k_proj = nn.Linear(dim, dim)
        self.v_proj = nn.Linear(dim, dim)

        self.out_proj = nn.Linear(dim, dim)
        self.dropout = nn.Dropout(dropout)
        self.norm = nn.LayerNorm(dim)

    def forward(self, x, edge_index):
        Q = self.q_proj(x).view(-1, self.heads, self.head_dim)
        K = self.k_proj(x).view(-1, self.heads, self.head_dim)
        V = self.v_proj(x).view(-1, self.heads, self.head_dim)

        out = self.propagate(edge_index, Q=Q, K=K, V=V, size=(x.size(0), x.size(0)))
        out = out.view(-1, self.dim)
        return self.norm(x + self.out_proj(out))

    def message(self, Q_i, K_j, V_j, index):
        # Standard attention
        attn = (Q_i * K_j).sum(-1) * self.scale  # (E, heads)

        # Neighborhood similarity gate (CORE)
        sim = F.cosine_similarity(Q_i, K_j, dim=-1)  # (E, heads)
        gate = torch.sigmoid(sim / self.tau)

        attn = attn * gate
        attn = softmax(attn, index)
        attn = self.dropout(attn)

        return V_j * attn.unsqueeze(-1)

In [21]:
class SnapshotEncoder(nn.Module):
    """
    Encodes a single temporal snapshot into node embeddings
    """
    def __init__(self, in_dim, hidden_dim, num_layers=2):
        super().__init__()
        self.input_proj = nn.Linear(in_dim, hidden_dim)
        self.layers = nn.ModuleList([
            SimilarityAwareConv(hidden_dim)
            for _ in range(num_layers)
        ])
        self.norm = nn.LayerNorm(hidden_dim)

    def forward(self, x, edge_index):
        x = self.input_proj(x)
        for layer in self.layers:
            x = layer(x, edge_index)
        return self.norm(x)

In [22]:
class TemporalSnapshotAttention(nn.Module):
    """
    Aggregates K snapshot embeddings using time gaps
    """
    def __init__(self, dim, num_heads=4):
        super().__init__()
        self.attn = nn.MultiheadAttention(
            embed_dim=dim,
            num_heads=num_heads,
            batch_first=True
        )
        self.time_mlp = nn.Sequential(
            nn.Linear(1, dim),
            nn.GELU(),
            nn.Linear(dim, dim)
        )
        self.norm = nn.LayerNorm(dim)

    def forward(self, h_snapshots, delta_t):
        """
        h_snapshots: (N, K, D)
        delta_t:     (K,) or (N, K)
        """
        if delta_t.dim() == 1:
            delta_t = delta_t.unsqueeze(0).repeat(h_snapshots.size(0), 1)

        time_bias = self.time_mlp(delta_t.unsqueeze(-1))  # (N, K, D)
        h = h_snapshots + time_bias

        out, _ = self.attn(h, h, h)
        out = self.norm(out + h)

        # Pool over snapshots
        return out.mean(dim=1)

In [23]:
class NATC_GNN(nn.Module):
    """
    Neighborhood-Aware Temporal Contrastive GNN
    """
    def __init__(self, in_dim, hidden_dim, num_snapshots, num_layers=2):
        super().__init__()
        self.num_snapshots = num_snapshots

        self.snapshot_encoder = SnapshotEncoder(
            in_dim=in_dim,
            hidden_dim=hidden_dim,
            num_layers=num_layers
        )

        self.temporal_agg = TemporalSnapshotAttention(hidden_dim)

        # Projection head for contrastive learning
        self.proj_head = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, hidden_dim)
        )

    def forward(self, snapshots, delta_t):
        """
        snapshots: list of K PyG Data objects
        delta_t:   (K,) time gaps between snapshots
        """
        h_list = []

        for data in snapshots:
            h = self.snapshot_encoder(data.x, data.edge_index)
            h_list.append(h)

        h_snap = torch.stack(h_list, dim=1)  # (N, K, D)

        z = self.temporal_agg(h_snap, delta_t)
        z = self.proj_head(z)

        return F.normalize(z, dim=-1), h_snap

In [25]:
"""
Drop-in replacement for your notebook.
Keeps: data, loaders, History, args, measure
Replaces: model, train(), test()

Just run these cells in order after your existing setup cells.
"""

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import MessagePassing
from torch_geometric.utils import softmax
from tqdm import tqdm


# ── Paste the full enhanced_gnn_v2.py here, or import it ──────────────────────
# from enhanced_gnn_v2 import (EnhancedTemporalGNN, train, test)
# ──────────────────────────────────────────────────────────────────────────────
# If running directly in notebook, paste the class definitions above this cell.


# ═══════════════════════════════════════════
# CELL: Build model + history (unchanged)
# ═══════════════════════════════════════════

try:
    del model
except NameError:
    pass

model = NATC_GNN(
    in_dim     = data.x.size(1),          # 1 for wikipedia
    hidden_dim = 225,     # 64
    num_layers = args.nums_layers,         # 2              # 8            # 0.3
    num_snapshots     = 6
).to(device)

# History is UNCHANGED — reuse your existing History class
try:
    del history
except NameError:
    pass

from mhiscl.history import History

history = History(
    data.num_nodes,
    args.num_timeslots,
    225,
    device=device,
    history_retrieve=args.history_retrieve,
    recurrent=args.recurrent,
).to(device)

print(model)
print(history)
print(f"Model params: {sum(p.numel() for p in model.parameters()):,}")




AssertionError: embed_dim must be divisible by num_heads

In [ ]:
import math
import copy
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import MessagePassing
from torch_geometric.utils import softmax
from tqdm import tqdm


# ═══════════════════════════════════════════════════════════════════════════════
# 1. Time2Vec — learns periodic + trend components from raw timestamps
# ═══════════════════════════════════════════════════════════════════════════════

class Time2Vec(nn.Module):
    """
    t → [w0*t + b0,  sin(w1*t + b1), ..., sin(wk*t + bk)]
    Linear term captures trend, sinusoids capture periodicity.
    Frequencies w are LEARNED — model discovers daily/weekly patterns.
    """
    def __init__(self, out_dim):
        super().__init__()
        # k periodic components + 1 linear
        k = out_dim - 1
        self.w0 = nn.Parameter(torch.randn(1) * 0.01)
        self.b0 = nn.Parameter(torch.zeros(1))
        self.W  = nn.Parameter(torch.randn(k) * 0.01)
        self.B  = nn.Parameter(torch.zeros(k))
        self.out_dim = out_dim

    def forward(self, t):
        """t: (N,) → (N, out_dim)"""
        t = t.float().unsqueeze(-1)                                    # (N, 1)
        linear   = t * self.w0 + self.b0                               # (N, 1)
        periodic = torch.sin(t * self.W + self.B)                      # (N, k)
        return torch.cat([linear, periodic], dim=-1)                   # (N, out_dim)


class TimeEncoder(nn.Module):
    """
    Full time encoding: Time2Vec → MLP projection → hidden_dim
    Also computes RELATIVE time gap between edge time and node's last seen time.
    """
    def __init__(self, hidden_dim):
        super().__init__()
        self.t2v     = Time2Vec(hidden_dim // 2)
        self.rel_t2v = Time2Vec(hidden_dim // 2)   # for relative time gap
        self.proj    = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, hidden_dim)
        )

    def forward(self, t_abs, t_rel=None):
        """
        t_abs: (E,) absolute edge timestamps
        t_rel: (E,) relative gap = edge_time - node_last_seen_time (optional)
        Returns: (E, hidden_dim)
        """
        abs_enc = self.t2v(t_abs)                                      # (E, h/2)
        if t_rel is not None:
            rel_enc = self.rel_t2v(t_rel)                              # (E, h/2)
        else:
            rel_enc = torch.zeros_like(abs_enc)
        combined = torch.cat([abs_enc, rel_enc], dim=-1)               # (E, h)
        return self.proj(combined)                                     # (E, h)


# ═══════════════════════════════════════════════════════════════════════════════
# 2. Recurrent Node Memory with Uncertainty
# ═══════════════════════════════════════════════════════════════════════════════

class RecurrentMemory(nn.Module):
    """
    Per-node GRU hidden state updated every time a node is seen.
    Tracks variance (uncertainty) as EMA of squared delta.
    Low variance = stable node = trustworthy memory.
    """
    def __init__(self, num_nodes, dim, momentum=0.9):
        super().__init__()
        self.gru      = nn.GRUCell(dim, dim)
        self.momentum = momentum
        self.register_buffer("hidden",   torch.zeros(num_nodes, dim))
        self.register_buffer("variance", torch.ones(num_nodes, dim) * 0.1)

    def read(self, idx):
        return self.hidden[idx], self.variance[idx]                    # (B,D), (B,D)

    def get_hidden(self, idx):
        return self.hidden[idx]

    @torch.no_grad()
    def write(self, idx, x):
        h_old = self.hidden[idx]
        h_new = self.gru(x.detach(), h_old)
        delta = h_new - h_old
        self.variance[idx] = (self.momentum * self.variance[idx]
                              + (1 - self.momentum) * delta.pow(2))
        self.hidden[idx] = h_new


# ═══════════════════════════════════════════════════════════════════════════════
# 3. Neighborhood Evolution Bank
# ═══════════════════════════════════════════════════════════════════════════════

class EvolutionBank(nn.Module):
    """
    Rolling window of K most recent neighbor mean embeddings per node.
    Used by TrustModule to see HOW a node's neighborhood has evolved.
    """
    def __init__(self, num_nodes, dim, window=6):
        super().__init__()
        self.window = window
        self.register_buffer("bank", torch.zeros(num_nodes, window, dim))
        self.register_buffer("ptr",  torch.zeros(num_nodes, dtype=torch.long))

    def read(self, idx):
        return self.bank[idx]                                          # (B, W, D)

    @torch.no_grad()
    def write(self, idx, emb):
        """emb: (B, D)"""
        for b in range(idx.size(0)):
            p = int(self.ptr[idx[b]]) % self.window
            self.bank[idx[b], p] = emb[b].detach()
            self.ptr[idx[b]] += 1


# ═══════════════════════════════════════════════════════════════════════════════
# 4. Trust Module — CORE CONTRIBUTION
#
# Trust(i→j) answers: "Given how i has behaved historically and how j's
# neighborhood has evolved, should i trust a message from j?"
#
# Mechanism:
#   Query  = node i's GRU hidden state (its history)
#   Keys   = node j's neighborhood evolution window (j's recent neighbors)
#   Values = same as keys
#   Cross-attention pools the evolution window → single trust scalar
#   Uncertainty from memory variance gates the final score
# ═══════════════════════════════════════════════════════════════════════════════

class TrustModule(nn.Module):
    def __init__(self, dim, num_heads=4, window=6):
        super().__init__()
        assert dim % num_heads == 0
        self.num_heads = num_heads
        self.head_dim  = dim // num_heads
        self.scale     = self.head_dim ** -0.5
        self.window    = window

        # Project node i's history → query
        self.q_proj = nn.Linear(dim, dim)
        # Project j's evolution window → keys and values
        self.k_proj = nn.Linear(dim, dim)
        self.v_proj = nn.Linear(dim, dim)

        # Learned time-decay: older evolution steps matter less
        self.log_decay = nn.Parameter(torch.tensor(-2.0))  # starts at e^-2 ≈ 0.13

        # Final trust scalar from pooled evolution context
        self.trust_out = nn.Sequential(
            nn.Linear(dim, dim // 2),
            nn.GELU(),
            nn.Linear(dim // 2, 1)
        )

    def forward(self, h_i, evo_j, var_i, var_j, step_weights=None, eps=1e-6):
        """
        h_i:    (E, dim)    — node i GRU hidden state
        evo_j:  (E, W, dim) — node j neighborhood evolution window
        var_i:  (E, dim)    — memory variance of i (uncertainty)
        var_j:  (E, dim)    — memory variance of j
        Returns: (E,) trust scores in [0, 1]
        """
        E, W, D = evo_j.shape
        nh, hd  = self.num_heads, self.head_dim

        # Query from i's recurrent history
        Q = self.q_proj(h_i).view(E, nh, hd)                         # (E, nh, hd)

        # Keys and values from j's evolution window
        evo_flat = evo_j.reshape(E * W, D)
        K = self.k_proj(evo_flat).view(E, W, nh, hd)                 # (E, W, nh, hd)
        V = self.v_proj(evo_flat).view(E, W, nh, hd)

        # Attention scores: i's history attends over j's evolution
        Q_exp = Q.unsqueeze(2)                                        # (E, nh, 1, hd)
        K_t   = K.permute(0, 2, 1, 3)                                # (E, nh, W, hd)
        scores = (Q_exp * K_t).sum(-1) * self.scale                  # (E, nh, W)

        # Time-decay bias: step 0 is oldest, step W-1 is newest
        # Newer steps should get higher weight
        steps     = torch.arange(W, device=h_i.device).float()
        decay_w   = torch.exp(self.log_decay.exp() * steps)          # (W,) increasing
        decay_w   = decay_w / decay_w.sum()
        scores    = scores + decay_w.view(1, 1, W)                   # broadcast

        attn   = F.softmax(scores, dim=-1)                           # (E, nh, W)
        V_t    = V.permute(0, 2, 1, 3)                               # (E, nh, W, hd)
        pooled = (attn.unsqueeze(-1) * V_t).sum(2)                   # (E, nh, hd)
        pooled = pooled.reshape(E, D)                                 # (E, dim)

        # Raw trust logit
        trust_logit = self.trust_out(pooled).squeeze(-1)             # (E,)

        # Uncertainty gating:
        # High variance in i → i's history is unreliable → lower trust weight
        # High variance in j → j's neighborhood is erratic → lower trust
        conf_i = 1.0 / (1.0 + var_i.mean(-1).sqrt().clamp(min=eps)) # (E,)
        conf_j = 1.0 / (1.0 + var_j.mean(-1).sqrt().clamp(min=eps)) # (E,)
        conf   = (conf_i * conf_j).sqrt()                            # geometric mean

        return torch.sigmoid(trust_logit + conf.log().clamp(min=-5)) # (E,) in [0,1]


# ═══════════════════════════════════════════════════════════════════════════════
# 5. Temporal Trust Attention Layer
# ═══════════════════════════════════════════════════════════════════════════════

class TemporalTrustAttention(MessagePassing):
    def __init__(self, dim, heads=8, dropout=0.1, window=6):
        super().__init__(aggr="add", node_dim=0)
        assert dim % heads == 0
        self.dim      = dim
        self.heads    = heads
        self.head_dim = dim // heads
        self.scale    = self.head_dim ** -0.5
        self.window   = window

        self.q_proj    = nn.Linear(dim, dim)
        self.k_proj    = nn.Linear(dim, dim)
        self.v_proj    = nn.Linear(dim, dim)
        self.o_proj    = nn.Linear(dim, dim)
        self.edge_proj = nn.Linear(dim, dim)   # edge features → key bias
        self.time_proj = nn.Linear(dim, dim)   # time encoding → key bias

        self.trust  = TrustModule(dim, num_heads=4, window=window)
        self.dropout = nn.Dropout(dropout)

        # Gated residual
        self.gate = nn.Linear(dim, 1)
        self.norm = nn.LayerNorm(dim)

        # Fusion with recurrent hidden state
        self.fusion = nn.Sequential(
            nn.Linear(dim * 2, dim),
            nn.GELU(),
            nn.Linear(dim, dim)
        )

        self.trust_cache = None

    def split(self, x):
        return x.view(-1, self.heads, self.head_dim)

    def forward(self, x, edge_index, edge_emb, time_emb, memory, evo_bank):
        N      = x.size(0)
        device = x.device

        Q = self.split(self.q_proj(x))                                # (N, h, hd)
        K = self.split(self.k_proj(x))
        V = self.split(self.v_proj(x))

        mem_vals, mem_vars = memory.read(torch.arange(N, device=device))
        evo_vals = evo_bank.read(torch.arange(N, device=device))      # (N, W, D)

        edge_emb_proj = self.split(self.edge_proj(edge_emb))          # (E, h, hd)
        time_emb_proj = self.split(self.time_proj(time_emb))          # (E, h, hd)

        out = self.propagate(
            edge_index,
            Q=Q, K=K, V=V,
            mem_val=mem_vals,
            mem_var=mem_vars,
            hidden=mem_vals,
            evo_vals=evo_vals,
            edge_emb=edge_emb_proj,
            time_emb=time_emb_proj,
            size=(N, N)
        )

        out = out.view(N, self.dim)

        # Fuse with recurrent hidden state
        h   = memory.get_hidden(torch.arange(N, device=device))
        out = self.fusion(torch.cat([out, h], dim=-1))
        out = self.o_proj(out)

        # Gated residual
        g = torch.sigmoid(self.gate(x))
        return self.norm(g * x + (1 - g) * out)

    def message(self, Q_i, K_j, V_j,
                mem_val_i, mem_val_j,
                mem_var_i, mem_var_j,
                hidden_i, hidden_j,
                evo_vals_j,
                edge_emb, time_emb,
                index):

        # Enrich keys with edge and time features
        K_j = K_j + edge_emb + time_emb                               # (E, h, hd)

        # Attention scores
        attn = (Q_i * K_j).sum(-1) * self.scale                      # (E, h)

        # Trust score from neighborhood evolution
        # Uses i's recurrent history vs j's evolution window
        trust = self.trust(
            hidden_i, evo_vals_j,
            mem_var_i, mem_var_j
        )                                                              # (E,)
        self.trust_cache = trust.detach()

        # Modulate attention by trust
        attn = attn * trust.unsqueeze(-1)                             # (E, h)
        attn = softmax(attn, index)
        attn = self.dropout(attn)

        return V_j * attn.unsqueeze(-1)                               # (E, h, hd)


# ═══════════════════════════════════════════════════════════════════════════════
# 6. Full Model
# ═══════════════════════════════════════════════════════════════════════════════

class EnhancedTemporalGNN(nn.Module):
    def __init__(self, num_nodes, in_dim, edge_dim, hidden_dim,
                 num_layers=2, heads=8, dropout=0.1,
                 window=6, memory_momentum=0.9):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.num_nodes  = num_nodes

        self.input_proj = nn.Linear(in_dim, hidden_dim)
        self.edge_enc   = nn.Linear(edge_dim, hidden_dim)
        self.time_enc   = TimeEncoder(hidden_dim)

        self.memory   = RecurrentMemory(num_nodes, hidden_dim, momentum=memory_momentum)
        self.evo_bank = EvolutionBank(num_nodes, hidden_dim, window=window)

        self.layers = nn.ModuleList([
            TemporalTrustAttention(hidden_dim, heads=heads,
                                   dropout=dropout, window=window)
            for _ in range(num_layers)
        ])

        self.final_norm = nn.LayerNorm(hidden_dim)
        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)

    def forward(self, batch):
        N          = batch.x.size(0)
        device     = batch.x.device
        edge_index = batch.edge_index
        src, dst   = edge_index

        x        = self.input_proj(batch.x)
        edge_emb = self.edge_enc(batch.msg)

        # Per-node last-seen time for relative encoding
        node_last_t = torch.zeros(N, device=device)
        node_last_t.scatter_reduce_(
            0, src, batch.t.float(), reduce='amax', include_self=True)

        # Relative time gap per edge: how long since src was last seen
        t_rel    = batch.t.float() - node_last_t[src]                 # (E,)
        time_emb = self.time_enc(batch.t, t_rel)                      # (E, hidden)

        for layer in self.layers:
            x = layer(x, edge_index, edge_emb, time_emb,
                      self.memory, self.evo_bank)

        x = self.final_norm(x)

        # Update memory and evolution bank
        self.memory.write(torch.arange(N, device=device), x)

        # Neighborhood mean for evolution bank
        neigh = torch.zeros(N, self.hidden_dim, device=device)
        cnt   = torch.zeros(N, 1, device=device)
        neigh.scatter_add_(0, dst.unsqueeze(1).expand(-1, self.hidden_dim), x[src])
        cnt.scatter_add_(0, dst.unsqueeze(1), torch.ones(src.size(0), 1, device=device))
        neigh = neigh / (cnt + 1e-6)
        self.evo_bank.write(dst.unique(), neigh[dst.unique()])

        return x



In [9]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import MessagePassing
from torch_geometric.utils import softmax
import math

class TimeEncoder(nn.Module):
    """Encodes timestamps into continuous vector representations to act as Keys."""
    def __init__(self, out_channels):
        super().__init__()
        self.out_channels = out_channels
        self.lin = nn.Linear(1, out_channels)

    def forward(self, t):
        # t shape: [E] -> [E, 1]
        return torch.cos(self.lin(t.view(-1, 1)))

class TrustAwareTemporalAttention(MessagePassing):
    """
    Neighborhood-aware attention that handles PyG TemporalData.
    """
    def __init__(self, node_in_dim, time_dim, msg_dim, out_dim):
        super().__init__(aggr='add', node_dim=0)
        
        # 1. Project 1D node features to a higher dimension for meaningful Cosine Similarity
        self.x_proj = nn.Linear(node_in_dim, out_dim)
        
        # 2. Linear layers for Q, K, V
        self.W_q = nn.Linear(out_dim, out_dim)
        self.W_k = nn.Linear(time_dim, out_dim)
        self.W_v = nn.Linear(msg_dim, out_dim)
        
        self.out_dim = out_dim

    def forward(self, x, src, dst, t_enc, msg):
        # Create edge_index from src and dst [2, E]
        edge_index = torch.stack([src, dst], dim=0)
        
        # Elevate node features to richer space
        x_proj = self.x_proj(x)
        
        # Query comes from target nodes
        Q = self.W_q(x_proj)
        
        return self.propagate(edge_index, Q=Q, x_proj=x_proj, t_enc=t_enc, msg=msg)

    def message(self, edge_index_i, edge_index_j, Q_i, x_proj_i, x_proj_j, t_enc, msg, index, ptr, size_i):
        # Key comes from the time encoder
        K = self.W_k(t_enc)
        # Value comes from the raw 172-dim messages
        V = self.W_v(msg)

        # Base Scaled Dot-Product Attention
        attention_score = (Q_i * K).sum(dim=-1) / math.sqrt(self.out_dim)
        
        # Trust Mechanism: Cosine similarity in the projected space
        sim = F.cosine_similarity(x_proj_i, x_proj_j, dim=-1)
        trust_weight = (1.0 + sim) / 2.0 
        
        # Modulate attention
        weighted_score = attention_score * trust_weight
        
        # Normalize across the neighborhood
        alpha = softmax(weighted_score, index, ptr, size_i)
        
        return V * alpha.unsqueeze(-1)


class TemporalAnomalyEncoder(nn.Module):
    """
    Wraps the attention mechanism to process the full TemporalData batch.
    """
    def __init__(self, node_in_dim=1, msg_dim=172, time_dim=64, hidden_dim=128):
        super().__init__()
        self.time_encoder = TimeEncoder(time_dim)
        self.attn = TrustAwareTemporalAttention(node_in_dim, time_dim, msg_dim, hidden_dim)
        
        # Optional: A final projection layer before contrastive learning
        self.final_proj = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim)
        )

    def forward(self, batch):
        # Extract from TemporalData
        x = batch.x.float()      # [599, 1]
        src = batch.src.long()   # [3321]
        dst = batch.dst.long()   # [3321]
        t = batch.t.float()      # [3321]
        msg = batch.msg.float()  # [3321, 172]
        
        # 1. Get Temporal Features (Acts as Key)
        t_enc = self.time_encoder(t) 
        
        # 2. Apply Trust-Aware Temporal Attention
        node_reps = self.attn(x, src, dst, t_enc, msg)
        
        # 3. Final projection for the contrastive loss space
        final_embeddings = self.final_proj(node_reps)
        
        # Slice out the embeddings for the target batch nodes (input_id)
        # Assuming batch.batch_size = 256, the first 256 nodes are the targets
        target_embeddings = final_embeddings[:batch.batch_size]
        
        return target_embeddings

In [10]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import MessagePassing
from torch_geometric.utils import softmax


# ═══════════════════════════════════════════════════════════════════════════════
# 1. Time2Vec — learns periodic + trend components from raw timestamps
# ═══════════════════════════════════════════════════════════════════════════════

class Time2Vec(nn.Module):
    """
    t → [w0*t + b0,  sin(w1*t + b1), ..., sin(wk*t + bk)]
    Frequencies are LEARNED — model discovers daily/weekly patterns.
    """
    def __init__(self, out_dim):
        super().__init__()
        k = out_dim - 1
        self.w0 = nn.Parameter(torch.randn(1) * 0.01)
        self.b0 = nn.Parameter(torch.zeros(1))
        self.W  = nn.Parameter(torch.randn(k) * 0.01)
        self.B  = nn.Parameter(torch.zeros(k))
        self.out_dim = out_dim

    def forward(self, t):
        t        = t.float().unsqueeze(-1)
        linear   = t * self.w0 + self.b0
        periodic = torch.sin(t * self.W + self.B)
        return torch.cat([linear, periodic], dim=-1)


class TimeEncoder(nn.Module):
    """
    Encodes both absolute timestamp and relative inter-step time delta.
    The inter-step delta tells the model HOW FAR APART two interactions are in time.
    """
    def __init__(self, hidden_dim):
        super().__init__()
        self.t2v_abs   = Time2Vec(hidden_dim // 2)
        self.t2v_delta = Time2Vec(hidden_dim // 2)  # inter-step delta encoding
        self.proj = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, hidden_dim)
        )

    def forward(self, t_abs, t_delta=None):
        """
        t_abs:   (E,) absolute timestamps
        t_delta: (E,) time gap since previous snapshot step or last interaction
        Returns: (E, hidden_dim)
        """
        abs_enc   = self.t2v_abs(t_abs)
        delta_enc = self.t2v_delta(t_delta) if t_delta is not None \
                    else torch.zeros_like(abs_enc)
        return self.proj(torch.cat([abs_enc, delta_enc], dim=-1))


# ═══════════════════════════════════════════════════════════════════════════════
# 2. Recurrent Node Memory with Uncertainty
# ═══════════════════════════════════════════════════════════════════════════════

class RecurrentMemory(nn.Module):
    """
    Per-node GRU memory updated at every interaction.
    Tracks variance (uncertainty): low variance = stable, trustworthy node.
    """
    def __init__(self, num_nodes, dim, momentum=0.9):
        super().__init__()
        self.gru      = nn.GRUCell(dim, dim)
        self.momentum = momentum
        self.register_buffer("hidden",   torch.zeros(num_nodes, dim))
        self.register_buffer("variance", torch.ones(num_nodes, dim) * 0.1)
        self.register_buffer("last_t",   torch.zeros(num_nodes))   # last seen time

    def read(self, idx):
        return self.hidden[idx], self.variance[idx]

    def get_hidden(self, idx):
        return self.hidden[idx]

    def get_last_t(self, idx):
        return self.last_t[idx]

    @torch.no_grad()
    def write(self, idx, x, t=None):
        h_old = self.hidden[idx]
        h_new = self.gru(x.detach(), h_old)
        delta = h_new - h_old
        self.variance[idx] = (self.momentum * self.variance[idx]
                              + (1 - self.momentum) * delta.pow(2))
        self.hidden[idx] = h_new
        if t is not None:
            self.last_t[idx] = t.float()


# ═══════════════════════════════════════════════════════════════════════════════
# 3. K-Step Temporal Snapshot Builder
#
# Splits edges into K consecutive temporal windows.
# Each window produces one "view" of every node's representation.
# The views encode WHAT a node looks like at each moment in time.
# ═══════════════════════════════════════════════════════════════════════════════

def build_temporal_snapshots(edge_index, edge_t, edge_msg, num_snapshots):
    """
    Partition edges into K equal-width time windows.
    Returns list of (edge_index_k, edge_t_k, edge_msg_k, t_min_k, t_max_k).
    """
    t_min, t_max = edge_t.min().item(), edge_t.max().item()
    if t_min == t_max:
        t_max = t_min + 1.0
    bins  = torch.linspace(t_min, t_max, num_snapshots + 1, device=edge_t.device)
    snaps = []
    for k in range(num_snapshots):
        lo, hi = bins[k], bins[k + 1]
        mask   = (edge_t >= lo) & (edge_t < hi if k < num_snapshots - 1 else edge_t <= hi)
        snaps.append((
            edge_index[:, mask],
            edge_t[mask],
            edge_msg[mask],
            lo,
            hi
        ))
    return snaps


# ═══════════════════════════════════════════════════════════════════════════════
# 4. Recurrent View Integrator with Time-Delta GRU
#
# Given K snapshot embeddings and their timestamps, this module runs a GRU
# that receives the inter-step time delta as part of its input.
# This allows the model to know that "3 days passed between steps 2 and 3"
# rather than treating all steps as uniformly spaced.
# ═══════════════════════════════════════════════════════════════════════════════

class TemporalViewIntegrator(nn.Module):
    """
    Integrates K node snapshot views into a single final representation
    using a GRU that explicitly receives inter-step time deltas.

    Input sequence:  [(h_1, Δt_1), (h_2, Δt_2), ..., (h_K, Δt_K)]
    where Δt_k = t_k - t_{k-1}  (inter-step time gap)

    The time delta is encoded via Time2Vec and concatenated with the
    snapshot embedding before each GRU step, so the model learns HOW
    time gaps modulate representation evolution.
    """
    def __init__(self, dim, num_views):
        super().__init__()
        self.dim       = dim
        self.num_views = num_views
        self.t2v_delta = Time2Vec(dim // 2)
        self.delta_proj = nn.Linear(dim // 2, dim)
        # GRU input = snapshot_emb + time_delta_enc
        self.gru        = nn.GRUCell(dim * 2, dim)
        # Multi-head attention pooling over all K views for final representation
        self.view_attn  = nn.MultiheadAttention(dim, num_heads=4, batch_first=True)
        self.cls_token  = nn.Parameter(torch.randn(1, 1, dim) * 0.02)
        self.out_norm   = nn.LayerNorm(dim)

    def forward(self, view_embs, view_times):
        """
        view_embs:  (N, K, dim)  — node embeddings at each snapshot step
        view_times: (K,)         — representative timestamp of each snapshot
        Returns:    (N, dim)     — final fused node representation
        """
        N, K, D = view_embs.shape
        device  = view_embs.device

        # Compute inter-step time deltas: Δt_k = t_k - t_{k-1}, Δt_0 = 0
        deltas    = torch.zeros(K, device=device)
        deltas[1:] = (view_times[1:] - view_times[:-1]).clamp(min=0)

        # GRU over views with time-delta conditioning
        h = torch.zeros(N, D, device=device)
        gru_out = []
        for k in range(K):
            delta_enc = self.delta_proj(self.t2v_delta(deltas[k:k+1].expand(N)))  # (N, D)
            inp       = torch.cat([view_embs[:, k, :], delta_enc], dim=-1)         # (N, 2D)
            h         = self.gru(inp, h)                                            # (N, D)
            gru_out.append(h)

        # Stack all GRU hidden states: (N, K, D)
        gru_seq = torch.stack(gru_out, dim=1)

        # Attention pooling: CLS token attends over all K views
        # This lets the model weight which snapshot steps matter most
        cls    = self.cls_token.expand(N, -1, -1)                      # (N, 1, D)
        seq    = torch.cat([cls, gru_seq], dim=1)                       # (N, K+1, D)
        out, _ = self.view_attn(seq, seq, seq)
        final  = self.out_norm(out[:, 0, :])                            # (N, D) from CLS

        return final


# ═══════════════════════════════════════════════════════════════════════════════
# 5. Neighborhood Evolution Bank
# ═══════════════════════════════════════════════════════════════════════════════

class EvolutionBank(nn.Module):
    """
    Rolling window of K most recent neighbor mean embeddings per node.
    Used by TrustModule to see HOW a node's neighborhood has evolved.
    """
    def __init__(self, num_nodes, dim, window=6):
        super().__init__()
        self.window = window
        self.register_buffer("bank", torch.zeros(num_nodes, window, dim))
        self.register_buffer("ptr",  torch.zeros(num_nodes, dtype=torch.long))

    def read(self, idx):
        return self.bank[idx]

    @torch.no_grad()
    def write(self, idx, emb):
        for b in range(idx.size(0)):
            p = int(self.ptr[idx[b]]) % self.window
            self.bank[idx[b], p] = emb[b].detach()
            self.ptr[idx[b]] += 1


# ═══════════════════════════════════════════════════════════════════════════════
# 6. Trust Module — anomaly-robust neighbor gating
#
# Trust(i→j) answers: given i's history and j's neighborhood evolution,
# should i trust a message from j?
#
# ANOMALY ROBUSTNESS:
# - If j is anomalous: its neighborhood evolution will be erratic (high variance)
#   → conf_j drops → trust drops → j's messages are down-weighted.
# - If i is anomalous: its own history is erratic (high var_i)
#   → conf_i drops → lower reliance on memory for gating
#   → IMPORTANTLY: anomalous i still gets proper attention (not suppressed),
#     but its trust gate is less confident so it won't blindly amplify neighbors.
# ═══════════════════════════════════════════════════════════════════════════════

class TrustModule(nn.Module):
    def __init__(self, dim, num_heads=4, window=6):
        super().__init__()
        self.num_heads = num_heads
        self.head_dim  = dim // num_heads
        self.scale     = self.head_dim ** -0.5
        self.window    = window

        self.q_proj  = nn.Linear(dim, dim)
        self.k_proj  = nn.Linear(dim, dim)
        self.v_proj  = nn.Linear(dim, dim)

        self.log_decay = nn.Parameter(torch.tensor(-2.0))

        self.trust_out = nn.Sequential(
            nn.Linear(dim, dim // 2),
            nn.GELU(),
            nn.Linear(dim // 2, 1)
        )

    def forward(self, h_i, evo_j, var_i, var_j, eps=1e-6):
        E, W, D = evo_j.shape
        nh, hd  = self.num_heads, self.head_dim

        Q      = self.q_proj(h_i).view(E, nh, hd)
        K      = self.k_proj(evo_j.reshape(E * W, D)).view(E, W, nh, hd)
        V      = self.v_proj(evo_j.reshape(E * W, D)).view(E, W, nh, hd)

        Q_exp  = Q.unsqueeze(2)
        K_t    = K.permute(0, 2, 1, 3)
        scores = (Q_exp * K_t).sum(-1) * self.scale

        steps   = torch.arange(W, device=h_i.device).float()
        decay_w = torch.exp(self.log_decay.exp() * steps)
        decay_w = decay_w / decay_w.sum()
        scores  = scores + decay_w.view(1, 1, W)

        attn   = F.softmax(scores, dim=-1)
        V_t    = V.permute(0, 2, 1, 3)
        pooled = (attn.unsqueeze(-1) * V_t).sum(2).reshape(E, D)

        trust_logit = self.trust_out(pooled).squeeze(-1)

        conf_i = 1.0 / (1.0 + var_i.mean(-1).sqrt().clamp(min=eps))
        conf_j = 1.0 / (1.0 + var_j.mean(-1).sqrt().clamp(min=eps))
        conf   = (conf_i * conf_j).sqrt()

        return torch.sigmoid(trust_logit + conf.log().clamp(min=-5))


# ═══════════════════════════════════════════════════════════════════════════════
# 7. Similarity-Gated Temporal Trust Attention Layer
#
# Improvements over baseline:
#   (a) SIMILARITY GATE: cosine similarity between i and j suppresses
#       structurally-dissimilar (potentially anomalous) neighbors.
#       This prevents normal nodes from being "poisoned" by anomalous neighbors.
#   (b) ANOMALY EXPOSURE: the similarity gate is SOFT, not hard.
#       Anomalous nodes still receive some signal from their neighbors,
#       preventing them from being camouflaged as normal nodes.
#       Their dissimilarity from normal neighbors is reflected in lower gate,
#       which indirectly flags them as structurally different.
#   (c) Trust + Similarity are combined multiplicatively: both must agree
#       for a message to have high weight.
# ═══════════════════════════════════════════════════════════════════════════════

class SimilarityGatedTrustAttention(MessagePassing):
    def __init__(self, dim, heads=8, dropout=0.1, window=6, sim_temp=0.5):
        super().__init__(aggr="add", node_dim=0)
        self.dim      = dim
        self.heads    = heads
        self.head_dim = dim // heads
        self.scale    = self.head_dim ** -0.5
        self.window   = window
        self.sim_temp = sim_temp   # temperature for similarity gate

        self.q_proj    = nn.Linear(dim, dim)
        self.k_proj    = nn.Linear(dim, dim)
        self.v_proj    = nn.Linear(dim, dim)
        self.o_proj    = nn.Linear(dim, dim)
        self.edge_proj = nn.Linear(dim, dim)
        self.time_proj = nn.Linear(dim, dim)

        self.trust   = TrustModule(dim, num_heads=4, window=window)
        self.dropout = nn.Dropout(dropout)

        # Learnable balance between trust and similarity
        self.gate_balance = nn.Parameter(torch.tensor(0.5))

        self.gate   = nn.Linear(dim, 1)
        self.norm   = nn.LayerNorm(dim)

        self.fusion = nn.Sequential(
            nn.Linear(dim * 2, dim),
            nn.GELU(),
            nn.Linear(dim, dim)
        )

        self.trust_cache = None
        self.sim_cache   = None

    def split(self, x):
        return x.view(-1, self.heads, self.head_dim)

    def forward(self, x, edge_index, edge_emb, time_emb, memory, evo_bank):
        N      = x.size(0)
        device = x.device

        Q = self.split(self.q_proj(x))
        K = self.split(self.k_proj(x))
        V = self.split(self.v_proj(x))

        mem_vals, mem_vars = memory.read(torch.arange(N, device=device))
        evo_vals           = evo_bank.read(torch.arange(N, device=device))

        edge_emb_proj = self.split(self.edge_proj(edge_emb))
        time_emb_proj = self.split(self.time_proj(time_emb))

        out = self.propagate(
            edge_index,
            Q=Q, K=K, V=V,
            x=x,
            mem_val=mem_vals,
            mem_var=mem_vars,
            hidden=mem_vals,
            evo_vals=evo_vals,
            edge_emb=edge_emb_proj,
            time_emb=time_emb_proj,
            size=(N, N)
        )

        out = out.view(N, self.dim)
        h   = memory.get_hidden(torch.arange(N, device=device))
        out = self.fusion(torch.cat([out, h], dim=-1))
        out = self.o_proj(out)

        g = torch.sigmoid(self.gate(x))
        return self.norm(g * x + (1 - g) * out)

    def message(self, Q_i, K_j, V_j, x_i, x_j,
                mem_val_i, mem_val_j,
                mem_var_i, mem_var_j,
                hidden_i,
                evo_vals_j,
                edge_emb, time_emb,
                index):

        K_j = K_j + edge_emb + time_emb

        # — Standard attention scores —
        attn = (Q_i * K_j).sum(-1) * self.scale          # (E, heads)

        # — Trust gate (from memory + evolution history) —
        trust = self.trust(hidden_i, evo_vals_j, mem_var_i, mem_var_j)  # (E,)
        self.trust_cache = trust.detach()

        # — Similarity gate (structural proximity in current embedding space) —
        # Cosine similarity between node i and node j in current step.
        # Anomalous nodes are structurally distant from their normal neighbors
        # → low similarity → down-weighted in aggregation.
        # This prevents normal nodes from being poisoned by anomalous neighbors.
        # Soft gating (not hard threshold) keeps anomalous nodes "visible":
        # their low similarity score flags them during downstream classification.
        sim = F.cosine_similarity(x_i, x_j, dim=-1)           # (E,)  in [-1, 1]
        sim = torch.sigmoid(sim / self.sim_temp)               # (E,)  in  (0, 1)
        self.sim_cache = sim.detach()

        # — Combine trust and similarity —
        # Learnable balance: in early training, equal weight; model adapts.
        alpha    = torch.sigmoid(self.gate_balance)
        combined = alpha * trust + (1 - alpha) * sim           # (E,)

        attn = attn * combined.unsqueeze(-1)                   # (E, heads)
        attn = softmax(attn, index)
        attn = self.dropout(attn)

        return V_j * attn.unsqueeze(-1)                        # (E, heads, head_dim)


# ═══════════════════════════════════════════════════════════════════════════════
# 8. Full Model — Multi-View Temporal GNN
#
# Pipeline per forward pass:
#   1. Split edges into K temporal snapshot windows
#   2. For each snapshot k:
#        a. Encode time (absolute + delta since prior snapshot)
#        b. Run SimilarityGatedTrustAttention to get node emb at snapshot k
#   3. TemporalViewIntegrator:
#        - GRU with inter-step time deltas integrates views sequentially
#        - CLS-token attention pooling produces final representation
#   4. Update recurrent memory + evolution bank
# ═══════════════════════════════════════════════════════════════════════════════

class MultiViewTemporalGNN(nn.Module):
    def __init__(self, num_nodes, in_dim, edge_dim, hidden_dim,
                 num_layers=2, heads=8, dropout=0.1,
                 num_snapshots=6, window=6, memory_momentum=0.9,
                 sim_temp=0.5):
        super().__init__()
        self.hidden_dim    = hidden_dim
        self.num_nodes     = num_nodes
        self.num_snapshots = num_snapshots

        self.input_proj = nn.Linear(in_dim, hidden_dim)
        self.edge_enc   = nn.Linear(edge_dim, hidden_dim)
        self.time_enc   = TimeEncoder(hidden_dim)

        self.memory   = RecurrentMemory(num_nodes, hidden_dim, momentum=memory_momentum)
        self.evo_bank = EvolutionBank(num_nodes, hidden_dim, window=window)

        # Shared attention layers applied at each snapshot step
        self.layers = nn.ModuleList([
            SimilarityGatedTrustAttention(
                hidden_dim, heads=heads, dropout=dropout,
                window=window, sim_temp=sim_temp
            )
            for _ in range(num_layers)
        ])

        # Multi-view integrator: fuses K snapshot representations
        self.view_integrator = TemporalViewIntegrator(hidden_dim, num_snapshots)

        self.final_norm = nn.LayerNorm(hidden_dim)
        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)

    def _run_snapshot(self, x, snap_edge_index, snap_t, snap_msg,
                      t_prev_center, device):
        """
        Process one temporal snapshot:
        - Encode time with delta from prior snapshot's center time
        - Run all attention layers
        Returns updated node embeddings (N, D)
        """
        if snap_edge_index.size(1) == 0:
            # Empty snapshot — return unchanged embeddings
            return x

        edge_emb = self.edge_enc(snap_msg)

        # Per-node last-seen time for relative within-snapshot encoding
        N = x.size(0)
        src = snap_edge_index[0]
        node_last_t = torch.zeros(N, device=device)
        node_last_t.scatter_reduce_(
            0, src, snap_t.float(), reduce='amax', include_self=True)
        t_rel = snap_t.float() - node_last_t[src]

        # Inter-snapshot delta: how much time has passed since last snapshot
        snap_center = snap_t.float().mean()
        t_delta_snap = (snap_center - t_prev_center).unsqueeze(0).expand(snap_t.size(0))

        # Time encoding = absolute + within-snapshot relative + inter-snapshot delta
        # We combine within-snapshot relative (t_rel) and inter-snapshot delta:
        # t_rel captures fine-grained within-step ordering
        # t_delta_snap captures coarse between-step gaps (fed to view integrator later)
        time_emb = self.time_enc(snap_t, t_rel)

        for layer in self.layers:
            x = layer(x, snap_edge_index, edge_emb, time_emb,
                      self.memory, self.evo_bank)

        return x, snap_center

    def forward(self, batch):
        N      = batch.x.size(0)
        device = batch.x.device

        x = self.input_proj(batch.x)

        # ── Split edges into K temporal snapshots ──
        snaps = build_temporal_snapshots(
            batch.edge_index, batch.t, batch.msg, self.num_snapshots
        )

        # ── Process each snapshot, collect views ──
        view_embs  = []   # node embeddings per snapshot (N, D)
        view_times = []   # representative timestamp per snapshot
        t_prev     = batch.t.float().min()

        for snap_edge_index, snap_t, snap_msg, t_lo, t_hi in snaps:
            x_snap, snap_center = self._run_snapshot(
                x.clone(),   # each snapshot starts from same base x
                snap_edge_index, snap_t, snap_msg,
                t_prev, device
            )
            view_embs.append(x_snap)
            view_times.append(snap_center)
            t_prev = snap_center

        # ── Stack views: (N, K, D) ──
        view_embs  = torch.stack(view_embs, dim=1)           # (N, K, D)
        view_times = torch.stack(view_times)                  # (K,)

        # ── Fuse K views with recurrent time-delta integration ──
        # GRU with inter-step Δt + attention pooling → final representation
        x_final = self.view_integrator(view_embs, view_times)
        x_final = self.final_norm(x_final)

        # ── Update recurrent memory and evolution bank ──
        t_per_node = torch.zeros(N, device=device)
        t_per_node.scatter_reduce_(
            0, batch.edge_index[0], batch.t.float(), reduce='amax', include_self=True)
        self.memory.write(torch.arange(N, device=device), x_final, t_per_node)

        # Neighborhood mean for evolution bank
        src, dst = batch.edge_index
        neigh = torch.zeros(N, self.hidden_dim, device=device)
        cnt   = torch.zeros(N, 1, device=device)
        neigh.scatter_add_(0, dst.unsqueeze(1).expand(-1, self.hidden_dim), x_final[src])
        cnt.scatter_add_(0, dst.unsqueeze(1), torch.ones(src.size(0), 1, device=device))
        neigh = neigh / (cnt + 1e-6)
        unique_dst = dst.unique()
        self.evo_bank.write(unique_dst, neigh[unique_dst])

        return x_final


# ═══════════════════════════════════════════════════════════════════════════════
# 9. Contrastive Projection Head (for downstream unsupervised anomaly detection)
# ═══════════════════════════════════════════════════════════════════════════════

class ContrastiveHead(nn.Module):
    """
    Standard 2-layer MLP projection head for contrastive learning.
    Projects node embeddings into a normalized space where contrastive
    loss is applied (e.g. NT-Xent, InfoNCE, or graph-level variants).
    """
    def __init__(self, hidden_dim, proj_dim=128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.GELU(),
            nn.LayerNorm(hidden_dim),
            nn.Linear(hidden_dim, proj_dim)
        )

    def forward(self, x):
        return F.normalize(self.net(x), dim=-1)


class FullModel(nn.Module):
    """
    Complete model = MultiViewTemporalGNN + ContrastiveHead.
    Call forward() to get node representations.
    Call project() to get contrastive projection for loss computation.
    """
    def __init__(self, num_nodes, in_dim, edge_dim, hidden_dim,
                 proj_dim=128, **gnn_kwargs):
        super().__init__()
        self.gnn  = MultiViewTemporalGNN(
            num_nodes, in_dim, edge_dim, hidden_dim, **gnn_kwargs
        )
        self.head = ContrastiveHead(hidden_dim, proj_dim)

    def forward(self, batch):
        return self.gnn(batch)

    def project(self, batch):
        h = self.gnn(batch)
        return h, self.head(h)

In [ ]:
"""
Drop-in replacement for your notebook.a
Keeps: data, loaders, History, args, measure
Replaces: model, train(), test()

Just run these cells in order after your existing setup cells.
"""

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import MessagePassing
from torch_geometric.utils import softmax
from tqdm import tqdm


# ── Paste the full enhanced_gnn_v2.py here, or import it ──────────────────────
# from enhanced_gnn_v2 import (EnhancedTemporalGNN, train, test)
# ──────────────────────────────────────────────────────────────────────────────
# If running directly in notebook, paste the class definitions above this cell.


# ═══════════════════════════════════════════
# CELL: Build model + history (unchanged)
# ═══════════════════════════════════════════

try:
    del model
except NameError:
    pass



model = MultiViewTemporalGNN(
    num_nodes  = data.num_nodes,
    in_dim     = data.x.size(1),          # 1 for wikipedia
    edge_dim   = data.msg.size(1),         # 172 for wikipedia
    hidden_dim = args.hidden_channels,     # 64
    num_layers = args.nums_layers,         # 2
    heads      = args.heads,               # 8
    dropout    = args.dropout,             # 0.3
    window     = 6,
    num_snapshots=6,
    memory_momentum = 0.9,
).to(device)

# History is UNCHANGED — reuse your existing History class
try:
    del history
except NameError:
    pass

from mhiscl.history import History

history = History(
    data.num_nodes,
    args.num_timeslots,
    args.hidden_channels,
    device=device,
    history_retrieve=args.history_retrieve,
    recurrent=args.recurrent,
).to(device)

print(model)
print(history)
print(f"Model params: {sum(p.numel() for p in model.parameters()):,}")




MultiViewTemporalGNN(
  (input_proj): Linear(in_features=1, out_features=64, bias=True)
  (edge_enc): Linear(in_features=172, out_features=64, bias=True)
  (time_enc): TimeEncoder(
    (t2v_abs): Time2Vec()
    (t2v_delta): Time2Vec()
    (proj): Sequential(
      (0): Linear(in_features=64, out_features=64, bias=True)
      (1): GELU(approximate='none')
      (2): Linear(in_features=64, out_features=64, bias=True)
    )
  )
  (memory): RecurrentMemory(
    (gru): GRUCell(64, 64)
  )
  (evo_bank): EvolutionBank()
  (layers): ModuleList(
    (0-1): 2 x SimilarityGatedTrustAttention()
  )
  (view_integrator): TemporalViewIntegrator(
    (t2v_delta): Time2Vec()
    (delta_proj): Linear(in_features=32, out_features=64, bias=True)
    (gru): GRUCell(128, 64)
    (view_attn): MultiheadAttention(
      (out_proj): NonDynamicallyQuantizableLinear(in_features=64, out_features=64, bias=True)
    )
    (out_norm): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
  )
  (final_norm): LayerNorm(

In [8]:
## Training Function 

def train(loader, model, history, optimizer, device, data, args):
    model.train()
    history.train()
    pbar = tqdm(loader)
    total_loss = 0
    for batch in pbar:
        batch = batch.to(device)
        optimizer.zero_grad()
        out = model(batch)
        src = batch.src[: batch.batch_size]
        out = out[src]
        idx = batch.input_id.cpu()
        num_neg = batch.batch_size
        neg_mem = history.get_history(torch.randint(
            0, history.num_nodes, size=(num_neg,)))
        raw_msg = torch.zeros(idx.size(0),1).to(device) 
        cur_mem, prev_mem = history(out, data.src[idx]) 

        if args.con=='his':
            loss = contrastive_loss(prev_mem, cur_mem, args.tau) 
            loss += torch.exp(cosine_similarity(cur_mem, neg_mem) 
                          ).sum(dim=1).log().mean()
        elif args.con=='stru':
            loss = contrastive_loss(out, cur_mem, args.tau) 
            loss += torch.exp(cosine_similarity(out, neg_mem)
                            ).sum(dim=1).log().mean()
        elif args.con=='all':
            loss = args.alpha*contrastive_loss(out, cur_mem, args.tau) + \
                contrastive_loss(prev_mem, cur_mem, args.tau)
            loss += torch.exp(cosine_similarity(cur_mem, neg_mem) 
                            ).sum(dim=1).log().mean()
            loss += args.alpha*torch.exp(cosine_similarity(out, neg_mem) 
                            ).sum(dim=1).log().mean()


        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        pbar.set_description(f"Train Loss = {loss.item():.4f}")

    return total_loss / len(loader)


## Test function

@torch.no_grad()
def test(loader, model, history, measure, device, data):
    preds = []
    labels = []
    model.eval()
    history.eval()
    for batch in tqdm(loader):
        torch.cuda.empty_cache()
        batch = batch.to(device)
        out = model(batch)
        src = batch.src[: batch.batch_size]
        out = out[src]
        label = batch.y[: batch.batch_size]
        idx = batch.input_id.cpu()
        raw_msg = torch.zeros(idx.size(0),1).to(device) 
        cur_mem, prev_mem = history(out, data.src[idx], update=True)
        if args.con=='his':
            pred = 1-torch.diag(cosine_similarity(cur_mem, prev_mem)).view(-1)
        elif args.con=='stru': 
            pred = 1-torch.diag(cosine_similarity(out, cur_mem)).view(-1)
        elif args.con=='all':
            p1 = torch.diag(cosine_similarity(out, cur_mem)).view(-1)
            p2 = torch.diag(cosine_similarity(cur_mem, prev_mem)).view(-1)
            pred = (1 - p1) + (1 - p2) 
            
        pred = pred.sigmoid() 
        torch.cuda.empty_cache()
        preds.append(pred.detach().cpu())
        labels.append(label.detach().cpu())
    preds = torch.cat(preds)
    preds[torch.isnan(preds)] = 0.0 
    labels = torch.cat(labels)
    return measure(labels[labels!=2], preds[labels!=2]) 




In [27]:
# ═══════════════════════════════════════════
# CELL: Training loop (drop-in replacement)
# ═══════════════════════════════════════════

best     = 0
best_val = 0

optimizer = torch.optim.Adam(
    list(model.parameters()) + list(history.parameters()),
    lr=args.lr,
    weight_decay=1e-5       # small L2 reg helps generalization
)

# Cosine LR schedule: smoothly decays over training
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=args.epochs, eta_min=args.lr * 0.1)

measure = Measure("auc")

for epoch in range(1, 1 + args.epochs):
    loss    = train(train_loader, model, history, optimizer, device, data, args)
    val_auc = test(valid_loader,  model, history, measure,   device, data)
    test_auc = test(test_loader,  model, history, measure,   device, data)
    scheduler.step()

    if val_auc > best_val:
        best_val = val_auc
        best     = test_auc

    print(
        f"Epoch: {epoch:03d}, Loss: {loss:.4f}, "
        f"Val AUC: {val_auc:.2%}, Test AUC: {test_auc:.2%}, Best AUC: {best:.2%}"
    )

100%|██████████| 93/93 [00:07<00:00, 12.75it/s]


Epoch: 001, Loss: 20.9191, Val AUC: 72.32%, Test AUC: 70.96%, Best AUC: 70.96%


100%|██████████| 93/93 [00:07<00:00, 12.76it/s]


Epoch: 002, Loss: 18.7347, Val AUC: 84.70%, Test AUC: 86.77%, Best AUC: 86.77%


100%|██████████| 93/93 [00:07<00:00, 12.92it/s]


...

Epoch: 006, Loss: 17.1335, Val AUC: 91.41%, Test AUC: 90.56%, Best AUC: 90.56%


100%|██████████| 93/93 [00:07<00:00, 13.04it/s]


Epoch: 007, Loss: 17.0007, Val AUC: 85.31%, Test AUC: 86.48%, Best AUC: 90.56%


Train Loss = 16.9515:  32%|███▏      | 137/431 [00:25<00:54,  5.42it/s]


KeyboardInterrupt: 

In [6]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import MessagePassing
from torch_geometric.utils import softmax


# ═══════════════════════════════════════════════════════════════════════════════
# 1. Time2Vec + TimeEncoder
# ═══════════════════════════════════════════════════════════════════════════════

class Time2Vec(nn.Module):
    def __init__(self, out_dim):
        super().__init__()
        k = out_dim - 1
        self.w0 = nn.Parameter(torch.randn(1) * 0.01)
        self.b0 = nn.Parameter(torch.zeros(1))
        self.W  = nn.Parameter(torch.randn(k) * 0.01)
        self.B  = nn.Parameter(torch.zeros(k))

    def forward(self, t):                                   # t: (N,)
        t        = t.float().unsqueeze(-1)                  # (N,1)
        linear   = t * self.w0 + self.b0                   # (N,1)
        periodic = torch.sin(t * self.W + self.B)          # (N,k)
        return torch.cat([linear, periodic], dim=-1)        # (N, out_dim)


class TimeEncoder(nn.Module):
    """
    Encodes absolute timestamp + inter-event delta (time since previous
    event for the same node).  Both are needed:
      - absolute t  → where in calendar time this event sits
      - delta t     → rhythm / burstiness of this node's activity
    """
    def __init__(self, hidden_dim):
        super().__init__()
        self.t2v_abs   = Time2Vec(hidden_dim // 2)
        self.t2v_delta = Time2Vec(hidden_dim // 2)
        self.proj = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, hidden_dim)
        )

    def forward(self, t_abs, t_delta=None):                 # both (E,)
        abs_enc   = self.t2v_abs(t_abs)
        delta_enc = self.t2v_delta(t_delta) if t_delta is not None \
                    else torch.zeros_like(abs_enc)
        return self.proj(torch.cat([abs_enc, delta_enc], dim=-1))   # (E, D)


# ═══════════════════════════════════════════════════════════════════════════════
# 2. Per-Node Activity Snapshot Builder
#
# KEY DESIGN DECISION:
#   Snapshots are built per node from that node's OWN interaction history.
#   Node i's K views reflect how i itself evolved, not global time bins.
#
#   Two strategies (controlled by `mode`):
#     'equal_count' — each view covers the same number of events
#                     good when activity is bursty / uneven
#     'equal_time'  — each view covers the same time span
#                     good when you care about calendar rhythm
#
#   Returns for each node:
#     view_edge_masks: list of K boolean masks over the node's local edges
#     view_t_centers:  (K,) representative timestamp per view
# ═══════════════════════════════════════════════════════════════════════════════

def build_per_node_snapshots(edge_index, edge_t, num_nodes, num_snapshots,
                              mode='equal_count'):
    """
    For every node i, partition its edges into K views.

    Returns:
        node_view_masks : list of length num_nodes, each entry is a
                          LongTensor of shape (K, E_i) — boolean mask
                          over that node's edges for each view.
                          We store as list[node] → list[view] → edge indices.
        node_view_centers: (num_nodes, K) — center timestamp of each view.
    """
    src = edge_index[0]
    E   = edge_index.size(1)

    # For each node, collect sorted edge indices
    # node_edges[i] = sorted indices into the global edge list
    node_edges = [[] for _ in range(num_nodes)]
    for e in range(E):
        node_edges[src[e].item()].append(e)
    # Also include edges where node is destination (it receives messages)
    dst = edge_index[1]
    for e in range(E):
        d = dst[e].item()
        if e not in node_edges[d]:        # avoid double-counting
            node_edges[d].append(e)

    node_view_edge_lists  = []   # node → [view_0_edges, view_1_edges, ...]
    node_view_centers     = torch.zeros(num_nodes, num_snapshots,
                                        device=edge_t.device)

    for i in range(num_nodes):
        edges_i = sorted(node_edges[i], key=lambda e: edge_t[e].item())
        views   = _split_edges_into_views(
            edges_i, edge_t, num_snapshots, mode
        )
        node_view_edge_lists.append(views)      # list of K lists of edge indices

        for k, v in enumerate(views):
            if len(v) > 0:
                node_view_centers[i, k] = edge_t[torch.tensor(v)].float().mean()
            else:
                # Empty view: inherit center from previous or default to 0
                node_view_centers[i, k] = (node_view_centers[i, k-1]
                                           if k > 0 else edge_t.float().mean())

    return node_view_edge_lists, node_view_centers


def _split_edges_into_views(sorted_edge_indices, edge_t, K, mode):
    """Split a sorted list of edge indices into K groups."""
    if len(sorted_edge_indices) == 0:
        return [[] for _ in range(K)]

    if mode == 'equal_count':
        # Each view gets ~equal number of events
        chunks = [[] for _ in range(K)]
        for idx, e in enumerate(sorted_edge_indices):
            k = min(int(idx * K / len(sorted_edge_indices)), K - 1)
            chunks[k].append(e)
        return chunks

    elif mode == 'equal_time':
        # Each view covers an equal time span
        t_vals  = [edge_t[e].item() for e in sorted_edge_indices]
        t_min, t_max = t_vals[0], t_vals[-1]
        if t_min == t_max:
            t_max = t_min + 1.0
        bins   = [t_min + (t_max - t_min) * k / K for k in range(K + 1)]
        chunks = [[] for _ in range(K)]
        for e, tv in zip(sorted_edge_indices, t_vals):
            k = min(int((tv - t_min) / (t_max - t_min + 1e-9) * K), K - 1)
            chunks[k].append(e)
        return chunks

    else:
        raise ValueError(f"Unknown mode {mode}")


# ═══════════════════════════════════════════════════════════════════════════════
# 3. Recurrent Node Memory with Uncertainty
# ═══════════════════════════════════════════════════════════════════════════════

class RecurrentMemory(nn.Module):
    def __init__(self, num_nodes, dim, momentum=0.9):
        super().__init__()
        self.gru      = nn.GRUCell(dim, dim)
        self.momentum = momentum
        self.register_buffer("hidden",   torch.zeros(num_nodes, dim))
        self.register_buffer("variance", torch.ones(num_nodes, dim) * 0.1)
        self.register_buffer("last_t",   torch.zeros(num_nodes))

    def read(self, idx):
        return self.hidden[idx], self.variance[idx]

    def get_hidden(self, idx):
        return self.hidden[idx]

    @torch.no_grad()
    def write(self, idx, x, t=None):
        h_old = self.hidden[idx]
        h_new = self.gru(x.detach(), h_old)
        delta = h_new - h_old
        self.variance[idx] = (self.momentum * self.variance[idx]
                              + (1 - self.momentum) * delta.pow(2))
        self.hidden[idx] = h_new
        if t is not None:
            self.last_t[idx] = t.float()


# ═══════════════════════════════════════════════════════════════════════════════
# 4. Neighborhood Evolution Bank
# ═══════════════════════════════════════════════════════════════════════════════

class EvolutionBank(nn.Module):
    def __init__(self, num_nodes, dim, window=6):
        super().__init__()
        self.window = window
        self.register_buffer("bank", torch.zeros(num_nodes, window, dim))
        self.register_buffer("ptr",  torch.zeros(num_nodes, dtype=torch.long))

    def read(self, idx):
        return self.bank[idx]

    @torch.no_grad()
    def write(self, idx, emb):
        for b in range(idx.size(0)):
            p = int(self.ptr[idx[b]]) % self.window
            self.bank[idx[b], p] = emb[b].detach()
            self.ptr[idx[b]] += 1


# ═══════════════════════════════════════════════════════════════════════════════
# 5. Trust Module
# ═══════════════════════════════════════════════════════════════════════════════

class TrustModule(nn.Module):
    def __init__(self, dim, num_heads=4, window=6):
        super().__init__()
        self.num_heads = num_heads
        self.head_dim  = dim // num_heads
        self.scale     = self.head_dim ** -0.5
        self.window    = window
        self.q_proj    = nn.Linear(dim, dim)
        self.k_proj    = nn.Linear(dim, dim)
        self.v_proj    = nn.Linear(dim, dim)
        self.log_decay = nn.Parameter(torch.tensor(-2.0))
        self.trust_out = nn.Sequential(
            nn.Linear(dim, dim // 2), nn.GELU(), nn.Linear(dim // 2, 1)
        )

    def forward(self, h_i, evo_j, var_i, var_j, eps=1e-6):
        E, W, D = evo_j.shape
        nh, hd  = self.num_heads, self.head_dim
        Q = self.q_proj(h_i).view(E, nh, hd)
        K = self.k_proj(evo_j.reshape(E * W, D)).view(E, W, nh, hd)
        V = self.v_proj(evo_j.reshape(E * W, D)).view(E, W, nh, hd)
        scores = (Q.unsqueeze(2) * K.permute(0,2,1,3)).sum(-1) * self.scale
        steps   = torch.arange(W, device=h_i.device).float()
        decay_w = torch.exp(self.log_decay.exp() * steps)
        scores  = scores + (decay_w / decay_w.sum()).view(1, 1, W)
        attn    = F.softmax(scores, dim=-1)
        pooled  = (attn.unsqueeze(-1) * V.permute(0,2,1,3)).sum(2).reshape(E, D)
        trust_logit = self.trust_out(pooled).squeeze(-1)
        conf_i = 1.0 / (1.0 + var_i.mean(-1).sqrt().clamp(min=eps))
        conf_j = 1.0 / (1.0 + var_j.mean(-1).sqrt().clamp(min=eps))
        conf   = (conf_i * conf_j).sqrt()
        return torch.sigmoid(trust_logit + conf.log().clamp(min=-5))


# ═══════════════════════════════════════════════════════════════════════════════
# 6. Similarity-Gated Trust Attention Layer
#    Applied independently to each snapshot view's edge set.
# ═══════════════════════════════════════════════════════════════════════════════

class SimilarityGatedTrustAttention(MessagePassing):
    def __init__(self, dim, heads=8, dropout=0.1, window=6, sim_temp=0.5):
        super().__init__(aggr="add", node_dim=0)
        self.dim, self.heads = dim, heads
        self.head_dim = dim // heads
        self.scale    = self.head_dim ** -0.5
        self.window   = window
        self.sim_temp = sim_temp

        self.q_proj    = nn.Linear(dim, dim)
        self.k_proj    = nn.Linear(dim, dim)
        self.v_proj    = nn.Linear(dim, dim)
        self.o_proj    = nn.Linear(dim, dim)
        self.edge_proj = nn.Linear(dim, dim)
        self.time_proj = nn.Linear(dim, dim)

        self.trust       = TrustModule(dim, num_heads=4, window=window)
        self.dropout     = nn.Dropout(dropout)
        self.gate_balance = nn.Parameter(torch.tensor(0.5))
        self.gate        = nn.Linear(dim, 1)
        self.norm        = nn.LayerNorm(dim)
        self.fusion      = nn.Sequential(
            nn.Linear(dim * 2, dim), nn.GELU(), nn.Linear(dim, dim)
        )

    def split(self, x):
        return x.view(-1, self.heads, self.head_dim)

    def forward(self, x, edge_index, edge_emb, time_emb, memory, evo_bank):
        """
        x:           (N, D)  current node features
        edge_index:  (2, E)  edges in THIS snapshot (already filtered)
        edge_emb:    (E, D)
        time_emb:    (E, D)
        Returns updated (N, D)
        """
        N      = x.size(0)
        device = x.device

        Q = self.split(self.q_proj(x))
        K = self.split(self.k_proj(x))
        V = self.split(self.v_proj(x))

        mem_vals, mem_vars = memory.read(torch.arange(N, device=device))
        evo_vals           = evo_bank.read(torch.arange(N, device=device))

        out = self.propagate(
            edge_index,
            Q=Q, K=K, V=V, x=x,
            mem_val=mem_vals, mem_var=mem_vars,
            hidden=mem_vals, evo_vals=evo_vals,
            edge_emb=self.split(self.edge_proj(edge_emb)),
            time_emb=self.split(self.time_proj(time_emb)),
            size=(N, N)
        )

        out = out.view(N, self.dim)
        h   = memory.get_hidden(torch.arange(N, device=device))
        out = self.fusion(torch.cat([out, h], dim=-1))
        out = self.o_proj(out)
        g   = torch.sigmoid(self.gate(x))
        return self.norm(g * x + (1 - g) * out)

    def message(self, Q_i, K_j, V_j, x_i, x_j,
                mem_var_i, mem_var_j, hidden_i, evo_vals_j,
                edge_emb, time_emb, index):
        K_j  = K_j + edge_emb + time_emb
        attn = (Q_i * K_j).sum(-1) * self.scale                   # (E, heads)

        trust = self.trust(hidden_i, evo_vals_j, mem_var_i, mem_var_j)
        sim   = torch.sigmoid(
            F.cosine_similarity(x_i, x_j, dim=-1) / self.sim_temp
        )
        alpha    = torch.sigmoid(self.gate_balance)
        combined = alpha * trust + (1 - alpha) * sim               # (E,)

        attn = softmax(attn * combined.unsqueeze(-1), index)
        attn = self.dropout(attn)
        return V_j * attn.unsqueeze(-1)


# ═══════════════════════════════════════════════════════════════════════════════
# 7. Per-Node View Encoder
#
# Given a node i and its K activity windows, this applies the attention layers
# to each window IN SEQUENCE, carrying the hidden state forward between views.
#
# Crucially, each view's time encoding uses:
#   t_abs   = absolute timestamp of each edge
#   t_delta = time since previous event FOR THIS NODE
#             (not a global delta — it's node i's personal inter-event gap)
#
# This captures each node's OWN temporal rhythm.
# ═══════════════════════════════════════════════════════════════════════════════

class PerNodeViewEncoder(nn.Module):
    """
    Runs attention layers on each of node i's K activity views.
    Carries x forward between views so later views build on earlier context.
    Returns (N, K, D) — one embedding per view per node.
    """
    def __init__(self, layers, edge_enc, time_enc):
        super().__init__()
        # These are shared with the parent model (not re-instantiated)
        self.layers    = layers
        self.edge_enc  = edge_enc
        self.time_enc  = time_enc

    def forward(self, x_init, node_view_edge_lists, node_view_centers,
                full_edge_index, full_t, full_msg, memory, evo_bank, num_snapshots):
        """
        x_init:                (N, D)
        node_view_edge_lists:  list[N] → list[K] → list of global edge indices
        node_view_centers:     (N, K) — center timestamp of each node's views
        full_edge_index:       (2, E_total)
        full_t:                (E_total,)
        full_msg:              (E_total, edge_dim) — raw edge features before encoding

        Returns:
            view_embs: (N, K, D)
        """
        N      = x_init.size(0)
        K      = num_snapshots
        device = x_init.device
        D      = x_init.size(-1)

        view_embs = torch.zeros(N, K, D, device=device)

        # We process all K views together in a batch-of-snapshots loop.
        # For each view k, we build a sub-graph containing only the edges
        # that belong to view k for at least one node.
        # This is more efficient than looping over individual nodes.

        x_current = x_init.clone()   # carries state across views

        for k in range(K):
            # Collect all edges that are in view k for any node
            view_k_edge_set = set()
            for i in range(N):
                view_k_edge_set.update(node_view_edge_lists[i][k])
            view_k_edges = sorted(view_k_edge_set)

            if len(view_k_edges) == 0:
                view_embs[:, k, :] = x_current
                continue

            eidx  = torch.tensor(view_k_edges, device=device, dtype=torch.long)
            sub_ei = full_edge_index[:, eidx]        # (2, E_k)
            sub_t  = full_t[eidx]                    # (E_k,)
            sub_msg = full_msg[eidx]                 # (E_k, edge_dim)

            # Per-edge inter-event delta:
            # For each edge, compute time since the previous edge of the same source
            # This is the node's personal activity rhythm, not a global delta
            src_nodes  = sub_ei[0]
            t_delta    = _compute_inter_event_delta(src_nodes, sub_t, N, device)

            edge_emb = self.edge_enc(sub_msg)                     # (E_k, D)
            time_emb = self.time_enc(sub_t, t_delta)              # (E_k, D)

            # Run all attention layers on this snapshot
            x_view = x_current.clone()
            for layer in self.layers:
                x_view = layer(x_view, sub_ei, edge_emb, time_emb, memory, evo_bank)

            view_embs[:, k, :] = x_view
            # Carry forward: next view starts from the updated state
            x_current = x_view

        return view_embs


def _compute_inter_event_delta(src_nodes, edge_t, num_nodes, device):
    """
    For each edge e with source node src[e] and timestamp t[e],
    compute t[e] - t[prev_edge_of_src[e]].
    First event per node gets delta=0.
    This measures the personal inter-event gap of each source node.
    """
    E      = src_nodes.size(0)
    delta  = torch.zeros(E, device=device)
    last_t = torch.full((num_nodes,), -1.0, device=device)

    for e in range(E):
        s = src_nodes[e].item()
        t = edge_t[e].float().item()
        if last_t[s] >= 0:
            delta[e] = t - last_t[s]
        last_t[s] = t

    return delta


# ═══════════════════════════════════════════════════════════════════════════════
# 8. Temporal View Integrator (GRU + attention over K views)
#
# After getting K per-node views, this module integrates them into
# a single final representation.
#
# The GRU explicitly receives the inter-VIEW time delta:
#   Δt_k = center_time(view_k) - center_time(view_{k-1})
# This is PER NODE: node i's Δt_k reflects how much of node i's
# personal time elapsed between its k-1'th and k'th activity windows.
# ═══════════════════════════════════════════════════════════════════════════════

class TemporalViewIntegrator(nn.Module):
    def __init__(self, dim, num_views):
        super().__init__()
        self.dim       = dim
        self.num_views = num_views
        self.t2v_delta  = Time2Vec(dim // 2)
        self.delta_proj = nn.Linear(dim // 2, dim)
        self.gru        = nn.GRUCell(dim * 2, dim)
        self.view_attn  = nn.MultiheadAttention(dim, num_heads=4, batch_first=True)
        self.cls_token  = nn.Parameter(torch.randn(1, 1, dim) * 0.02)
        self.out_norm   = nn.LayerNorm(dim)

    def forward(self, view_embs, node_view_centers):
        """
        view_embs:         (N, K, D)
        node_view_centers: (N, K)  — each node's own view timestamps
        Returns:           (N, D)
        """
        N, K, D = view_embs.shape
        device  = view_embs.device

        # Inter-view time deltas — PER NODE
        # shape (N, K): Δt[i, k] = center[i,k] - center[i,k-1]
        view_deltas        = torch.zeros(N, K, device=device)
        view_deltas[:, 1:] = (node_view_centers[:, 1:]
                              - node_view_centers[:, :-1]).clamp(min=0)

        h = torch.zeros(N, D, device=device)
        gru_out = []

        for k in range(K):
            # Encode each node's individual inter-view gap at step k
            delta_enc = self.delta_proj(
                self.t2v_delta(view_deltas[:, k])           # (N, D/2) → (N, D)
            )
            inp = torch.cat([view_embs[:, k, :], delta_enc], dim=-1)   # (N, 2D)
            h   = self.gru(inp, h)
            gru_out.append(h)

        gru_seq = torch.stack(gru_out, dim=1)                # (N, K, D)

        # CLS token attends over all K views to weight temporal moments
        cls    = self.cls_token.expand(N, -1, -1)            # (N, 1, D)
        seq    = torch.cat([cls, gru_seq], dim=1)            # (N, K+1, D)
        out, _ = self.view_attn(seq, seq, seq)
        return self.out_norm(out[:, 0, :])                   # (N, D)


# ═══════════════════════════════════════════════════════════════════════════════
# 9. Full Model
# ═══════════════════════════════════════════════════════════════════════════════

class MultiViewTemporalGNN(nn.Module):
    """
    Full pipeline:
      1. Build per-node activity snapshots (each node's OWN K windows)
      2. Encode each snapshot view with SimilarityGatedTrustAttention
         — trust + similarity gates applied at EVERY view
         — state carried forward between views
      3. Integrate K views with GRU using per-node inter-view time deltas
      4. CLS attention pooling → final (N, D) representation
      5. Update recurrent memory + evolution bank
    """
    def __init__(self, num_nodes, in_dim, edge_dim, hidden_dim,
                 num_layers=2, heads=8, dropout=0.1,
                 num_snapshots=6, window=6, memory_momentum=0.9,
                 sim_temp=0.5, snapshot_mode='equal_count'):
        super().__init__()
        self.hidden_dim    = hidden_dim
        self.num_nodes     = num_nodes
        self.num_snapshots = num_snapshots
        self.snapshot_mode = snapshot_mode

        self.input_proj = nn.Linear(in_dim, hidden_dim)
        self.edge_enc   = nn.Linear(edge_dim, hidden_dim)
        self.time_enc   = TimeEncoder(hidden_dim)

        self.memory   = RecurrentMemory(num_nodes, hidden_dim, momentum=memory_momentum)
        self.evo_bank = EvolutionBank(num_nodes, hidden_dim, window=window)

        self.layers = nn.ModuleList([
            SimilarityGatedTrustAttention(
                hidden_dim, heads=heads, dropout=dropout,
                window=window, sim_temp=sim_temp
            )
            for _ in range(num_layers)
        ])

        self.view_encoder = PerNodeViewEncoder(
            self.layers, self.edge_enc, self.time_enc
        )

        self.view_integrator = TemporalViewIntegrator(hidden_dim, num_snapshots)

        self.final_norm = nn.LayerNorm(hidden_dim)
        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)

    def forward(self, batch):
        """
        batch.x          : (N, in_dim)
        batch.edge_index : (2, E)
        batch.t          : (E,)   edge timestamps
        batch.msg        : (E, edge_dim) raw edge features
        """
        N      = batch.x.size(0)
        device = batch.x.device

        x = self.input_proj(batch.x)

        # ── 1. Build per-node K-view snapshots ──
        # Each node gets its own K windows based on ITS OWN activity
        node_view_edge_lists, node_view_centers = build_per_node_snapshots(
            batch.edge_index, batch.t, N,
            self.num_snapshots, mode=self.snapshot_mode
        )
        # node_view_centers: (N, K) — different per node!

        # ── 2. Encode each view with trust+similarity attention ──
        # Trust and similarity gates are applied INSIDE each view
        # State (x) is carried forward from view k to view k+1
        view_embs = self.view_encoder(
            x, node_view_edge_lists, node_view_centers,
            batch.edge_index, batch.t, batch.msg,
            self.memory, self.evo_bank, self.num_snapshots
        )
        # view_embs: (N, K, D)

        # ── 3. Integrate K views with per-node inter-view time deltas ──
        x_final = self.view_integrator(view_embs, node_view_centers)
        x_final = self.final_norm(x_final)

        # ── 4. Update memory and evolution bank ──
        t_per_node = torch.zeros(N, device=device)
        t_per_node.scatter_reduce_(
            0, batch.edge_index[0], batch.t.float(),
            reduce='amax', include_self=True
        )
        self.memory.write(torch.arange(N, device=device), x_final, t_per_node)

        src, dst = batch.edge_index
        neigh = torch.zeros(N, self.hidden_dim, device=device)
        cnt   = torch.zeros(N, 1, device=device)
        neigh.scatter_add_(0, dst.unsqueeze(1).expand(-1, self.hidden_dim), x_final[src])
        cnt.scatter_add_(0, dst.unsqueeze(1), torch.ones(src.size(0), 1, device=device))
        neigh   = neigh / (cnt + 1e-6)
        unique_dst = dst.unique()
        self.evo_bank.write(unique_dst, neigh[unique_dst])

        return x_final


# ═══════════════════════════════════════════════════════════════════════════════
# 10. Contrastive Projection Head
# ═══════════════════════════════════════════════════════════════════════════════

class ContrastiveHead(nn.Module):
    def __init__(self, hidden_dim, proj_dim=128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.GELU(),
            nn.LayerNorm(hidden_dim),
            nn.Linear(hidden_dim, proj_dim)
        )

    def forward(self, x):
        return F.normalize(self.net(x), dim=-1)


class FullModel(nn.Module):
    def __init__(self, num_nodes, in_dim, edge_dim, hidden_dim,
                 proj_dim=128, **gnn_kwargs):
        super().__init__()
        self.gnn  = MultiViewTemporalGNN(
            num_nodes, in_dim, edge_dim, hidden_dim, **gnn_kwargs
        )
        self.head = ContrastiveHead(hidden_dim, proj_dim)

    def forward(self, batch):
        return self.gnn(batch)

    def project(self, batch):
        h = self.gnn(batch)
        return h, self.head(h)

In [7]:
"""
Drop-in replacement for your notebook.a
Keeps: data, loaders, History, args, measure
Replaces: model, train(), test()

Just run these cells in order after your existing setup cells.
"""

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import MessagePassing
from torch_geometric.utils import softmax
from tqdm import tqdm


# ── Paste the full enhanced_gnn_v2.py here, or import it ──────────────────────
# from enhanced_gnn_v2 import (EnhancedTemporalGNN, train, test)
# ──────────────────────────────────────────────────────────────────────────────
# If running directly in notebook, paste the class definitions above this cell.


# ═══════════════════════════════════════════
# CELL: Build model + history (unchanged)
# ═══════════════════════════════════════════

try:
    del model
except NameError:
    pass



model = MultiViewTemporalGNN(
    num_nodes  = data.num_nodes,
    in_dim     = data.x.size(1),          # 1 for wikipedia
    edge_dim   = data.msg.size(1),         # 172 for wikipedia
    hidden_dim = args.hidden_channels,     # 64
    num_layers = args.nums_layers,         # 2
    heads      = args.heads,               # 8
    dropout    = args.dropout,  
    snapshot_mode='equal_count',           # 0.3
    window     = 4,
    num_snapshots=4,
    memory_momentum = 0.9,
).to(device)

# History is UNCHANGED — reuse your existing History class
try:
    del history
except NameError:
    pass

from mhiscl.history import History

history = History(
    data.num_nodes,
    args.num_timeslots,
    args.hidden_channels,
    device=device,
    history_retrieve=args.history_retrieve,
    recurrent=args.recurrent,
).to(device)

print(model)
print(history)
print(f"Model params: {sum(p.numel() for p in model.parameters()):,}")




MultiViewTemporalGNN(
  (input_proj): Linear(in_features=1, out_features=64, bias=True)
  (edge_enc): Linear(in_features=172, out_features=64, bias=True)
  (time_enc): TimeEncoder(
    (t2v_abs): Time2Vec()
    (t2v_delta): Time2Vec()
    (proj): Sequential(
      (0): Linear(in_features=64, out_features=64, bias=True)
      (1): GELU(approximate='none')
      (2): Linear(in_features=64, out_features=64, bias=True)
    )
  )
  (memory): RecurrentMemory(
    (gru): GRUCell(64, 64)
  )
  (evo_bank): EvolutionBank()
  (layers): ModuleList(
    (0-1): 2 x SimilarityGatedTrustAttention()
  )
  (view_encoder): PerNodeViewEncoder(
    (layers): ModuleList(
      (0-1): 2 x SimilarityGatedTrustAttention()
    )
    (edge_enc): Linear(in_features=172, out_features=64, bias=True)
    (time_enc): TimeEncoder(
      (t2v_abs): Time2Vec()
      (t2v_delta): Time2Vec()
      (proj): Sequential(
        (0): Linear(in_features=64, out_features=64, bias=True)
        (1): GELU(approximate='none')
    

In [9]:
# ═══════════════════════════════════════════
# CELL: Training loop (drop-in replacement)
# ═══════════════════════════════════════════

best     = 0
best_val = 0

optimizer = torch.optim.Adam(
    list(model.parameters()) + list(history.parameters()),
    lr=args.lr,
    weight_decay=1e-5       # small L2 reg helps generalization
)

# Cosine LR schedule: smoothly decays over training
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=args.epochs, eta_min=args.lr * 0.1)

measure = Measure("auc")

for epoch in range(1, 1 + args.epochs):
    loss    = train(train_loader, model, history, optimizer, device, data, args)
    val_auc = test(valid_loader,  model, history, measure,   device, data)
    test_auc = test(test_loader,  model, history, measure,   device, data)
    scheduler.step()

    if val_auc > best_val:
        best_val = val_auc
        best     = test_auc

    print(
        f"Epoch: {epoch:03d}, Loss: {loss:.4f}, "
        f"Val AUC: {val_auc:.2%}, Test AUC: {test_auc:.2%}, Best AUC: {best:.2%}"
    )

100%|██████████| 93/93 [01:19<00:00,  1.17it/s]


Epoch: 001, Loss: 15.1242, Val AUC: 76.11%, Test AUC: 88.23%, Best AUC: 88.23%


100%|██████████| 93/93 [01:19<00:00,  1.17it/s]


Epoch: 002, Loss: 12.4374, Val AUC: 79.73%, Test AUC: 89.73%, Best AUC: 89.73%


 12%|█▏        | 11/93 [00:09<01:14,  1.10it/s]


KeyboardInterrupt: 

### Temproal embedding in message passing

In [18]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import MessagePassing
from torch_geometric.utils import softmax


# ═══════════════════════════════════════════════════════════════════════════════
# 1. Time2Vec + TimeEncoder
# ═══════════════════════════════════════════════════════════════════════════════

class Time2Vec(nn.Module):
    """
    Encodes a scalar time value into a vector:
      [linear_trend, sin(w1*t+b1), ..., sin(wk*t+bk)]
    Frequencies are learned — the model discovers the relevant temporal scales
    (e.g. hourly, daily, weekly patterns) from data.
    """
    def __init__(self, out_dim):
        super().__init__()
        k = out_dim - 1
        self.w0 = nn.Parameter(torch.randn(1) * 0.01)
        self.b0 = nn.Parameter(torch.zeros(1))
        self.W  = nn.Parameter(torch.randn(k) * 0.01)
        self.B  = nn.Parameter(torch.zeros(k))

    def forward(self, t):
        """t: (N,) → (N, out_dim)"""
        t        = t.float().unsqueeze(-1)
        linear   = t * self.w0 + self.b0
        periodic = torch.sin(t * self.W + self.B)
        return torch.cat([linear, periodic], dim=-1)


class TimeEncoder(nn.Module):
    """
    Projects a time delta into hidden_dim via Time2Vec + MLP.
    Used in multiple places:
      - State decay gate  (Δt since node's last interaction)
      - Joint attention   (Δt since this specific edge last fired)
      - Temporal consistency gate (deviation from expected gap)
    """
    def __init__(self, hidden_dim):
        super().__init__()
        self.t2v  = Time2Vec(hidden_dim)
        self.proj = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, hidden_dim),
        )

    def forward(self, t):
        """t: (N,) → (N, hidden_dim)"""
        return self.proj(self.t2v(t))


# ═══════════════════════════════════════════════════════════════════════════════
# 2. Node-Level State Decay
#
# A node that has not been seen for a long time should have a "faded" memory.
# Before aggregating new neighbor messages, we decay node i's hidden state
# by a learned gate that depends on how much time has elapsed since i's
# last interaction.
#
# h_i_decayed = decay_gate(h_i, Δt_i) ⊙ h_i
#
# Intuition for anomaly detection:
#   - Normal nodes: regular activity → moderate decay → stable representations
#   - Anomalous nodes: sudden bursts after silence → large Δt → strong decay
#     followed by aggressive update → representation shift is detectable
# ═══════════════════════════════════════════════════════════════════════════════

class StateDecayGate(nn.Module):
    """
    Computes a per-dimension decay gate:
        decay = sigmoid(W_h @ h_i + W_t @ time_enc(Δt_i) + b)
        h_decayed = decay ⊙ h_i

    Per-dimension (not scalar) decay allows different aspects of the
    node's memory to fade at different rates.
    """
    def __init__(self, hidden_dim):
        super().__init__()
        self.time_enc = TimeEncoder(hidden_dim)
        self.W_h = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.W_t = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.b   = nn.Parameter(torch.zeros(hidden_dim))
        self.norm = nn.LayerNorm(hidden_dim)

    def forward(self, h, delta_t):
        """
        h:       (N, D)  current node hidden states
        delta_t: (N,)    time elapsed since each node's last interaction
        Returns: (N, D)  decayed hidden states
        """
        t_enc = self.time_enc(delta_t)                         # (N, D)
        gate  = torch.sigmoid(self.W_h(h) + self.W_t(t_enc) + self.b)
        return self.norm(gate * h)


# ═══════════════════════════════════════════════════════════════════════════════
# 3. Edge-Pair Temporal Memory
#
# Tracks, for each directed edge (i→j), the history of interaction gaps:
#   - mean gap  μ_ij  (EMA)
#   - variance  σ²_ij (EMA of squared deviations)
#
# Used by the Temporal Consistency Gate to compute "surprise":
#   surprise = |Δt_ij - μ_ij| / (sqrt(σ²_ij) + ε)
#
# High surprise → this interaction is temporally anomalous
#   → down-weighted in aggregation
#   → the pattern is preserved in the representation as a structural anomaly signal
#
# Storage: O(unique_edges) — only pairs that have been seen ≥ 2 times have stats.
# For unseen pairs, we fall back to node-level statistics.
# ═══════════════════════════════════════════════════════════════════════════════

class EdgePairTemporalMemory:
    """
    Non-parametric buffer tracking per-edge-pair timing statistics.
    Stored as plain tensors keyed by (src, dst) pair index.
    Updated in-place after each forward pass (no gradients needed).
    """
    def __init__(self, num_nodes, momentum=0.9, device='cpu'):
        self.num_nodes = num_nodes
        self.momentum  = momentum
        self.device    = device

        # Flat index: pair (i,j) → i * num_nodes + j
        max_pairs = num_nodes * num_nodes
        self.last_t   = torch.full((max_pairs,), -1.0, device=device)
        self.mean_gap = torch.zeros(max_pairs, device=device)
        self.var_gap  = torch.ones(max_pairs, device=device) * 1.0  # start uncertain
        self.count    = torch.zeros(max_pairs, dtype=torch.long, device=device)

    def pair_idx(self, src, dst):
        return src * self.num_nodes + dst

    def get_surprise(self, src, dst, t_current, eps=1e-6):
        """
        For each edge (src_e, dst_e) at time t_e, compute:
            Δt       = t_e - last_t[src_e, dst_e]
            surprise = |Δt - μ| / (σ + ε)
        For first-time edges (never seen before), surprise = 1.0 (neutral).
        Returns: (E,) surprise scores ≥ 0
        """
        pidx     = self.pair_idx(src, dst)
        last     = self.last_t[pidx]
        seen     = (last >= 0)

        delta_t  = torch.zeros_like(t_current)
        delta_t[seen] = (t_current[seen] - last[seen]).clamp(min=0)

        mu  = self.mean_gap[pidx]
        sig = self.var_gap[pidx].sqrt()

        surprise = torch.ones_like(t_current)            # default: neutral
        surprise[seen] = ((delta_t[seen] - mu[seen]).abs() / (sig[seen] + eps))

        return surprise, delta_t

    @torch.no_grad()
    def update(self, src, dst, t_current, delta_t):
        """Update EMA statistics for observed edges."""
        pidx = self.pair_idx(src, dst)
        seen = (self.last_t[pidx] >= 0)

        # Only update EMA for pairs seen at least once before
        if seen.any():
            idx_seen  = pidx[seen]
            dt_seen   = delta_t[seen]
            old_mu    = self.mean_gap[idx_seen]
            old_var   = self.var_gap[idx_seen]
            new_mu    = self.momentum * old_mu + (1 - self.momentum) * dt_seen
            new_var   = self.momentum * old_var + (1 - self.momentum) * (dt_seen - old_mu).pow(2)
            self.mean_gap[idx_seen] = new_mu
            self.var_gap[idx_seen]  = new_var

        # Always update last seen time and count
        self.last_t[pidx]  = t_current
        self.count[pidx]  += 1

    def to(self, device):
        self.device    = device
        self.last_t    = self.last_t.to(device)
        self.mean_gap  = self.mean_gap.to(device)
        self.var_gap   = self.var_gap.to(device)
        self.count     = self.count.to(device)
        return self


# ═══════════════════════════════════════════════════════════════════════════════
# 4. Node-Level Recurrent Memory
#
# Stores per-node GRU hidden state + variance (uncertainty).
# Also tracks each node's last-seen timestamp for state decay.
# ═══════════════════════════════════════════════════════════════════════════════

class RecurrentMemory(nn.Module):
    def __init__(self, num_nodes, dim, momentum=0.9):
        super().__init__()
        self.gru      = nn.GRUCell(dim, dim)
        self.momentum = momentum
        self.register_buffer("hidden",   torch.zeros(num_nodes, dim))
        self.register_buffer("variance", torch.ones(num_nodes, dim) * 0.1)
        self.register_buffer("last_t",   torch.full((num_nodes,), -1.0))

    def read(self, idx):
        """Returns (hidden, variance) for requested node indices."""
        return self.hidden[idx], self.variance[idx]

    def get_delta_t(self, idx, t_now):
        """
        Per-node time since last interaction.
        Returns (N,) delta, with 0 for nodes never seen before.
        """
        last = self.last_t[idx]
        delta = torch.where(last >= 0, (t_now - last).clamp(min=0),
                            torch.zeros_like(t_now))
        return delta

    @torch.no_grad()
    def write(self, idx, x, t_now):
        h_old = self.hidden[idx]
        h_new = self.gru(x.detach(), h_old)
        delta = h_new - h_old
        self.variance[idx] = (self.momentum * self.variance[idx]
                              + (1 - self.momentum) * delta.pow(2))
        self.hidden[idx]  = h_new
        self.last_t[idx]  = t_now


# ═══════════════════════════════════════════════════════════════════════════════
# 5. Neighborhood Evolution Bank
#
# Rolling window of K most recent neighborhood mean embeddings per node.
# Used by TrustModule to query: how has j's neighborhood evolved recently?
# ═══════════════════════════════════════════════════════════════════════════════

class EvolutionBank(nn.Module):
    def __init__(self, num_nodes, dim, window=6):
        super().__init__()
        self.window = window
        self.register_buffer("bank", torch.zeros(num_nodes, window, dim))
        self.register_buffer("ptr",  torch.zeros(num_nodes, dtype=torch.long))

    def read(self, idx):
        return self.bank[idx]                              # (B, W, D)

    @torch.no_grad()
    def write(self, idx, emb):
        for b in range(idx.size(0)):
            p = int(self.ptr[idx[b]]) % self.window
            self.bank[idx[b], p] = emb[b].detach()
            self.ptr[idx[b]] += 1


# ═══════════════════════════════════════════════════════════════════════════════
# 6. Trust Module
#
# Cross-attention between node i's GRU history and node j's neighborhood
# evolution window. Uncertainty-gated via memory variance.
# Keeps your original design — it works well.
# ═══════════════════════════════════════════════════════════════════════════════

class TrustModule(nn.Module):
    def __init__(self, dim, num_heads=4, window=6):
        super().__init__()
        self.num_heads = num_heads
        self.head_dim  = dim // num_heads
        self.scale     = self.head_dim ** -0.5
        self.window    = window
        self.q_proj    = nn.Linear(dim, dim)
        self.k_proj    = nn.Linear(dim, dim)
        self.v_proj    = nn.Linear(dim, dim)
        self.log_decay = nn.Parameter(torch.tensor(-2.0))
        self.trust_out = nn.Sequential(
            nn.Linear(dim, dim // 2), nn.GELU(), nn.Linear(dim // 2, 1)
        )

    def forward(self, h_i, evo_j, var_i, var_j, eps=1e-6):
        """
        h_i:   (E, D)    node i recurrent hidden state
        evo_j: (E, W, D) node j neighborhood evolution window
        Returns: (E,) trust scores in [0,1]
        """
        E, W, D = evo_j.shape
        nh, hd  = self.num_heads, self.head_dim

        Q = self.q_proj(h_i).view(E, nh, hd)
        K = self.k_proj(evo_j.reshape(E * W, D)).view(E, W, nh, hd)
        V = self.v_proj(evo_j.reshape(E * W, D)).view(E, W, nh, hd)

        scores = (Q.unsqueeze(2) * K.permute(0, 2, 1, 3)).sum(-1) * self.scale

        steps   = torch.arange(W, device=h_i.device).float()
        decay_w = torch.exp(self.log_decay.exp() * steps)
        scores  = scores + (decay_w / decay_w.sum()).view(1, 1, W)

        attn   = F.softmax(scores, dim=-1)
        pooled = (attn.unsqueeze(-1) * V.permute(0, 2, 1, 3)).sum(2).reshape(E, D)

        trust_logit = self.trust_out(pooled).squeeze(-1)
        conf_i = 1.0 / (1.0 + var_i.mean(-1).sqrt().clamp(min=eps))
        conf_j = 1.0 / (1.0 + var_j.mean(-1).sqrt().clamp(min=eps))
        conf   = (conf_i * conf_j).sqrt()

        return torch.sigmoid(trust_logit + conf.log().clamp(min=-5))


# ═══════════════════════════════════════════════════════════════════════════════
# 7. Joint Temporal-Structural Attention Layer  ← CORE
#
# Three interlocking gates, all computed jointly in a single message-passing step:
#
#   GATE 1 — Trust (structural-temporal history):
#     Does i's recurrent history match j's neighborhood evolution pattern?
#     → Uses TrustModule (cross-attention over EvolutionBank)
#
#   GATE 2 — Structural Similarity:
#     Are i and j currently similar in embedding space?
#     → Cosine similarity with learned temperature
#     → Prevents anomalous neighbors from poisoning normal nodes
#     → Keeps anomalous nodes distinguishable (soft gate, not hard cut)
#
#   GATE 3 — Temporal Consistency:
#     Is this interaction happening at the expected time?
#     → surprise = |actual_gap - mean_gap| / std_gap
#     → High surprise → interaction is temporally anomalous → down-weighted
#     → The surprise score itself is an auxiliary anomaly signal
#
#   ATTENTION SCORE (joint, not factored):
#     score = (Q_i · K_j) / sqrt(d)          ← structural query-key match
#           + w_t · (time_enc(Δt_ij) · v_t)  ← temporal recency bonus
#     This means time and structure are fused in ONE score, not multiplied separately.
#     A neighbor is highly attended if it is BOTH structurally relevant AND recent.
#
#   FINAL MESSAGE WEIGHT:
#     weight = softmax(score) * trust * sim_gate * temporal_gate
# ═══════════════════════════════════════════════════════════════════════════════

class JointTemporalStructuralAttention(MessagePassing):
    def __init__(self, dim, heads=8, dropout=0.1, window=6, sim_temp=0.5,
                 surprise_lambda=2.0):
        super().__init__(aggr="add", node_dim=0)
        assert dim % heads == 0
        self.dim      = dim
        self.heads    = heads
        self.head_dim = dim // heads
        self.scale    = self.head_dim ** -0.5
        self.sim_temp = sim_temp
        self.surprise_lambda = surprise_lambda

        # Standard attention projections
        self.q_proj = nn.Linear(dim, dim)
        self.k_proj = nn.Linear(dim, dim)
        self.v_proj = nn.Linear(dim, dim)
        self.o_proj = nn.Linear(dim, dim)

        # Edge feature projection (enriches keys)
        self.edge_proj = nn.Linear(dim, dim)

        # Temporal recency: Δt_ij encoded and projected to a single scalar per head
        # Added directly to attention score so time and structure are jointly scored
        self.time_enc      = TimeEncoder(dim)
        self.time_to_score = nn.Linear(dim, heads)    # (E, heads) recency bias

        # Trust module
        self.trust = TrustModule(dim, num_heads=4, window=window)

        # Temporal consistency gate
        # Takes surprise score → scalar gate per edge
        self.surprise_gate = nn.Sequential(
            nn.Linear(1, 16), nn.GELU(), nn.Linear(16, 1), nn.Sigmoid()
        )
        # Alternatively, use closed-form: sigmoid(-λ * surprise)
        # We use a learned MLP so the model can learn non-monotone surprise responses

        # Learnable balancing weights between the three gates
        # Initialized equally; model learns which gate matters most
        self.log_gate_weights = nn.Parameter(torch.zeros(3))   # [trust, sim, temporal]

        self.dropout = nn.Dropout(dropout)
        self.gate    = nn.Linear(dim, 1)
        self.norm    = nn.LayerNorm(dim)
        self.fusion  = nn.Sequential(
            nn.Linear(dim * 2, dim), nn.GELU(), nn.Linear(dim, dim)
        )

        # Expose anomaly signals for downstream use
        self.last_trust_scores    = None
        self.last_sim_scores      = None
        self.last_surprise_scores = None
        self.last_combined_gates  = None

    def split(self, x):
        return x.view(-1, self.heads, self.head_dim)

    def forward(self, x, edge_index, edge_emb, delta_t_edge,
                surprise, memory, evo_bank):
        """
        x:             (N, D)  current node embeddings (already decay-gated)
        edge_index:    (2, E)
        edge_emb:      (E, D)  encoded edge features
        delta_t_edge:  (E,)    time since this (src,dst) pair last interacted
        surprise:      (E,)    temporal surprise score for each edge
        memory:        RecurrentMemory
        evo_bank:      EvolutionBank
        """
        N      = x.size(0)
        device = x.device

        Q = self.split(self.q_proj(x))    # (N, heads, head_dim)
        K = self.split(self.k_proj(x))
        V = self.split(self.v_proj(x))

        mem_h, mem_var = memory.read(torch.arange(N, device=device))
        evo_vals       = evo_bank.read(torch.arange(N, device=device))

        # Temporal recency bias: encode edge-level Δt_ij
        time_enc_edge = self.time_enc(delta_t_edge)          # (E, D)
        time_bias     = self.time_to_score(time_enc_edge)    # (E, heads)

        out = self.propagate(
            edge_index,
            x=x,
            Q=Q, K=K, V=V,
            mem_h=mem_h, mem_var=mem_var,
            evo_vals=evo_vals,
            edge_emb=self.split(self.edge_proj(edge_emb)),   # (E, heads, head_dim)
            time_bias=time_bias,
            surprise=surprise,
            size=(N, N)
        )

        out = out.view(N, self.dim)

        # Fuse aggregated message with node's recurrent hidden state
        h   = memory.read(torch.arange(N, device=device))[0]
        out = self.fusion(torch.cat([out, h], dim=-1))
        out = self.o_proj(out)

        # Gated residual: balance between old embedding and new aggregation
        g = torch.sigmoid(self.gate(x))
        return self.norm(g * x + (1 - g) * out)

    def message(self, Q_i, K_j, V_j, x_i, x_j,
                mem_h_i, mem_h_j, mem_var_i, mem_var_j,
                evo_vals_j, edge_emb, time_bias, surprise, index):
        """
        All tensors subscripted _i or _j are already gathered by PyG
        for the source/destination of each edge.
        """
        # ── Joint temporal-structural attention score ──
        # Enrich keys with edge features (edge type, weight, etc.)
        K_j_enriched = K_j + edge_emb                       # (E, heads, head_dim)

        # Structural component: query-key dot product
        attn_struct = (Q_i * K_j_enriched).sum(-1) * self.scale  # (E, heads)

        # Temporal component: recency bias added directly to score
        # time_bias: (E, heads) — high for recent interactions
        attn_score = attn_struct + time_bias                 # (E, heads)  ← JOINT

        # ── Gate 1: Trust (structural-temporal history) ──
        trust = self.trust(mem_h_i, evo_vals_j, mem_var_i, mem_var_j)  # (E,)

        # ── Gate 2: Structural similarity ──
        # Cosine similarity in current embedding space
        # Soft gate: anomalous neighbors are down-weighted but NOT cut off,
        # so their distinctiveness is preserved in the aggregated representation
        sim = torch.sigmoid(
            F.cosine_similarity(x_i, x_j, dim=-1) / self.sim_temp
        )                                                    # (E,) in (0,1)

        # ── Gate 3: Temporal consistency ──
        # Learned response to surprise (not just sigmoid(-λ*s))
        # Allows model to learn: "moderate surprise is ok, extreme surprise is not"
        temporal_gate = self.surprise_gate(
            surprise.unsqueeze(-1)
        ).squeeze(-1)                                        # (E,)

        # ── Combine gates with learned weights ──
        # Softmax-normalize so total gate is in comparable range
        gate_w = F.softmax(self.log_gate_weights, dim=0)    # (3,) sums to 1
        combined_gate = (gate_w[0] * trust
                       + gate_w[1] * sim
                       + gate_w[2] * temporal_gate)         # (E,)

        # Cache for anomaly scoring downstream
        self.last_trust_scores    = trust.detach()
        self.last_sim_scores      = sim.detach()
        self.last_surprise_scores = surprise.detach()
        self.last_combined_gates  = combined_gate.detach()

        # ── Apply combined gate and normalize ──
        attn_score = attn_score * combined_gate.unsqueeze(-1)  # (E, heads)
        attn_score = softmax(attn_score, index)                # normalize over neighbors
        attn_score = self.dropout(attn_score)

        return V_j * attn_score.unsqueeze(-1)                  # (E, heads, head_dim)


# ═══════════════════════════════════════════════════════════════════════════════
# 8. Full Model
#
# Forward pass:
#   1. Project node features
#   2. Compute per-node Δt (time since last interaction) → state decay
#   3. Compute per-edge Δt_ij (time since this pair last interacted) + surprise
#   4. Update edge-pair temporal memory
#   5. Run L layers of JointTemporalStructuralAttention
#   6. Update recurrent node memory + evolution bank
#   7. Return node embeddings + anomaly signals
# ═══════════════════════════════════════════════════════════════════════════════

class TemporalStructuralGNN(nn.Module):
    def __init__(self, num_nodes, in_dim, edge_dim, hidden_dim,
                 num_layers=2, heads=8, dropout=0.1,
                 window=6, memory_momentum=0.9,
                 sim_temp=0.5, surprise_lambda=2.0):
        super().__init__()
        self.num_nodes  = num_nodes
        self.hidden_dim = hidden_dim

        # Input projections
        self.input_proj = nn.Linear(in_dim, hidden_dim)
        self.edge_enc   = nn.Linear(edge_dim, hidden_dim)

        # State decay (applied before aggregation)
        self.decay_gate = StateDecayGate(hidden_dim)

        # Persistent memories
        self.memory   = RecurrentMemory(num_nodes, hidden_dim, momentum=memory_momentum)
        self.evo_bank = EvolutionBank(num_nodes, hidden_dim, window=window)
        # Edge-pair temporal memory is NOT an nn.Module (no parameters, just buffers)
        self.pair_memory = None   # initialized in forward (needs device)

        # Attention layers
        self.layers = nn.ModuleList([
            JointTemporalStructuralAttention(
                hidden_dim, heads=heads, dropout=dropout,
                window=window, sim_temp=sim_temp,
                surprise_lambda=surprise_lambda
            )
            for _ in range(num_layers)
        ])

        self.final_norm = nn.LayerNorm(hidden_dim)
        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)

    def _get_pair_memory(self, device):
        if self.pair_memory is None:
            self.pair_memory = EdgePairTemporalMemory(self.num_nodes, device=device)
        if self.pair_memory.device != device:
            self.pair_memory = self.pair_memory.to(device)
        return self.pair_memory

    def forward(self, batch):
        """
        batch.x          : (N, in_dim)
        batch.edge_index : (2, E)
        batch.t          : (E,)   edge timestamps (float, larger = later)
        batch.msg        : (E, edge_dim)

        Returns:
            x_final      : (N, hidden_dim)  node representations
            anomaly_info : dict with per-edge anomaly signals
        """
        N      = batch.x.size(0)
        device = batch.x.device
        src, dst = batch.edge_index

        # ── Step 1: Project inputs ──
        x        = self.input_proj(batch.x)                   # (N, D)
        edge_emb = self.edge_enc(batch.msg)                   # (E, D)
        t        = batch.t.float()

        # ── Step 2: State decay — per-node Δt ──
        # For each node, how long since it was last involved in any interaction?
        # We use the minimum timestamp in this batch as "t_now" reference
        t_now_per_node = torch.zeros(N, device=device)
        t_now_per_node.scatter_reduce_(
            0, src, t, reduce='amax', include_self=True
        )
        # Also consider edges where node is destination
        t_now_per_node.scatter_reduce_(
            0, dst, t, reduce='amax', include_self=True
        )

        # Per-node Δt: time elapsed since last interaction in memory
        delta_t_node = self.memory.get_delta_t(
            torch.arange(N, device=device), t_now_per_node
        )                                                      # (N,)

        # Apply state decay to node embeddings before aggregation
        # The decay modulates the node's OWN memory contribution, not neighbors'
        x = self.decay_gate(x, delta_t_node)                  # (N, D)

        # ── Step 3: Per-edge temporal consistency ──
        pair_mem = self._get_pair_memory(device)
        surprise, delta_t_edge = pair_mem.get_surprise(src, dst, t)
        # surprise:      (E,)  — how anomalous is this edge's timing?
        # delta_t_edge:  (E,)  — actual gap since this pair last interacted

        # ── Step 4: Update edge-pair memory ──
        pair_mem.update(src, dst, t, delta_t_edge)

        # ── Step 5: Run attention layers ──
        for layer in self.layers:
            x = layer(
                x, batch.edge_index, edge_emb,
                delta_t_edge, surprise,
                self.memory, self.evo_bank
            )

        x = self.final_norm(x)

        # ── Step 6: Update recurrent node memory ──
        self.memory.write(
            torch.arange(N, device=device), x, t_now_per_node
        )

        # ── Step 7: Update evolution bank ──
        neigh = torch.zeros(N, self.hidden_dim, device=device)
        cnt   = torch.zeros(N, 1, device=device)
        neigh.scatter_add_(0, dst.unsqueeze(1).expand(-1, self.hidden_dim), x[src])
        cnt.scatter_add_(0, dst.unsqueeze(1), torch.ones(src.size(0), 1, device=device))
        neigh /= cnt + 1e-6
        unique_dst = dst.unique()
        self.evo_bank.write(unique_dst, neigh[unique_dst])

        # ── Step 8: Collect anomaly signals ──
        # These can be used as auxiliary features or direct anomaly scores
        # alongside contrastive loss during training
        last_layer = self.layers[-1]
        anomaly_info = {
            # Per-edge signals
            "surprise":       last_layer.last_surprise_scores,  # (E,) temporal anomaly
            "trust":          last_layer.last_trust_scores,     # (E,) low = distrusted
            "sim":            last_layer.last_sim_scores,       # (E,) low = structurally distant
            "combined_gate":  last_layer.last_combined_gates,   # (E,) overall suspicion

            # Per-node signals derived from edges
            # Max surprise among all edges involving node i → how anomalous is i's timing?
            "node_max_surprise": _scatter_max(surprise, src, N),
            # Mean gate weight received by each node → how much do neighbors trust i?
            "node_mean_gate":    _scatter_mean(last_layer.last_combined_gates, dst, N),
        }

        return x


def _scatter_max(src_vals, idx, N):
    """Max of src_vals for each target index. (E,) → (N,)"""
    out = torch.zeros(N, device=src_vals.device)
    out.scatter_reduce_(0, idx, src_vals, reduce='amax', include_self=True)
    return out


def _scatter_mean(src_vals, idx, N):
    """Mean of src_vals for each target index. (E,) → (N,)"""
    out = torch.zeros(N, device=src_vals.device)
    cnt = torch.zeros(N, device=src_vals.device)
    out.scatter_add_(0, idx, src_vals)
    cnt.scatter_add_(0, idx, torch.ones_like(src_vals))
    return out / (cnt + 1e-6)


# ═══════════════════════════════════════════════════════════════════════════════
# 9. Contrastive Projection Head
#
# For unsupervised anomaly detection via contrastive learning.
# Normal nodes: similar embeddings across augmented views
# Anomalous nodes: embeddings that are outliers in projection space
# ═══════════════════════════════════════════════════════════════════════════════

class ContrastiveHead(nn.Module):
    def __init__(self, hidden_dim, proj_dim=128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.GELU(),
            nn.LayerNorm(hidden_dim),
            nn.Linear(hidden_dim, proj_dim),
        )

    def forward(self, x):
        return F.normalize(self.net(x), dim=-1)


# ═══════════════════════════════════════════════════════════════════════════════
# 10. Full Pipeline
# ═══════════════════════════════════════════════════════════════════════════════

class FullModel(nn.Module):
    """
    GNN + contrastive head.

    Training usage:
        h, anomaly_info = model.encode(batch)
        z = model.project(h)
        loss = contrastive_loss(z, z_augmented)
        # Optionally add auxiliary loss on anomaly_info signals

    Inference:
        h, anomaly_info = model.encode(batch)
        # h → feed to anomaly scorer (e.g. KNN distance, LOF, GMM)
        # anomaly_info["node_max_surprise"] → direct temporal anomaly score
        # anomaly_info["node_mean_gate"]    → direct structural anomaly score
    """
    def __init__(self, num_nodes, in_dim, edge_dim, hidden_dim,
                 proj_dim=128, **gnn_kwargs):
        super().__init__()
        self.gnn  = TemporalStructuralGNN(
            num_nodes, in_dim, edge_dim, hidden_dim, **gnn_kwargs
        )
        self.head = ContrastiveHead(hidden_dim, proj_dim)

    def encode(self, batch):
        return self.gnn(batch)            # (N, D), anomaly_info

    def project(self, h):
        return self.head(h)              # (N, proj_dim), L2-normalized

    def forward(self, batch):
        h, info = self.encode(batch)
        z = self.project(h)
        return h, z, info

In [19]:
"""
Drop-in replacement for your notebook.a
Keeps: data, loaders, History, args, measure
Replaces: model, train(), test()

Just run these cells in order after your existing setup cells.
"""

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import MessagePassing
from torch_geometric.utils import softmax
from tqdm import tqdm


# ── Paste the full enhanced_gnn_v2.py here, or import it ──────────────────────
# from enhanced_gnn_v2 import (EnhancedTemporalGNN, train, test)
# ──────────────────────────────────────────────────────────────────────────────
# If running directly in notebook, paste the class definitions above this cell.


# ═══════════════════════════════════════════
# CELL: Build model + history (unchanged)
# ═══════════════════════════════════════════

try:
    del model
except NameError:
    pass




model = TemporalStructuralGNN(
    num_nodes  = data.num_nodes,
    in_dim     = data.x.size(1),          # 1 for wikipedia
    edge_dim   = data.msg.size(1),         # 172 for wikipedia
    hidden_dim = args.hidden_channels,     # 64
    num_layers = args.nums_layers,         # 2
    heads      = args.heads,               # 8
    dropout    = args.dropout,  
    window     = 4,
    memory_momentum = 0.9,
).to(device)

# History is UNCHANGED — reuse your existing History class
try:
    del history
except NameError:
    pass

from mhiscl.history import History

history = History(
    data.num_nodes,
    args.num_timeslots,
    args.hidden_channels,
    device=device,
    history_retrieve=args.history_retrieve,
    recurrent=args.recurrent,
).to(device)

print(model)
print(history)
print(f"Model params: {sum(p.numel() for p in model.parameters()):,}")




TemporalStructuralGNN(
  (input_proj): Linear(in_features=1, out_features=64, bias=True)
  (edge_enc): Linear(in_features=172, out_features=64, bias=True)
  (decay_gate): StateDecayGate(
    (time_enc): TimeEncoder(
      (t2v): Time2Vec()
      (proj): Sequential(
        (0): Linear(in_features=64, out_features=64, bias=True)
        (1): GELU(approximate='none')
        (2): Linear(in_features=64, out_features=64, bias=True)
      )
    )
    (W_h): Linear(in_features=64, out_features=64, bias=False)
    (W_t): Linear(in_features=64, out_features=64, bias=False)
    (norm): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
  )
  (memory): RecurrentMemory(
    (gru): GRUCell(64, 64)
  )
  (evo_bank): EvolutionBank()
  (layers): ModuleList(
    (0-1): 2 x JointTemporalStructuralAttention()
  )
  (final_norm): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
)
History(
  (recurrent_network): GRUCell(64, 64)
  (lin): Sequential(
    (0): Linear(-1, 64, bias=True)
    (1): ELU(alp

In [20]:
# ═══════════════════════════════════════════
# CELL: Training loop (drop-in replacement)
# ═══════════════════════════════════════════

best     = 0
best_val = 0

optimizer = torch.optim.Adam(
    list(model.parameters()) + list(history.parameters()),
    lr=args.lr,
    weight_decay=1e-5       # small L2 reg helps generalization
)

# Cosine LR schedule: smoothly decays over training
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=args.epochs, eta_min=args.lr * 0.1)

measure = Measure("auc")

for epoch in range(1, 1 + args.epochs):
    loss    = train(train_loader, model, history, optimizer, device, data, args)
    val_auc = test(valid_loader,  model, history, measure,   device, data)
    test_auc = test(test_loader,  model, history, measure,   device, data)
    scheduler.step()

    if val_auc > best_val:
        best_val = val_auc
        best     = test_auc

    print(
        f"Epoch: {epoch:03d}, Loss: {loss:.4f}, "
        f"Val AUC: {val_auc:.2%}, Test AUC: {test_auc:.2%}, Best AUC: {best:.2%}"
    )

100%|██████████| 93/93 [00:04<00:00, 19.35it/s]


Epoch: 001, Loss: 17.4696, Val AUC: 82.52%, Test AUC: 84.92%, Best AUC: 84.92%


100%|██████████| 93/93 [00:04<00:00, 19.78it/s]


Epoch: 002, Loss: 13.5701, Val AUC: 80.47%, Test AUC: 88.58%, Best AUC: 84.92%


100%|██████████| 93/93 [00:04<00:00, 19.92it/s]


...

Epoch: 025, Loss: 11.5326, Val AUC: 82.47%, Test AUC: 67.56%, Best AUC: 84.29%


100%|██████████| 93/93 [00:04<00:00, 20.09it/s]


Epoch: 026, Loss: 11.5639, Val AUC: 72.33%, Test AUC: 75.72%, Best AUC: 84.29%


Train Loss = 11.6651:  18%|█▊        | 79/431 [00:07<00:34, 10.19it/s]


KeyboardInterrupt: 

In [21]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import MessagePassing
from torch_geometric.utils import softmax


# ═══════════════════════════════════════════════════════════════════════════════
# 1. Time2Vec
# ═══════════════════════════════════════════════════════════════════════════════

class Time2Vec(nn.Module):
    """
    Encodes a scalar time value into a learned vector representation:
      [linear_trend, sin(w1*t+b1), ..., sin(wk*t+bk)]
    Frequencies are learned from data.
    """
    def __init__(self, out_dim):
        super().__init__()
        k        = out_dim - 1
        self.w0  = nn.Parameter(torch.randn(1) * 0.01)
        self.b0  = nn.Parameter(torch.zeros(1))
        self.W   = nn.Parameter(torch.randn(k) * 0.01)
        self.B   = nn.Parameter(torch.zeros(k))

    def forward(self, t):                              # t: (N,)
        t        = t.float().unsqueeze(-1)
        linear   = t * self.w0 + self.b0
        periodic = torch.sin(t * self.W + self.B)
        return torch.cat([linear, periodic], dim=-1)   # (N, out_dim)


class TimeEncoder(nn.Module):
    def __init__(self, hidden_dim):
        super().__init__()
        self.t2v  = Time2Vec(hidden_dim)
        self.proj = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, hidden_dim),
        )

    def forward(self, t):                              # t: (N,) → (N, D)
        return self.proj(self.t2v(t))


# ═══════════════════════════════════════════════════════════════════════════════
# 2. State Decay Gate
#
# Before aggregating neighbor messages, decay node i's hidden state
# by how much time has passed since its last interaction.
#
# h_decayed = sigmoid(W_h·h + W_t·time_enc(Δt_node)) ⊙ h
#
# Why this matters for representation:
#   The decayed state becomes part of the node's representation context.
#   A node that has been silent for a long time will have a strongly decayed
#   state — when it receives new messages, the update will be large, creating
#   a representation shift. This shift IS the anomaly signal, baked into h.
# ═══════════════════════════════════════════════════════════════════════════════

class StateDecayGate(nn.Module):
    def __init__(self, hidden_dim):
        super().__init__()
        self.time_enc = TimeEncoder(hidden_dim)
        self.W_h      = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.W_t      = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.b        = nn.Parameter(torch.zeros(hidden_dim))
        self.norm     = nn.LayerNorm(hidden_dim)

    def forward(self, h, delta_t_node):                # h: (N,D), delta_t: (N,)
        t_enc = self.time_enc(delta_t_node)            # (N, D)
        gate  = torch.sigmoid(self.W_h(h) + self.W_t(t_enc) + self.b)
        return self.norm(gate * h)                     # (N, D)


# ═══════════════════════════════════════════════════════════════════════════════
# 3. Edge-Pair Temporal Memory (non-parametric buffer)
#
# Tracks per (src, dst) pair:
#   - last seen time
#   - EMA of interaction gap  μ_ij
#   - EMA of gap variance     σ²_ij
#
# Computes "surprise" for each edge:
#   surprise = |Δt_ij - μ_ij| / (σ_ij + ε)
#
# Surprise is NOT returned as a separate output — it is encoded and
# injected into the node representation via the attention mechanism.
# ═══════════════════════════════════════════════════════════════════════════════

class EdgePairTemporalMemory:
    def __init__(self, num_nodes, momentum=0.9, device='cpu'):
        self.num_nodes = num_nodes
        self.momentum  = momentum
        self.device    = device
        M = num_nodes * num_nodes
        self.last_t   = torch.full((M,), -1.0,  device=device)
        self.mean_gap = torch.zeros(M,           device=device)
        self.var_gap  = torch.ones(M,            device=device)  # start uncertain
        self.count    = torch.zeros(M, dtype=torch.long, device=device)

    def _idx(self, src, dst):
        return src * self.num_nodes + dst

    def get_surprise_and_delta(self, src, dst, t, eps=1e-6):
        """
        Returns:
          surprise:     (E,) how anomalous is this edge's timing
          delta_t_edge: (E,) actual gap since this pair last interacted
        """
        pidx    = self._idx(src, dst)
        last    = self.last_t[pidx]
        seen    = last >= 0

        delta_t = torch.zeros_like(t)
        delta_t[seen] = (t[seen] - last[seen]).clamp(min=0)

        mu      = self.mean_gap[pidx]
        sig     = self.var_gap[pidx].sqrt()

        # Unseen pairs: neutral surprise = 0 (no prior expectation)
        surprise = torch.zeros_like(t)
        surprise[seen] = ((delta_t[seen] - mu[seen]).abs() / (sig[seen] + eps))

        return surprise, delta_t

    @torch.no_grad()
    def update(self, src, dst, t, delta_t):
        pidx = self._idx(src, dst)
        seen = self.last_t[pidx] >= 0
        if seen.any():
            i        = pidx[seen]
            dt       = delta_t[seen]
            old_mu   = self.mean_gap[i]
            self.mean_gap[i] = self.momentum * old_mu + (1 - self.momentum) * dt
            self.var_gap[i]  = (self.momentum * self.var_gap[i]
                                + (1 - self.momentum) * (dt - old_mu).pow(2))
        self.last_t[pidx]  = t
        self.count[pidx]  += 1

    def to(self, device):
        self.device   = device
        self.last_t   = self.last_t.to(device)
        self.mean_gap = self.mean_gap.to(device)
        self.var_gap  = self.var_gap.to(device)
        self.count    = self.count.to(device)
        return self


# ═══════════════════════════════════════════════════════════════════════════════
# 4. Recurrent Node Memory
# ═══════════════════════════════════════════════════════════════════════════════

class RecurrentMemory(nn.Module):
    def __init__(self, num_nodes, dim, momentum=0.9):
        super().__init__()
        self.gru      = nn.GRUCell(dim, dim)
        self.momentum = momentum
        self.register_buffer("hidden",  torch.zeros(num_nodes, dim))
        self.register_buffer("last_t",  torch.full((num_nodes,), -1.0))

    def get_hidden(self, idx):
        return self.hidden[idx]

    def get_delta_t(self, idx, t_now):
        last  = self.last_t[idx]
        return torch.where(last >= 0,
                           (t_now - last).clamp(min=0),
                           torch.zeros_like(t_now))

    @torch.no_grad()
    def write(self, idx, x, t_now):
        self.hidden[idx] = self.gru(x.detach(), self.hidden[idx])
        self.last_t[idx] = t_now


# ═══════════════════════════════════════════════════════════════════════════════
# 5. Joint Temporal-Structural Attention  ← CORE LAYER
#
# Single gate: SIMILARITY
#   - Similar neighbors teach each other (contrastive learning rationale)
#   - Dissimilar (anomalous) neighbors are down-weighted
#   - Soft gate: anomalous nodes still receive signal, they just receive less
#     → their embedding stays distinctly different from normal nodes
#
# Attention score is JOINT (not factored):
#   score = (Q_i · K_j) / √d          ← structural: who is relevant
#         + time_bias(Δt_ij)           ← temporal: how recent is j
#         + surprise_bias(surprise_ij) ← anomaly: how unexpected is this interaction
#
# All three signals collapse into ONE score, so the final node representation
# is a single vector that encodes structure + time + anomaly signal together.
#
# After aggregation, a GRU fuses the aggregated message with the node's
# recurrent history, producing a final representation that also carries
# the node's longitudinal behavior across batches.
# ═══════════════════════════════════════════════════════════════════════════════

class JointTemporalStructuralAttention(MessagePassing):
    def __init__(self, dim, heads=8, dropout=0.1, sim_temp=0.5):
        super().__init__(aggr="add", node_dim=0)
        assert dim % heads == 0
        self.dim      = dim
        self.heads    = heads
        self.head_dim = dim // heads
        self.scale    = self.head_dim ** -0.5
        self.sim_temp = sim_temp

        self.q_proj = nn.Linear(dim, dim)
        self.k_proj = nn.Linear(dim, dim)
        self.v_proj = nn.Linear(dim, dim)
        self.o_proj = nn.Linear(dim, dim)

        # Edge features enrich the keys
        self.edge_proj = nn.Linear(dim, dim)

        # Temporal recency: Δt_ij → per-head score bias
        self.time_enc      = TimeEncoder(dim)
        self.time_to_score = nn.Linear(dim, heads)   # (E, heads) additive bias

        # Temporal surprise: surprise_ij → per-head score bias
        # Learned so model can have non-monotone response to surprise
        # e.g. mild surprise is normal, extreme surprise is anomalous
        self.surprise_enc      = nn.Sequential(
            nn.Linear(1, dim), nn.GELU(), nn.Linear(dim, dim)
        )
        self.surprise_to_score = nn.Linear(dim, heads)  # (E, heads) additive bias

        self.dropout = nn.Dropout(dropout)

        # Gated residual between old embedding and aggregated message
        self.gate = nn.Linear(dim, 1)

        # GRU fuses aggregated message with recurrent history
        # This is where longitudinal behavior (across batches) meets
        # current structural+temporal context
        self.gru  = nn.GRUCell(dim, dim)

        self.norm = nn.LayerNorm(dim)

    def split(self, x):
        return x.view(-1, self.heads, self.head_dim)

    def forward(self, x, edge_index, edge_emb,
                delta_t_edge, surprise, memory):
        """
        x:             (N, D)  node embeddings (already decay-gated)
        edge_index:    (2, E)
        edge_emb:      (E, D)  encoded edge features
        delta_t_edge:  (E,)    time since this (src,dst) pair last interacted
        surprise:      (E,)    temporal surprise for each edge
        memory:        RecurrentMemory
        """
        N      = x.size(0)
        device = x.device

        Q = self.split(self.q_proj(x))   # (N, heads, head_dim)
        K = self.split(self.k_proj(x))
        V = self.split(self.v_proj(x))

        # Encode Δt_ij and surprise into per-head score biases
        time_bias     = self.time_to_score(
            self.time_enc(delta_t_edge)
        )                                                    # (E, heads)
        surprise_bias = self.surprise_to_score(
            self.surprise_enc(surprise.unsqueeze(-1))
        )                                                    # (E, heads)

        # Aggregate messages
        agg = self.propagate(
            edge_index,
            x=x,
            Q=Q, K=K, V=V,
            edge_emb=self.split(self.edge_proj(edge_emb)),  # (E, heads, head_dim)
            time_bias=time_bias,
            surprise_bias=surprise_bias,
            size=(N, N)
        )                                                    # (N, heads, head_dim)
        agg = agg.view(N, self.dim)                         # (N, D)

        # Project aggregated message
        agg = self.o_proj(agg)

        # ── Fuse with recurrent history via GRU ──
        # The GRU input is the aggregated structural+temporal message.
        # The GRU hidden state carries the node's longitudinal behavior.
        # Output = representation that knows BOTH current context AND history.
        h_prev = memory.get_hidden(torch.arange(N, device=device))
        h_new  = self.gru(agg, h_prev)                      # (N, D)

        # Gated residual: how much does new context update the embedding?
        g   = torch.sigmoid(self.gate(x))
        out = self.norm(g * x + (1 - g) * h_new)

        return out                                           # (N, D)

    def message(self, Q_i, K_j, V_j, x_i, x_j,
                edge_emb, time_bias, surprise_bias, index):

        # Enrich keys with edge features
        K_j = K_j + edge_emb                                # (E, heads, head_dim)

        # ── Joint attention score ──
        # Structural: Q·K measures relevance of j to i
        score = (Q_i * K_j).sum(-1) * self.scale            # (E, heads)
        # + Temporal recency: recent interactions get higher score
        score = score + time_bias                            # (E, heads)
        # + Surprise: anomalous timing shifts the score (learned direction)
        score = score + surprise_bias                        # (E, heads)
        # All three fused into one score → one softmax → one set of weights

        # ── Similarity gate ──
        # Similar nodes teach each other more.
        # Dissimilar (potentially anomalous) neighbors contribute less.
        # Soft: they still contribute, just less — their dissimilarity
        # IS the signal that separates them in representation space.
        sim  = torch.sigmoid(
            F.cosine_similarity(x_i, x_j, dim=-1) / self.sim_temp
        )                                                    # (E,)

        score = score * sim.unsqueeze(-1)                   # (E, heads)
        score = softmax(score, index)
        score = self.dropout(score)

        return V_j * score.unsqueeze(-1)                    # (E, heads, head_dim)


# ═══════════════════════════════════════════════════════════════════════════════
# 6. Full Model
#
# Everything collapses into ONE node representation vector h ∈ R^D that encodes:
#
#   STRUCTURAL  — which nodes i interacts with and how similar they are
#                 (similarity-gated attention, query-key structural score)
#
#   TEMPORAL    — when interactions happen, how bursty/regular the node is
#                 (state decay, Δt_node, Δt_edge recency bias)
#
#   ANOMALY     — whether interactions are temporally surprising
#                 (surprise bias in attention score)
#                 whether the node is structurally isolated
#                 (low similarity gate → less message → representation drift)
#
#   HISTORY     — how the node has evolved over time across batches
#                 (GRU with recurrent memory)
#
# No separate anomaly head. No separate outputs. One vector.
# ═══════════════════════════════════════════════════════════════════════════════

class TemporalStructuralGNN(nn.Module):
    def __init__(self, num_nodes, in_dim, edge_dim, hidden_dim,
                 num_layers=2, heads=8, dropout=0.1,
                 memory_momentum=0.9, sim_temp=0.5):
        super().__init__()
        self.num_nodes  = num_nodes
        self.hidden_dim = hidden_dim

        self.input_proj = nn.Linear(in_dim, hidden_dim)
        self.edge_enc   = nn.Linear(edge_dim, hidden_dim)

        # State decay: fades node memory based on inactivity
        self.decay_gate = StateDecayGate(hidden_dim)

        # Recurrent node memory (persistent across batches)
        self.memory = RecurrentMemory(num_nodes, hidden_dim, momentum=memory_momentum)

        # Edge-pair temporal memory (non-parametric, persistent across batches)
        self._pair_mem = None

        # Attention layers
        self.layers = nn.ModuleList([
            JointTemporalStructuralAttention(
                hidden_dim, heads=heads, dropout=dropout, sim_temp=sim_temp
            )
            for _ in range(num_layers)
        ])

        self.final_norm = nn.LayerNorm(hidden_dim)
        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)

    def _pair_memory(self, device):
        if self._pair_mem is None:
            self._pair_mem = EdgePairTemporalMemory(self.num_nodes, device=device)
        if self._pair_mem.device != device:
            self._pair_mem = self._pair_mem.to(device)
        return self._pair_mem

    def forward(self, batch):
        """
        batch.x          : (N, in_dim)
        batch.edge_index : (2, E)
        batch.t          : (E,)   edge timestamps
        batch.msg        : (E, edge_dim)

        Returns: (N, hidden_dim) — final node representations
        """
        N      = batch.x.size(0)
        device = batch.x.device
        src, dst = batch.edge_index
        t        = batch.t.float()

        # ── 1. Project inputs ──
        x        = self.input_proj(batch.x)     # (N, D)
        edge_emb = self.edge_enc(batch.msg)     # (E, D)

        # ── 2. Per-node Δt → state decay ──
        # Each node's last-seen time from memory; current time = max edge time touching it
        t_now = torch.zeros(N, device=device)
        t_now.scatter_reduce_(0, src, t, reduce='amax', include_self=True)
        t_now.scatter_reduce_(0, dst, t, reduce='amax', include_self=True)

        delta_t_node = self.memory.get_delta_t(torch.arange(N, device=device), t_now)
        x = self.decay_gate(x, delta_t_node)   # (N, D) — state fades with inactivity

        # ── 3. Per-edge surprise + Δt_edge ──
        pm = self._pair_memory(device)
        surprise, delta_t_edge = pm.get_surprise_and_delta(src, dst, t)
        pm.update(src, dst, t, delta_t_edge)
        # surprise: (E,) — 0=expected timing, high=anomalous timing

        # ── 4. Attention layers ──
        # Each layer fuses structure + time + surprise into node embeddings
        for layer in self.layers:
            x = layer(x, batch.edge_index, edge_emb,
                      delta_t_edge, surprise, self.memory)

        x = self.final_norm(x)

        # ── 5. Update recurrent memory ──
        self.memory.write(torch.arange(N, device=device), x, t_now)

        return x   # (N, D) — everything encoded in one vector


# ═══════════════════════════════════════════════════════════════════════════════
# 7. Contrastive Projection Head + Full Pipeline
# ═══════════════════════════════════════════════════════════════════════════════

class ContrastiveHead(nn.Module):
    def __init__(self, hidden_dim, proj_dim=128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.GELU(),
            nn.LayerNorm(hidden_dim),
            nn.Linear(hidden_dim, proj_dim),
        )

    def forward(self, x):
        return F.normalize(self.net(x), dim=-1)   # (N, proj_dim)


class FullModel(nn.Module):
    """
    Usage:

      # Training
      h = model.encode(batch)          # (N, D) — rich node representations
      z = model.project(h)             # (N, proj_dim) — for contrastive loss

      # Inference
      h = model.encode(batch)          # (N, D) — feed to anomaly detector
                                       #          (KNN, LOF, GMM, etc.)
    """
    def __init__(self, num_nodes, in_dim, edge_dim, hidden_dim,
                 proj_dim=128, **gnn_kwargs):
        super().__init__()
        self.gnn  = TemporalStructuralGNN(
            num_nodes, in_dim, edge_dim, hidden_dim, **gnn_kwargs
        )
        self.head = ContrastiveHead(hidden_dim, proj_dim)

    def encode(self, batch):
        return self.gnn(batch)         # (N, D)

    def project(self, h):
        return self.head(h)            # (N, proj_dim)

    def forward(self, batch):
        h = self.encode(batch)
        z = self.project(h)
        return h, z

In [23]:
"""
Drop-in replacement for your notebook.a
Keeps: data, loaders, History, args, measure
Replaces: model, train(), test()

Just run these cells in order after your existing setup cells.
"""

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import MessagePassing
from torch_geometric.utils import softmax
from tqdm import tqdm


# ── Paste the full enhanced_gnn_v2.py here, or import it ──────────────────────
# from enhanced_gnn_v2 import (EnhancedTemporalGNN, train, test)
# ──────────────────────────────────────────────────────────────────────────────
# If running directly in notebook, paste the class definitions above this cell.


# ═══════════════════════════════════════════
# CELL: Build model + history (unchanged)
# ═══════════════════════════════════════════

try:
    del model
except NameError:
    pass




model = TemporalStructuralGNN(
    num_nodes  = data.num_nodes,
    in_dim     = data.x.size(1),          # 1 for wikipedia
    edge_dim   = data.msg.size(1),         # 172 for wikipedia
    hidden_dim = args.hidden_channels,     # 64
    num_layers = args.nums_layers,         # 2
    heads      = args.heads,               # 8
    dropout    = args.dropout,  
    memory_momentum = 0.9,
).to(device)

# History is UNCHANGED — reuse your existing History class
try:
    del history
except NameError:
    pass

from mhiscl.history import History

history = History(
    data.num_nodes,
    args.num_timeslots,
    args.hidden_channels,
    device=device,
    history_retrieve=args.history_retrieve,
    recurrent=args.recurrent,
).to(device)

print(model)
print(history)
print(f"Model params: {sum(p.numel() for p in model.parameters()):,}")




TemporalStructuralGNN(
  (input_proj): Linear(in_features=1, out_features=64, bias=True)
  (edge_enc): Linear(in_features=172, out_features=64, bias=True)
  (decay_gate): StateDecayGate(
    (time_enc): TimeEncoder(
      (t2v): Time2Vec()
      (proj): Sequential(
        (0): Linear(in_features=64, out_features=64, bias=True)
        (1): GELU(approximate='none')
        (2): Linear(in_features=64, out_features=64, bias=True)
      )
    )
    (W_h): Linear(in_features=64, out_features=64, bias=False)
    (W_t): Linear(in_features=64, out_features=64, bias=False)
    (norm): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
  )
  (memory): RecurrentMemory(
    (gru): GRUCell(64, 64)
  )
  (layers): ModuleList(
    (0-1): 2 x JointTemporalStructuralAttention()
  )
  (final_norm): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
)
History(
  (recurrent_network): GRUCell(64, 64)
  (lin): Sequential(
    (0): Linear(-1, 64, bias=True)
    (1): ELU(alpha=1.0)
    (2): Linear(64, 64

In [24]:
# ═══════════════════════════════════════════
# CELL: Training loop (drop-in replacement)
# ═══════════════════════════════════════════

best     = 0
best_val = 0

optimizer = torch.optim.Adam(
    list(model.parameters()) + list(history.parameters()),
    lr=args.lr,
    weight_decay=1e-5       # small L2 reg helps generalization
)

# Cosine LR schedule: smoothly decays over training
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=args.epochs, eta_min=args.lr * 0.1)

measure = Measure("auc")

for epoch in range(1, 1 + args.epochs):
    loss    = train(train_loader, model, history, optimizer, device, data, args)
    val_auc = test(valid_loader,  model, history, measure,   device, data)
    test_auc = test(test_loader,  model, history, measure,   device, data)
    scheduler.step()

    if val_auc > best_val:
        best_val = val_auc
        best     = test_auc

    print(
        f"Epoch: {epoch:03d}, Loss: {loss:.4f}, "
        f"Val AUC: {val_auc:.2%}, Test AUC: {test_auc:.2%}, Best AUC: {best:.2%}"
    )

100%|██████████| 93/93 [00:02<00:00, 45.32it/s]


Epoch: 001, Loss: 18.1165, Val AUC: 77.14%, Test AUC: 84.04%, Best AUC: 84.04%


100%|██████████| 93/93 [00:01<00:00, 47.44it/s]


Epoch: 002, Loss: 14.6717, Val AUC: 84.26%, Test AUC: 86.37%, Best AUC: 86.37%


100%|██████████| 93/93 [00:01<00:00, 48.27it/s]


...

Epoch: 033, Loss: 11.6732, Val AUC: 64.78%, Test AUC: 53.64%, Best AUC: 86.42%


100%|██████████| 93/93 [00:01<00:00, 54.35it/s]


Epoch: 034, Loss: 11.6221, Val AUC: 49.51%, Test AUC: 45.46%, Best AUC: 86.42%


100%|██████████| 93/93 [00:01<00:00, 52.49it/s]


Epoch: 035, Loss: 11.7040, Val AUC: 51.69%, Test AUC: 46.24%, Best AUC: 86.42%


In [1]:
"""
Temporal Trust GNN for Unsupervised Anomaly Detection
=====================================================
Produces node representations for downstream contrastive learning.
No labels required — fully unsupervised.

Message Aggregation (TemporalTrustAttention):

  [1] CROSS-SOURCE ATTENTION
      Q comes from EDGE embeddings  (what does this interaction ask for?)
      K comes from NODE features x  (who is this neighbour?)
      The same neighbour j is weighted differently depending on the
      type/context of the edge, making attention interaction-aware.

  [2] RoPE ON MESSAGES  (MessageRoPE)
      After scoring, the value message V_j is rotated by its edge
      timestamp using continuous-time RoPE BEFORE aggregation.
      Time is therefore baked into the message CONTENT, not just
      the score — messages from different times aggregate to different
      vectors even with identical node features.

  [3] SIMILARITY-BASED TRUST  (StructuralTrustModule)
      trust(i->j) in (0, 2):
        trust > 1 : i and j are structurally similar -> amplify
        trust < 1 : i and j are dissimilar           -> attenuate
      Both normal<->normal and anomalous<->anomalous get high trust.
      The model self-segregates: each node learns from its own "type".
      This naturally separates normal and anomalous representations.

  [4] TEMPORAL MESSAGE BANK  (TemporalMessageBank) — NEW
      A per-node buffer stores the last M aggregated messages.
      After each aggregation, cross-attention reads over this buffer:
        Q = current aggregated message
        K/V = past M stored messages
      Output: temporally-smoothed message that blends current with
      historical incoming signals. Detects sudden anomalous changes
      in what a node is receiving (erratic = anomalous influence).

Trust signals (StructuralTrustModule):
  [A] cos(h_i, h_j)              current GRU state similarity
  [B] degree profile similarity  structural role similarity
  [C] K-view trajectory sim      RoPE+Time2Vec encoded snapshot comparison
      trust = 1 + tanh(MLP([A, B, C]))   in (0, 2)
"""

import torch
import torch.nn as nn
import torch.nn.functional as F
from typing import Optional, Tuple
from torch_geometric.nn import MessagePassing, TransformerConv
from torch_geometric.utils import softmax, degree


# ═══════════════════════════════════════════════════════════════════════════════
# 1.  Time2Vec
# ═══════════════════════════════════════════════════════════════════════════════

class Time2Vec(nn.Module):
    """
    Encodes a scalar timestamp into a vector:
        [w0*t + b0,  sin(w1*t + b1), ..., sin(w_{k}*t + b_{k})]
    Linear term = trend.  Sinusoids = periodicity.  All frequencies learned.
    """
    def __init__(self, out_dim: int):
        super().__init__()
        assert out_dim >= 2
        k = out_dim - 1
        self.w0 = nn.Parameter(torch.randn(1) * 0.01)
        self.b0 = nn.Parameter(torch.zeros(1))
        self.W  = nn.Parameter(torch.randn(k) * 0.01)
        self.B  = nn.Parameter(torch.zeros(k))

    def forward(self, t: torch.Tensor) -> torch.Tensor:
        """t: (*,) -> (*, out_dim)"""
        t   = t.float().unsqueeze(-1)
        lin = t * self.w0 + self.b0
        per = torch.sin(t * self.W + self.B)
        return torch.cat([lin, per], dim=-1)


# ═══════════════════════════════════════════════════════════════════════════════
# 2.  TimeEncoder  (edge-level: absolute + relative timestamp -> hidden_dim)
# ═══════════════════════════════════════════════════════════════════════════════

class TimeEncoder(nn.Module):
    def __init__(self, hidden_dim: int):
        super().__init__()
        half = hidden_dim // 2
        self.t2v_abs = Time2Vec(half)
        self.t2v_rel = Time2Vec(half)
        self.proj = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, hidden_dim),
        )

    def forward(self, t_abs: torch.Tensor,
                t_rel: Optional[torch.Tensor] = None) -> torch.Tensor:
        """(E,), (E,) -> (E, hidden_dim)"""
        a = self.t2v_abs(t_abs)
        r = self.t2v_rel(t_rel) if t_rel is not None else torch.zeros_like(a)
        return self.proj(torch.cat([a, r], dim=-1))


# ═══════════════════════════════════════════════════════════════════════════════
# 3.  RecurrentMemory  (per-node GRU hidden state)
# ═══════════════════════════════════════════════════════════════════════════════

class RecurrentMemory(nn.Module):
    """
    Maintains one GRU hidden state per node, updated after each snapshot.
    The hidden state is the running summary of the node's full history.
    """
    def __init__(self, num_nodes: int, dim: int, momentum: float = 0.9):
        super().__init__()
        self.gru      = nn.GRUCell(dim, dim)
        self.momentum = momentum
        self.register_buffer("hidden",   torch.zeros(num_nodes, dim))
        self.register_buffer("variance", torch.ones(num_nodes, dim) * 0.1)

    def read(self, idx: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        return self.hidden[idx], self.variance[idx]

    def get_hidden(self, idx: torch.Tensor) -> torch.Tensor:
        return self.hidden[idx]

    @torch.no_grad()
    def write(self, idx: torch.Tensor, x: torch.Tensor):
        h_old = self.hidden[idx]
        h_new = self.gru(x.detach(), h_old)
        delta = h_new - h_old
        self.variance[idx] = (self.momentum * self.variance[idx]
                              + (1 - self.momentum) * delta.pow(2))
        self.hidden[idx] = h_new


# ═══════════════════════════════════════════════════════════════════════════════
# 4.  SnapshotGNN  — TransformerConv-based node encoder
#
#  Produces the snapshot embedding stored in the EvolutionBank.
#  Uses PyG's TransformerConv which performs full transformer-style
#  multi-head attention between a node and its one-hop neighbours,
#  incorporating edge_attr directly into the key/value computation:
#
#      alpha(i,j) = softmax( (W_Q * x_i)^T (W_K * x_j + W_E * edge_attr) )
#      out_i      = W_O * sum_j [ alpha(i,j) * (W_V * x_j + W_E * edge_attr) ]
#
#  Inputs: node features (x) and edge features (edge_attr) ONLY.
#  Time is NOT used here — it is handled separately in the positional
#  encoding step when comparing K-view trajectories in the trust module.
#
#  Two TransformerConv layers with a residual connection give the model
#  capacity to capture both direct neighbours and two-hop structure.
# ═══════════════════════════════════════════════════════════════════════════════

class SnapshotGNN(nn.Module):
    """
    Transformer-based snapshot encoder using PyG's TransformerConv.

    Produces a rich node embedding from node features and edge attributes,
    capturing neighbourhood structure via transformer-style attention.
    Time is deliberately excluded — it is injected separately during
    the K-view positional encoding in StructuralTrustModule.

    Args:
        dim      : hidden dimension (input, output, and internal)
        heads    : number of attention heads for TransformerConv
        dropout  : dropout on attention weights
    """
    def __init__(self, dim: int, heads: int = 4, dropout: float = 0.1):
        super().__init__()
        assert dim % heads == 0, "dim must be divisible by heads"

        # Layer 1: node features + edge_attr -> dim
        self.conv1 = TransformerConv(
            in_channels  = dim,
            out_channels = dim // heads,
            heads        = heads,
            dropout      = dropout,
            edge_dim     = dim,        # edge_attr dimension matches hidden dim
            concat       = True,       # concat heads -> dim
            beta         = True,       # learnable skip connection inside conv
        )
        self.norm1 = nn.LayerNorm(dim)

        # Layer 2: refine with a second transformer hop
        self.conv2 = TransformerConv(
            in_channels  = dim,
            out_channels = dim // heads,
            heads        = heads,
            dropout      = dropout,
            edge_dim     = dim,
            concat       = True,
            beta         = True,
        )
        self.norm2 = nn.LayerNorm(dim)

        # Final projection + residual gate
        self.out_proj = nn.Linear(dim, dim)
        self.gate     = nn.Linear(dim, 1)

    def forward(self,
                x:          torch.Tensor,   # (N, dim)   projected node features
                edge_index: torch.Tensor,   # (2, E)
                edge_attr:  torch.Tensor,   # (E, dim)   projected edge features
                ) -> torch.Tensor:          # (N, dim)
        # Layer 1
        h = self.norm1(F.gelu(self.conv1(x, edge_index, edge_attr)))  # (N, dim)
        # Layer 2 with residual
        h2 = self.norm2(F.gelu(self.conv2(h, edge_index, edge_attr))) # (N, dim)
        h  = h + h2                                                    # residual

        # Gated output projection
        out = self.out_proj(h)
        g   = torch.sigmoid(self.gate(x))
        return g * x + (1 - g) * out                                  # (N, dim)


# ═══════════════════════════════════════════════════════════════════════════════
# 5.  EvolutionBank  — stores last K snapshot embeddings + timestamps per node
#
#  After each forward pass, the snapshot embedding produced by SnapshotGNN
#  is written here alongside its timestamp.
#  Shape: bank (num_nodes, K, dim),  times (num_nodes, K)
# ═══════════════════════════════════════════════════════════════════════════════

class EvolutionBank(nn.Module):
    def __init__(self, num_nodes: int, dim: int, window: int = 8):
        super().__init__()
        self.window = window
        self.dim    = dim
        self.register_buffer("bank",  torch.zeros(num_nodes, window, dim))
        self.register_buffer("times", torch.zeros(num_nodes, window))
        self.register_buffer("ptr",   torch.zeros(num_nodes, dtype=torch.long))

    def read(self, idx: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        """Returns (B, K, dim) embeddings and (B, K) timestamps."""
        return self.bank[idx], self.times[idx]

    @torch.no_grad()
    def write(self, idx: torch.Tensor, emb: torch.Tensor, t: torch.Tensor):
        """
        idx : (B,)    node indices
        emb : (B, D)  snapshot embedding from SnapshotGNN
        t   : (B,)    timestamp of this snapshot
        """
        for b in range(idx.size(0)):
            node = int(idx[b])
            p    = int(self.ptr[node]) % self.window
            self.bank[node,  p] = emb[b].detach()
            self.times[node, p] = float(t[b])
            self.ptr[node] += 1

    @torch.no_grad()
    def reset(self):
        self.bank.zero_()
        self.times.zero_()
        self.ptr.zero_()


# ═══════════════════════════════════════════════════════════════════════════════
# 6.  TemporalPositionalEncoding  — RoPE + Time2Vec for K-view snapshots
#
#  Applies two complementary positional encodings to each of the K
#  snapshot embeddings before trajectory comparison:
#
#  (a) RoPE (Rotary Position Encoding) adapted for continuous timestamps:
#      Pairs of dimensions in the embedding are rotated by an angle θ = t / T_d
#      where T_d is a learned per-dimension timescale.
#      This injects RELATIVE temporal information: the dot product between
#      two time-encoded views naturally captures their temporal distance.
#      If views i_k and j_k have similar timestamps, their rotated embeddings
#      will be close; if timestamps differ greatly, they will diverge.
#
#  (b) Time2Vec (additive):
#      Encodes the absolute timestamp as [linear, sin(w1*t), ..., sin(wk*t)]
#      and adds a learned projection of it to the embedding.
#      This captures absolute periodicity (daily/weekly patterns).
#
#  Combined: view_k = RoPE(emb_k, t_k) + Linear(Time2Vec(t_k))
#  The two encodings are complementary:
#      RoPE    captures relative temporal distance between views
#      Time2Vec captures absolute temporal position and periodicity
# ═══════════════════════════════════════════════════════════════════════════════

class TemporalPositionalEncoding(nn.Module):
    """
    Applies RoPE + Time2Vec positional encoding to K snapshot embeddings.

    Args:
        dim     : embedding dimension (must be even for RoPE)
        t2v_dim : dimension of Time2Vec encoding before projection
    """
    def __init__(self, dim: int, t2v_dim: int = 16):
        super().__init__()
        assert dim % 2 == 0, "dim must be even for RoPE"
        self.dim     = dim
        self.t2v_dim = t2v_dim

        # RoPE: learned per-dimension timescale (log-space for numerical stability)
        # Shape (dim//2,): one timescale per pair of dimensions
        self.log_timescales = nn.Parameter(
            torch.linspace(0, 9, dim // 2)  # init: timescales from 1 to e^9 ≈ 8000
        )

        # Time2Vec: absolute timestamp encoding
        self.t2v     = Time2Vec(t2v_dim)
        self.t2v_proj = nn.Linear(t2v_dim, dim)

    def _apply_rope(self,
                    embs:  torch.Tensor,   # (B, K, dim)
                    times: torch.Tensor,   # (B, K)
                    ) -> torch.Tensor:     # (B, K, dim)
        """
        Rotate each embedding by an angle proportional to its timestamp.
        For each pair of dimensions (2d, 2d+1):
            theta_d  = t / exp(log_timescales[d])
            [e_{2d}, e_{2d+1}] -> [e_{2d}*cos(theta) - e_{2d+1}*sin(theta),
                                   e_{2d}*sin(theta) + e_{2d+1}*cos(theta)]
        """
        B, K, D = embs.shape
        timescales = self.log_timescales.exp()                       # (D//2,)

        # angles: (B, K, D//2)
        angles = times.unsqueeze(-1) / timescales.unsqueeze(0).unsqueeze(0)

        cos_a = angles.cos()   # (B, K, D//2)
        sin_a = angles.sin()

        # Split embedding into even/odd dimensions
        e_even = embs[..., 0::2]   # (B, K, D//2)
        e_odd  = embs[..., 1::2]

        # Apply rotation
        rot_even = e_even * cos_a - e_odd * sin_a
        rot_odd  = e_even * sin_a + e_odd * cos_a

        # Interleave back: (B, K, D)
        rotated = torch.stack([rot_even, rot_odd], dim=-1)           # (B, K, D//2, 2)
        return rotated.reshape(B, K, D)

    def forward(self,
                embs:  torch.Tensor,   # (B, K, dim)
                times: torch.Tensor,   # (B, K)
                ) -> torch.Tensor:     # (B, K, dim)
        B, K, D = embs.shape

        # (a) RoPE: rotate embedding in-place by timestamp angle
        rope_embs = self._apply_rope(embs, times)                    # (B, K, dim)

        # (b) Time2Vec: additive absolute time signal
        t_flat  = times.reshape(B * K)                               # (B*K,)
        t2v_enc = self.t2v(t_flat).view(B, K, self.t2v_dim)         # (B, K, t2v_dim)
        t2v_add = self.t2v_proj(t2v_enc)                             # (B, K, dim)

        # Combine: rotated embedding + absolute time encoding
        return rope_embs + t2v_add                                   # (B, K, dim)


# ═══════════════════════════════════════════════════════════════════════════════
# 7.  StructuralTrustModule  — CORE CONTRIBUTION
#
#  For every edge (i -> j), computes trust(i, j) in (0, 2).
#
#  [A] EMBEDDING SIMILARITY
#      cos(h_i, h_j) using current GRU hidden states.
#
#  [B] DEGREE PROFILE SIMILARITY
#      1 - |deg_i - deg_j| / (deg_i + deg_j)
#
#  [C] K-VIEW TRAJECTORY SIMILARITY
#      Each node has K snapshot embeddings in EvolutionBank (from TransformerConv).
#      Apply TemporalPositionalEncoding to each view:
#          view_k = RoPE(emb_k, t_k) + Time2Vec_proj(t_k)
#      Then compare position-aligned:
#          traj_sim = mean_k cos(view_i_k, view_j_k)
#
#  FUSION:
#      trust = 1 + tanh( MLP([A, B, C]) )   in (0, 2)
# ═══════════════════════════════════════════════════════════════════════════════

class StructuralTrustModule(nn.Module):
    def __init__(self, dim: int, window: int = 8, t2v_dim: int = 16):
        super().__init__()
        self.window = window

        # Temporal positional encoding: RoPE + Time2Vec
        self.time_pe = TemporalPositionalEncoding(dim, t2v_dim=t2v_dim)

        # Final MLP: 3 scalar signals -> trust logit
        h = max(dim // 4, 16)
        self.trust_mlp = nn.Sequential(
            nn.Linear(3, h),
            nn.GELU(),
            nn.Linear(h, h),
            nn.GELU(),
            nn.Linear(h, 1),
        )

    def forward(
        self,
        h_i:    torch.Tensor,   # (E, dim)    current GRU hidden state of i
        h_j:    torch.Tensor,   # (E, dim)    current GRU hidden state of j
        deg_i:  torch.Tensor,   # (E,)        in-degree of i
        deg_j:  torch.Tensor,   # (E,)        in-degree of j
        embs_i: torch.Tensor,   # (E, K, dim) last K snapshot embeddings of i
        embs_j: torch.Tensor,   # (E, K, dim) last K snapshot embeddings of j
        t_i:    torch.Tensor,   # (E, K)      timestamps of i's K snapshots
        t_j:    torch.Tensor,   # (E, K)      timestamps of j's K snapshots
        eps:    float = 1e-6,
    ) -> torch.Tensor:          # (E,) in (0, 2)

        # ── [A] Embedding similarity ──────────────────────────────────────────
        emb_cos = (F.normalize(h_i, dim=-1, eps=eps)
                   * F.normalize(h_j, dim=-1, eps=eps)
                   ).sum(-1, keepdim=True)                           # (E, 1)

        # ── [B] Degree profile similarity ─────────────────────────────────────
        di, dj  = deg_i.float(), deg_j.float()
        deg_sim = (1.0 - (di - dj).abs() / (di + dj + eps)
                   ).unsqueeze(-1)                                   # (E, 1)

        # ── [C] K-view trajectory similarity ─────────────────────────────────
        # Apply RoPE + Time2Vec to each of the K snapshot embeddings.
        # After encoding, view_k carries both the structural content of the
        # snapshot AND a rotation / additive signal from its timestamp.
        views_i = self.time_pe(embs_i, t_i)                         # (E, K, dim)
        views_j = self.time_pe(embs_j, t_j)

        # Position-aligned cosine similarity, averaged over K
        vi_n     = F.normalize(views_i, dim=-1, eps=eps)
        vj_n     = F.normalize(views_j, dim=-1, eps=eps)
        traj_sim = (vi_n * vj_n).sum(-1).mean(-1, keepdim=True)     # (E, 1)

        # ── Fuse and map to (0, 2) ────────────────────────────────────────────
        features    = torch.cat([emb_cos, deg_sim, traj_sim], dim=-1)  # (E, 3)
        trust_logit = self.trust_mlp(features).squeeze(-1)             # (E,)
        return 1.0 + torch.tanh(trust_logit)                          # (E,) in (0,2)


# ═══════════════════════════════════════════════════════════════════════════════
# 8.  MessageRoPE  — applies continuous-time RoPE directly to message vectors
#
#  Unlike the TemporalPositionalEncoding used in the trust module (which
#  operates on stored K-view snapshots), this module applies RoPE to the
#  LIVE message vector being passed from j to i along an edge, using the
#  edge's own timestamp as the rotation angle.
#
#  Why apply RoPE to messages (not just to queries/keys)?
#  ────────────────────────────────────────────────────────
#  In standard transformer attention, RoPE is applied to Q and K so that
#  the dot product Q·K reflects relative position.  Here we go further:
#  the VALUE (message content) is also rotated by the edge timestamp.
#  This means:
#    - Two messages with the same content but different timestamps will
#      aggregate to DIFFERENT vectors at the target node.
#    - The target node's representation is intrinsically time-aware,
#      not just score-aware.
#    - Recent messages occupy a different angular region from old ones,
#      so the GRU memory update naturally distinguishes temporal recency.
#
#  Shared log_timescales with the trust module's TemporalPositionalEncoding
#  are NOT used — this module has its own learned timescales tuned for
#  the message aggregation regime (shorter timescales work better here
#  since edge timestamps are absolute, not relative gaps).
# ═══════════════════════════════════════════════════════════════════════════════

class MessageRoPE(nn.Module):
    """
    Applies continuous-time RoPE to a batch of message vectors.

    Rotates pairs of dimensions by angles proportional to the edge timestamp:
        theta_d = t_edge / exp(log_timescale_d)
        msg[2d]   ->  msg[2d]   * cos(theta_d) - msg[2d+1] * sin(theta_d)
        msg[2d+1] ->  msg[2d]   * sin(theta_d) + msg[2d+1] * cos(theta_d)

    The dot product between two RoPE-encoded messages naturally reflects
    their temporal distance: same-time messages stay aligned, cross-time
    messages rotate apart in proportion to the timestamp gap.
    """
    def __init__(self, dim: int):
        super().__init__()
        assert dim % 2 == 0
        # Shorter timescales than K-view RoPE — edge-level granularity
        self.log_timescales = nn.Parameter(
            torch.linspace(0, 6, dim // 2)    # timescales: 1 → e^6 ≈ 400
        )

    def forward(self,
                msg: torch.Tensor,   # (E, dim)
                t:   torch.Tensor,   # (E,)   edge timestamps
                ) -> torch.Tensor:   # (E, dim)
        timescales = self.log_timescales.exp()              # (D//2,)
        angles     = t.float().unsqueeze(-1) / timescales  # (E, D//2)

        cos_a = angles.cos()   # (E, D//2)
        sin_a = angles.sin()

        m_even = msg[:, 0::2]  # (E, D//2)
        m_odd  = msg[:, 1::2]

        rot_even = m_even * cos_a - m_odd * sin_a
        rot_odd  = m_even * sin_a + m_odd * cos_a

        # Interleave back to (E, D)
        rotated = torch.stack([rot_even, rot_odd], dim=-1)  # (E, D//2, 2)
        return rotated.reshape(msg.shape)


# ═══════════════════════════════════════════════════════════════════════════════
# 9.  TemporalMessageBank  — NEW ADVANCED MECHANISM
#
#  Standard message passing aggregates ALL neighbours in one flat sum/mean.
#  This loses two things:
#    1. TEMPORAL ORDER: messages from t=10 and t=1000 are mixed equally.
#    2. TEMPORAL CONTEXT: the node has no memory of what it received before.
#
#  The TemporalMessageBank fixes both:
#    - After each forward pass, the aggregated message vector (already
#      RoPE-encoded) is stored in a per-node circular buffer of size M.
#    - Before the final update, a cross-attention reads over this buffer:
#        Query  = current aggregated message  (what I just received)
#        Key/V  = past M buffered messages    (what I've received before)
#      This lets the node ask: "is what I'm receiving NOW consistent with
#      my recent message history, or is it a sudden anomalous change?"
#    - The attention output is a temporally-smoothed message that blends
#      current and historical incoming signals.
#
#  Effect on anomaly detection:
#    Normal nodes receive consistent messages over time → strong cross-attn
#    Anomalous nodes receive erratic messages           → weak cross-attn
#    Central node near anomaly gets a divergent current msg vs history
#    → the bank effectively provides a "temporal message baseline"
# ═══════════════════════════════════════════════════════════════════════════════

class TemporalMessageBank(nn.Module):
    """
    Per-node circular buffer of past aggregated messages with
    cross-attention retrieval.

    Args:
        num_nodes : total nodes in the graph
        dim       : message / hidden dimension
        bank_size : M, number of past messages to store per node
    """
    def __init__(self, num_nodes: int, dim: int, bank_size: int = 8):
        super().__init__()
        self.bank_size = bank_size
        self.dim       = dim
        # Stored past messages per node: (num_nodes, M, dim)
        self.register_buffer("bank", torch.zeros(num_nodes, bank_size, dim))
        self.register_buffer("ptr",  torch.zeros(num_nodes, dtype=torch.long))

        # Cross-attention: current msg as Q, past bank as K and V
        self.q_proj = nn.Linear(dim, dim)
        self.k_proj = nn.Linear(dim, dim)
        self.v_proj = nn.Linear(dim, dim)
        self.scale  = dim ** -0.5
        self.out_proj = nn.Linear(dim, dim)
        self.norm     = nn.LayerNorm(dim)

    @torch.no_grad()
    def write(self, idx: torch.Tensor, msg: torch.Tensor):
        """Write current aggregated message into the circular buffer."""
        for b in range(idx.size(0)):
            node = int(idx[b])
            p    = int(self.ptr[node]) % self.bank_size
            self.bank[node, p] = msg[b].detach()
            self.ptr[node] += 1

    def read_and_attend(self,
                        idx:     torch.Tensor,   # (N,)
                        cur_msg: torch.Tensor,   # (N, dim)
                        ) -> torch.Tensor:       # (N, dim)
        """
        Cross-attention: current message queries the historical bank.

        Q = Linear(cur_msg)          (N, dim)
        K = Linear(bank[idx])        (N, M, dim)
        V = Linear(bank[idx])        (N, M, dim)

        output = softmax(Q K^T / sqrt(d)) V    (N, dim)

        Then blend: alpha * output + (1-alpha) * cur_msg
        where alpha is learned (starts near 0.5).
        """
        past = self.bank[idx]                                        # (N, M, dim)
        N, M, D = past.shape

        Q  = self.q_proj(cur_msg).unsqueeze(1)                       # (N, 1, dim)
        K  = self.k_proj(past)                                       # (N, M, dim)
        V  = self.v_proj(past)

        # Scaled dot-product attention over M past messages
        attn = torch.bmm(Q, K.transpose(1, 2)) * self.scale         # (N, 1, M)
        attn = F.softmax(attn, dim=-1)
        ctx  = torch.bmm(attn, V).squeeze(1)                        # (N, dim)

        # Residual: blend retrieved history with current message
        out = self.out_proj(ctx)
        return self.norm(cur_msg + out)                              # (N, dim)

    @torch.no_grad()
    def reset(self):
        self.bank.zero_()
        self.ptr.zero_()


# ═══════════════════════════════════════════════════════════════════════════════
# 10.  TemporalTrustAttention  (one message-passing layer)
#
#  Four upgrades vs the previous version:
#
#  [1] CROSS-SOURCE ATTENTION
#      Q comes from EDGE embeddings  (what is this interaction asking for?)
#      K comes from NODE features x  (who is this neighbour?)
#      This makes the attention interaction-aware:  the same neighbour j
#      will be weighted differently depending on the TYPE of edge (e.g.
#      a payment edge vs a message edge queries j differently).
#
#  [2] RoPE ON MESSAGES  (MessageRoPE)
#      After computing V_j (the value to send), apply RoPE with the edge
#      timestamp BEFORE aggregation.  So the aggregated sum at node i is
#      a timestamp-rotated superposition of neighbour values — time is
#      baked into the message content, not just the score.
#
#  [3] SIMILARITY-BASED TRUST  (StructuralTrustModule, unchanged formula)
#      Trust(i→j) ∈ (0,2) — amplifies SIMILAR nodes (normal↔normal or
#      anomalous↔anomalous) and attenuates DISSIMILAR ones.
#      This means the model self-segregates: nodes learn mostly from
#      their own "type", which separates the representation spaces of
#      normal and anomalous nodes without needing any labels.
#
#  [4] TEMPORAL MESSAGE BANK  (TemporalMessageBank)
#      After aggregation, cross-attend over the node's M most recently
#      stored aggregated messages.  Provides a temporal context window
#      over received messages, enabling the node to detect sudden changes
#      in what it is receiving (a hallmark of anomalous influence).
# ═══════════════════════════════════════════════════════════════════════════════

class TemporalTrustAttention(MessagePassing):
    def __init__(self, dim: int, heads: int = 8, dropout: float = 0.1,
                 window: int = 8, t2v_dim: int = 16,
                 num_nodes: int = 0, bank_size: int = 8):
        super().__init__(aggr="add", node_dim=0)
        assert dim % heads == 0
        self.dim      = dim
        self.heads    = heads
        self.head_dim = dim // heads

        # ── Cross-source projections ──────────────────────────────────────────
        # Q from edge embedding (interaction type)
        self.q_edge_proj = nn.Linear(dim, dim)
        # K from node features (neighbour identity)
        self.k_node_proj = nn.Linear(dim, dim)
        # V from node features (what neighbour sends)
        self.v_proj      = nn.Linear(dim, dim)
        self.o_proj      = nn.Linear(dim, dim)

        # Scale factor for dot-product attention
        self.scale = self.head_dim ** -0.5

        # ── RoPE on messages ──────────────────────────────────────────────────
        self.msg_rope = MessageRoPE(dim)

        # ── Trust ─────────────────────────────────────────────────────────────
        self.trust   = StructuralTrustModule(dim, window=window, t2v_dim=t2v_dim)

        # ── Temporal message bank ─────────────────────────────────────────────
        # Only instantiated if num_nodes > 0
        self.msg_bank = (TemporalMessageBank(num_nodes, dim, bank_size=bank_size)
                         if num_nodes > 0 else None)

        self.dropout = nn.Dropout(dropout)
        self.gate    = nn.Linear(dim, 1)
        self.norm    = nn.LayerNorm(dim)
        self.fusion  = nn.Sequential(
            nn.Linear(dim * 2, dim),
            nn.GELU(),
            nn.Linear(dim, dim),
        )

        self._trust_cache: Optional[torch.Tensor] = None

    # --------------------------------------------------------------------------
    def forward(self,
                x:          torch.Tensor,       # (N, dim)
                edge_index: torch.Tensor,       # (2, E)
                edge_emb:   torch.Tensor,       # (E, dim)  raw (not split)
                time_emb:   torch.Tensor,       # (E, dim)  raw
                edge_t:     torch.Tensor,       # (E,)      edge timestamps
                memory:     RecurrentMemory,
                evo_bank:   EvolutionBank,
                deg:        torch.Tensor,       # (N,)
                ) -> torch.Tensor:

        N    = x.size(0)
        nidx = torch.arange(N, device=x.device)

        # ── Cross-source Q, K, V ──────────────────────────────────────────────
        # Q: from edge embedding — per edge, split into heads
        Q_edge = self.q_edge_proj(edge_emb).view(-1, self.heads, self.head_dim)
        # K: from node features — per node, split into heads
        K_node = self.k_node_proj(x).view(N, self.heads, self.head_dim)
        # V: from node features — per node, split into heads
        V_node = self.v_proj(x).view(N, self.heads, self.head_dim)

        mem_vals, _ = memory.read(nidx)
        embs_all, times_all = evo_bank.read(nidx)

        out = self.propagate(
            edge_index,
            x=x,
            Q_edge=Q_edge,         # (E, heads, hd) — already per-edge
            K_node=K_node,         # (N, heads, hd) — will be indexed as K_node_j
            V_node=V_node,         # (N, heads, hd) — will be indexed as V_node_j
            mem_val=mem_vals,
            deg=deg,
            embs=embs_all,
            times=times_all,
            edge_t=edge_t,         # (E,) raw timestamps for MessageRoPE
            size=(N, N),
        )
        out = out.view(N, self.dim)                                  # (N, dim)

        # ── Temporal Message Bank: retrieve temporal context ──────────────────
        if self.msg_bank is not None:
            out = self.msg_bank.read_and_attend(nidx, out)          # (N, dim)
            self.msg_bank.write(nidx, out)

        # ── Fuse with GRU memory + gated residual ─────────────────────────────
        h   = memory.get_hidden(nidx)
        out = self.fusion(torch.cat([out, h], dim=-1))
        out = self.o_proj(out)

        g = torch.sigmoid(self.gate(x))
        return self.norm(g * x + (1 - g) * out)

    # --------------------------------------------------------------------------
    def message(self,
                x_i:        torch.Tensor,   # (E, dim)   central node features
                x_j:        torch.Tensor,   # (E, dim)   neighbour features
                Q_edge:     torch.Tensor,   # (E, heads, head_dim)  from edge
                K_node_j:   torch.Tensor,   # (E, heads, head_dim)  from node j
                V_node_j:   torch.Tensor,   # (E, heads, head_dim)  from node j
                mem_val_i:  torch.Tensor,   # (E, dim)
                mem_val_j:  torch.Tensor,
                deg_i:      torch.Tensor,   # (E,)
                deg_j:      torch.Tensor,
                embs_i:     torch.Tensor,   # (E, K, dim)
                embs_j:     torch.Tensor,
                times_i:    torch.Tensor,   # (E, K)
                times_j:    torch.Tensor,
                edge_t:     torch.Tensor,   # (E,)   edge timestamps
                index:      torch.Tensor,   # (E,)   target indices for softmax
                ) -> torch.Tensor:

        E = x_i.size(0)

        # ── [1] Cross-source attention score ─────────────────────────────────
        # Q from edge embedding asks: "given this interaction, how relevant is j?"
        # K from node j answers:     "here is what I am structurally"
        # Dot product: (E, heads, hd) x (E, heads, hd) -> (E, heads)
        scores = (Q_edge * K_node_j).sum(-1) * self.scale            # (E, heads)

        # ── [2] Similarity-based trust ────────────────────────────────────────
        # trust(i→j) ∈ (0, 2): amplifies similar pairs (normal↔normal or
        # anomalous↔anomalous), attenuates dissimilar pairs.
        # Self-segregation: the model learns to cluster by type unsupervised.
        trust = self.trust(
            h_i    = mem_val_i,
            h_j    = mem_val_j,
            deg_i  = deg_i,
            deg_j  = deg_j,
            embs_i = embs_i,
            embs_j = embs_j,
            t_i    = times_i,
            t_j    = times_j,
        )                                                            # (E,) in (0,2)
        self._trust_cache = trust.detach()

        # Scale scores by trust BEFORE softmax:
        # - Similar j's (trust>1) get amplified relative scores
        # - Dissimilar j's (trust<1) get suppressed relative scores
        # After softmax, each central node i will draw mostly from
        # neighbours that are structurally similar to itself.
        scores = scores * trust.unsqueeze(-1)                        # (E, heads)
        scores = softmax(scores, index)                              # (E, heads)
        scores = self.dropout(scores)

        # ── [3] Compute value and apply RoPE with edge timestamp ──────────────
        # V comes from node j's features
        msg = V_node_j * scores.unsqueeze(-1)                        # (E, heads, hd)
        msg = msg.reshape(E, self.dim)                               # (E, dim)

        # Apply RoPE to the message using the edge's actual timestamp.
        # This rotates the message content by the time it was sent, so
        # the aggregated sum at i carries intrinsic temporal information.
        # Messages from t=100 and t=1000 will aggregate to different vectors
        # even if their content (V_j) was identical.
        msg = self.msg_rope(msg, edge_t)                             # (E, dim)

        return msg                                                   # (E, dim)


# ═══════════════════════════════════════════════════════════════════════════════
# 11.  EnhancedTemporalGNN  — full model
# ═══════════════════════════════════════════════════════════════════════════════

class EnhancedTemporalGNN(nn.Module):
    """
    Full temporal GNN with K-view trajectory trust for representation learning.

    Expected batch fields:
        batch.x          (N, in_dim)    node features
        batch.edge_index (2, E)         directed edges [src, dst]
        batch.msg        (E, edge_dim)  edge / message features
        batch.t          (E,)           edge timestamps

    Returns:
        x  (N, hidden_dim)  node representations
    """
    def __init__(
        self,
        num_nodes:       int,
        in_dim:          int,
        edge_dim:        int,
        hidden_dim:      int,
        num_layers:      int   = 2,
        heads:           int   = 8,
        dropout:         float = 0.1,
        window:          int   = 8,
        t2v_dim:         int   = 16,
        bank_size:       int   = 8,
        memory_momentum: float = 0.9,
    ):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.num_nodes  = num_nodes
        self.window     = window

        # Input projections
        self.input_proj = nn.Linear(in_dim, hidden_dim)
        self.edge_enc   = nn.Linear(edge_dim, hidden_dim)
        self.time_enc   = TimeEncoder(hidden_dim)

        # TransformerConv snapshot encoder: produces embeddings stored in EvolutionBank
        self.snapshot_gnn = SnapshotGNN(hidden_dim, heads=heads, dropout=dropout)

        # Persistent state
        self.memory   = RecurrentMemory(num_nodes, hidden_dim, momentum=memory_momentum)
        self.evo_bank = EvolutionBank(num_nodes, hidden_dim, window=window)

        # Main message-passing layers
        self.layers = nn.ModuleList([
            TemporalTrustAttention(
                hidden_dim, heads=heads, dropout=dropout,
                window=window, t2v_dim=t2v_dim,
                num_nodes=num_nodes, bank_size=bank_size,
            )
            for _ in range(num_layers)
        ])

        self.final_norm = nn.LayerNorm(hidden_dim)
        self._init_weights()

    # --------------------------------------------------------------------------
    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)

    # --------------------------------------------------------------------------
    def forward(self, batch) -> torch.Tensor:
        N          = batch.x.size(0)
        device     = batch.x.device
        edge_index = batch.edge_index
        src, dst   = edge_index

        # ── Encode inputs ─────────────────────────────────────────────────────
        x        = self.input_proj(batch.x)                         # (N, hidden)
        edge_emb = self.edge_enc(batch.msg)                         # (E, hidden)

        # ── Node degrees ──────────────────────────────────────────────────────
        deg = degree(dst, num_nodes=N, dtype=torch.float).clamp(min=1.0)

        # ── Temporal encoding (for TimeEncoder / GRU context) ─────────────────
        node_last_t = torch.zeros(N, device=device)
        node_last_t.scatter_reduce_(
            0, src, batch.t.float(), reduce='amax', include_self=True
        )
        t_rel    = batch.t.float() - node_last_t[src]
        time_emb = self.time_enc(batch.t, t_rel)                    # (E, hidden)

        # ── Compute snapshot embedding for EvolutionBank ──────────────────────
        # TransformerConv operates directly on node features and edge_attr.
        # No time signal here — time is injected in TemporalPositionalEncoding.
        snapshot_emb = self.snapshot_gnn(x, edge_index, edge_emb)   # (N, hidden)

        # ── Message passing layers ─────────────────────────────────────────────
        # Pass raw edge timestamps (batch.t) separately for MessageRoPE
        for layer in self.layers:
            x = layer(x, edge_index, edge_emb, time_emb,
                      batch.t.float(),                               # edge_t for RoPE
                      self.memory, self.evo_bank, deg)

        x = self.final_norm(x)

        # ── Update persistent state ───────────────────────────────────────────
        nidx      = torch.arange(N, device=device)
        current_t = batch.t.float().max().expand(N)

        # Write snapshot to EvolutionBank (the rich contextual embedding)
        self.evo_bank.write(nidx, snapshot_emb, current_t)
        # Update GRU memory
        self.memory.write(nidx, x)

        return x

    # --------------------------------------------------------------------------
    def get_trust_scores(self, layer_idx: int = -1) -> Optional[torch.Tensor]:
        """Cached trust scores (E,) in (0, 2) from last forward()."""
        return self.layers[layer_idx]._trust_cache

    @torch.no_grad()
    def reset_memory(self):
        self.memory.hidden.zero_()
        self.memory.variance.fill_(0.1)
        self.evo_bank.reset()
        for layer in self.layers:
            if layer.msg_bank is not None:
                layer.msg_bank.reset()


# ═══════════════════════════════════════════════════════════════════════════════
# Quick sanity check
# ═══════════════════════════════════════════════════════════════════════════════
if __name__ == "__main__":
    import types

    torch.manual_seed(42)

    N, E      = 50, 120
    in_dim    = 32
    edge_dim  = 16
    hidden    = 64
    num_nodes = 100
    K         = 8

    def make_batch(t_offset=0.0):
        return types.SimpleNamespace(
            x          = torch.randn(N, in_dim),
            edge_index = torch.randint(0, N, (2, E)),
            msg        = torch.randn(E, edge_dim),
            t          = torch.rand(E) * 1000 + t_offset,
        )

    model = EnhancedTemporalGNN(
        num_nodes=num_nodes, in_dim=in_dim, edge_dim=edge_dim,
        hidden_dim=hidden, num_layers=2, heads=4,
        window=K, t2v_dim=16, bank_size=8,
    )

    for step in range(3):
        emb   = model(make_batch(step * 1000))
        trust = model.get_trust_scores()
        print(f"[pass {step+1}] emb: {emb.shape} | "
              f"trust: [{trust.min():.4f}, {trust.max():.4f}] "
              f"mean={trust.mean():.4f}")
        assert trust.min() > 0.0 and trust.max() < 2.0

    print("\nAll checks passed. Trust is in (0, 2).")

[pass 1] emb: torch.Size([50, 64]) | trust: [0.9042, 0.9562] mean=0.9173
[pass 2] emb: torch.Size([50, 64]) | trust: [0.8813, 0.9786] mean=0.9242
[pass 3] emb: torch.Size([50, 64]) | trust: [0.8873, 0.9833] mean=0.9272

All checks passed. Trust is in (0, 2).


In [ ]:
"""
Temporal Trust GNN for Unsupervised Anomaly Detection
=====================================================
Produces node representations for downstream contrastive learning.
No labels required — fully unsupervised.

Message Aggregation (TemporalTrustAttention):

  [1] CROSS-SOURCE ATTENTION
      Q comes from EDGE embeddings  (what does this interaction ask for?)
      K comes from NODE features x  (who is this neighbour?)
      The same neighbour j is weighted differently depending on the
      type/context of the edge, making attention interaction-aware.

  [2] RoPE ON MESSAGES  (MessageRoPE)
      After scoring, the value message V_j is rotated by its edge
      timestamp using continuous-time RoPE BEFORE aggregation.
      Time is therefore baked into the message CONTENT, not just
      the score — messages from different times aggregate to different
      vectors even with identical node features.

  [3] SIMILARITY-BASED TRUST  (StructuralTrustModule)
      trust(i->j) in (0, 2):
        trust > 1 : i and j are structurally similar -> amplify
        trust < 1 : i and j are dissimilar           -> attenuate
      Both normal<->normal and anomalous<->anomalous get high trust.
      The model self-segregates: each node learns from its own "type".
      This naturally separates normal and anomalous representations.

  [4] TEMPORAL MESSAGE BANK  (TemporalMessageBank) — NEW
      A per-node buffer stores the last M aggregated messages.
      After each aggregation, cross-attention reads over this buffer:
        Q = current aggregated message
        K/V = past M stored messages
      Output: temporally-smoothed message that blends current with
      historical incoming signals. Detects sudden anomalous changes
      in what a node is receiving (erratic = anomalous influence).

Trust signals (StructuralTrustModule):
  [A] cos(h_i, h_j)              current GRU state similarity
  [B] degree profile similarity  structural role similarity
  [C] K-view trajectory sim      RoPE+Time2Vec encoded snapshot comparison
      trust = 1 + tanh(MLP([A, B, C]))   in (0, 2)
"""

import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from typing import Optional, Tuple
from torch_geometric.nn import MessagePassing, TransformerConv
from torch_geometric.utils import softmax, degree


# ═══════════════════════════════════════════════════════════════════════════════
# 1.  Time2Vec
# ═══════════════════════════════════════════════════════════════════════════════

class Time2Vec(nn.Module):
    """
    Encodes a scalar timestamp into a vector:
        [w0*t + b0,  sin(w1*t + b1), ..., sin(w_{k}*t + b_{k})]
    Linear term = trend.  Sinusoids = periodicity.  All frequencies learned.
    """
    def __init__(self, out_dim: int):
        super().__init__()
        assert out_dim >= 2
        k = out_dim - 1
        self.w0 = nn.Parameter(torch.randn(1) * 0.01)
        self.b0 = nn.Parameter(torch.zeros(1))
        self.W  = nn.Parameter(torch.randn(k) * 0.01)
        self.B  = nn.Parameter(torch.zeros(k))

    def forward(self, t: torch.Tensor) -> torch.Tensor:
        """t: (*,) -> (*, out_dim)"""
        t   = t.float().unsqueeze(-1)
        lin = t * self.w0 + self.b0
        per = torch.sin(t * self.W + self.B)
        return torch.cat([lin, per], dim=-1)


# ═══════════════════════════════════════════════════════════════════════════════
# 2.  TimeEncoder  (edge-level: absolute + relative timestamp -> hidden_dim)
# ═══════════════════════════════════════════════════════════════════════════════

class TimeEncoder(nn.Module):
    def __init__(self, hidden_dim: int):
        super().__init__()
        half = hidden_dim // 2
        self.t2v_abs = Time2Vec(half)
        self.t2v_rel = Time2Vec(half)
        self.proj = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, hidden_dim),
        )

    def forward(self, t_abs: torch.Tensor,
                t_rel: Optional[torch.Tensor] = None) -> torch.Tensor:
        """(E,), (E,) -> (E, hidden_dim)"""
        a = self.t2v_abs(t_abs)
        r = self.t2v_rel(t_rel) if t_rel is not None else torch.zeros_like(a)
        return self.proj(torch.cat([a, r], dim=-1))


# ═══════════════════════════════════════════════════════════════════════════════
# 3.  RecurrentMemory  (per-node GRU hidden state)
# ═══════════════════════════════════════════════════════════════════════════════

class RecurrentMemory(nn.Module):
    """
    Maintains one GRU hidden state per node, updated after each snapshot.
    The hidden state is the running summary of the node's full history.
    """
    def __init__(self, num_nodes: int, dim: int, momentum: float = 0.9):
        super().__init__()
        self.gru      = nn.GRUCell(dim, dim)
        self.momentum = momentum
        self.register_buffer("hidden",   torch.zeros(num_nodes, dim))
        self.register_buffer("variance", torch.ones(num_nodes, dim) * 0.1)

    def read(self, idx: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        return self.hidden[idx], self.variance[idx]

    def get_hidden(self, idx: torch.Tensor) -> torch.Tensor:
        return self.hidden[idx]

    @torch.no_grad()
    def write(self, idx: torch.Tensor, x: torch.Tensor):
        h_old = self.hidden[idx]
        h_new = self.gru(x.detach(), h_old)
        delta = h_new - h_old
        self.variance[idx] = (self.momentum * self.variance[idx]
                              + (1 - self.momentum) * delta.pow(2))
        self.hidden[idx] = h_new


# ═══════════════════════════════════════════════════════════════════════════════
# 4.  SnapshotGNN  — TransformerConv-based node encoder
#
#  Produces the snapshot embedding stored in the EvolutionBank.
#  Uses PyG's TransformerConv which performs full transformer-style
#  multi-head attention between a node and its one-hop neighbours,
#  incorporating edge_attr directly into the key/value computation:
#
#      alpha(i,j) = softmax( (W_Q * x_i)^T (W_K * x_j + W_E * edge_attr) )
#      out_i      = W_O * sum_j [ alpha(i,j) * (W_V * x_j + W_E * edge_attr) ]
#
#  Inputs: node features (x) and edge features (edge_attr) ONLY.
#  Time is NOT used here — it is handled separately in the positional
#  encoding step when comparing K-view trajectories in the trust module.
#
#  Two TransformerConv layers with a residual connection give the model
#  capacity to capture both direct neighbours and two-hop structure.
# ═══════════════════════════════════════════════════════════════════════════════

class SnapshotGNN(nn.Module):
    """
    Transformer-based snapshot encoder using PyG's TransformerConv.

    Produces a rich node embedding from node features and edge attributes,
    capturing neighbourhood structure via transformer-style attention.
    Time is deliberately excluded — it is injected separately during
    the K-view positional encoding in StructuralTrustModule.

    Args:
        dim      : hidden dimension (input, output, and internal)
        heads    : number of attention heads for TransformerConv
        dropout  : dropout on attention weights
    """
    def __init__(self, dim: int, heads: int = 4, dropout: float = 0.1):
        super().__init__()
        assert dim % heads == 0, "dim must be divisible by heads"

        # Layer 1: node features + edge_attr -> dim
        self.conv1 = TransformerConv(
            in_channels  = dim,
            out_channels = dim // heads,
            heads        = heads,
            dropout      = dropout,
            edge_dim     = dim,        # edge_attr dimension matches hidden dim
            concat       = True,       # concat heads -> dim
            beta         = True,       # learnable skip connection inside conv
        )
        self.norm1 = nn.LayerNorm(dim)

        # Layer 2: refine with a second transformer hop
        self.conv2 = TransformerConv(
            in_channels  = dim,
            out_channels = dim // heads,
            heads        = heads,
            dropout      = dropout,
            edge_dim     = dim,
            concat       = True,
            beta         = True,
        )
        self.norm2 = nn.LayerNorm(dim)

        # Final projection + residual gate
        self.out_proj = nn.Linear(dim, dim)
        self.gate     = nn.Linear(dim, 1)

    def forward(self,
                x:          torch.Tensor,   # (N, dim)   projected node features
                edge_index: torch.Tensor,   # (2, E)
                edge_attr:  torch.Tensor,   # (E, dim)   projected edge features
                ) -> torch.Tensor:          # (N, dim)
        # Layer 1
        h = self.norm1(F.gelu(self.conv1(x, edge_index, edge_attr)))  # (N, dim)
        # Layer 2 with residual
        h2 = self.norm2(F.gelu(self.conv2(h, edge_index, edge_attr))) # (N, dim)
        h  = h + h2                                                    # residual

        # Gated output projection
        out = self.out_proj(h)
        g   = torch.sigmoid(self.gate(x))
        return g * x + (1 - g) * out                                  # (N, dim)


# ═══════════════════════════════════════════════════════════════════════════════
# 5.  EvolutionBank  — stores last K snapshot embeddings + timestamps per node
#
#  After each forward pass, the snapshot embedding produced by SnapshotGNN
#  is written here alongside its timestamp.
#  Shape: bank (num_nodes, K, dim),  times (num_nodes, K)
# ═══════════════════════════════════════════════════════════════════════════════

class EvolutionBank(nn.Module):
    def __init__(self, num_nodes: int, dim: int, window: int = 8):
        super().__init__()
        self.window = window
        self.dim    = dim
        self.register_buffer("bank",  torch.zeros(num_nodes, window, dim))
        self.register_buffer("times", torch.zeros(num_nodes, window))
        self.register_buffer("ptr",   torch.zeros(num_nodes, dtype=torch.long))

    def read(self, idx: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        """Returns (B, K, dim) embeddings and (B, K) timestamps."""
        return self.bank[idx], self.times[idx]

    @torch.no_grad()
    def write(self, idx: torch.Tensor, emb: torch.Tensor, t: torch.Tensor):
        """
        idx : (B,)    node indices
        emb : (B, D)  snapshot embedding from SnapshotGNN
        t   : (B,)    timestamp of this snapshot
        """
        for b in range(idx.size(0)):
            node = int(idx[b])
            p    = int(self.ptr[node]) % self.window
            self.bank[node,  p] = emb[b].detach()
            self.times[node, p] = float(t[b])
            self.ptr[node] += 1

    @torch.no_grad()
    def reset(self):
        self.bank.zero_()
        self.times.zero_()
        self.ptr.zero_()


# ═══════════════════════════════════════════════════════════════════════════════
# 6.  TemporalPositionalEncoding  — per-head multi-resolution temporal encoding
#
#  Applies temporal encoding to K-view snapshot sequences for the trust module.
#
#  KEY IDEA: split the embedding into H temporal heads, each head covering a
#  DIFFERENT temporal resolution band. Head 0 uses fast-rotating RoPE
#  (captures short-range patterns, e.g. hourly), head H-1 uses slow-rotating
#  RoPE (captures long-range drift, e.g. weekly). This gives the model multiple
#  simultaneous temporal "lenses" operating in parallel, similar to how
#  multi-head attention gives multiple geometric viewpoints.
#
#  Per head h:
#    (a) RoPE with head-specific timescale band:
#            θ_{h,d} = t / T_{h,d}
#        T_{h,d} are sampled from a sub-range of [T_min, T_max] assigned to head h.
#        Head h rotates dimensions (h*hd) to ((h+1)*hd) with timescales in
#        [T_min + h*(T_max-T_min)/H, T_min + (h+1)*(T_max-T_min)/H].
#
#    (b) Multi-scale Fourier (additive, shared across heads):
#        Complete Fourier basis (sin + cos) over F log-uniform frequencies,
#        projected to dim. Captures absolute periodic patterns independently
#        of the RoPE rotation.
#
#    (c) Recency decay gate (per-head learned λ_h):
#        g_{h,k} = σ( -λ_h * Δt_k + b_h )
#        Different heads decay at different rates — fast heads strongly
#        discount old views, slow heads treat all views more equally.
#
#  Combined per head h, view k:
#      view_{h,k} = g_{h,k} * RoPE_h(emb_{h,k}, t_k) + Fourier_proj(t_k)[h*hd:(h+1)*hd]
# ═══════════════════════════════════════════════════════════════════════════════

class TemporalPositionalEncoding(nn.Module):
    """
    Per-head multi-resolution temporal positional encoding for K-view sequences.

    Args:
        dim       : total embedding dimension (must be even, divisible by t_heads)
        t_heads   : number of temporal resolution heads
        n_fourier : Fourier frequency bands (sin+cos each)
        t_min_log : log of minimum timescale (default: 0 → T=1)
        t_max_log : log of maximum timescale (default: 9 → T≈8000)
    """
    def __init__(self, dim: int, t_heads: int = 4, n_fourier: int = 32,
                 t_min_log: float = 0.0, t_max_log: float = 9.0):
        super().__init__()
        assert dim % 2 == 0 and dim % t_heads == 0
        self.dim     = dim
        self.t_heads = t_heads
        self.head_d  = dim // t_heads   # dims per temporal head
        assert self.head_d % 2 == 0,   "head_d must be even for RoPE"

        # ── (a) Per-head RoPE timescales ──────────────────────────────────────
        # Each head h gets timescales from a dedicated sub-range of [t_min, t_max].
        # Head 0: fast (captures fine-grained recency)
        # Head H-1: slow (captures long-range historical trends)
        rope_params = []
        for h in range(t_heads):
            lo = t_min_log + h       * (t_max_log - t_min_log) / t_heads
            hi = t_min_log + (h + 1) * (t_max_log - t_min_log) / t_heads
            rope_params.append(torch.linspace(lo, hi, self.head_d // 2))
        # Stack to (t_heads, head_d//2), learned
        self.rope_log_ts = nn.Parameter(torch.stack(rope_params, dim=0))

        # ── (b) Multi-scale Fourier (shared, additive) ────────────────────────
        freqs = torch.logspace(-4, 1, n_fourier)
        self.register_buffer("fourier_freqs", freqs)
        self.fourier_proj = nn.Linear(2 * n_fourier, dim)

        # ── (c) Per-head recency decay gate ───────────────────────────────────
        # Each head has its own learned decay rate λ_h and bias b_h.
        # Fast heads (low h): init high λ → strong recency emphasis
        # Slow heads (high h): init low λ → weaker decay
        init_log_decay = torch.linspace(1.0, -1.0, t_heads)         # head 0 fastest
        self.log_decay  = nn.Parameter(init_log_decay)               # (t_heads,)
        self.decay_bias = nn.Parameter(torch.ones(t_heads))          # (t_heads,)

    # ──────────────────────────────────────────────────────────────────────────
    def _rope_per_head(self,
                       embs:  torch.Tensor,   # (B, K, dim)
                       times: torch.Tensor,   # (B, K)
                       ) -> torch.Tensor:     # (B, K, dim)
        """Apply per-head RoPE: each head h rotates its slice of the embedding
        with head-h-specific timescales."""
        B, K, D = embs.shape
        out_parts = []
        for h in range(self.t_heads):
            # Slice embedding for this head: (B, K, head_d)
            sl = embs[..., h * self.head_d : (h + 1) * self.head_d]
            ts  = self.rope_log_ts[h].exp()                          # (head_d//2,)
            ang = times.unsqueeze(-1) / ts                           # (B, K, hd//2)
            cos_a, sin_a = ang.cos(), ang.sin()
            e_even = sl[..., 0::2]
            e_odd  = sl[..., 1::2]
            rot = torch.stack(
                [e_even * cos_a - e_odd * sin_a,
                 e_even * sin_a + e_odd * cos_a], dim=-1
            ).reshape(B, K, self.head_d)
            out_parts.append(rot)
        return torch.cat(out_parts, dim=-1)                          # (B, K, dim)

    # ──────────────────────────────────────────────────────────────────────────
    def _fourier(self, times: torch.Tensor) -> torch.Tensor:
        """(B, K) -> (B, K, dim)"""
        B, K = times.shape
        t    = times.reshape(B * K, 1).float()
        phi  = t * self.fourier_freqs.unsqueeze(0)                   # (BK, F)
        feat = torch.cat([phi.sin(), phi.cos()], dim=-1)             # (BK, 2F)
        return self.fourier_proj(feat).view(B, K, self.dim)          # (B, K, D)

    # ──────────────────────────────────────────────────────────────────────────
    def _decay_gates(self, times: torch.Tensor) -> torch.Tensor:
        """(B, K) -> (B, K, dim)  — per-head gate broadcast across head dims"""
        t_max = times.max(dim=-1, keepdim=True).values               # (B, 1)
        delta = (t_max - times).clamp(min=0.0)                      # (B, K)
        lam   = self.log_decay.exp()                                 # (t_heads,)
        bias  = self.decay_bias                                      # (t_heads,)
        # gate per head: (B, K, t_heads)
        gate  = torch.sigmoid(
            -lam.unsqueeze(0).unsqueeze(0) * delta.unsqueeze(-1)
            + bias.unsqueeze(0).unsqueeze(0)
        )
        # Broadcast to full dim: each head's gate applies to its head_d dims
        gate_full = gate.repeat_interleave(self.head_d, dim=-1)      # (B, K, dim)
        return gate_full

    # ──────────────────────────────────────────────────────────────────────────
    def forward(self,
                embs:  torch.Tensor,   # (B, K, dim)
                times: torch.Tensor,   # (B, K)
                ) -> torch.Tensor:     # (B, K, dim)
        rope_out   = self._rope_per_head(embs, times)                # (B, K, dim)
        gates      = self._decay_gates(times)                        # (B, K, dim)
        fourier_out = self._fourier(times)                           # (B, K, dim)
        # Each head's RoPE output is gated by its own recency rate,
        # then the absolute Fourier signal is added on top.
        return gates * rope_out + fourier_out                        # (B, K, dim)


# ═══════════════════════════════════════════════════════════════════════════════
# 7.  StructuralTrustModule  — K-VIEW SINKHORN TRAJECTORY SIMILARITY
#
#  Computes trust(i→j) ∈ (0, 2) using ONLY the K-view trajectory signal.
#
#  WHY SINKHORN INSTEAD OF CROSS-ATTENTION?
#  ─────────────────────────────────────────
#  Cross-attention (softmax over rows) lets each of i's views attend to ALL
#  of j's views independently.  Two problems:
#    1. Multiple views of i can "compete" for the same view of j, pulling
#       the match towards a few dominant modes.
#    2. No constraint that the overall matching is globally consistent —
#       a single anomalous view in j can dominate every row.
#
#  Sinkhorn optimal transport solves both:
#    - Finds the globally optimal SOFT ASSIGNMENT matrix T ∈ R^{K×K}
#      between i's K views and j's K views.
#    - Row margins: T·1 = u (how much each view of i is matched)
#    - Column margins: T^T·1 = v (how much each view of j is used)
#    - With uniform margins (u=v=1/K), every view contributes equally.
#    - Minimises sum_{k,k'} C_{k,k'} * T_{k,k'}
#      where C_{k,k'} = 1 - cos(view_i_k, view_j_k') is the COST.
#    - Solved iteratively via Sinkhorn-Knopp (log-domain for stability).
#
#  After finding T, the trajectory similarity is:
#      traj_sim = sum_{k,k'} T_{k,k'} * (1 - C_{k,k'})    (E,) ∈ [-1,1]
#               = sum_{k,k'} T_{k,k'} * cos(vi_k, vj_k')
#  This is the EXPECTED cosine similarity under the optimal transport plan —
#  a true measure of how "alignable" the two trajectories are.
# ═══════════════════════════════════════════════════════════════════════════════

class StructuralTrustModule(nn.Module):
    """
    K-view trajectory trust via Sinkhorn Optimal Transport.

    Args:
        dim         : embedding dimension
        window      : K, number of stored snapshots
        t_heads     : temporal resolution heads for TemporalPositionalEncoding
        sinkhorn_eps: entropy regularisation (smaller = sharper assignment)
        sinkhorn_iters: number of Sinkhorn iterations (5-10 is usually enough)
    """
    def __init__(self, dim: int, window: int = 8,
                 t2v_dim: int = 16,          # kept for API compat, unused
                 t_heads: int = 4,
                 sinkhorn_eps: float = 0.05,
                 sinkhorn_iters: int = 7):
        super().__init__()
        self.window         = window
        self.sinkhorn_eps   = sinkhorn_eps
        self.sinkhorn_iters = sinkhorn_iters

        # Per-head multi-resolution temporal positional encoding
        self.time_pe = TemporalPositionalEncoding(dim, t_heads=t_heads)

        # Lightweight projection to compare views in a lower-dim space
        # This decouples the comparison space from the full hidden dim,
        # reducing computation in the K×K cost matrix.
        proj_dim = max(dim // 2, 32)
        self.proj = nn.Linear(dim, proj_dim, bias=False)

        # Maps OT similarity scalar → trust logit
        self.trust_head = nn.Sequential(
            nn.Linear(1, max(dim // 8, 8)),
            nn.GELU(),
            nn.Linear(max(dim // 8, 8), 1),
        )

    # ──────────────────────────────────────────────────────────────────────────
    @staticmethod
    def _sinkhorn_log(log_C: torch.Tensor,
                      eps: float, n_iter: int) -> torch.Tensor:
        """
        Log-domain Sinkhorn: finds the optimal transport plan T for cost C.

        log_C : (B, K, K)  log of cost matrix (here log(1 - cos_sim + ε))
        Returns log_T : (B, K, K)  log of transport plan

        Log-domain is numerically stable for small eps.
        The algorithm alternates normalising rows and columns of log_T.
        """
        B, K, _ = log_C.shape
        log_a = torch.full((B, K), -math.log(K),
                           device=log_C.device, dtype=log_C.dtype)  # uniform source
        log_b = torch.full((B, K), -math.log(K),
                           device=log_C.device, dtype=log_C.dtype)  # uniform target

        log_T = -log_C / eps    # initialise with Gibbs kernel
        for _ in range(n_iter):
            # Row normalisation: T·1 = a
            log_T = log_T - torch.logsumexp(log_T, dim=2, keepdim=True) + log_a.unsqueeze(2)
            # Column normalisation: T^T·1 = b
            log_T = log_T - torch.logsumexp(log_T, dim=1, keepdim=True) + log_b.unsqueeze(1)
        return log_T   # (B, K, K)

    # ──────────────────────────────────────────────────────────────────────────
    def forward(
        self,
        embs_i: torch.Tensor,   # (E, K, dim)  last K snapshot embeddings of i
        embs_j: torch.Tensor,   # (E, K, dim)  last K snapshot embeddings of j
        t_i:    torch.Tensor,   # (E, K)       timestamps of i's K snapshots
        t_j:    torch.Tensor,   # (E, K)       timestamps of j's K snapshots
        eps:    float = 1e-6,
    ) -> torch.Tensor:          # (E,) ∈ (0, 2)

        E, K, D = embs_i.shape

        # Step 1: per-head multi-resolution temporal PE
        # Fast heads emphasise recency, slow heads treat all views equally.
        vi = self.time_pe(embs_i, t_i)   # (E, K, dim)
        vj = self.time_pe(embs_j, t_j)

        # Step 2: project to lower-dim comparison space and normalise
        vi_p = F.normalize(self.proj(vi.reshape(E * K, D)),
                           dim=-1, eps=eps).view(E, K, -1)           # (E, K, proj_dim)
        vj_p = F.normalize(self.proj(vj.reshape(E * K, D)),
                           dim=-1, eps=eps).view(E, K, -1)

        # Step 3: full K×K pairwise cosine similarity
        sim = torch.bmm(vi_p, vj_p.transpose(1, 2))                 # (E, K, K)

        # Step 4: Sinkhorn OT — finds globally optimal soft assignment
        log_C = torch.log((1.0 - sim).clamp(min=1e-6))
        log_T = self._sinkhorn_log(log_C, self.sinkhorn_eps,
                                   self.sinkhorn_iters)
        T = log_T.exp()                                              # (E, K, K)

        # Step 5: OT-weighted expected cosine similarity
        traj_sim = (T * sim).sum(dim=(1, 2))                         # (E,) ∈ [-1, 1]

        # Step 6: trust ∈ (0, 2)
        logit = self.trust_head(traj_sim.unsqueeze(-1)).squeeze(-1)
        return 1.0 + torch.tanh(logit)


# ═══════════════════════════════════════════════════════════════════════════════
# 8.  MessageTemporalEncoding  — enhanced per-head RoPE + Fourier + decay gate
#
#  Applied to live edge messages (E, dim) before aggregation at node i.
#
#  Four components, all applied together:
#
#  (a) TIMESTAMP NORMALISATION
#      Raw timestamps can be arbitrarily large absolute values (e.g. Unix time).
#      Large t values cause RoPE angles to wrap many times, losing information.
#      We apply learnable normalisation:  t_norm = (t - mu) / (sigma + ε)
#      where mu and sigma are EMA statistics updated during training.
#      This keeps angles in a well-behaved range regardless of the time scale.
#
#  (b) PER-HEAD RoPE
#      Message is split into t_heads segments. Each segment is rotated by
#      RoPE with a head-specific timescale sub-range. Some heads capture
#      fine-grained recency (hourly), others capture long-range patterns.
#
#  (c) MULTI-SCALE FOURIER (additive, shared)
#      Absolute temporal position signal via sin/cos of log-uniform frequencies.
#
#  (d) PER-HEAD TEMPORAL DECAY GATE
#      g_h = σ( -λ_h * |t_norm| + b_h )   ∈ (0, 1)
#      Each head has a learned decay rate λ_h. Fast heads (large λ) strongly
#      down-weight messages that are "old" relative to the mean timestamp.
#      Slow heads weight all messages more uniformly.
#      The gated message for head h: g_h * rope_h(msg_h) + (1-g_h) * msg_h
#      This is a soft interpolation between time-rotated and original content,
#      letting slow heads preserve content while fast heads inject recency.
# ═══════════════════════════════════════════════════════════════════════════════

class MessageTemporalEncoding(nn.Module):
    """
    Per-head multi-resolution RoPE + Fourier + decay gate for edge messages.

    Args:
        dim       : message dimension (even, divisible by t_heads)
        t_heads   : number of temporal resolution heads
        n_fourier : Fourier frequency bands (sin+cos each)
        ema_decay : momentum for running timestamp statistics
    """
    def __init__(self, dim: int, t_heads: int = 4, n_fourier: int = 16,
                 ema_decay: float = 0.99):
        super().__init__()
        assert dim % 2 == 0 and dim % t_heads == 0
        self.dim      = dim
        self.t_heads  = t_heads
        self.head_d   = dim // t_heads
        assert self.head_d % 2 == 0

        # ── (a) Learnable timestamp normalisation ─────────────────────────────
        # Running statistics for EMA normalisation of raw timestamps
        self.ema_decay = ema_decay
        self.register_buffer("t_mean", torch.zeros(1))
        self.register_buffer("t_var",  torch.ones(1))
        # Learnable affine rescale after normalisation
        self.t_scale = nn.Parameter(torch.ones(1))
        self.t_shift = nn.Parameter(torch.zeros(1))

        # ── (b) Per-head RoPE timescales ──────────────────────────────────────
        # Shorter range for messages: angles of normalised timestamps
        rope_params = []
        T_MIN, T_MAX = 0.0, 4.0   # timescales 1 → e^4 ≈ 55 (on normalised scale)
        for h in range(t_heads):
            lo = T_MIN + h       * (T_MAX - T_MIN) / t_heads
            hi = T_MIN + (h + 1) * (T_MAX - T_MIN) / t_heads
            rope_params.append(torch.linspace(lo, hi, self.head_d // 2))
        self.rope_log_ts = nn.Parameter(torch.stack(rope_params, dim=0))  # (H, hd//2)

        # ── (c) Multi-scale Fourier (shared, additive) ────────────────────────
        freqs = torch.logspace(-3, 1, n_fourier)
        self.register_buffer("fourier_freqs", freqs)
        self.fourier_proj = nn.Linear(2 * n_fourier, dim)

        # ── (d) Per-head temporal decay gate ──────────────────────────────────
        # λ_h: learned decay rate per head; fast heads init to larger values
        init_log_decay = torch.linspace(1.5, -0.5, t_heads)  # head 0 = fastest
        self.log_decay  = nn.Parameter(init_log_decay)        # (t_heads,)
        self.decay_bias = nn.Parameter(torch.ones(t_heads))   # (t_heads,)

    @torch.no_grad()
    def _update_stats(self, t: torch.Tensor):
        """EMA update of running mean and variance of timestamps."""
        mu  = t.float().mean()
        var = t.float().var().clamp(min=1e-4)
        self.t_mean = self.ema_decay * self.t_mean + (1 - self.ema_decay) * mu
        self.t_var  = self.ema_decay * self.t_var  + (1 - self.ema_decay) * var

    def _normalise_t(self, t: torch.Tensor) -> torch.Tensor:
        """Normalise timestamps to zero-mean, unit-variance, then rescale."""
        t_norm = (t.float() - self.t_mean) / (self.t_var.sqrt() + 1e-6)
        return self.t_scale * t_norm + self.t_shift      # (E,)

    def forward(self,
                msg: torch.Tensor,   # (E, dim)
                t:   torch.Tensor,   # (E,)   raw edge timestamps
                ) -> torch.Tensor:   # (E, dim)
        E = msg.size(0)

        # Update running stats during training
        if self.training:
            self._update_stats(t)

        # Normalise timestamps
        t_n = self._normalise_t(t)                                   # (E,)

        # ── (b) Per-head RoPE on normalised timestamps ────────────────────────
        rope_parts = []
        for h in range(self.t_heads):
            sl  = msg[:, h * self.head_d : (h + 1) * self.head_d]   # (E, hd)
            ts  = self.rope_log_ts[h].exp()                          # (hd//2,)
            ang = t_n.unsqueeze(-1) / ts                             # (E, hd//2)
            cos_a, sin_a = ang.cos(), ang.sin()
            m_even = sl[:, 0::2]
            m_odd  = sl[:, 1::2]
            rot = torch.stack(
                [m_even * cos_a - m_odd * sin_a,
                 m_even * sin_a + m_odd * cos_a], dim=-1
            ).reshape(E, self.head_d)
            rope_parts.append(rot)
        rope_out = torch.cat(rope_parts, dim=-1)                     # (E, dim)

        # ── (c) Multi-scale Fourier (additive, absolute time) ─────────────────
        phi    = t_n.unsqueeze(-1) * self.fourier_freqs              # (E, F)
        feat   = torch.cat([phi.sin(), phi.cos()], dim=-1)           # (E, 2F)
        fourier = self.fourier_proj(feat)                            # (E, dim)

        # ── (d) Per-head temporal decay gate ──────────────────────────────────
        # g_h = σ( -λ_h * |t_norm| + b_h )
        # Fast heads (large λ) strongly gate old messages toward identity.
        lam  = self.log_decay.exp()                                  # (t_heads,)
        gate = torch.sigmoid(
            -lam.unsqueeze(0) * t_n.abs().unsqueeze(-1)
            + self.decay_bias.unsqueeze(0)
        )                                                            # (E, t_heads)
        # Broadcast gate to full dim (each head covers head_d dims)
        gate_full = gate.repeat_interleave(self.head_d, dim=-1)      # (E, dim)
        # Gated blend: interpolate between rotated and original content
        gated_rope = gate_full * rope_out + (1 - gate_full) * msg    # (E, dim)

        return gated_rope + fourier                                   # (E, dim)


# ═══════════════════════════════════════════════════════════════════════════════
# 9.  GRN  — Gated Residual Network (from Temporal Fusion Transformer, Lim 2021)
#
#  A proven building block for temporal modelling. Replaces plain nn.Linear
#  for Q, K, V projections in TemporalTrustAttention.
#
#  GRN(x, c) = LayerNorm( x + W2 · ELU(W1 · [x, c] + b1) ⊙ σ(W_gate · [x, c]) )
#
#  Two important properties:
#    1. CONTEXT GATING: an optional context vector c (here: the node's GRU
#       memory) modulates the output via a sigmoid gate.  This means the
#       projection adapts to the node's historical state without needing
#       a full hypernetwork.
#    2. ELU NONLINEARITY: unlike GELU (smooth everywhere), ELU has exact
#       linearity for positive inputs and a negative saturation floor.
#       This is beneficial for temporal data where some features are truly
#       linear (trend) and others saturate (periodic amplitude).
#    3. SKIP CONNECTION: the residual allows the GRN to be identity-like
#       when no transformation is needed, which helps early training.
# ═══════════════════════════════════════════════════════════════════════════════

class GRN(nn.Module):
    """
    Gated Residual Network projection with optional context conditioning.

    Args:
        in_dim  : input dimension
        out_dim : output dimension
        ctx_dim : context dimension (None = no context gating)
        dropout : dropout before residual add
    """
    def __init__(self, in_dim: int, out_dim: int,
                 ctx_dim: Optional[int] = None, dropout: float = 0.1):
        super().__init__()
        self.in_dim  = in_dim
        self.out_dim = out_dim
        ctx = ctx_dim or 0

        # Main transformation: [x, c] -> hidden -> out
        self.W1 = nn.Linear(in_dim + ctx, out_dim)
        self.W2 = nn.Linear(out_dim, out_dim)

        # Context gate: [x, c] -> gate ∈ (0,1) controlling residual weight
        self.W_gate = nn.Linear(in_dim + ctx, out_dim)

        # Skip connection projection (needed when in_dim ≠ out_dim)
        self.skip = (nn.Linear(in_dim, out_dim, bias=False)
                     if in_dim != out_dim else nn.Identity())

        self.norm    = nn.LayerNorm(out_dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self,
                x:   torch.Tensor,              # (B, in_dim)
                ctx: Optional[torch.Tensor] = None  # (B, ctx_dim)
                ) -> torch.Tensor:              # (B, out_dim)
        if ctx is not None:
            inp = torch.cat([x, ctx], dim=-1)   # (B, in_dim + ctx_dim)
        else:
            inp = x

        # Gated transformation
        h    = F.elu(self.W1(inp))              # (B, out_dim)
        h    = self.dropout(self.W2(h))         # (B, out_dim)
        gate = torch.sigmoid(self.W_gate(inp))  # (B, out_dim) ∈ (0,1)

        # Gated residual: gate blends transformed output with skip
        out = gate * h + (1 - gate) * self.skip(x)
        return self.norm(out)                   # (B, out_dim)


# ═══════════════════════════════════════════════════════════════════════════════
# 10.  TemporalTrustAttention  (one message-passing layer)
#
#  [1] GRN-PROJECTED Q / K / V  (context-conditioned via GRU memory)
#      Q  = GRN(edge_emb, ctx=mem_i)  interaction queries through node history
#      K  = GRN(x_j,      ctx=mem_j)  node j's key adapted to its history
#      V  = GRN(x_j,      ctx=mem_j)  value also history-conditioned
#
#  [2] RoPE APPLIED TO Q AND K BEFORE SCORING
#      This is the mathematically correct placement for RoPE.
#      Rotating Q and K by the edge timestamp means that the dot product
#      Q·K naturally captures relative temporal position:
#          (R_t Q)·(R_t K) = Q^T R_{0} K = Q^T K   (same time: unrotated)
#          (R_{t1} Q)·(R_{t2} K) = Q^T R_{t2-t1} K  (different times: rotated)
#      The score between Q and K reflects BOTH semantic compatibility AND
#      temporal proximity. Temporally distant Q-K pairs will have a
#      phase-shifted dot product, naturally reducing their score.
#      Per-head timescales are used (same as MessageTemporalEncoding).
#
#  [3] PER-HEAD BILINEAR SCORING  (replaces identity dot-product)
#      Standard dot-product: score = Q^T K  (uses identity metric)
#      Bilinear form:        score = Q^T W_h K  (each head learns its own metric)
#      W_h ∈ R^{hd×hd} is a learned per-head compatibility matrix.
#      This allows each head to discover a different notion of compatibility
#      between the edge query and the node key — e.g. one head may focus on
#      frequency features, another on structural role features.
#      To keep computation light, W_h is parameterised as a low-rank outer
#      product: W_h = U_h V_h^T where U_h, V_h ∈ R^{hd×r}, rank r << hd.
#
#  [4] SINKHORN TRUST  (K-view OT only, no h/deg)
#
#  [5] PER-HEAD MULTI-RESOLUTION MESSAGE TEMPORAL ENCODING  (with decay gate)
#
#  [6] TEMPORAL MESSAGE BANK
# ═══════════════════════════════════════════════════════════════════════════════

class TemporalTrustAttention(MessagePassing):
    def __init__(self, dim: int, heads: int = 8, dropout: float = 0.1,
                 window: int = 8, t2v_dim: int = 16,
                 num_nodes: int = 0, bank_size: int = 8,
                 t_heads: int = 4, bilinear_rank: int = 8):
        super().__init__(aggr="add", node_dim=0)
        assert dim % heads == 0
        self.dim        = dim
        self.heads      = heads
        self.head_dim   = dim // heads
        self.t_heads    = t_heads

        # ── [1] GRN projections (context-conditioned) ─────────────────────────
        self.q_grn  = GRN(dim, dim, ctx_dim=dim, dropout=dropout)
        self.k_grn  = GRN(dim, dim, ctx_dim=dim, dropout=dropout)
        self.v_grn  = GRN(dim, dim, ctx_dim=dim, dropout=dropout)
        self.o_proj = nn.Linear(dim, dim)

        # ── [2] Per-head RoPE on Q and K ─────────────────────────────────────
        # Timescales shared with message encoding for consistency
        # Each head gets its own sub-range of [0, 4] on normalised timestamps
        rope_params = []
        T_MIN, T_MAX = 0.0, 4.0
        hd = self.head_dim
        assert hd % 2 == 0
        for h in range(heads):
            lo = T_MIN + h       * (T_MAX - T_MIN) / heads
            hi = T_MIN + (h + 1) * (T_MAX - T_MIN) / heads
            rope_params.append(torch.linspace(lo, hi, hd // 2))
        self.qk_rope_log_ts = nn.Parameter(torch.stack(rope_params, dim=0))  # (H, hd//2)

        # Running timestamp stats for normalisation (shared with MessageTemporalEncoding)
        self.register_buffer("t_mean_qk", torch.zeros(1))
        self.register_buffer("t_var_qk",  torch.ones(1))
        self.t_scale_qk = nn.Parameter(torch.ones(1))
        self.t_shift_qk = nn.Parameter(torch.zeros(1))
        self.ema_decay = 0.99

        # ── [3] Per-head bilinear scoring (low-rank) ──────────────────────────
        # W_h = U_h @ V_h^T  where U_h, V_h ∈ R^{hd × r}
        # Parameterised as two (heads, hd, rank) tensors.
        # score_h = Q_h^T W_h K_h = (Q_h^T U_h)(V_h^T K_h) — factored efficiently.
        r = bilinear_rank
        self.bilin_U = nn.Parameter(torch.randn(heads, hd, r) * (r ** -0.5))
        self.bilin_V = nn.Parameter(torch.randn(heads, hd, r) * (r ** -0.5))
        self.scale   = r ** -0.5   # normalise by rank, not head_dim

        # ── [4] Sinkhorn trust (K-view only) ─────────────────────────────────
        self.trust = StructuralTrustModule(dim, window=window, t_heads=t_heads)

        # ── [5] Message temporal encoding (with decay gate) ───────────────────
        self.msg_time_enc = MessageTemporalEncoding(dim, t_heads=t_heads, n_fourier=16)

        # ── [6] Temporal message bank ─────────────────────────────────────────
        self.msg_bank = (TemporalMessageBank(num_nodes, dim, bank_size=bank_size)
                         if num_nodes > 0 else None)

        self.dropout = nn.Dropout(dropout)
        self.gate    = nn.Linear(dim, 1)
        self.norm    = nn.LayerNorm(dim)
        self.fusion  = nn.Sequential(
            nn.Linear(dim * 2, dim),
            nn.GELU(),
            nn.Linear(dim, dim),
        )
        self._trust_cache: Optional[torch.Tensor] = None

    # ──────────────────────────────────────────────────────────────────────────
    @torch.no_grad()
    def _update_t_stats(self, t: torch.Tensor):
        mu  = t.float().mean()
        var = t.float().var().clamp(min=1e-4)
        self.t_mean_qk = self.ema_decay * self.t_mean_qk + (1 - self.ema_decay) * mu
        self.t_var_qk  = self.ema_decay * self.t_var_qk  + (1 - self.ema_decay) * var

    def _normalise_t(self, t: torch.Tensor) -> torch.Tensor:
        t_n = (t.float() - self.t_mean_qk) / (self.t_var_qk.sqrt() + 1e-6)
        return self.t_scale_qk * t_n + self.t_shift_qk

    def _apply_rope_to_qk(self,
                           qk:  torch.Tensor,   # (E, heads, head_dim)
                           t_n: torch.Tensor,   # (E,)  normalised timestamps
                           ) -> torch.Tensor:   # (E, heads, head_dim)
        """
        Apply per-head RoPE to Q or K using normalised edge timestamps.
        Each head h rotates its head_dim vector by its own timescale band.
        """
        E = qk.size(0)
        parts = []
        for h in range(self.heads):
            sl  = qk[:, h, :]                                        # (E, hd)
            ts  = self.qk_rope_log_ts[h].exp()                       # (hd//2,)
            ang = t_n.unsqueeze(-1) / ts                             # (E, hd//2)
            cos_a, sin_a = ang.cos(), ang.sin()
            e_even = sl[:, 0::2]
            e_odd  = sl[:, 1::2]
            rot = torch.stack(
                [e_even * cos_a - e_odd * sin_a,
                 e_even * sin_a + e_odd * cos_a], dim=-1
            ).reshape(E, self.head_dim)
            parts.append(rot)
        return torch.stack(parts, dim=1)                             # (E, H, hd)

    # ──────────────────────────────────────────────────────────────────────────
    def forward(self,
                x:          torch.Tensor,   # (N, dim)
                edge_index: torch.Tensor,   # (2, E)
                edge_emb:   torch.Tensor,   # (E, dim)
                time_emb:   torch.Tensor,   # (E, dim) unused, kept for API
                edge_t:     torch.Tensor,   # (E,)  raw timestamps
                memory:     RecurrentMemory,
                evo_bank:   EvolutionBank,
                deg:        torch.Tensor,   # (N,)
                ) -> torch.Tensor:

        N    = x.size(0)
        nidx = torch.arange(N, device=x.device)
        src  = edge_index[0]

        # Update running timestamp statistics
        if self.training:
            self._update_t_stats(edge_t)
        t_n = self._normalise_t(edge_t)                             # (E,)

        mem_vals, _ = memory.read(nidx)
        embs_all, times_all = evo_bank.read(nidx)

        # GRN-projected Q/K/V
        mem_src = mem_vals[src]
        Q = self.q_grn(edge_emb, mem_src).view(-1, self.heads, self.head_dim)
        K = self.k_grn(x, mem_vals).view(N, self.heads, self.head_dim)
        V = self.v_grn(x, mem_vals).view(N, self.heads, self.head_dim)

        # Apply RoPE to Q (edge-level, already (E, H, hd))
        Q_rope = self._apply_rope_to_qk(Q, t_n)                     # (E, H, hd)
        # K will be indexed as K_j inside message() — apply RoPE there

        out = self.propagate(
            edge_index,
            x=x,
            Q_rope=Q_rope,
            K=K,
            V=V,
            mem_val=mem_vals,
            embs=embs_all,
            times=times_all,
            edge_t=edge_t,
            t_n=t_n,
            size=(N, N),
        )
        out = out.view(N, self.dim)

        # Temporal message bank
        if self.msg_bank is not None:
            out = self.msg_bank.read_and_attend(nidx, out)
            self.msg_bank.write(nidx, out)

        # Fuse with GRU memory + gated residual
        h   = memory.get_hidden(nidx)
        out = self.fusion(torch.cat([out, h], dim=-1))
        out = self.o_proj(out)
        g   = torch.sigmoid(self.gate(x))
        return self.norm(g * x + (1 - g) * out)

    # ──────────────────────────────────────────────────────────────────────────
    def message(self,
                x_i:      torch.Tensor,   # (E, dim)
                x_j:      torch.Tensor,
                Q_rope:   torch.Tensor,   # (E, H, hd)  Q already RoPE-encoded
                K_j:      torch.Tensor,   # (E, H, hd)  K from node j (raw)
                V_j:      torch.Tensor,   # (E, H, hd)
                mem_val_i: torch.Tensor,  # (E, dim)
                mem_val_j: torch.Tensor,
                embs_i:   torch.Tensor,   # (E, K, dim)
                embs_j:   torch.Tensor,
                times_i:  torch.Tensor,   # (E, K)
                times_j:  torch.Tensor,
                edge_t:   torch.Tensor,   # (E,)
                t_n:      torch.Tensor,   # (E,)  normalised timestamps
                index:    torch.Tensor,
                ) -> torch.Tensor:

        E = x_i.size(0)

        # ── [2] Apply RoPE to K_j using normalised timestamps ─────────────────
        # After rotation, Q_rope · K_j_rope reflects both semantic match AND
        # temporal proximity between the edge time and j's contribution time.
        K_j_rope = self._apply_rope_to_qk(K_j, t_n)                 # (E, H, hd)

        # ── [3] Per-head bilinear score: score_h = (Q^T U_h)(V_h^T K) ────────
        # Factored bilinear: avoids materialising the full hd×hd matrix.
        # Q^T U_h : (E, r)   and   V_h^T K : (E, r)
        # score_h = dot product of these two r-dim vectors.
        scores = torch.zeros(E, self.heads, device=x_i.device)
        for h in range(self.heads):
            q_h = Q_rope[:, h, :]                                    # (E, hd)
            k_h = K_j_rope[:, h, :]                                  # (E, hd)
            U_h = self.bilin_U[h]                                    # (hd, r)
            V_h = self.bilin_V[h]                                    # (hd, r)
            # (E, r) · (E, r) summed → (E,)
            scores[:, h] = ((q_h @ U_h) * (k_h @ V_h)).sum(-1) * self.scale

        # ── [4] Sinkhorn trust (K-view only) ─────────────────────────────────
        trust = self.trust(
            embs_i=embs_i, embs_j=embs_j,
            t_i=times_i,   t_j=times_j,
        )                                                            # (E,) ∈ (0,2)
        self._trust_cache = trust.detach()

        # Trust-scaled softmax
        scores = scores * trust.unsqueeze(-1)
        scores = softmax(scores, index)
        scores = self.dropout(scores)

        # ── [5] Value + multi-resolution temporal message encoding ────────────
        msg = (V_j * scores.unsqueeze(-1)).reshape(E, self.dim)
        msg = self.msg_time_enc(msg, edge_t)                         # (E, dim)
        return msg
        self.q_proj  = nn.Linear(dim, dim)
        self.kv_proj = nn.Linear(dim, dim * 2)  # fused K and V projection

        # Single scalar from attended similarity → trust logit
        # Input: traj_sim scalar (1,) → logit
        self.trust_head = nn.Sequential(
            nn.Linear(1, max(dim // 8, 8)),
            nn.GELU(),
            nn.Linear(max(dim // 8, 8), 1),
        )

    def forward(
        self,
        h_i:    torch.Tensor,   # (E, dim)   kept for API compatibility, unused
        h_j:    torch.Tensor,   # (E, dim)   kept for API compatibility, unused
        deg_i:  torch.Tensor,   # (E,)       kept for API compatibility, unused
        deg_j:  torch.Tensor,   # (E,)       kept for API compatibility, unused
        embs_i: torch.Tensor,   # (E, K, dim)
        embs_j: torch.Tensor,   # (E, K, dim)
        t_i:    torch.Tensor,   # (E, K)
        t_j:    torch.Tensor,   # (E, K)
        eps:    float = 1e-6,
    ) -> torch.Tensor:           # (E,) ∈ (0, 2)

        E, K, D = embs_i.shape

        # ── Step 1: apply temporal positional encoding to both sequences ──────
        # Each view gets: g_k * RoPE(emb_k, t_k) + Fourier(t_k)
        # The recency gate g_k ensures the model focuses on recent views.
        vi = self.time_pe(embs_i, t_i)   # (E, K, dim)
        vj = self.time_pe(embs_j, t_j)   # (E, K, dim)

        # ── Step 2: multi-head cross-attention  (i queries j) ─────────────────
        # Q from i's time-encoded views:  what has i experienced?
        # K, V from j's time-encoded views: what has j experienced?
        #
        # For each of i's K states, the attention finds which of j's K states
        # is most structurally and temporally compatible.
        # Because views are RoPE-encoded, attention weights naturally decay
        # for pairs that are far apart in time.
        Q  = self.q_proj(vi.reshape(E * K, D)).view(E, K, self.attn_heads, self.head_dim)
        KV = self.kv_proj(vj.reshape(E * K, D)).view(E, K, 2, self.attn_heads, self.head_dim)
        Kj = KV[:, :, 0]   # (E, K, heads, hd)
        Vj = KV[:, :, 1]   # (E, K, heads, hd)

        # Transpose to (E, heads, K, hd) for batched matmul
        Q  = Q.permute(0, 2, 1, 3)   # (E, heads, K_i, hd)
        Kj = Kj.permute(0, 2, 1, 3)  # (E, heads, K_j, hd)
        Vj = Vj.permute(0, 2, 1, 3)  # (E, heads, K_j, hd)

        # Attention scores: (E, heads, K_i, K_j)
        attn = torch.matmul(Q, Kj.transpose(-2, -1)) * self.scale
        attn = F.softmax(attn, dim=-1)

        # Attended output: (E, heads, K_i, hd) -> (E, K_i, dim)
        attended = torch.matmul(attn, Vj)              # (E, heads, K_i, hd)
        attended = attended.permute(0, 2, 1, 3).reshape(E, K, D)

        # ── Step 3: trajectory similarity ────────────────────────────────────
        # How well does j's trajectory explain i's trajectory?
        # High similarity → i and j have been evolving the same way.
        vi_n   = F.normalize(vi,       dim=-1, eps=eps)  # (E, K, dim)
        att_n  = F.normalize(attended, dim=-1, eps=eps)  # (E, K, dim)
        traj_sim = (vi_n * att_n).sum(-1).mean(-1)       # (E,) ∈ [-1, 1]

        # ── Step 4: map to trust ∈ (0, 2) ────────────────────────────────────
        logit = self.trust_head(traj_sim.unsqueeze(-1)).squeeze(-1)  # (E,)
        return 1.0 + torch.tanh(logit)                               # (E,) ∈ (0, 2)


# ═══════════════════════════════════════════════════════════════════════════════
# 8.  MessageRoPE  — applies continuous-time RoPE directly to message vectors
#
#  Unlike the TemporalPositionalEncoding used in the trust module (which
#  operates on stored K-view snapshots), this module applies RoPE to the
#  LIVE message vector being passed from j to i along an edge, using the
#  edge's own timestamp as the rotation angle.
#
#  Why apply RoPE to messages (not just to queries/keys)?
#  ────────────────────────────────────────────────────────
#  In standard transformer attention, RoPE is applied to Q and K so that
#  the dot product Q·K reflects relative position.  Here we go further:
#  the VALUE (message content) is also rotated by the edge timestamp.
#  This means:
#    - Two messages with the same content but different timestamps will
#      aggregate to DIFFERENT vectors at the target node.
#    - The target node's representation is intrinsically time-aware,
#      not just score-aware.
#    - Recent messages occupy a different angular region from old ones,
#      so the GRU memory update naturally distinguishes temporal recency.
#
#  Shared log_timescales with the trust module's TemporalPositionalEncoding
#  are NOT used — this module has its own learned timescales tuned for
#  the message aggregation regime (shorter timescales work better here
#  since edge timestamps are absolute, not relative gaps).
# ═══════════════════════════════════════════════════════════════════════════════

class MessageRoPE(nn.Module):
    """
    Applies continuous-time RoPE to a batch of message vectors.

    Rotates pairs of dimensions by angles proportional to the edge timestamp:
        theta_d = t_edge / exp(log_timescale_d)
        msg[2d]   ->  msg[2d]   * cos(theta_d) - msg[2d+1] * sin(theta_d)
        msg[2d+1] ->  msg[2d]   * sin(theta_d) + msg[2d+1] * cos(theta_d)

    The dot product between two RoPE-encoded messages naturally reflects
    their temporal distance: same-time messages stay aligned, cross-time
    messages rotate apart in proportion to the timestamp gap.
    """
    def __init__(self, dim: int):
        super().__init__()
        assert dim % 2 == 0
        # Shorter timescales than K-view RoPE — edge-level granularity
        self.log_timescales = nn.Parameter(
            torch.linspace(0, 6, dim // 2)    # timescales: 1 → e^6 ≈ 400
        )

    def forward(self,
                msg: torch.Tensor,   # (E, dim)
                t:   torch.Tensor,   # (E,)   edge timestamps
                ) -> torch.Tensor:   # (E, dim)
        timescales = self.log_timescales.exp()              # (D//2,)
        angles     = t.float().unsqueeze(-1) / timescales  # (E, D//2)

        cos_a = angles.cos()   # (E, D//2)
        sin_a = angles.sin()

        m_even = msg[:, 0::2]  # (E, D//2)
        m_odd  = msg[:, 1::2]

        rot_even = m_even * cos_a - m_odd * sin_a
        rot_odd  = m_even * sin_a + m_odd * cos_a

        # Interleave back to (E, D)
        rotated = torch.stack([rot_even, rot_odd], dim=-1)  # (E, D//2, 2)
        return rotated.reshape(msg.shape)


# ═══════════════════════════════════════════════════════════════════════════════
# 9.  TemporalMessageBank  — NEW ADVANCED MECHANISM
#
#  Standard message passing aggregates ALL neighbours in one flat sum/mean.
#  This loses two things:
#    1. TEMPORAL ORDER: messages from t=10 and t=1000 are mixed equally.
#    2. TEMPORAL CONTEXT: the node has no memory of what it received before.
#
#  The TemporalMessageBank fixes both:
#    - After each forward pass, the aggregated message vector (already
#      RoPE-encoded) is stored in a per-node circular buffer of size M.
#    - Before the final update, a cross-attention reads over this buffer:
#        Query  = current aggregated message  (what I just received)
#        Key/V  = past M buffered messages    (what I've received before)
#      This lets the node ask: "is what I'm receiving NOW consistent with
#      my recent message history, or is it a sudden anomalous change?"
#    - The attention output is a temporally-smoothed message that blends
#      current and historical incoming signals.
#
#  Effect on anomaly detection:
#    Normal nodes receive consistent messages over time → strong cross-attn
#    Anomalous nodes receive erratic messages           → weak cross-attn
#    Central node near anomaly gets a divergent current msg vs history
#    → the bank effectively provides a "temporal message baseline"
# ═══════════════════════════════════════════════════════════════════════════════

class TemporalMessageBank(nn.Module):
    """
    Per-node circular buffer of past aggregated messages with
    cross-attention retrieval.

    Args:
        num_nodes : total nodes in the graph
        dim       : message / hidden dimension
        bank_size : M, number of past messages to store per node
    """
    def __init__(self, num_nodes: int, dim: int, bank_size: int = 8):
        super().__init__()
        self.bank_size = bank_size
        self.dim       = dim
        # Stored past messages per node: (num_nodes, M, dim)
        self.register_buffer("bank", torch.zeros(num_nodes, bank_size, dim))
        self.register_buffer("ptr",  torch.zeros(num_nodes, dtype=torch.long))

        # Cross-attention: current msg as Q, past bank as K and V
        self.q_proj = nn.Linear(dim, dim)
        self.k_proj = nn.Linear(dim, dim)
        self.v_proj = nn.Linear(dim, dim)
        self.scale  = dim ** -0.5
        self.out_proj = nn.Linear(dim, dim)
        self.norm     = nn.LayerNorm(dim)

    @torch.no_grad()
    def write(self, idx: torch.Tensor, msg: torch.Tensor):
        """Write current aggregated message into the circular buffer."""
        for b in range(idx.size(0)):
            node = int(idx[b])
            p    = int(self.ptr[node]) % self.bank_size
            self.bank[node, p] = msg[b].detach()
            self.ptr[node] += 1

    def read_and_attend(self,
                        idx:     torch.Tensor,   # (N,)
                        cur_msg: torch.Tensor,   # (N, dim)
                        ) -> torch.Tensor:       # (N, dim)
        """
        Cross-attention: current message queries the historical bank.

        Q = Linear(cur_msg)          (N, dim)
        K = Linear(bank[idx])        (N, M, dim)
        V = Linear(bank[idx])        (N, M, dim)

        output = softmax(Q K^T / sqrt(d)) V    (N, dim)

        Then blend: alpha * output + (1-alpha) * cur_msg
        where alpha is learned (starts near 0.5).
        """
        past = self.bank[idx]                                        # (N, M, dim)
        N, M, D = past.shape

        Q  = self.q_proj(cur_msg).unsqueeze(1)                       # (N, 1, dim)
        K  = self.k_proj(past)                                       # (N, M, dim)
        V  = self.v_proj(past)

        # Scaled dot-product attention over M past messages
        attn = torch.bmm(Q, K.transpose(1, 2)) * self.scale         # (N, 1, M)
        attn = F.softmax(attn, dim=-1)
        ctx  = torch.bmm(attn, V).squeeze(1)                        # (N, dim)

        # Residual: blend retrieved history with current message
        out = self.out_proj(ctx)
        return self.norm(cur_msg + out)                              # (N, dim)

    @torch.no_grad()
    def reset(self):
        self.bank.zero_()
        self.ptr.zero_()


# ═══════════════════════════════════════════════════════════════════════════════
# 10.  TemporalTrustAttention  (one message-passing layer)
#
#  Upgrades vs vanilla attention:
#
#  [1] MEMORY-CONDITIONED Q/K/V  (Hyper-network projections)
#      Standard attention uses fixed weight matrices W_Q, W_K, W_V.
#      Here, the projection weights are DYNAMICALLY generated from each
#      node's GRU memory state — inspired by hypernetwork attention used
#      in DyGFormer and CAW.
#
#      For each node i:
#        ΔW_Q(i) = reshape( HyperNet_Q( memory_i ) )   (dim × dim)
#        W_Q(i)  = W_Q_base + ΔW_Q(i)
#      Then: Q_i = x_i @ W_Q(i)^T
#
#      This means nodes with different histories produce different Q/K/V
#      projections — two nodes with the same current features but different
#      pasts will attend differently.  Anomalous nodes, whose memory is
#      consistently erratic, develop distinctive projection patterns that
#      separate them from normal nodes without supervision.
#
#      To keep it efficient, the hyper-net produces a LOW-RANK delta:
#        ΔW = U @ V^T   where U ∈ R^{dim×r}, V ∈ R^{dim×r}, r << dim
#      This avoids full dim×dim output from the hyper-net.
#
#  [2] CROSS-SOURCE ATTENTION (retained from previous version)
#      Q from EDGE (interaction type), K from NODE (neighbour identity).
#      The hyper-net conditions the edge-Q projection on the central
#      node's memory, making the query node-specific.
#
#  [3] RoPE ON MESSAGES + Fourier additive (MessageTemporalEncoding)
#      Upgraded: RoPE rotates the message, multi-scale Fourier is added.
#      Same design philosophy as TemporalPositionalEncoding but applied
#      to live message vectors at edge-level granularity.
#
#  [4] SIMILARITY-BASED TRUST + TEMPORAL MESSAGE BANK  (unchanged)
# ═══════════════════════════════════════════════════════════════════════════════

class HyperProjection(nn.Module):
    """
    Generates a low-rank delta ΔW = U @ V^T conditioned on a context vector,
    then applies (W_base + ΔW) @ x^T to project x.

    This is a lightweight hypernetwork: instead of a full dim×dim weight matrix
    per node, we produce rank-r perturbations from the node's memory state.

    Args:
        dim   : input and output dimension
        rank  : rank of the delta (r << dim, default 8)
        ctx_dim: dimension of the conditioning context (memory state)
    """
    def __init__(self, dim: int, rank: int = 8, ctx_dim: Optional[int] = None):
        super().__init__()
        ctx = ctx_dim or dim
        self.dim  = dim
        self.rank = rank

        # Base (shared) projection — standard Linear
        self.W_base = nn.Linear(dim, dim, bias=True)

        # Hyper-net: context -> (U, V) for low-rank delta
        # Output: dim*rank for U, dim*rank for V
        self.hyper = nn.Sequential(
            nn.Linear(ctx, dim),
            nn.GELU(),
            nn.Linear(dim, 2 * dim * rank),
        )

    def forward(self,
                x:   torch.Tensor,   # (B, dim)  vectors to project
                ctx: torch.Tensor,   # (B, ctx_dim)  conditioning context
                ) -> torch.Tensor:   # (B, dim)
        B = x.size(0)

        # Base projection
        out_base = self.W_base(x)                                    # (B, dim)

        # Low-rank delta from hyper-net
        uv   = self.hyper(ctx)                                       # (B, 2*dim*rank)
        U, V = uv.split(self.dim * self.rank, dim=-1)
        U    = U.view(B, self.dim, self.rank)                        # (B, dim, r)
        V    = V.view(B, self.dim, self.rank)                        # (B, dim, r)
        # ΔW = U @ V^T  shape (B, dim, dim)
        # Apply: x @ ΔW^T = x @ V @ U^T
        delta = (x.unsqueeze(1) @ V).squeeze(1) @ U.transpose(1, 2).mean(0)
        # Efficient form: x (B,dim) @ V (B,dim,r) -> (B,r) @ U^T mean (r,dim) -> (B,dim)
        x_proj = (x.unsqueeze(1) @ V).squeeze(1)                    # (B, r)
        # Average U across batch for the delta (memory-conditioned direction)
        delta  = x_proj @ U.mean(0).t()                             # (B, dim)

        return out_base + delta                                      # (B, dim)


class MessageTemporalEncoding(nn.Module):
    """
    Applies RoPE + multi-scale Fourier to live edge message vectors.
    Same design as TemporalPositionalEncoding but for 2D (E, dim) messages.

    Args:
        dim       : message dimension
        n_fourier : number of Fourier frequency bands
    """
    def __init__(self, dim: int, n_fourier: int = 16):
        super().__init__()
        assert dim % 2 == 0
        self.dim = dim

        # RoPE timescales (shorter range for edge-level timestamps)
        self.rope_log_ts = nn.Parameter(
            torch.linspace(0.0, 6.0, dim // 2)    # 1 → e^6 ≈ 400
        )

        # Multi-scale Fourier (fixed freqs, learned projection)
        freqs = torch.logspace(-3, 1, n_fourier)
        self.register_buffer("fourier_freqs", freqs)
        self.fourier_proj = nn.Linear(2 * n_fourier, dim)

    def forward(self,
                msg: torch.Tensor,   # (E, dim)
                t:   torch.Tensor,   # (E,)
                ) -> torch.Tensor:   # (E, dim)
        # ── RoPE rotation ─────────────────────────────────────────────────────
        ts     = self.rope_log_ts.exp()                              # (D//2,)
        angles = t.float().unsqueeze(-1) / ts                       # (E, D//2)
        cos_a, sin_a = angles.cos(), angles.sin()

        m_even = msg[:, 0::2]
        m_odd  = msg[:, 1::2]
        rot    = torch.stack(
            [m_even * cos_a - m_odd * sin_a,
             m_even * sin_a + m_odd * cos_a], dim=-1
        ).reshape(msg.shape)                                         # (E, dim)

        # ── Multi-scale Fourier (additive) ────────────────────────────────────
        phi  = t.float().unsqueeze(-1) * self.fourier_freqs         # (E, F)
        feat = torch.cat([phi.sin(), phi.cos()], dim=-1)             # (E, 2F)
        fourier = self.fourier_proj(feat)                            # (E, dim)

        return rot + fourier                                         # (E, dim)


class TemporalTrustAttention(MessagePassing):
    def __init__(self, dim: int, heads: int = 8, dropout: float = 0.1,
                 window: int = 8, t2v_dim: int = 16,
                 num_nodes: int = 0, bank_size: int = 8,
                 hyper_rank: int = 8):
        super().__init__(aggr="add", node_dim=0)
        assert dim % heads == 0
        self.dim      = dim
        self.heads    = heads
        self.head_dim = dim // heads
        self.scale    = self.head_dim ** -0.5

        # ── Memory-conditioned hyper-projected Q, K, V ────────────────────────
        # Q: edge embedding conditioned on i's memory
        #    (the query adapts to who i is historically)
        self.q_hyper = HyperProjection(dim, rank=hyper_rank, ctx_dim=dim)
        # K: neighbour node j conditioned on j's memory
        #    (j's key reflects its own historical identity)
        self.k_hyper = HyperProjection(dim, rank=hyper_rank, ctx_dim=dim)
        # V: neighbour node j conditioned on j's memory
        #    (what j sends is shaped by its own history)
        self.v_hyper = HyperProjection(dim, rank=hyper_rank, ctx_dim=dim)

        self.o_proj = nn.Linear(dim, dim)

        # ── Temporal message encoding (RoPE + Fourier on live messages) ───────
        self.msg_time_enc = MessageTemporalEncoding(dim, n_fourier=16)

        # ── Trust ─────────────────────────────────────────────────────────────
        self.trust = StructuralTrustModule(
            dim, window=window, t2v_dim=t2v_dim
        )

        # ── Temporal message bank ─────────────────────────────────────────────
        self.msg_bank = (TemporalMessageBank(num_nodes, dim, bank_size=bank_size)
                         if num_nodes > 0 else None)

        self.dropout = nn.Dropout(dropout)
        self.gate    = nn.Linear(dim, 1)
        self.norm    = nn.LayerNorm(dim)
        self.fusion  = nn.Sequential(
            nn.Linear(dim * 2, dim),
            nn.GELU(),
            nn.Linear(dim, dim),
        )

        self._trust_cache: Optional[torch.Tensor] = None

    # --------------------------------------------------------------------------
    def forward(self,
                x:          torch.Tensor,       # (N, dim)
                edge_index: torch.Tensor,       # (2, E)
                edge_emb:   torch.Tensor,       # (E, dim)
                time_emb:   torch.Tensor,       # (E, dim)  unused, kept for API
                edge_t:     torch.Tensor,       # (E,)  raw timestamps
                memory:     RecurrentMemory,
                evo_bank:   EvolutionBank,
                deg:        torch.Tensor,       # (N,)
                ) -> torch.Tensor:

        N    = x.size(0)
        nidx = torch.arange(N, device=x.device)
        src, dst = edge_index

        mem_vals, _ = memory.read(nidx)                              # (N, dim)
        embs_all, times_all = evo_bank.read(nidx)

        # ── Memory-conditioned Q from edge embedding ──────────────────────────
        # Q is produced from the edge embedding, but conditioned on the
        # CENTRAL node i's memory — so the same edge will produce a different
        # query depending on who i is and what its history looks like.
        mem_src = mem_vals[src]                                      # (E, dim)
        Q_edge = self.q_hyper(edge_emb, mem_src
                              ).view(-1, self.heads, self.head_dim)  # (E, h, hd)

        # K, V from node features — conditioned on each node's own memory
        K_node = self.k_hyper(x, mem_vals).view(N, self.heads, self.head_dim)
        V_node = self.v_hyper(x, mem_vals).view(N, self.heads, self.head_dim)

        out = self.propagate(
            edge_index,
            x=x,
            Q_edge=Q_edge,
            K_node=K_node,
            V_node=V_node,
            mem_val=mem_vals,
            deg=deg,
            embs=embs_all,
            times=times_all,
            edge_t=edge_t,
            size=(N, N),
        )
        out = out.view(N, self.dim)

        # ── Temporal Message Bank ─────────────────────────────────────────────
        if self.msg_bank is not None:
            out = self.msg_bank.read_and_attend(nidx, out)
            self.msg_bank.write(nidx, out)

        # ── Fuse with GRU memory + gated residual ─────────────────────────────
        h   = memory.get_hidden(nidx)
        out = self.fusion(torch.cat([out, h], dim=-1))
        out = self.o_proj(out)

        g = torch.sigmoid(self.gate(x))
        return self.norm(g * x + (1 - g) * out)

    # --------------------------------------------------------------------------
    def message(self,
                x_i:        torch.Tensor,   # (E, dim)
                x_j:        torch.Tensor,   # (E, dim)
                Q_edge:     torch.Tensor,   # (E, heads, head_dim)
                K_node_j:   torch.Tensor,   # (E, heads, head_dim)
                V_node_j:   torch.Tensor,   # (E, heads, head_dim)
                mem_val_i:  torch.Tensor,   # (E, dim)
                mem_val_j:  torch.Tensor,
                deg_i:      torch.Tensor,   # (E,)
                deg_j:      torch.Tensor,
                embs_i:     torch.Tensor,   # (E, K, dim)
                embs_j:     torch.Tensor,
                times_i:    torch.Tensor,   # (E, K)
                times_j:    torch.Tensor,
                edge_t:     torch.Tensor,   # (E,)
                index:      torch.Tensor,
                ) -> torch.Tensor:

        E = x_i.size(0)

        # ── [1] Cross-source attention score (memory-conditioned) ─────────────
        # Q from edge (what does this interaction ask for, given who i is?)
        # K from node j (what does j structurally offer, given its history?)
        scores = (Q_edge * K_node_j).sum(-1) * self.scale            # (E, heads)

        # ── [2] K-view trajectory trust ───────────────────────────────────────
        trust = self.trust(
            h_i=mem_val_i, h_j=mem_val_j,
            deg_i=deg_i,   deg_j=deg_j,
            embs_i=embs_i, embs_j=embs_j,
            t_i=times_i,   t_j=times_j,
        )                                                            # (E,) in (0,2)
        self._trust_cache = trust.detach()

        # Trust-scaled softmax
        scores = scores * trust.unsqueeze(-1)
        scores = softmax(scores, index)
        scores = self.dropout(scores)

        # ── [3] Value + temporal message encoding ─────────────────────────────
        msg = V_node_j * scores.unsqueeze(-1)                        # (E, h, hd)
        msg = msg.reshape(E, self.dim)                               # (E, dim)

        # Apply RoPE + Fourier to bake edge timestamp into message content
        msg = self.msg_time_enc(msg, edge_t)                         # (E, dim)

        return msg


# ═══════════════════════════════════════════════════════════════════════════════
# 11.  EnhancedTemporalGNN  — full model
# ═══════════════════════════════════════════════════════════════════════════════

class EnhancedTemporalGNN(nn.Module):
    """
    Full temporal GNN with K-view trajectory trust for representation learning.

    Expected batch fields:
        batch.x          (N, in_dim)    node features
        batch.edge_index (2, E)         directed edges [src, dst]
        batch.msg        (E, edge_dim)  edge / message features
        batch.t          (E,)           edge timestamps

    Returns:
        x  (N, hidden_dim)  node representations
    """
    def __init__(
        self,
        num_nodes:       int,
        in_dim:          int,
        edge_dim:        int,
        hidden_dim:      int,
        num_layers:      int   = 2,
        heads:           int   = 8,
        dropout:         float = 0.1,
        window:          int   = 8,
        t2v_dim:         int   = 16,
        bank_size:       int   = 8,
        bilinear_rank:   int   = 8,
        memory_momentum: float = 0.9,
    ):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.num_nodes  = num_nodes
        self.window     = window

        # Input projections
        self.input_proj = nn.Linear(in_dim, hidden_dim)
        self.edge_enc   = nn.Linear(edge_dim, hidden_dim)
        self.time_enc   = TimeEncoder(hidden_dim)

        # TransformerConv snapshot encoder: produces embeddings stored in EvolutionBank
        self.snapshot_gnn = SnapshotGNN(hidden_dim, heads=heads, dropout=dropout)

        # Persistent state
        self.memory   = RecurrentMemory(num_nodes, hidden_dim, momentum=memory_momentum)
        self.evo_bank = EvolutionBank(num_nodes, hidden_dim, window=window)

        # Main message-passing layers
        self.layers = nn.ModuleList([
            TemporalTrustAttention(
                hidden_dim, heads=heads, dropout=dropout,
                window=window, t2v_dim=t2v_dim,
                num_nodes=num_nodes, bank_size=bank_size,
                t_heads=4, bilinear_rank=bilinear_rank,
            )
            for _ in range(num_layers)
        ])

        self.final_norm = nn.LayerNorm(hidden_dim)
        self._init_weights()

    # --------------------------------------------------------------------------
    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)

    # --------------------------------------------------------------------------
    def forward(self, batch) -> torch.Tensor:
        N          = batch.x.size(0)
        device     = batch.x.device
        edge_index = batch.edge_index
        src, dst   = edge_index

        # ── Encode inputs ─────────────────────────────────────────────────────
        x        = self.input_proj(batch.x)                         # (N, hidden)
        edge_emb = self.edge_enc(batch.msg)                         # (E, hidden)

        # ── Node degrees ──────────────────────────────────────────────────────
        deg = degree(dst, num_nodes=N, dtype=torch.float).clamp(min=1.0)

        # ── Temporal encoding (for TimeEncoder / GRU context) ─────────────────
        node_last_t = torch.zeros(N, device=device)
        node_last_t.scatter_reduce_(
            0, src, batch.t.float(), reduce='amax', include_self=True
        )
        t_rel    = batch.t.float() - node_last_t[src]
        time_emb = self.time_enc(batch.t, t_rel)                    # (E, hidden)

        # ── Compute snapshot embedding for EvolutionBank ──────────────────────
        # TransformerConv operates directly on node features and edge_attr.
        # No time signal here — time is injected in TemporalPositionalEncoding.
        snapshot_emb = self.snapshot_gnn(x, edge_index, edge_emb)   # (N, hidden)

        # ── Message passing layers ─────────────────────────────────────────────
        # Pass raw edge timestamps (batch.t) separately for RoPE and decay gate
        for layer in self.layers:
            x = layer(x, edge_index, edge_emb, time_emb,
                      batch.t.float(),                               # edge_t
                      self.memory, self.evo_bank, deg)

        x = self.final_norm(x)

        # ── Update persistent state ───────────────────────────────────────────
        nidx      = torch.arange(N, device=device)
        current_t = batch.t.float().max().expand(N)

        # Write snapshot to EvolutionBank (the rich contextual embedding)
        self.evo_bank.write(nidx, snapshot_emb, current_t)
        # Update GRU memory
        self.memory.write(nidx, x)

        return x

    # --------------------------------------------------------------------------
    def get_trust_scores(self, layer_idx: int = -1) -> Optional[torch.Tensor]:
        """Cached trust scores (E,) in (0, 2) from last forward()."""
        return self.layers[layer_idx]._trust_cache

    @torch.no_grad()
    def reset_memory(self):
        self.memory.hidden.zero_()
        self.memory.variance.fill_(0.1)
        self.evo_bank.reset()
        for layer in self.layers:
            if layer.msg_bank is not None:
                layer.msg_bank.reset()


# ═══════════════════════════════════════════════════════════════════════════════
# Quick sanity check
# ═══════════════════════════════════════════════════════════════════════════════
if __name__ == "__main__":
    import types

    torch.manual_seed(42)

    N, E      = 50, 120
    in_dim    = 32
    edge_dim  = 16
    hidden    = 64
    num_nodes = 100
    K         = 8

    def make_batch(t_offset=0.0):
        return types.SimpleNamespace(
            x          = torch.randn(N, in_dim),
            edge_index = torch.randint(0, N, (2, E)),
            msg        = torch.randn(E, edge_dim),
            t          = torch.rand(E) * 1000 + t_offset,
        )

    model = EnhancedTemporalGNN(
        num_nodes=num_nodes, in_dim=in_dim, edge_dim=edge_dim,
        hidden_dim=hidden, num_layers=2, heads=4,
        window=K, t2v_dim=16, bank_size=8, bilinear_rank=8,
    )

    for step in range(3):
        emb   = model(make_batch(step * 1000))
        trust = model.get_trust_scores()
        print(f"[pass {step+1}] emb: {emb.shape} | "
              f"trust: [{trust.min():.4f}, {trust.max():.4f}] "
              f"mean={trust.mean():.4f}")
        assert trust.min() > 0.0 and trust.max() < 2.0

    print("\nAll checks passed. Trust is in (0, 2).")

In [1]:
"""
Temporal Trust GNN for Unsupervised Anomaly Detection
=====================================================
Produces node representations for downstream contrastive learning.
No labels required — fully unsupervised.

Message Aggregation (TemporalTrustAttention):

  [1] CROSS-SOURCE ATTENTION
      Q comes from EDGE embeddings  (what does this interaction ask for?)
      K comes from NODE features x  (who is this neighbour?)
      The same neighbour j is weighted differently depending on the
      type/context of the edge, making attention interaction-aware.

  [2] RoPE ON MESSAGES  (MessageRoPE)
      After scoring, the value message V_j is rotated by its edge
      timestamp using continuous-time RoPE BEFORE aggregation.
      Time is therefore baked into the message CONTENT, not just
      the score — messages from different times aggregate to different
      vectors even with identical node features.

  [3] SIMILARITY-BASED TRUST  (StructuralTrustModule)
      trust(i->j) in (0, 2):
        trust > 1 : i and j are structurally similar -> amplify
        trust < 1 : i and j are dissimilar           -> attenuate
      Both normal<->normal and anomalous<->anomalous get high trust.
      The model self-segregates: each node learns from its own "type".
      This naturally separates normal and anomalous representations.

  [4] TEMPORAL MESSAGE BANK  (TemporalMessageBank) — NEW
      A per-node buffer stores the last M aggregated messages.
      After each aggregation, cross-attention reads over this buffer:
        Q = current aggregated message
        K/V = past M stored messages
      Output: temporally-smoothed message that blends current with
      historical incoming signals. Detects sudden anomalous changes
      in what a node is receiving (erratic = anomalous influence).

Trust signals (StructuralTrustModule):
  [A] cos(h_i, h_j)              current GRU state similarity
  [B] degree profile similarity  structural role similarity
  [C] K-view trajectory sim      RoPE+Time2Vec encoded snapshot comparison
      trust = 1 + tanh(MLP([A, B, C]))   in (0, 2)
"""

import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from typing import Optional, Tuple
from torch_geometric.nn import MessagePassing, TransformerConv
from torch_geometric.utils import softmax, degree


# ═══════════════════════════════════════════════════════════════════════════════
# 1.  Time2Vec
# ═══════════════════════════════════════════════════════════════════════════════

class Time2Vec(nn.Module):
    """
    Encodes a scalar timestamp into a vector:
        [w0*t + b0,  sin(w1*t + b1), ..., sin(w_{k}*t + b_{k})]
    Linear term = trend.  Sinusoids = periodicity.  All frequencies learned.
    """
    def __init__(self, out_dim: int):
        super().__init__()
        assert out_dim >= 2
        k = out_dim - 1
        self.w0 = nn.Parameter(torch.randn(1) * 0.01)
        self.b0 = nn.Parameter(torch.zeros(1))
        self.W  = nn.Parameter(torch.randn(k) * 0.01)
        self.B  = nn.Parameter(torch.zeros(k))

    def forward(self, t: torch.Tensor) -> torch.Tensor:
        """t: (*,) -> (*, out_dim)"""
        t   = t.float().unsqueeze(-1)
        lin = t * self.w0 + self.b0
        per = torch.sin(t * self.W + self.B)
        return torch.cat([lin, per], dim=-1)


# ═══════════════════════════════════════════════════════════════════════════════
# 2.  TimeEncoder  (edge-level: absolute + relative timestamp -> hidden_dim)
# ═══════════════════════════════════════════════════════════════════════════════

class TimeEncoder(nn.Module):
    def __init__(self, hidden_dim: int):
        super().__init__()
        half = hidden_dim // 2
        self.t2v_abs = Time2Vec(half)
        self.t2v_rel = Time2Vec(half)
        self.proj = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, hidden_dim),
        )

    def forward(self, t_abs: torch.Tensor,
                t_rel: Optional[torch.Tensor] = None) -> torch.Tensor:
        """(E,), (E,) -> (E, hidden_dim)"""
        a = self.t2v_abs(t_abs)
        r = self.t2v_rel(t_rel) if t_rel is not None else torch.zeros_like(a)
        return self.proj(torch.cat([a, r], dim=-1))


# ═══════════════════════════════════════════════════════════════════════════════
# 3.  RecurrentMemory  (per-node GRU hidden state)
# ═══════════════════════════════════════════════════════════════════════════════

class RecurrentMemory(nn.Module):
    """
    Maintains one GRU hidden state per node, updated after each snapshot.
    The hidden state is the running summary of the node's full history.
    """
    def __init__(self, num_nodes: int, dim: int, momentum: float = 0.9):
        super().__init__()
        self.gru      = nn.GRUCell(dim, dim)
        self.momentum = momentum
        self.register_buffer("hidden",   torch.zeros(num_nodes, dim))
        self.register_buffer("variance", torch.ones(num_nodes, dim) * 0.1)

    def read(self, idx: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        return self.hidden[idx], self.variance[idx]

    def get_hidden(self, idx: torch.Tensor) -> torch.Tensor:
        return self.hidden[idx]

    @torch.no_grad()
    def write(self, idx: torch.Tensor, x: torch.Tensor):
        h_old = self.hidden[idx]
        h_new = self.gru(x.detach(), h_old)
        delta = h_new - h_old
        self.variance[idx] = (self.momentum * self.variance[idx]
                              + (1 - self.momentum) * delta.pow(2))
        self.hidden[idx] = h_new


# ═══════════════════════════════════════════════════════════════════════════════
# 4.  SnapshotGNN  — TransformerConv-based node encoder
#
#  Produces the snapshot embedding stored in the EvolutionBank.
#  Uses PyG's TransformerConv which performs full transformer-style
#  multi-head attention between a node and its one-hop neighbours,
#  incorporating edge_attr directly into the key/value computation:
#
#      alpha(i,j) = softmax( (W_Q * x_i)^T (W_K * x_j + W_E * edge_attr) )
#      out_i      = W_O * sum_j [ alpha(i,j) * (W_V * x_j + W_E * edge_attr) ]
#
#  Inputs: node features (x) and edge features (edge_attr) ONLY.
#  Time is NOT used here — it is handled separately in the positional
#  encoding step when comparing K-view trajectories in the trust module.
#
#  Two TransformerConv layers with a residual connection give the model
#  capacity to capture both direct neighbours and two-hop structure.
# ═══════════════════════════════════════════════════════════════════════════════

class SnapshotGNN(nn.Module):
    """
    Transformer-based snapshot encoder using PyG's TransformerConv.

    Produces a rich node embedding from node features and edge attributes,
    capturing neighbourhood structure via transformer-style attention.
    Time is deliberately excluded — it is injected separately during
    the K-view positional encoding in StructuralTrustModule.

    Args:
        dim      : hidden dimension (input, output, and internal)
        heads    : number of attention heads for TransformerConv
        dropout  : dropout on attention weights
    """
    def __init__(self, dim: int, heads: int = 4, dropout: float = 0.1):
        super().__init__()
        assert dim % heads == 0, "dim must be divisible by heads"

        # Layer 1: node features + edge_attr -> dim
        self.conv1 = TransformerConv(
            in_channels  = dim,
            out_channels = dim // heads,
            heads        = heads,
            dropout      = dropout,
            edge_dim     = dim,        # edge_attr dimension matches hidden dim
            concat       = True,       # concat heads -> dim
            beta         = True,       # learnable skip connection inside conv
        )
        self.norm1 = nn.LayerNorm(dim)

        # Layer 2: refine with a second transformer hop
        self.conv2 = TransformerConv(
            in_channels  = dim,
            out_channels = dim // heads,
            heads        = heads,
            dropout      = dropout,
            edge_dim     = dim,
            concat       = True,
            beta         = True,
        )
        self.norm2 = nn.LayerNorm(dim)

        # Final projection + residual gate
        self.out_proj = nn.Linear(dim, dim)
        self.gate     = nn.Linear(dim, 1)

    def forward(self,
                x:          torch.Tensor,   # (N, dim)   projected node features
                edge_index: torch.Tensor,   # (2, E)
                edge_attr:  torch.Tensor,   # (E, dim)   projected edge features
                ) -> torch.Tensor:          # (N, dim)
        # Layer 1
        h = self.norm1(F.gelu(self.conv1(x, edge_index, edge_attr)))  # (N, dim)
        # Layer 2 with residual
        h2 = self.norm2(F.gelu(self.conv2(h, edge_index, edge_attr))) # (N, dim)
        h  = h + h2                                                    # residual

        # Gated output projection
        out = self.out_proj(h)
        g   = torch.sigmoid(self.gate(x))
        return g * x + (1 - g) * out                                  # (N, dim)


# ═══════════════════════════════════════════════════════════════════════════════
# 5.  EvolutionBank  — stores last K snapshot embeddings + timestamps per node
#
#  After each forward pass, the snapshot embedding produced by SnapshotGNN
#  is written here alongside its timestamp.
#  Shape: bank (num_nodes, K, dim),  times (num_nodes, K)
# ═══════════════════════════════════════════════════════════════════════════════

class EvolutionBank(nn.Module):
    def __init__(self, num_nodes: int, dim: int, window: int = 8):
        super().__init__()
        self.window = window
        self.dim    = dim
        self.register_buffer("bank",  torch.zeros(num_nodes, window, dim))
        self.register_buffer("times", torch.zeros(num_nodes, window))
        self.register_buffer("ptr",   torch.zeros(num_nodes, dtype=torch.long))

    def read(self, idx: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        """Returns (B, K, dim) embeddings and (B, K) timestamps."""
        return self.bank[idx], self.times[idx]

    @torch.no_grad()
    def write(self, idx: torch.Tensor, emb: torch.Tensor, t: torch.Tensor):
        """
        idx : (B,)    node indices
        emb : (B, D)  snapshot embedding from SnapshotGNN
        t   : (B,)    timestamp of this snapshot
        """
        for b in range(idx.size(0)):
            node = int(idx[b])
            p    = int(self.ptr[node]) % self.window
            self.bank[node,  p] = emb[b].detach()
            self.times[node, p] = float(t[b])
            self.ptr[node] += 1

    @torch.no_grad()
    def reset(self):
        self.bank.zero_()
        self.times.zero_()
        self.ptr.zero_()


# ═══════════════════════════════════════════════════════════════════════════════
# 6.  TemporalPositionalEncoding  — per-head multi-resolution temporal encoding
#
#  Applies temporal encoding to K-view snapshot sequences for the trust module.
#
#  KEY IDEA: split the embedding into H temporal heads, each head covering a
#  DIFFERENT temporal resolution band. Head 0 uses fast-rotating RoPE
#  (captures short-range patterns, e.g. hourly), head H-1 uses slow-rotating
#  RoPE (captures long-range drift, e.g. weekly). This gives the model multiple
#  simultaneous temporal "lenses" operating in parallel, similar to how
#  multi-head attention gives multiple geometric viewpoints.
#
#  Per head h:
#    (a) RoPE with head-specific timescale band:
#            θ_{h,d} = t / T_{h,d}
#        T_{h,d} are sampled from a sub-range of [T_min, T_max] assigned to head h.
#        Head h rotates dimensions (h*hd) to ((h+1)*hd) with timescales in
#        [T_min + h*(T_max-T_min)/H, T_min + (h+1)*(T_max-T_min)/H].
#
#    (b) Multi-scale Fourier (additive, shared across heads):
#        Complete Fourier basis (sin + cos) over F log-uniform frequencies,
#        projected to dim. Captures absolute periodic patterns independently
#        of the RoPE rotation.
#
#    (c) Recency decay gate (per-head learned λ_h):
#        g_{h,k} = σ( -λ_h * Δt_k + b_h )
#        Different heads decay at different rates — fast heads strongly
#        discount old views, slow heads treat all views more equally.
#
#  Combined per head h, view k:
#      view_{h,k} = g_{h,k} * RoPE_h(emb_{h,k}, t_k) + Fourier_proj(t_k)[h*hd:(h+1)*hd]
# ═══════════════════════════════════════════════════════════════════════════════

class TemporalPositionalEncoding(nn.Module):
    """
    Per-head multi-resolution temporal positional encoding for K-view sequences.

    Args:
        dim       : total embedding dimension (must be even, divisible by t_heads)
        t_heads   : number of temporal resolution heads
        n_fourier : Fourier frequency bands (sin+cos each)
        t_min_log : log of minimum timescale (default: 0 → T=1)
        t_max_log : log of maximum timescale (default: 9 → T≈8000)
    """
    def __init__(self, dim: int, t_heads: int = 4, n_fourier: int = 32,
                 t_min_log: float = 0.0, t_max_log: float = 9.0):
        super().__init__()
        assert dim % 2 == 0 and dim % t_heads == 0
        self.dim     = dim
        self.t_heads = t_heads
        self.head_d  = dim // t_heads   # dims per temporal head
        assert self.head_d % 2 == 0,   "head_d must be even for RoPE"

        # ── (a) Per-head RoPE timescales ──────────────────────────────────────
        # Each head h gets timescales from a dedicated sub-range of [t_min, t_max].
        # Head 0: fast (captures fine-grained recency)
        # Head H-1: slow (captures long-range historical trends)
        rope_params = []
        for h in range(t_heads):
            lo = t_min_log + h       * (t_max_log - t_min_log) / t_heads
            hi = t_min_log + (h + 1) * (t_max_log - t_min_log) / t_heads
            rope_params.append(torch.linspace(lo, hi, self.head_d // 2))
        # Stack to (t_heads, head_d//2), learned
        self.rope_log_ts = nn.Parameter(torch.stack(rope_params, dim=0))

        # ── (b) Multi-scale Fourier (shared, additive) ────────────────────────
        freqs = torch.logspace(-4, 1, n_fourier)
        self.register_buffer("fourier_freqs", freqs)
        self.fourier_proj = nn.Linear(2 * n_fourier, dim)

        # ── (c) Per-head recency decay gate ───────────────────────────────────
        # Each head has its own learned decay rate λ_h and bias b_h.
        # Fast heads (low h): init high λ → strong recency emphasis
        # Slow heads (high h): init low λ → weaker decay
        init_log_decay = torch.linspace(1.0, -1.0, t_heads)         # head 0 fastest
        self.log_decay  = nn.Parameter(init_log_decay)               # (t_heads,)
        self.decay_bias = nn.Parameter(torch.ones(t_heads))          # (t_heads,)

    # ──────────────────────────────────────────────────────────────────────────
    def _rope_per_head(self,
                       embs:  torch.Tensor,   # (B, K, dim)
                       times: torch.Tensor,   # (B, K)
                       ) -> torch.Tensor:     # (B, K, dim)
        """Apply per-head RoPE: each head h rotates its slice of the embedding
        with head-h-specific timescales."""
        B, K, D = embs.shape
        out_parts = []
        for h in range(self.t_heads):
            # Slice embedding for this head: (B, K, head_d)
            sl = embs[..., h * self.head_d : (h + 1) * self.head_d]
            ts  = self.rope_log_ts[h].exp()                          # (head_d//2,)
            ang = times.unsqueeze(-1) / ts                           # (B, K, hd//2)
            cos_a, sin_a = ang.cos(), ang.sin()
            e_even = sl[..., 0::2]
            e_odd  = sl[..., 1::2]
            rot = torch.stack(
                [e_even * cos_a - e_odd * sin_a,
                 e_even * sin_a + e_odd * cos_a], dim=-1
            ).reshape(B, K, self.head_d)
            out_parts.append(rot)
        return torch.cat(out_parts, dim=-1)                          # (B, K, dim)

    # ──────────────────────────────────────────────────────────────────────────
    def _fourier(self, times: torch.Tensor) -> torch.Tensor:
        """(B, K) -> (B, K, dim)"""
        B, K = times.shape
        t    = times.reshape(B * K, 1).float()
        phi  = t * self.fourier_freqs.unsqueeze(0)                   # (BK, F)
        feat = torch.cat([phi.sin(), phi.cos()], dim=-1)             # (BK, 2F)
        return self.fourier_proj(feat).view(B, K, self.dim)          # (B, K, D)

    # ──────────────────────────────────────────────────────────────────────────
    def _decay_gates(self, times: torch.Tensor) -> torch.Tensor:
        """(B, K) -> (B, K, dim)  — per-head gate broadcast across head dims"""
        t_max = times.max(dim=-1, keepdim=True).values               # (B, 1)
        delta = (t_max - times).clamp(min=0.0)                      # (B, K)
        lam   = self.log_decay.exp()                                 # (t_heads,)
        bias  = self.decay_bias                                      # (t_heads,)
        # gate per head: (B, K, t_heads)
        gate  = torch.sigmoid(
            -lam.unsqueeze(0).unsqueeze(0) * delta.unsqueeze(-1)
            + bias.unsqueeze(0).unsqueeze(0)
        )
        # Broadcast to full dim: each head's gate applies to its head_d dims
        gate_full = gate.repeat_interleave(self.head_d, dim=-1)      # (B, K, dim)
        return gate_full

    # ──────────────────────────────────────────────────────────────────────────
    def forward(self,
                embs:  torch.Tensor,   # (B, K, dim)
                times: torch.Tensor,   # (B, K)
                ) -> torch.Tensor:     # (B, K, dim)
        rope_out   = self._rope_per_head(embs, times)                # (B, K, dim)
        gates      = self._decay_gates(times)                        # (B, K, dim)
        fourier_out = self._fourier(times)                           # (B, K, dim)
        # Each head's RoPE output is gated by its own recency rate,
        # then the absolute Fourier signal is added on top.
        return gates * rope_out + fourier_out                        # (B, K, dim)


# ═══════════════════════════════════════════════════════════════════════════════
# 7.  StructuralTrustModule  — K-VIEW SINKHORN TRAJECTORY SIMILARITY
#
#  Computes trust(i→j) ∈ (0, 2) using ONLY the K-view trajectory signal.
#
#  WHY SINKHORN INSTEAD OF CROSS-ATTENTION?
#  ─────────────────────────────────────────
#  Cross-attention (softmax over rows) lets each of i's views attend to ALL
#  of j's views independently.  Two problems:
#    1. Multiple views of i can "compete" for the same view of j, pulling
#       the match towards a few dominant modes.
#    2. No constraint that the overall matching is globally consistent —
#       a single anomalous view in j can dominate every row.
#
#  Sinkhorn optimal transport solves both:
#    - Finds the globally optimal SOFT ASSIGNMENT matrix T ∈ R^{K×K}
#      between i's K views and j's K views.
#    - Row margins: T·1 = u (how much each view of i is matched)
#    - Column margins: T^T·1 = v (how much each view of j is used)
#    - With uniform margins (u=v=1/K), every view contributes equally.
#    - Minimises sum_{k,k'} C_{k,k'} * T_{k,k'}
#      where C_{k,k'} = 1 - cos(view_i_k, view_j_k') is the COST.
#    - Solved iteratively via Sinkhorn-Knopp (log-domain for stability).
#
#  After finding T, the trajectory similarity is:
#      traj_sim = sum_{k,k'} T_{k,k'} * (1 - C_{k,k'})    (E,) ∈ [-1,1]
#               = sum_{k,k'} T_{k,k'} * cos(vi_k, vj_k')
#  This is the EXPECTED cosine similarity under the optimal transport plan —
#  a true measure of how "alignable" the two trajectories are.
# ═══════════════════════════════════════════════════════════════════════════════

class StructuralTrustModule(nn.Module):
    """
    K-view trajectory trust via Sinkhorn Optimal Transport.

    Args:
        dim         : embedding dimension
        window      : K, number of stored snapshots
        t_heads     : temporal resolution heads for TemporalPositionalEncoding
        sinkhorn_eps: entropy regularisation (smaller = sharper assignment)
        sinkhorn_iters: number of Sinkhorn iterations (5-10 is usually enough)
    """
    def __init__(self, dim: int, window: int = 8,
                 t2v_dim: int = 16,          # kept for API compat, unused
                 t_heads: int = 4,
                 sinkhorn_eps: float = 0.05,
                 sinkhorn_iters: int = 7):
        super().__init__()
        self.window         = window
        self.sinkhorn_eps   = sinkhorn_eps
        self.sinkhorn_iters = sinkhorn_iters

        # Per-head multi-resolution temporal positional encoding
        self.time_pe = TemporalPositionalEncoding(dim, t_heads=t_heads)

        # Lightweight projection to compare views in a lower-dim space
        # This decouples the comparison space from the full hidden dim,
        # reducing computation in the K×K cost matrix.
        proj_dim = max(dim // 2, 32)
        self.proj = nn.Linear(dim, proj_dim, bias=False)

        # Maps OT similarity scalar → trust logit
        self.trust_head = nn.Sequential(
            nn.Linear(1, max(dim // 8, 8)),
            nn.GELU(),
            nn.Linear(max(dim // 8, 8), 1),
        )

    # ──────────────────────────────────────────────────────────────────────────
    @staticmethod
    def _sinkhorn_log(log_C: torch.Tensor,
                      eps: float, n_iter: int) -> torch.Tensor:
        """
        Log-domain Sinkhorn: finds the optimal transport plan T for cost C.

        log_C : (B, K, K)  log of cost matrix (here log(1 - cos_sim + ε))
        Returns log_T : (B, K, K)  log of transport plan

        Log-domain is numerically stable for small eps.
        The algorithm alternates normalising rows and columns of log_T.
        """
        B, K, _ = log_C.shape
        log_a = torch.full((B, K), -math.log(K),
                           device=log_C.device, dtype=log_C.dtype)  # uniform source
        log_b = torch.full((B, K), -math.log(K),
                           device=log_C.device, dtype=log_C.dtype)  # uniform target

        log_T = -log_C / eps    # initialise with Gibbs kernel
        for _ in range(n_iter):
            # Row normalisation: T·1 = a
            log_T = log_T - torch.logsumexp(log_T, dim=2, keepdim=True) + log_a.unsqueeze(2)
            # Column normalisation: T^T·1 = b
            log_T = log_T - torch.logsumexp(log_T, dim=1, keepdim=True) + log_b.unsqueeze(1)
        return log_T   # (B, K, K)

    # ──────────────────────────────────────────────────────────────────────────
    def forward(
        self,
        embs_i: torch.Tensor,   # (E, K, dim)  last K snapshot embeddings of i
        embs_j: torch.Tensor,   # (E, K, dim)  last K snapshot embeddings of j
        t_i:    torch.Tensor,   # (E, K)       timestamps of i's K snapshots
        t_j:    torch.Tensor,   # (E, K)       timestamps of j's K snapshots
        eps:    float = 1e-6,
    ) -> torch.Tensor:          # (E,) ∈ (0, 2)

        E, K, D = embs_i.shape

        # Step 1: per-head multi-resolution temporal PE
        # Fast heads emphasise recency, slow heads treat all views equally.
        vi = self.time_pe(embs_i, t_i)   # (E, K, dim)
        vj = self.time_pe(embs_j, t_j)

        # Step 2: project to lower-dim comparison space and normalise
        vi_p = F.normalize(self.proj(vi.reshape(E * K, D)),
                           dim=-1, eps=eps).view(E, K, -1)           # (E, K, proj_dim)
        vj_p = F.normalize(self.proj(vj.reshape(E * K, D)),
                           dim=-1, eps=eps).view(E, K, -1)

        # Step 3: full K×K pairwise cosine similarity
        sim = torch.bmm(vi_p, vj_p.transpose(1, 2))                 # (E, K, K)

        # Step 4: Sinkhorn OT — finds globally optimal soft assignment
        log_C = torch.log((1.0 - sim).clamp(min=1e-6))
        log_T = self._sinkhorn_log(log_C, self.sinkhorn_eps,
                                   self.sinkhorn_iters)
        T = log_T.exp()                                              # (E, K, K)

        # Step 5: OT-weighted expected cosine similarity
        traj_sim = (T * sim).sum(dim=(1, 2))                         # (E,) ∈ [-1, 1]

        # Step 6: trust ∈ (0, 2)
        logit = self.trust_head(traj_sim.unsqueeze(-1)).squeeze(-1)
        return 1.0 + torch.tanh(logit)


# ═══════════════════════════════════════════════════════════════════════════════
# 8.  MessageTemporalEncoding  — enhanced per-head RoPE + Fourier + decay gate
#
#  Applied to live edge messages (E, dim) before aggregation at node i.
#
#  Four components, all applied together:
#
#  (a) TIMESTAMP NORMALISATION
#      Raw timestamps can be arbitrarily large absolute values (e.g. Unix time).
#      Large t values cause RoPE angles to wrap many times, losing information.
#      We apply learnable normalisation:  t_norm = (t - mu) / (sigma + ε)
#      where mu and sigma are EMA statistics updated during training.
#      This keeps angles in a well-behaved range regardless of the time scale.
#
#  (b) PER-HEAD RoPE
#      Message is split into t_heads segments. Each segment is rotated by
#      RoPE with a head-specific timescale sub-range. Some heads capture
#      fine-grained recency (hourly), others capture long-range patterns.
#
#  (c) MULTI-SCALE FOURIER (additive, shared)
#      Absolute temporal position signal via sin/cos of log-uniform frequencies.
#
#  (d) PER-HEAD TEMPORAL DECAY GATE
#      g_h = σ( -λ_h * |t_norm| + b_h )   ∈ (0, 1)
#      Each head has a learned decay rate λ_h. Fast heads (large λ) strongly
#      down-weight messages that are "old" relative to the mean timestamp.
#      Slow heads weight all messages more uniformly.
#      The gated message for head h: g_h * rope_h(msg_h) + (1-g_h) * msg_h
#      This is a soft interpolation between time-rotated and original content,
#      letting slow heads preserve content while fast heads inject recency.
# ═══════════════════════════════════════════════════════════════════════════════

class MessageTemporalEncoding(nn.Module):
    """
    Per-head multi-resolution RoPE + Fourier + decay gate for edge messages.

    Args:
        dim       : message dimension (even, divisible by t_heads)
        t_heads   : number of temporal resolution heads
        n_fourier : Fourier frequency bands (sin+cos each)
        ema_decay : momentum for running timestamp statistics
    """
    def __init__(self, dim: int, t_heads: int = 4, n_fourier: int = 16,
                 ema_decay: float = 0.99):
        super().__init__()
        assert dim % 2 == 0 and dim % t_heads == 0
        self.dim      = dim
        self.t_heads  = t_heads
        self.head_d   = dim // t_heads
        assert self.head_d % 2 == 0

        # ── (a) Learnable timestamp normalisation ─────────────────────────────
        # Running statistics for EMA normalisation of raw timestamps
        self.ema_decay = ema_decay
        self.register_buffer("t_mean", torch.zeros(1))
        self.register_buffer("t_var",  torch.ones(1))
        # Learnable affine rescale after normalisation
        self.t_scale = nn.Parameter(torch.ones(1))
        self.t_shift = nn.Parameter(torch.zeros(1))

        # ── (b) Per-head RoPE timescales ──────────────────────────────────────
        # Shorter range for messages: angles of normalised timestamps
        rope_params = []
        T_MIN, T_MAX = 0.0, 4.0   # timescales 1 → e^4 ≈ 55 (on normalised scale)
        for h in range(t_heads):
            lo = T_MIN + h       * (T_MAX - T_MIN) / t_heads
            hi = T_MIN + (h + 1) * (T_MAX - T_MIN) / t_heads
            rope_params.append(torch.linspace(lo, hi, self.head_d // 2))
        self.rope_log_ts = nn.Parameter(torch.stack(rope_params, dim=0))  # (H, hd//2)

        # ── (c) Multi-scale Fourier (shared, additive) ────────────────────────
        freqs = torch.logspace(-3, 1, n_fourier)
        self.register_buffer("fourier_freqs", freqs)
        self.fourier_proj = nn.Linear(2 * n_fourier, dim)

        # ── (d) Per-head temporal decay gate ──────────────────────────────────
        # λ_h: learned decay rate per head; fast heads init to larger values
        init_log_decay = torch.linspace(1.5, -0.5, t_heads)  # head 0 = fastest
        self.log_decay  = nn.Parameter(init_log_decay)        # (t_heads,)
        self.decay_bias = nn.Parameter(torch.ones(t_heads))   # (t_heads,)

    @torch.no_grad()
    def _update_stats(self, t: torch.Tensor):
        """EMA update of running mean and variance of timestamps."""
        mu  = t.float().mean()
        var = t.float().var().clamp(min=1e-4)
        self.t_mean = self.ema_decay * self.t_mean + (1 - self.ema_decay) * mu
        self.t_var  = self.ema_decay * self.t_var  + (1 - self.ema_decay) * var

    def _normalise_t(self, t: torch.Tensor) -> torch.Tensor:
        """Normalise timestamps to zero-mean, unit-variance, then rescale."""
        t_norm = (t.float() - self.t_mean) / (self.t_var.sqrt() + 1e-6)
        return self.t_scale * t_norm + self.t_shift      # (E,)

    def forward(self,
                msg: torch.Tensor,   # (E, dim)
                t:   torch.Tensor,   # (E,)   raw edge timestamps
                ) -> torch.Tensor:   # (E, dim)
        E = msg.size(0)

        # Update running stats during training
        if self.training:
            self._update_stats(t)

        # Normalise timestamps
        t_n = self._normalise_t(t)                                   # (E,)

        # ── (b) Per-head RoPE on normalised timestamps ────────────────────────
        rope_parts = []
        for h in range(self.t_heads):
            sl  = msg[:, h * self.head_d : (h + 1) * self.head_d]   # (E, hd)
            ts  = self.rope_log_ts[h].exp()                          # (hd//2,)
            ang = t_n.unsqueeze(-1) / ts                             # (E, hd//2)
            cos_a, sin_a = ang.cos(), ang.sin()
            m_even = sl[:, 0::2]
            m_odd  = sl[:, 1::2]
            rot = torch.stack(
                [m_even * cos_a - m_odd * sin_a,
                 m_even * sin_a + m_odd * cos_a], dim=-1
            ).reshape(E, self.head_d)
            rope_parts.append(rot)
        rope_out = torch.cat(rope_parts, dim=-1)                     # (E, dim)

        # ── (c) Multi-scale Fourier (additive, absolute time) ─────────────────
        phi    = t_n.unsqueeze(-1) * self.fourier_freqs              # (E, F)
        feat   = torch.cat([phi.sin(), phi.cos()], dim=-1)           # (E, 2F)
        fourier = self.fourier_proj(feat)                            # (E, dim)

        # ── (d) Per-head temporal decay gate ──────────────────────────────────
        # g_h = σ( -λ_h * |t_norm| + b_h )
        # Fast heads (large λ) strongly gate old messages toward identity.
        lam  = self.log_decay.exp()                                  # (t_heads,)
        gate = torch.sigmoid(
            -lam.unsqueeze(0) * t_n.abs().unsqueeze(-1)
            + self.decay_bias.unsqueeze(0)
        )                                                            # (E, t_heads)
        # Broadcast gate to full dim (each head covers head_d dims)
        gate_full = gate.repeat_interleave(self.head_d, dim=-1)      # (E, dim)
        # Gated blend: interpolate between rotated and original content
        gated_rope = gate_full * rope_out + (1 - gate_full) * msg    # (E, dim)

        return gated_rope + fourier                                   # (E, dim)


# ═══════════════════════════════════════════════════════════════════════════════
# 9.  GRN  — Gated Residual Network (from Temporal Fusion Transformer, Lim 2021)
#
#  A proven building block for temporal modelling. Replaces plain nn.Linear
#  for Q, K, V projections in TemporalTrustAttention.
#
#  GRN(x, c) = LayerNorm( x + W2 · ELU(W1 · [x, c] + b1) ⊙ σ(W_gate · [x, c]) )
#
#  Two important properties:
#    1. CONTEXT GATING: an optional context vector c (here: the node's GRU
#       memory) modulates the output via a sigmoid gate.  This means the
#       projection adapts to the node's historical state without needing
#       a full hypernetwork.
#    2. ELU NONLINEARITY: unlike GELU (smooth everywhere), ELU has exact
#       linearity for positive inputs and a negative saturation floor.
#       This is beneficial for temporal data where some features are truly
#       linear (trend) and others saturate (periodic amplitude).
#    3. SKIP CONNECTION: the residual allows the GRN to be identity-like
#       when no transformation is needed, which helps early training.
# ═══════════════════════════════════════════════════════════════════════════════

class GRN(nn.Module):
    """
    Gated Residual Network projection with optional context conditioning.

    Args:
        in_dim  : input dimension
        out_dim : output dimension
        ctx_dim : context dimension (None = no context gating)
        dropout : dropout before residual add
    """
    def __init__(self, in_dim: int, out_dim: int,
                 ctx_dim: Optional[int] = None, dropout: float = 0.1):
        super().__init__()
        self.in_dim  = in_dim
        self.out_dim = out_dim
        ctx = ctx_dim or 0

        # Main transformation: [x, c] -> hidden -> out
        self.W1 = nn.Linear(in_dim + ctx, out_dim)
        self.W2 = nn.Linear(out_dim, out_dim)

        # Context gate: [x, c] -> gate ∈ (0,1) controlling residual weight
        self.W_gate = nn.Linear(in_dim + ctx, out_dim)

        # Skip connection projection (needed when in_dim ≠ out_dim)
        self.skip = (nn.Linear(in_dim, out_dim, bias=False)
                     if in_dim != out_dim else nn.Identity())

        self.norm    = nn.LayerNorm(out_dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self,
                x:   torch.Tensor,              # (B, in_dim)
                ctx: Optional[torch.Tensor] = None  # (B, ctx_dim)
                ) -> torch.Tensor:              # (B, out_dim)
        if ctx is not None:
            inp = torch.cat([x, ctx], dim=-1)   # (B, in_dim + ctx_dim)
        else:
            inp = x

        # Gated transformation
        h    = F.elu(self.W1(inp))              # (B, out_dim)
        h    = self.dropout(self.W2(h))         # (B, out_dim)
        gate = torch.sigmoid(self.W_gate(inp))  # (B, out_dim) ∈ (0,1)

        # Gated residual: gate blends transformed output with skip
        out = gate * h + (1 - gate) * self.skip(x)
        return self.norm(out)                   # (B, out_dim)


# ═══════════════════════════════════════════════════════════════════════════════
# 10.  TemporalTrustAttention  (one message-passing layer)
#
#  [1] GRN-PROJECTED Q / K / V  (context-conditioned via GRU memory)
#      Q  = GRN(edge_emb, ctx=mem_i)  interaction queries through node history
#      K  = GRN(x_j,      ctx=mem_j)  node j's key adapted to its history
#      V  = GRN(x_j,      ctx=mem_j)  value also history-conditioned
#
#  [2] RoPE APPLIED TO Q AND K BEFORE SCORING
#      This is the mathematically correct placement for RoPE.
#      Rotating Q and K by the edge timestamp means that the dot product
#      Q·K naturally captures relative temporal position:
#          (R_t Q)·(R_t K) = Q^T R_{0} K = Q^T K   (same time: unrotated)
#          (R_{t1} Q)·(R_{t2} K) = Q^T R_{t2-t1} K  (different times: rotated)
#      The score between Q and K reflects BOTH semantic compatibility AND
#      temporal proximity. Temporally distant Q-K pairs will have a
#      phase-shifted dot product, naturally reducing their score.
#      Per-head timescales are used (same as MessageTemporalEncoding).
#
#  [3] PER-HEAD BILINEAR SCORING  (replaces identity dot-product)
#      Standard dot-product: score = Q^T K  (uses identity metric)
#      Bilinear form:        score = Q^T W_h K  (each head learns its own metric)
#      W_h ∈ R^{hd×hd} is a learned per-head compatibility matrix.
#      This allows each head to discover a different notion of compatibility
#      between the edge query and the node key — e.g. one head may focus on
#      frequency features, another on structural role features.
#      To keep computation light, W_h is parameterised as a low-rank outer
#      product: W_h = U_h V_h^T where U_h, V_h ∈ R^{hd×r}, rank r << hd.
#
#  [4] SINKHORN TRUST  (K-view OT only, no h/deg)
#
#  [5] PER-HEAD MULTI-RESOLUTION MESSAGE TEMPORAL ENCODING  (with decay gate)
#
#  [6] TEMPORAL MESSAGE BANK
# ═══════════════════════════════════════════════════════════════════════════════

class TemporalTrustAttention(MessagePassing):
    def __init__(self, dim: int, heads: int = 8, dropout: float = 0.1,
                 window: int = 8, t2v_dim: int = 16,
                 num_nodes: int = 0, bank_size: int = 8,
                 t_heads: int = 4, bilinear_rank: int = 8):
        super().__init__(aggr="add", node_dim=0)
        assert dim % heads == 0
        self.dim        = dim
        self.heads      = heads
        self.head_dim   = dim // heads
        self.t_heads    = t_heads

        # ── [1] GRN projections (context-conditioned) ─────────────────────────
        self.q_grn  = GRN(dim, dim, ctx_dim=dim, dropout=dropout)
        self.k_grn  = GRN(dim, dim, ctx_dim=dim, dropout=dropout)
        self.v_grn  = GRN(dim, dim, ctx_dim=dim, dropout=dropout)
        self.o_proj = nn.Linear(dim, dim)

        # ── [2] Per-head RoPE on Q and K ─────────────────────────────────────
        # Timescales shared with message encoding for consistency
        # Each head gets its own sub-range of [0, 4] on normalised timestamps
        rope_params = []
        T_MIN, T_MAX = 0.0, 4.0
        hd = self.head_dim
        assert hd % 2 == 0
        for h in range(heads):
            lo = T_MIN + h       * (T_MAX - T_MIN) / heads
            hi = T_MIN + (h + 1) * (T_MAX - T_MIN) / heads
            rope_params.append(torch.linspace(lo, hi, hd // 2))
        self.qk_rope_log_ts = nn.Parameter(torch.stack(rope_params, dim=0))  # (H, hd//2)

        # Running timestamp stats for normalisation (shared with MessageTemporalEncoding)
        self.register_buffer("t_mean_qk", torch.zeros(1))
        self.register_buffer("t_var_qk",  torch.ones(1))
        self.t_scale_qk = nn.Parameter(torch.ones(1))
        self.t_shift_qk = nn.Parameter(torch.zeros(1))
        self.ema_decay = 0.99

        # ── [3] Per-head bilinear scoring (low-rank) ──────────────────────────
        # W_h = U_h @ V_h^T  where U_h, V_h ∈ R^{hd × r}
        # Parameterised as two (heads, hd, rank) tensors.
        # score_h = Q_h^T W_h K_h = (Q_h^T U_h)(V_h^T K_h) — factored efficiently.
        r = bilinear_rank
        self.bilin_U = nn.Parameter(torch.randn(heads, hd, r) * (r ** -0.5))
        self.bilin_V = nn.Parameter(torch.randn(heads, hd, r) * (r ** -0.5))
        self.scale   = r ** -0.5   # normalise by rank, not head_dim

        # ── [4] Sinkhorn trust (K-view only) ─────────────────────────────────
        self.trust = StructuralTrustModule(dim, window=window, t_heads=t_heads)

        # ── [5] Message temporal encoding (with decay gate) ───────────────────
        self.msg_time_enc = MessageTemporalEncoding(dim, t_heads=t_heads, n_fourier=16)

        # ── [6] Temporal message bank ─────────────────────────────────────────
        self.msg_bank = (TemporalMessageBank(num_nodes, dim, bank_size=bank_size)
                         if num_nodes > 0 else None)

        self.dropout = nn.Dropout(dropout)
        self.gate    = nn.Linear(dim, 1)
        self.norm    = nn.LayerNorm(dim)
        self.fusion  = nn.Sequential(
            nn.Linear(dim * 2, dim),
            nn.GELU(),
            nn.Linear(dim, dim),
        )
        self._trust_cache: Optional[torch.Tensor] = None

    # ──────────────────────────────────────────────────────────────────────────
    @torch.no_grad()
    def _update_t_stats(self, t: torch.Tensor):
        mu  = t.float().mean()
        var = t.float().var().clamp(min=1e-4)
        self.t_mean_qk = self.ema_decay * self.t_mean_qk + (1 - self.ema_decay) * mu
        self.t_var_qk  = self.ema_decay * self.t_var_qk  + (1 - self.ema_decay) * var

    def _normalise_t(self, t: torch.Tensor) -> torch.Tensor:
        t_n = (t.float() - self.t_mean_qk) / (self.t_var_qk.sqrt() + 1e-6)
        return self.t_scale_qk * t_n + self.t_shift_qk

    def _apply_rope_to_qk(self,
                           qk:  torch.Tensor,   # (E, heads, head_dim)
                           t_n: torch.Tensor,   # (E,)  normalised timestamps
                           ) -> torch.Tensor:   # (E, heads, head_dim)
        """
        Apply per-head RoPE to Q or K using normalised edge timestamps.
        Each head h rotates its head_dim vector by its own timescale band.
        """
        E = qk.size(0)
        parts = []
        for h in range(self.heads):
            sl  = qk[:, h, :]                                        # (E, hd)
            ts  = self.qk_rope_log_ts[h].exp()                       # (hd//2,)
            ang = t_n.unsqueeze(-1) / ts                             # (E, hd//2)
            cos_a, sin_a = ang.cos(), ang.sin()
            e_even = sl[:, 0::2]
            e_odd  = sl[:, 1::2]
            rot = torch.stack(
                [e_even * cos_a - e_odd * sin_a,
                 e_even * sin_a + e_odd * cos_a], dim=-1
            ).reshape(E, self.head_dim)
            parts.append(rot)
        return torch.stack(parts, dim=1)                             # (E, H, hd)

    # ──────────────────────────────────────────────────────────────────────────
    def forward(self,
                x:          torch.Tensor,   # (N, dim)
                edge_index: torch.Tensor,   # (2, E)
                edge_emb:   torch.Tensor,   # (E, dim)
                time_emb:   torch.Tensor,   # (E, dim) unused, kept for API
                edge_t:     torch.Tensor,   # (E,)  raw timestamps
                memory:     RecurrentMemory,
                evo_bank:   EvolutionBank,
                deg:        torch.Tensor,   # (N,)
                ) -> torch.Tensor:

        N    = x.size(0)
        nidx = torch.arange(N, device=x.device)
        src  = edge_index[0]

        # Update running timestamp statistics
        if self.training:
            self._update_t_stats(edge_t)
        t_n = self._normalise_t(edge_t)                             # (E,)

        mem_vals, _ = memory.read(nidx)
        embs_all, times_all = evo_bank.read(nidx)

        # GRN-projected Q/K/V
        mem_src = mem_vals[src]
        Q = self.q_grn(edge_emb, mem_src).view(-1, self.heads, self.head_dim)
        K = self.k_grn(x, mem_vals).view(N, self.heads, self.head_dim)
        V = self.v_grn(x, mem_vals).view(N, self.heads, self.head_dim)

        # Apply RoPE to Q (edge-level, already (E, H, hd))
        Q_rope = self._apply_rope_to_qk(Q, t_n)                     # (E, H, hd)
        # K will be indexed as K_j inside message() — apply RoPE there

        out = self.propagate(
            edge_index,
            x=x,
            Q_rope=Q_rope,
            K=K,
            V=V,
            mem_val=mem_vals,
            embs=embs_all,
            times=times_all,
            edge_t=edge_t,
            t_n=t_n,
            size=(N, N),
        )
        out = out.view(N, self.dim)

        # Temporal message bank
        if self.msg_bank is not None:
            out = self.msg_bank.read_and_attend(nidx, out)
            self.msg_bank.write(nidx, out)

        # Fuse with GRU memory + gated residual
        h   = memory.get_hidden(nidx)
        out = self.fusion(torch.cat([out, h], dim=-1))
        out = self.o_proj(out)
        g   = torch.sigmoid(self.gate(x))
        return self.norm(g * x + (1 - g) * out)

    # ──────────────────────────────────────────────────────────────────────────
    def message(self,
                x_i:      torch.Tensor,   # (E, dim)
                x_j:      torch.Tensor,
                Q_rope:   torch.Tensor,   # (E, H, hd)  Q already RoPE-encoded
                K_j:      torch.Tensor,   # (E, H, hd)  K from node j (raw)
                V_j:      torch.Tensor,   # (E, H, hd)
                mem_val_i: torch.Tensor,  # (E, dim)
                mem_val_j: torch.Tensor,
                embs_i:   torch.Tensor,   # (E, K, dim)
                embs_j:   torch.Tensor,
                times_i:  torch.Tensor,   # (E, K)
                times_j:  torch.Tensor,
                edge_t:   torch.Tensor,   # (E,)
                t_n:      torch.Tensor,   # (E,)  normalised timestamps
                index:    torch.Tensor,
                ) -> torch.Tensor:

        E = x_i.size(0)

        # ── [2] Apply RoPE to K_j using normalised timestamps ─────────────────
        # After rotation, Q_rope · K_j_rope reflects both semantic match AND
        # temporal proximity between the edge time and j's contribution time.
        K_j_rope = self._apply_rope_to_qk(K_j, t_n)                 # (E, H, hd)

        # ── [3] Per-head bilinear score: score_h = (Q^T U_h)(V_h^T K) ────────
        # Factored bilinear: avoids materialising the full hd×hd matrix.
        # Q^T U_h : (E, r)   and   V_h^T K : (E, r)
        # score_h = dot product of these two r-dim vectors.
        scores = torch.zeros(E, self.heads, device=x_i.device)
        for h in range(self.heads):
            q_h = Q_rope[:, h, :]                                    # (E, hd)
            k_h = K_j_rope[:, h, :]                                  # (E, hd)
            U_h = self.bilin_U[h]                                    # (hd, r)
            V_h = self.bilin_V[h]                                    # (hd, r)
            # (E, r) · (E, r) summed → (E,)
            scores[:, h] = ((q_h @ U_h) * (k_h @ V_h)).sum(-1) * self.scale

        # ── [4] Sinkhorn trust (K-view only) ─────────────────────────────────
        trust = self.trust(
            embs_i=embs_i, embs_j=embs_j,
            t_i=times_i,   t_j=times_j,
        )                                                            # (E,) ∈ (0,2)
        self._trust_cache = trust.detach()

        # Trust-scaled softmax
        scores = scores * trust.unsqueeze(-1)
        scores = softmax(scores, index)
        scores = self.dropout(scores)

        # ── [5] Value + multi-resolution temporal message encoding ────────────
        msg = (V_j * scores.unsqueeze(-1)).reshape(E, self.dim)
        msg = self.msg_time_enc(msg, edge_t)                         # (E, dim)
        return msg
        self.q_proj  = nn.Linear(dim, dim)
        self.kv_proj = nn.Linear(dim, dim * 2)  # fused K and V projection

        # Single scalar from attended similarity → trust logit
        # Input: traj_sim scalar (1,) → logit
        self.trust_head = nn.Sequential(
            nn.Linear(1, max(dim // 8, 8)),
            nn.GELU(),
            nn.Linear(max(dim // 8, 8), 1),
        )

    def forward(
        self,
        h_i:    torch.Tensor,   # (E, dim)   kept for API compatibility, unused
        h_j:    torch.Tensor,   # (E, dim)   kept for API compatibility, unused
        deg_i:  torch.Tensor,   # (E,)       kept for API compatibility, unused
        deg_j:  torch.Tensor,   # (E,)       kept for API compatibility, unused
        embs_i: torch.Tensor,   # (E, K, dim)
        embs_j: torch.Tensor,   # (E, K, dim)
        t_i:    torch.Tensor,   # (E, K)
        t_j:    torch.Tensor,   # (E, K)
        eps:    float = 1e-6,
    ) -> torch.Tensor:           # (E,) ∈ (0, 2)

        E, K, D = embs_i.shape

        # ── Step 1: apply temporal positional encoding to both sequences ──────
        # Each view gets: g_k * RoPE(emb_k, t_k) + Fourier(t_k)
        # The recency gate g_k ensures the model focuses on recent views.
        vi = self.time_pe(embs_i, t_i)   # (E, K, dim)
        vj = self.time_pe(embs_j, t_j)   # (E, K, dim)

        # ── Step 2: multi-head cross-attention  (i queries j) ─────────────────
        # Q from i's time-encoded views:  what has i experienced?
        # K, V from j's time-encoded views: what has j experienced?
        #
        # For each of i's K states, the attention finds which of j's K states
        # is most structurally and temporally compatible.
        # Because views are RoPE-encoded, attention weights naturally decay
        # for pairs that are far apart in time.
        Q  = self.q_proj(vi.reshape(E * K, D)).view(E, K, self.attn_heads, self.head_dim)
        KV = self.kv_proj(vj.reshape(E * K, D)).view(E, K, 2, self.attn_heads, self.head_dim)
        Kj = KV[:, :, 0]   # (E, K, heads, hd)
        Vj = KV[:, :, 1]   # (E, K, heads, hd)

        # Transpose to (E, heads, K, hd) for batched matmul
        Q  = Q.permute(0, 2, 1, 3)   # (E, heads, K_i, hd)
        Kj = Kj.permute(0, 2, 1, 3)  # (E, heads, K_j, hd)
        Vj = Vj.permute(0, 2, 1, 3)  # (E, heads, K_j, hd)

        # Attention scores: (E, heads, K_i, K_j)
        attn = torch.matmul(Q, Kj.transpose(-2, -1)) * self.scale
        attn = F.softmax(attn, dim=-1)

        # Attended output: (E, heads, K_i, hd) -> (E, K_i, dim)
        attended = torch.matmul(attn, Vj)              # (E, heads, K_i, hd)
        attended = attended.permute(0, 2, 1, 3).reshape(E, K, D)

        # ── Step 3: trajectory similarity ────────────────────────────────────
        # How well does j's trajectory explain i's trajectory?
        # High similarity → i and j have been evolving the same way.
        vi_n   = F.normalize(vi,       dim=-1, eps=eps)  # (E, K, dim)
        att_n  = F.normalize(attended, dim=-1, eps=eps)  # (E, K, dim)
        traj_sim = (vi_n * att_n).sum(-1).mean(-1)       # (E,) ∈ [-1, 1]

        # ── Step 4: map to trust ∈ (0, 2) ────────────────────────────────────
        logit = self.trust_head(traj_sim.unsqueeze(-1)).squeeze(-1)  # (E,)
        return 1.0 + torch.tanh(logit)                               # (E,) ∈ (0, 2)


# ═══════════════════════════════════════════════════════════════════════════════
# 8.  MessageRoPE  — applies continuous-time RoPE directly to message vectors
#
#  Unlike the TemporalPositionalEncoding used in the trust module (which
#  operates on stored K-view snapshots), this module applies RoPE to the
#  LIVE message vector being passed from j to i along an edge, using the
#  edge's own timestamp as the rotation angle.
#
#  Why apply RoPE to messages (not just to queries/keys)?
#  ────────────────────────────────────────────────────────
#  In standard transformer attention, RoPE is applied to Q and K so that
#  the dot product Q·K reflects relative position.  Here we go further:
#  the VALUE (message content) is also rotated by the edge timestamp.
#  This means:
#    - Two messages with the same content but different timestamps will
#      aggregate to DIFFERENT vectors at the target node.
#    - The target node's representation is intrinsically time-aware,
#      not just score-aware.
#    - Recent messages occupy a different angular region from old ones,
#      so the GRU memory update naturally distinguishes temporal recency.
#
#  Shared log_timescales with the trust module's TemporalPositionalEncoding
#  are NOT used — this module has its own learned timescales tuned for
#  the message aggregation regime (shorter timescales work better here
#  since edge timestamps are absolute, not relative gaps).
# ═══════════════════════════════════════════════════════════════════════════════

class MessageRoPE(nn.Module):
    """
    Applies continuous-time RoPE to a batch of message vectors.

    Rotates pairs of dimensions by angles proportional to the edge timestamp:
        theta_d = t_edge / exp(log_timescale_d)
        msg[2d]   ->  msg[2d]   * cos(theta_d) - msg[2d+1] * sin(theta_d)
        msg[2d+1] ->  msg[2d]   * sin(theta_d) + msg[2d+1] * cos(theta_d)

    The dot product between two RoPE-encoded messages naturally reflects
    their temporal distance: same-time messages stay aligned, cross-time
    messages rotate apart in proportion to the timestamp gap.
    """
    def __init__(self, dim: int):
        super().__init__()
        assert dim % 2 == 0
        # Shorter timescales than K-view RoPE — edge-level granularity
        self.log_timescales = nn.Parameter(
            torch.linspace(0, 6, dim // 2)    # timescales: 1 → e^6 ≈ 400
        )

    def forward(self,
                msg: torch.Tensor,   # (E, dim)
                t:   torch.Tensor,   # (E,)   edge timestamps
                ) -> torch.Tensor:   # (E, dim)
        timescales = self.log_timescales.exp()              # (D//2,)
        angles     = t.float().unsqueeze(-1) / timescales  # (E, D//2)

        cos_a = angles.cos()   # (E, D//2)
        sin_a = angles.sin()

        m_even = msg[:, 0::2]  # (E, D//2)
        m_odd  = msg[:, 1::2]

        rot_even = m_even * cos_a - m_odd * sin_a
        rot_odd  = m_even * sin_a + m_odd * cos_a

        # Interleave back to (E, D)
        rotated = torch.stack([rot_even, rot_odd], dim=-1)  # (E, D//2, 2)
        return rotated.reshape(msg.shape)


# ═══════════════════════════════════════════════════════════════════════════════
# 9.  TemporalMessageBank  — NEW ADVANCED MECHANISM
#
#  Standard message passing aggregates ALL neighbours in one flat sum/mean.
#  This loses two things:
#    1. TEMPORAL ORDER: messages from t=10 and t=1000 are mixed equally.
#    2. TEMPORAL CONTEXT: the node has no memory of what it received before.
#
#  The TemporalMessageBank fixes both:
#    - After each forward pass, the aggregated message vector (already
#      RoPE-encoded) is stored in a per-node circular buffer of size M.
#    - Before the final update, a cross-attention reads over this buffer:
#        Query  = current aggregated message  (what I just received)
#        Key/V  = past M buffered messages    (what I've received before)
#      This lets the node ask: "is what I'm receiving NOW consistent with
#      my recent message history, or is it a sudden anomalous change?"
#    - The attention output is a temporally-smoothed message that blends
#      current and historical incoming signals.
#
#  Effect on anomaly detection:
#    Normal nodes receive consistent messages over time → strong cross-attn
#    Anomalous nodes receive erratic messages           → weak cross-attn
#    Central node near anomaly gets a divergent current msg vs history
#    → the bank effectively provides a "temporal message baseline"
# ═══════════════════════════════════════════════════════════════════════════════

class TemporalMessageBank(nn.Module):
    """
    Per-node circular buffer of past aggregated messages with
    cross-attention retrieval.

    Args:
        num_nodes : total nodes in the graph
        dim       : message / hidden dimension
        bank_size : M, number of past messages to store per node
    """
    def __init__(self, num_nodes: int, dim: int, bank_size: int = 8):
        super().__init__()
        self.bank_size = bank_size
        self.dim       = dim
        # Stored past messages per node: (num_nodes, M, dim)
        self.register_buffer("bank", torch.zeros(num_nodes, bank_size, dim))
        self.register_buffer("ptr",  torch.zeros(num_nodes, dtype=torch.long))

        # Cross-attention: current msg as Q, past bank as K and V
        self.q_proj = nn.Linear(dim, dim)
        self.k_proj = nn.Linear(dim, dim)
        self.v_proj = nn.Linear(dim, dim)
        self.scale  = dim ** -0.5
        self.out_proj = nn.Linear(dim, dim)
        self.norm     = nn.LayerNorm(dim)

    @torch.no_grad()
    def write(self, idx: torch.Tensor, msg: torch.Tensor):
        """Write current aggregated message into the circular buffer."""
        for b in range(idx.size(0)):
            node = int(idx[b])
            p    = int(self.ptr[node]) % self.bank_size
            self.bank[node, p] = msg[b].detach()
            self.ptr[node] += 1

    def read_and_attend(self,
                        idx:     torch.Tensor,   # (N,)
                        cur_msg: torch.Tensor,   # (N, dim)
                        ) -> torch.Tensor:       # (N, dim)
        """
        Cross-attention: current message queries the historical bank.

        Q = Linear(cur_msg)          (N, dim)
        K = Linear(bank[idx])        (N, M, dim)
        V = Linear(bank[idx])        (N, M, dim)

        output = softmax(Q K^T / sqrt(d)) V    (N, dim)

        Then blend: alpha * output + (1-alpha) * cur_msg
        where alpha is learned (starts near 0.5).
        """
        past = self.bank[idx]                                        # (N, M, dim)
        N, M, D = past.shape

        Q  = self.q_proj(cur_msg).unsqueeze(1)                       # (N, 1, dim)
        K  = self.k_proj(past)                                       # (N, M, dim)
        V  = self.v_proj(past)

        # Scaled dot-product attention over M past messages
        attn = torch.bmm(Q, K.transpose(1, 2)) * self.scale         # (N, 1, M)
        attn = F.softmax(attn, dim=-1)
        ctx  = torch.bmm(attn, V).squeeze(1)                        # (N, dim)

        # Residual: blend retrieved history with current message
        out = self.out_proj(ctx)
        return self.norm(cur_msg + out)                              # (N, dim)

    @torch.no_grad()
    def reset(self):
        self.bank.zero_()
        self.ptr.zero_()


# ═══════════════════════════════════════════════════════════════════════════════
# 10.  TemporalTrustAttention  (one message-passing layer)
#
#  Upgrades vs vanilla attention:
#
#  [1] MEMORY-CONDITIONED Q/K/V  (Hyper-network projections)
#      Standard attention uses fixed weight matrices W_Q, W_K, W_V.
#      Here, the projection weights are DYNAMICALLY generated from each
#      node's GRU memory state — inspired by hypernetwork attention used
#      in DyGFormer and CAW.
#
#      For each node i:
#        ΔW_Q(i) = reshape( HyperNet_Q( memory_i ) )   (dim × dim)
#        W_Q(i)  = W_Q_base + ΔW_Q(i)
#      Then: Q_i = x_i @ W_Q(i)^T
#
#      This means nodes with different histories produce different Q/K/V
#      projections — two nodes with the same current features but different
#      pasts will attend differently.  Anomalous nodes, whose memory is
#      consistently erratic, develop distinctive projection patterns that
#      separate them from normal nodes without supervision.
#
#      To keep it efficient, the hyper-net produces a LOW-RANK delta:
#        ΔW = U @ V^T   where U ∈ R^{dim×r}, V ∈ R^{dim×r}, r << dim
#      This avoids full dim×dim output from the hyper-net.
#
#  [2] CROSS-SOURCE ATTENTION (retained from previous version)
#      Q from EDGE (interaction type), K from NODE (neighbour identity).
#      The hyper-net conditions the edge-Q projection on the central
#      node's memory, making the query node-specific.
#
#  [3] RoPE ON MESSAGES + Fourier additive (MessageTemporalEncoding)
#      Upgraded: RoPE rotates the message, multi-scale Fourier is added.
#      Same design philosophy as TemporalPositionalEncoding but applied
#      to live message vectors at edge-level granularity.
#
#  [4] SIMILARITY-BASED TRUST + TEMPORAL MESSAGE BANK  (unchanged)
# ═══════════════════════════════════════════════════════════════════════════════

class HyperProjection(nn.Module):
    """
    Generates a low-rank delta ΔW = U @ V^T conditioned on a context vector,
    then applies (W_base + ΔW) @ x^T to project x.

    This is a lightweight hypernetwork: instead of a full dim×dim weight matrix
    per node, we produce rank-r perturbations from the node's memory state.

    Args:
        dim   : input and output dimension
        rank  : rank of the delta (r << dim, default 8)
        ctx_dim: dimension of the conditioning context (memory state)
    """
    def __init__(self, dim: int, rank: int = 8, ctx_dim: Optional[int] = None):
        super().__init__()
        ctx = ctx_dim or dim
        self.dim  = dim
        self.rank = rank

        # Base (shared) projection — standard Linear
        self.W_base = nn.Linear(dim, dim, bias=True)

        # Hyper-net: context -> (U, V) for low-rank delta
        # Output: dim*rank for U, dim*rank for V
        self.hyper = nn.Sequential(
            nn.Linear(ctx, dim),
            nn.GELU(),
            nn.Linear(dim, 2 * dim * rank),
        )

    def forward(self,
                x:   torch.Tensor,   # (B, dim)  vectors to project
                ctx: torch.Tensor,   # (B, ctx_dim)  conditioning context
                ) -> torch.Tensor:   # (B, dim)
        B = x.size(0)

        # Base projection
        out_base = self.W_base(x)                                    # (B, dim)

        # Low-rank delta from hyper-net
        uv   = self.hyper(ctx)                                       # (B, 2*dim*rank)
        U, V = uv.split(self.dim * self.rank, dim=-1)
        U    = U.view(B, self.dim, self.rank)                        # (B, dim, r)
        V    = V.view(B, self.dim, self.rank)                        # (B, dim, r)
        # ΔW = U @ V^T  shape (B, dim, dim)
        # Apply: x @ ΔW^T = x @ V @ U^T
        delta = (x.unsqueeze(1) @ V).squeeze(1) @ U.transpose(1, 2).mean(0)
        # Efficient form: x (B,dim) @ V (B,dim,r) -> (B,r) @ U^T mean (r,dim) -> (B,dim)
        x_proj = (x.unsqueeze(1) @ V).squeeze(1)                    # (B, r)
        # Average U across batch for the delta (memory-conditioned direction)
        delta  = x_proj @ U.mean(0).t()                             # (B, dim)

        return out_base + delta                                      # (B, dim)


class MessageTemporalEncoding(nn.Module):
    """
    Applies RoPE + multi-scale Fourier to live edge message vectors.
    Same design as TemporalPositionalEncoding but for 2D (E, dim) messages.

    Args:
        dim       : message dimension
        n_fourier : number of Fourier frequency bands
    """
    def __init__(self, dim: int, n_fourier: int = 16):
        super().__init__()
        assert dim % 2 == 0
        self.dim = dim

        # RoPE timescales (shorter range for edge-level timestamps)
        self.rope_log_ts = nn.Parameter(
            torch.linspace(0.0, 6.0, dim // 2)    # 1 → e^6 ≈ 400
        )

        # Multi-scale Fourier (fixed freqs, learned projection)
        freqs = torch.logspace(-3, 1, n_fourier)
        self.register_buffer("fourier_freqs", freqs)
        self.fourier_proj = nn.Linear(2 * n_fourier, dim)

    def forward(self,
                msg: torch.Tensor,   # (E, dim)
                t:   torch.Tensor,   # (E,)
                ) -> torch.Tensor:   # (E, dim)
        # ── RoPE rotation ─────────────────────────────────────────────────────
        ts     = self.rope_log_ts.exp()                              # (D//2,)
        angles = t.float().unsqueeze(-1) / ts                       # (E, D//2)
        cos_a, sin_a = angles.cos(), angles.sin()

        m_even = msg[:, 0::2]
        m_odd  = msg[:, 1::2]
        rot    = torch.stack(
            [m_even * cos_a - m_odd * sin_a,
             m_even * sin_a + m_odd * cos_a], dim=-1
        ).reshape(msg.shape)                                         # (E, dim)

        # ── Multi-scale Fourier (additive) ────────────────────────────────────
        phi  = t.float().unsqueeze(-1) * self.fourier_freqs         # (E, F)
        feat = torch.cat([phi.sin(), phi.cos()], dim=-1)             # (E, 2F)
        fourier = self.fourier_proj(feat)                            # (E, dim)

        return rot + fourier                                         # (E, dim)


class TemporalTrustAttention(MessagePassing):
    def __init__(self, dim: int, heads: int = 8, dropout: float = 0.1,
                 window: int = 8, t2v_dim: int = 16,
                 num_nodes: int = 0, bank_size: int = 8,
                 hyper_rank: int = 8):
        super().__init__(aggr="add", node_dim=0)
        assert dim % heads == 0
        self.dim      = dim
        self.heads    = heads
        self.head_dim = dim // heads
        self.scale    = self.head_dim ** -0.5

        # ── Memory-conditioned hyper-projected Q, K, V ────────────────────────
        # Q: edge embedding conditioned on i's memory
        #    (the query adapts to who i is historically)
        self.q_hyper = HyperProjection(dim, rank=hyper_rank, ctx_dim=dim)
        # K: neighbour node j conditioned on j's memory
        #    (j's key reflects its own historical identity)
        self.k_hyper = HyperProjection(dim, rank=hyper_rank, ctx_dim=dim)
        # V: neighbour node j conditioned on j's memory
        #    (what j sends is shaped by its own history)
        self.v_hyper = HyperProjection(dim, rank=hyper_rank, ctx_dim=dim)

        self.o_proj = nn.Linear(dim, dim)

        # ── Temporal message encoding (RoPE + Fourier on live messages) ───────
        self.msg_time_enc = MessageTemporalEncoding(dim, n_fourier=16)

        # ── Trust ─────────────────────────────────────────────────────────────
        self.trust = StructuralTrustModule(
            dim, window=window, t2v_dim=t2v_dim
        )

        # ── Temporal message bank ─────────────────────────────────────────────
        self.msg_bank = (TemporalMessageBank(num_nodes, dim, bank_size=bank_size)
                         if num_nodes > 0 else None)

        self.dropout = nn.Dropout(dropout)
        self.gate    = nn.Linear(dim, 1)
        self.norm    = nn.LayerNorm(dim)
        self.fusion  = nn.Sequential(
            nn.Linear(dim * 2, dim),
            nn.GELU(),
            nn.Linear(dim, dim),
        )

        self._trust_cache: Optional[torch.Tensor] = None

    # --------------------------------------------------------------------------
    def forward(self,
                x:          torch.Tensor,       # (N, dim)
                edge_index: torch.Tensor,       # (2, E)
                edge_emb:   torch.Tensor,       # (E, dim)
                time_emb:   torch.Tensor,       # (E, dim)  unused, kept for API
                edge_t:     torch.Tensor,       # (E,)  raw timestamps
                memory:     RecurrentMemory,
                evo_bank:   EvolutionBank,
                deg:        torch.Tensor,       # (N,)
                ) -> torch.Tensor:

        N    = x.size(0)
        nidx = torch.arange(N, device=x.device)
        src, dst = edge_index

        mem_vals, _ = memory.read(nidx)                              # (N, dim)
        embs_all, times_all = evo_bank.read(nidx)

        # ── Memory-conditioned Q from edge embedding ──────────────────────────
        # Q is produced from the edge embedding, but conditioned on the
        # CENTRAL node i's memory — so the same edge will produce a different
        # query depending on who i is and what its history looks like.
        mem_src = mem_vals[src]                                      # (E, dim)
        Q_edge = self.q_hyper(edge_emb, mem_src
                              ).view(-1, self.heads, self.head_dim)  # (E, h, hd)

        # K, V from node features — conditioned on each node's own memory
        K_node = self.k_hyper(x, mem_vals).view(N, self.heads, self.head_dim)
        V_node = self.v_hyper(x, mem_vals).view(N, self.heads, self.head_dim)

        out = self.propagate(
            edge_index,
            x=x,
            Q_edge=Q_edge,
            K_node=K_node,
            V_node=V_node,
            mem_val=mem_vals,
            deg=deg,
            embs=embs_all,
            times=times_all,
            edge_t=edge_t,
            size=(N, N),
        )
        out = out.view(N, self.dim)

        # ── Temporal Message Bank ─────────────────────────────────────────────
        if self.msg_bank is not None:
            out = self.msg_bank.read_and_attend(nidx, out)
            self.msg_bank.write(nidx, out)

        # ── Fuse with GRU memory + gated residual ─────────────────────────────
        h   = memory.get_hidden(nidx)
        out = self.fusion(torch.cat([out, h], dim=-1))
        out = self.o_proj(out)

        g = torch.sigmoid(self.gate(x))
        return self.norm(g * x + (1 - g) * out)

    # --------------------------------------------------------------------------
    def message(self,
                x_i:        torch.Tensor,   # (E, dim)
                x_j:        torch.Tensor,   # (E, dim)
                Q_edge:     torch.Tensor,   # (E, heads, head_dim)
                K_node_j:   torch.Tensor,   # (E, heads, head_dim)
                V_node_j:   torch.Tensor,   # (E, heads, head_dim)
                mem_val_i:  torch.Tensor,   # (E, dim)
                mem_val_j:  torch.Tensor,
                deg_i:      torch.Tensor,   # (E,)
                deg_j:      torch.Tensor,
                embs_i:     torch.Tensor,   # (E, K, dim)
                embs_j:     torch.Tensor,
                times_i:    torch.Tensor,   # (E, K)
                times_j:    torch.Tensor,
                edge_t:     torch.Tensor,   # (E,)
                index:      torch.Tensor,
                ) -> torch.Tensor:

        E = x_i.size(0)

        # ── [1] Cross-source attention score (memory-conditioned) ─────────────
        # Q from edge (what does this interaction ask for, given who i is?)
        # K from node j (what does j structurally offer, given its history?)
        scores = (Q_edge * K_node_j).sum(-1) * self.scale            # (E, heads)

        # ── [2] K-view trajectory trust ───────────────────────────────────────
        trust = self.trust(
            h_i=mem_val_i, h_j=mem_val_j,
            deg_i=deg_i,   deg_j=deg_j,
            embs_i=embs_i, embs_j=embs_j,
            t_i=times_i,   t_j=times_j,
        )                                                            # (E,) in (0,2)
        self._trust_cache = trust.detach()

        # Trust-scaled softmax
        scores = scores * trust.unsqueeze(-1)
        scores = softmax(scores, index)
        scores = self.dropout(scores)

        # ── [3] Value + temporal message encoding ─────────────────────────────
        msg = V_node_j * scores.unsqueeze(-1)                        # (E, h, hd)
        msg = msg.reshape(E, self.dim)                               # (E, dim)

        # Apply RoPE + Fourier to bake edge timestamp into message content
        msg = self.msg_time_enc(msg, edge_t)                         # (E, dim)

        return msg


# ═══════════════════════════════════════════════════════════════════════════════
# 11.  EnhancedTemporalGNN  — full model
# ═══════════════════════════════════════════════════════════════════════════════

class EnhancedTemporalGNN(nn.Module):
    """
    Full temporal GNN with K-view trajectory trust for representation learning.

    Expected batch fields:
        batch.x          (N, in_dim)    node features
        batch.edge_index (2, E)         directed edges [src, dst]
        batch.msg        (E, edge_dim)  edge / message features
        batch.t          (E,)           edge timestamps

    Returns:
        x  (N, hidden_dim)  node representations
    """
    def __init__(
        self,
        num_nodes:       int,
        in_dim:          int,
        edge_dim:        int,
        hidden_dim:      int,
        num_layers:      int   = 2,
        heads:           int   = 8,
        dropout:         float = 0.1,
        window:          int   = 8,
        t2v_dim:         int   = 16,
        bank_size:       int   = 8,
        bilinear_rank:   int   = 8,
        memory_momentum: float = 0.9,
    ):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.num_nodes  = num_nodes
        self.window     = window

        # Input projections
        self.input_proj = nn.Linear(in_dim, hidden_dim)
        self.edge_enc   = nn.Linear(edge_dim, hidden_dim)
        self.time_enc   = TimeEncoder(hidden_dim)

        # TransformerConv snapshot encoder: produces embeddings stored in EvolutionBank
        self.snapshot_gnn = SnapshotGNN(hidden_dim, heads=heads, dropout=dropout)

        # Persistent state
        self.memory   = RecurrentMemory(num_nodes, hidden_dim, momentum=memory_momentum)
        self.evo_bank = EvolutionBank(num_nodes, hidden_dim, window=window)

        # Main message-passing layers
        self.layers = nn.ModuleList([
            TemporalTrustAttention(
                hidden_dim, heads=heads, dropout=dropout,
                window=window, t2v_dim=t2v_dim,
                num_nodes=num_nodes, bank_size=bank_size,
                t_heads=4, bilinear_rank=bilinear_rank,
            )
            for _ in range(num_layers)
        ])

        self.final_norm = nn.LayerNorm(hidden_dim)
        self._init_weights()

    # --------------------------------------------------------------------------
    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)

    # --------------------------------------------------------------------------
    def forward(self, batch) -> torch.Tensor:
        N          = batch.x.size(0)
        device     = batch.x.device
        edge_index = batch.edge_index
        src, dst   = edge_index

        # ── Encode inputs ─────────────────────────────────────────────────────
        x        = self.input_proj(batch.x)                         # (N, hidden)
        edge_emb = self.edge_enc(batch.msg)                         # (E, hidden)

        # ── Node degrees ──────────────────────────────────────────────────────
        deg = degree(dst, num_nodes=N, dtype=torch.float).clamp(min=1.0)

        # ── Temporal encoding (for TimeEncoder / GRU context) ─────────────────
        node_last_t = torch.zeros(N, device=device)
        node_last_t.scatter_reduce_(
            0, src, batch.t.float(), reduce='amax', include_self=True
        )
        t_rel    = batch.t.float() - node_last_t[src]
        time_emb = self.time_enc(batch.t, t_rel)                    # (E, hidden)

        # ── Compute snapshot embedding for EvolutionBank ──────────────────────
        # TransformerConv operates directly on node features and edge_attr.
        # No time signal here — time is injected in TemporalPositionalEncoding.
        snapshot_emb = self.snapshot_gnn(x, edge_index, edge_emb)   # (N, hidden)

        # ── Message passing layers ─────────────────────────────────────────────
        # Pass raw edge timestamps (batch.t) separately for RoPE and decay gate
        for layer in self.layers:
            x = layer(x, edge_index, edge_emb, time_emb,
                      batch.t.float(),                               # edge_t
                      self.memory, self.evo_bank, deg)

        x = self.final_norm(x)

        # ── Update persistent state ───────────────────────────────────────────
        nidx      = torch.arange(N, device=device)
        current_t = batch.t.float().max().expand(N)

        # Write snapshot to EvolutionBank (the rich contextual embedding)
        self.evo_bank.write(nidx, snapshot_emb, current_t)
        # Update GRU memory
        self.memory.write(nidx, x)

        return x

    # --------------------------------------------------------------------------
    def get_trust_scores(self, layer_idx: int = -1) -> Optional[torch.Tensor]:
        """Cached trust scores (E,) in (0, 2) from last forward()."""
        return self.layers[layer_idx]._trust_cache

    @torch.no_grad()
    def reset_memory(self):
        self.memory.hidden.zero_()
        self.memory.variance.fill_(0.1)
        self.evo_bank.reset()
        for layer in self.layers:
            if layer.msg_bank is not None:
                layer.msg_bank.reset()


# ═══════════════════════════════════════════════════════════════════════════════
# Quick sanity check
# ═══════════════════════════════════════════════════════════════════════════════
if __name__ == "__main__":
    import types

    torch.manual_seed(42)

    N, E      = 50, 120
    in_dim    = 32
    edge_dim  = 16
    hidden    = 64
    num_nodes = 100
    K         = 8

    def make_batch(t_offset=0.0):
        return types.SimpleNamespace(
            x          = torch.randn(N, in_dim),
            edge_index = torch.randint(0, N, (2, E)),
            msg        = torch.randn(E, edge_dim),
            t          = torch.rand(E) * 1000 + t_offset,
        )

    model = EnhancedTemporalGNN(
        num_nodes=num_nodes, in_dim=in_dim, edge_dim=edge_dim,
        hidden_dim=hidden, num_layers=2, heads=4,
        window=K, t2v_dim=16, bank_size=8, bilinear_rank=8,
    )

    for step in range(3):
        emb   = model(make_batch(step * 1000))
        trust = model.get_trust_scores()
        print(f"[pass {step+1}] emb: {emb.shape} | "
              f"trust: [{trust.min():.4f}, {trust.max():.4f}] "
              f"mean={trust.mean():.4f}")
        assert trust.min() > 0.0 and trust.max() < 2.0

    print("\nAll checks passed. Trust is in (0, 2).")

TypeError: __init__() got an unexpected keyword argument 't_heads'

In [1]:
"""
Temporal Trust GNN for Unsupervised Anomaly Detection
=====================================================
Produces node representations for downstream contrastive learning.
No labels required — fully unsupervised.

Message Aggregation (TemporalTrustAttention):

  [1] CROSS-SOURCE ATTENTION
      Q comes from EDGE embeddings  (what does this interaction ask for?)
      K comes from NODE features x  (who is this neighbour?)
      The same neighbour j is weighted differently depending on the
      type/context of the edge, making attention interaction-aware.

  [2] RoPE ON MESSAGES  (MessageRoPE)
      After scoring, the value message V_j is rotated by its edge
      timestamp using continuous-time RoPE BEFORE aggregation.
      Time is therefore baked into the message CONTENT, not just
      the score — messages from different times aggregate to different
      vectors even with identical node features.

  [3] SIMILARITY-BASED TRUST  (StructuralTrustModule)
      trust(i->j) in (0, 2):
        trust > 1 : i and j are structurally similar -> amplify
        trust < 1 : i and j are dissimilar           -> attenuate
      Both normal<->normal and anomalous<->anomalous get high trust.
      The model self-segregates: each node learns from its own "type".
      This naturally separates normal and anomalous representations.

  [4] TEMPORAL MESSAGE BANK  (TemporalMessageBank) — NEW
      A per-node buffer stores the last M aggregated messages.
      After each aggregation, cross-attention reads over this buffer:
        Q = current aggregated message
        K/V = past M stored messages
      Output: temporally-smoothed message that blends current with
      historical incoming signals. Detects sudden anomalous changes
      in what a node is receiving (erratic = anomalous influence).

Trust signals (StructuralTrustModule):
  [A] cos(h_i, h_j)              current GRU state similarity
  [B] degree profile similarity  structural role similarity
  [C] K-view trajectory sim      RoPE+Time2Vec encoded snapshot comparison
      trust = 1 + tanh(MLP([A, B, C]))   in (0, 2)
"""

import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from typing import Optional, Tuple
from torch_geometric.nn import MessagePassing, TransformerConv
from torch_geometric.utils import softmax, degree


# ═══════════════════════════════════════════════════════════════════════════════
# 1.  Time2Vec
# ═══════════════════════════════════════════════════════════════════════════════

class Time2Vec(nn.Module):
    """
    Encodes a scalar timestamp into a vector:
        [w0*t + b0,  sin(w1*t + b1), ..., sin(w_{k}*t + b_{k})]
    Linear term = trend.  Sinusoids = periodicity.  All frequencies learned.
    """
    def __init__(self, out_dim: int):
        super().__init__()
        assert out_dim >= 2
        k = out_dim - 1
        self.w0 = nn.Parameter(torch.randn(1) * 0.01)
        self.b0 = nn.Parameter(torch.zeros(1))
        self.W  = nn.Parameter(torch.randn(k) * 0.01)
        self.B  = nn.Parameter(torch.zeros(k))

    def forward(self, t: torch.Tensor) -> torch.Tensor:
        """t: (*,) -> (*, out_dim)"""
        t   = t.float().unsqueeze(-1)
        lin = t * self.w0 + self.b0
        per = torch.sin(t * self.W + self.B)
        return torch.cat([lin, per], dim=-1)


# ═══════════════════════════════════════════════════════════════════════════════
# 2.  TimeEncoder  (edge-level: absolute + relative timestamp -> hidden_dim)
# ═══════════════════════════════════════════════════════════════════════════════

class TimeEncoder(nn.Module):
    def __init__(self, hidden_dim: int):
        super().__init__()
        half = hidden_dim // 2
        self.t2v_abs = Time2Vec(half)
        self.t2v_rel = Time2Vec(half)
        self.proj = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, hidden_dim),
        )

    def forward(self, t_abs: torch.Tensor,
                t_rel: Optional[torch.Tensor] = None) -> torch.Tensor:
        """(E,), (E,) -> (E, hidden_dim)"""
        a = self.t2v_abs(t_abs)
        r = self.t2v_rel(t_rel) if t_rel is not None else torch.zeros_like(a)
        return self.proj(torch.cat([a, r], dim=-1))


# ═══════════════════════════════════════════════════════════════════════════════
# 3.  RecurrentMemory  (per-node GRU hidden state)
# ═══════════════════════════════════════════════════════════════════════════════

class RecurrentMemory(nn.Module):
    """
    Maintains one GRU hidden state per node, updated after each snapshot.
    The hidden state is the running summary of the node's full history.
    """
    def __init__(self, num_nodes: int, dim: int, momentum: float = 0.9):
        super().__init__()
        self.gru      = nn.GRUCell(dim, dim)
        self.momentum = momentum
        self.register_buffer("hidden",   torch.zeros(num_nodes, dim))
        self.register_buffer("variance", torch.ones(num_nodes, dim) * 0.1)

    def read(self, idx: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        return self.hidden[idx], self.variance[idx]

    def get_hidden(self, idx: torch.Tensor) -> torch.Tensor:
        return self.hidden[idx]

    @torch.no_grad()
    def write(self, idx: torch.Tensor, x: torch.Tensor):
        h_old = self.hidden[idx]
        h_new = self.gru(x.detach(), h_old)
        delta = h_new - h_old
        self.variance[idx] = (self.momentum * self.variance[idx]
                              + (1 - self.momentum) * delta.pow(2))
        self.hidden[idx] = h_new


# ═══════════════════════════════════════════════════════════════════════════════
# 4.  SnapshotGNN  — TransformerConv-based node encoder
#
#  Produces the snapshot embedding stored in the EvolutionBank.
#  Uses PyG's TransformerConv which performs full transformer-style
#  multi-head attention between a node and its one-hop neighbours,
#  incorporating edge_attr directly into the key/value computation:
#
#      alpha(i,j) = softmax( (W_Q * x_i)^T (W_K * x_j + W_E * edge_attr) )
#      out_i      = W_O * sum_j [ alpha(i,j) * (W_V * x_j + W_E * edge_attr) ]
#
#  Inputs: node features (x) and edge features (edge_attr) ONLY.
#  Time is NOT used here — it is handled separately in the positional
#  encoding step when comparing K-view trajectories in the trust module.
#
#  Two TransformerConv layers with a residual connection give the model
#  capacity to capture both direct neighbours and two-hop structure.
# ═══════════════════════════════════════════════════════════════════════════════

class SnapshotGNN(nn.Module):
    """
    Transformer-based snapshot encoder using PyG's TransformerConv.

    Produces a rich node embedding from node features and edge attributes,
    capturing neighbourhood structure via transformer-style attention.
    Time is deliberately excluded — it is injected separately during
    the K-view positional encoding in StructuralTrustModule.

    Args:
        dim      : hidden dimension (input, output, and internal)
        heads    : number of attention heads for TransformerConv
        dropout  : dropout on attention weights
    """
    def __init__(self, dim: int, heads: int = 4, dropout: float = 0.1):
        super().__init__()
        assert dim % heads == 0, "dim must be divisible by heads"

        # Layer 1: node features + edge_attr -> dim
        self.conv1 = TransformerConv(
            in_channels  = dim,
            out_channels = dim // heads,
            heads        = heads,
            dropout      = dropout,
            edge_dim     = dim,        # edge_attr dimension matches hidden dim
            concat       = True,       # concat heads -> dim
            beta         = True,       # learnable skip connection inside conv
        )
        self.norm1 = nn.LayerNorm(dim)

        # Layer 2: refine with a second transformer hop
        self.conv2 = TransformerConv(
            in_channels  = dim,
            out_channels = dim // heads,
            heads        = heads,
            dropout      = dropout,
            edge_dim     = dim,
            concat       = True,
            beta         = True,
        )
        self.norm2 = nn.LayerNorm(dim)

        # Final projection + residual gate
        self.out_proj = nn.Linear(dim, dim)
        self.gate     = nn.Linear(dim, 1)

    def forward(self,
                x:          torch.Tensor,   # (N, dim)   projected node features
                edge_index: torch.Tensor,   # (2, E)
                edge_attr:  torch.Tensor,   # (E, dim)   projected edge features
                ) -> torch.Tensor:          # (N, dim)
        # Layer 1
        h = self.norm1(F.gelu(self.conv1(x, edge_index, edge_attr)))  # (N, dim)
        # Layer 2 with residual
        h2 = self.norm2(F.gelu(self.conv2(h, edge_index, edge_attr))) # (N, dim)
        h  = h + h2                                                    # residual

        # Gated output projection
        out = self.out_proj(h)
        g   = torch.sigmoid(self.gate(x))
        return g * x + (1 - g) * out                                  # (N, dim)


# ═══════════════════════════════════════════════════════════════════════════════
# 5.  EvolutionBank  — stores last K snapshot embeddings + timestamps per node
#
#  After each forward pass, the snapshot embedding produced by SnapshotGNN
#  is written here alongside its timestamp.
#  Shape: bank (num_nodes, K, dim),  times (num_nodes, K)
# ═══════════════════════════════════════════════════════════════════════════════

class EvolutionBank(nn.Module):
    def __init__(self, num_nodes: int, dim: int, window: int = 8):
        super().__init__()
        self.window = window
        self.dim    = dim
        self.register_buffer("bank",  torch.zeros(num_nodes, window, dim))
        self.register_buffer("times", torch.zeros(num_nodes, window))
        self.register_buffer("ptr",   torch.zeros(num_nodes, dtype=torch.long))

    def read(self, idx: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        """Returns (B, K, dim) embeddings and (B, K) timestamps."""
        return self.bank[idx], self.times[idx]

    @torch.no_grad()
    def write(self, idx: torch.Tensor, emb: torch.Tensor, t: torch.Tensor):
        """
        idx : (B,)    node indices
        emb : (B, D)  snapshot embedding from SnapshotGNN
        t   : (B,)    timestamp of this snapshot
        """
        for b in range(idx.size(0)):
            node = int(idx[b])
            p    = int(self.ptr[node]) % self.window
            self.bank[node,  p] = emb[b].detach()
            self.times[node, p] = float(t[b])
            self.ptr[node] += 1

    @torch.no_grad()
    def reset(self):
        self.bank.zero_()
        self.times.zero_()
        self.ptr.zero_()


# ═══════════════════════════════════════════════════════════════════════════════
# 6.  TemporalPositionalEncoding  — per-head multi-resolution temporal encoding
#
#  Applies temporal encoding to K-view snapshot sequences for the trust module.
#
#  KEY IDEA: split the embedding into H temporal heads, each head covering a
#  DIFFERENT temporal resolution band. Head 0 uses fast-rotating RoPE
#  (captures short-range patterns, e.g. hourly), head H-1 uses slow-rotating
#  RoPE (captures long-range drift, e.g. weekly). This gives the model multiple
#  simultaneous temporal "lenses" operating in parallel, similar to how
#  multi-head attention gives multiple geometric viewpoints.
#
#  Per head h:
#    (a) RoPE with head-specific timescale band:
#            θ_{h,d} = t / T_{h,d}
#        T_{h,d} are sampled from a sub-range of [T_min, T_max] assigned to head h.
#        Head h rotates dimensions (h*hd) to ((h+1)*hd) with timescales in
#        [T_min + h*(T_max-T_min)/H, T_min + (h+1)*(T_max-T_min)/H].
#
#    (b) Multi-scale Fourier (additive, shared across heads):
#        Complete Fourier basis (sin + cos) over F log-uniform frequencies,
#        projected to dim. Captures absolute periodic patterns independently
#        of the RoPE rotation.
#
#    (c) Recency decay gate (per-head learned λ_h):
#        g_{h,k} = σ( -λ_h * Δt_k + b_h )
#        Different heads decay at different rates — fast heads strongly
#        discount old views, slow heads treat all views more equally.
#
#  Combined per head h, view k:
#      view_{h,k} = g_{h,k} * RoPE_h(emb_{h,k}, t_k) + Fourier_proj(t_k)[h*hd:(h+1)*hd]
# ═══════════════════════════════════════════════════════════════════════════════

class TemporalPositionalEncoding(nn.Module):
    """
    Per-head multi-resolution temporal positional encoding for K-view sequences.

    Args:
        dim       : total embedding dimension (must be even, divisible by t_heads)
        t_heads   : number of temporal resolution heads
        n_fourier : Fourier frequency bands (sin+cos each)
        t_min_log : log of minimum timescale (default: 0 → T=1)
        t_max_log : log of maximum timescale (default: 9 → T≈8000)
    """
    def __init__(self, dim: int, t_heads: int = 4, n_fourier: int = 32,
                 t_min_log: float = 0.0, t_max_log: float = 9.0):
        super().__init__()
        assert dim % 2 == 0 and dim % t_heads == 0
        self.dim     = dim
        self.t_heads = t_heads
        self.head_d  = dim // t_heads   # dims per temporal head
        assert self.head_d % 2 == 0,   "head_d must be even for RoPE"

        # ── (a) Per-head RoPE timescales ──────────────────────────────────────
        # Each head h gets timescales from a dedicated sub-range of [t_min, t_max].
        # Head 0: fast (captures fine-grained recency)
        # Head H-1: slow (captures long-range historical trends)
        rope_params = []
        for h in range(t_heads):
            lo = t_min_log + h       * (t_max_log - t_min_log) / t_heads
            hi = t_min_log + (h + 1) * (t_max_log - t_min_log) / t_heads
            rope_params.append(torch.linspace(lo, hi, self.head_d // 2))
        # Stack to (t_heads, head_d//2), learned
        self.rope_log_ts = nn.Parameter(torch.stack(rope_params, dim=0))

        # ── (b) Multi-scale Fourier (shared, additive) ────────────────────────
        freqs = torch.logspace(-4, 1, n_fourier)
        self.register_buffer("fourier_freqs", freqs)
        self.fourier_proj = nn.Linear(2 * n_fourier, dim)

        # ── (c) Per-head recency decay gate ───────────────────────────────────
        # Each head has its own learned decay rate λ_h and bias b_h.
        # Fast heads (low h): init high λ → strong recency emphasis
        # Slow heads (high h): init low λ → weaker decay
        init_log_decay = torch.linspace(1.0, -1.0, t_heads)         # head 0 fastest
        self.log_decay  = nn.Parameter(init_log_decay)               # (t_heads,)
        self.decay_bias = nn.Parameter(torch.ones(t_heads))          # (t_heads,)

    # ──────────────────────────────────────────────────────────────────────────
    def _rope_per_head(self,
                       embs:  torch.Tensor,   # (B, K, dim)
                       times: torch.Tensor,   # (B, K)
                       ) -> torch.Tensor:     # (B, K, dim)
        """Apply per-head RoPE: each head h rotates its slice of the embedding
        with head-h-specific timescales."""
        B, K, D = embs.shape
        out_parts = []
        for h in range(self.t_heads):
            # Slice embedding for this head: (B, K, head_d)
            sl = embs[..., h * self.head_d : (h + 1) * self.head_d]
            ts  = self.rope_log_ts[h].exp()                          # (head_d//2,)
            ang = times.unsqueeze(-1) / ts                           # (B, K, hd//2)
            cos_a, sin_a = ang.cos(), ang.sin()
            e_even = sl[..., 0::2]
            e_odd  = sl[..., 1::2]
            rot = torch.stack(
                [e_even * cos_a - e_odd * sin_a,
                 e_even * sin_a + e_odd * cos_a], dim=-1
            ).reshape(B, K, self.head_d)
            out_parts.append(rot)
        return torch.cat(out_parts, dim=-1)                          # (B, K, dim)

    # ──────────────────────────────────────────────────────────────────────────
    def _fourier(self, times: torch.Tensor) -> torch.Tensor:
        """(B, K) -> (B, K, dim)"""
        B, K = times.shape
        t    = times.reshape(B * K, 1).float()
        phi  = t * self.fourier_freqs.unsqueeze(0)                   # (BK, F)
        feat = torch.cat([phi.sin(), phi.cos()], dim=-1)             # (BK, 2F)
        return self.fourier_proj(feat).view(B, K, self.dim)          # (B, K, D)

    # ──────────────────────────────────────────────────────────────────────────
    def _decay_gates(self, times: torch.Tensor) -> torch.Tensor:
        """(B, K) -> (B, K, dim)  — per-head gate broadcast across head dims"""
        t_max = times.max(dim=-1, keepdim=True).values               # (B, 1)
        delta = (t_max - times).clamp(min=0.0)                      # (B, K)
        lam   = self.log_decay.exp()                                 # (t_heads,)
        bias  = self.decay_bias                                      # (t_heads,)
        # gate per head: (B, K, t_heads)
        gate  = torch.sigmoid(
            -lam.unsqueeze(0).unsqueeze(0) * delta.unsqueeze(-1)
            + bias.unsqueeze(0).unsqueeze(0)
        )
        # Broadcast to full dim: each head's gate applies to its head_d dims
        gate_full = gate.repeat_interleave(self.head_d, dim=-1)      # (B, K, dim)
        return gate_full

    # ──────────────────────────────────────────────────────────────────────────
    def forward(self,
                embs:  torch.Tensor,   # (B, K, dim)
                times: torch.Tensor,   # (B, K)
                ) -> torch.Tensor:     # (B, K, dim)
        rope_out   = self._rope_per_head(embs, times)                # (B, K, dim)
        gates      = self._decay_gates(times)                        # (B, K, dim)
        fourier_out = self._fourier(times)                           # (B, K, dim)
        # Each head's RoPE output is gated by its own recency rate,
        # then the absolute Fourier signal is added on top.
        return gates * rope_out + fourier_out                        # (B, K, dim)


# ═══════════════════════════════════════════════════════════════════════════════
# 7.  StructuralTrustModule  — K-VIEW SINKHORN TRAJECTORY SIMILARITY
#
#  Computes trust(i→j) ∈ (0, 2) using ONLY the K-view trajectory signal.
#
#  WHY SINKHORN INSTEAD OF CROSS-ATTENTION?
#  ─────────────────────────────────────────
#  Cross-attention (softmax over rows) lets each of i's views attend to ALL
#  of j's views independently.  Two problems:
#    1. Multiple views of i can "compete" for the same view of j, pulling
#       the match towards a few dominant modes.
#    2. No constraint that the overall matching is globally consistent —
#       a single anomalous view in j can dominate every row.
#
#  Sinkhorn optimal transport solves both:
#    - Finds the globally optimal SOFT ASSIGNMENT matrix T ∈ R^{K×K}
#      between i's K views and j's K views.
#    - Row margins: T·1 = u (how much each view of i is matched)
#    - Column margins: T^T·1 = v (how much each view of j is used)
#    - With uniform margins (u=v=1/K), every view contributes equally.
#    - Minimises sum_{k,k'} C_{k,k'} * T_{k,k'}
#      where C_{k,k'} = 1 - cos(view_i_k, view_j_k') is the COST.
#    - Solved iteratively via Sinkhorn-Knopp (log-domain for stability).
#
#  After finding T, the trajectory similarity is:
#      traj_sim = sum_{k,k'} T_{k,k'} * (1 - C_{k,k'})    (E,) ∈ [-1,1]
#               = sum_{k,k'} T_{k,k'} * cos(vi_k, vj_k')
#  This is the EXPECTED cosine similarity under the optimal transport plan —
#  a true measure of how "alignable" the two trajectories are.
# ═══════════════════════════════════════════════════════════════════════════════

class StructuralTrustModule(nn.Module):
    """
    K-view trajectory trust via Sinkhorn Optimal Transport.

    Args:
        dim         : embedding dimension
        window      : K, number of stored snapshots
        t_heads     : temporal resolution heads for TemporalPositionalEncoding
        sinkhorn_eps: entropy regularisation (smaller = sharper assignment)
        sinkhorn_iters: number of Sinkhorn iterations (5-10 is usually enough)
    """
    def __init__(self, dim: int, window: int = 8,
                 t2v_dim: int = 16,          # kept for API compat, unused
                 t_heads: int = 4,
                 sinkhorn_eps: float = 0.05,
                 sinkhorn_iters: int = 7):
        super().__init__()
        self.window         = window
        self.sinkhorn_eps   = sinkhorn_eps
        self.sinkhorn_iters = sinkhorn_iters

        # Per-head multi-resolution temporal positional encoding
        self.time_pe = TemporalPositionalEncoding(dim, t_heads=t_heads)

        # Lightweight projection to compare views in a lower-dim space
        # This decouples the comparison space from the full hidden dim,
        # reducing computation in the K×K cost matrix.
        proj_dim = max(dim // 2, 32)
        self.proj = nn.Linear(dim, proj_dim, bias=False)

        # Maps OT similarity scalar → trust logit
        self.trust_head = nn.Sequential(
            nn.Linear(1, max(dim // 8, 8)),
            nn.GELU(),
            nn.Linear(max(dim // 8, 8), 1),
        )

    # ──────────────────────────────────────────────────────────────────────────
    @staticmethod
    def _sinkhorn_log(log_C: torch.Tensor,
                      eps: float, n_iter: int) -> torch.Tensor:
        """
        Log-domain Sinkhorn: finds the optimal transport plan T for cost C.

        log_C : (B, K, K)  log of cost matrix (here log(1 - cos_sim + ε))
        Returns log_T : (B, K, K)  log of transport plan

        Log-domain is numerically stable for small eps.
        The algorithm alternates normalising rows and columns of log_T.
        """
        B, K, _ = log_C.shape
        log_a = torch.full((B, K), -math.log(K),
                           device=log_C.device, dtype=log_C.dtype)  # uniform source
        log_b = torch.full((B, K), -math.log(K),
                           device=log_C.device, dtype=log_C.dtype)  # uniform target

        log_T = -log_C / eps    # initialise with Gibbs kernel
        for _ in range(n_iter):
            # Row normalisation: T·1 = a
            log_T = log_T - torch.logsumexp(log_T, dim=2, keepdim=True) + log_a.unsqueeze(2)
            # Column normalisation: T^T·1 = b
            log_T = log_T - torch.logsumexp(log_T, dim=1, keepdim=True) + log_b.unsqueeze(1)
        return log_T   # (B, K, K)

    # ──────────────────────────────────────────────────────────────────────────
    def forward(
        self,
        embs_i: torch.Tensor,   # (E, K, dim)  last K snapshot embeddings of i
        embs_j: torch.Tensor,   # (E, K, dim)  last K snapshot embeddings of j
        t_i:    torch.Tensor,   # (E, K)       timestamps of i's K snapshots
        t_j:    torch.Tensor,   # (E, K)       timestamps of j's K snapshots
        eps:    float = 1e-6,
    ) -> torch.Tensor:          # (E,) ∈ (0, 2)

        E, K, D = embs_i.shape

        # Step 1: per-head multi-resolution temporal PE
        # Fast heads emphasise recency, slow heads treat all views equally.
        vi = self.time_pe(embs_i, t_i)   # (E, K, dim)
        vj = self.time_pe(embs_j, t_j)

        # Step 2: project to lower-dim comparison space and normalise
        vi_p = F.normalize(self.proj(vi.reshape(E * K, D)),
                           dim=-1, eps=eps).view(E, K, -1)           # (E, K, proj_dim)
        vj_p = F.normalize(self.proj(vj.reshape(E * K, D)),
                           dim=-1, eps=eps).view(E, K, -1)

        # Step 3: full K×K pairwise cosine similarity
        sim = torch.bmm(vi_p, vj_p.transpose(1, 2))                 # (E, K, K)

        # Step 4: Sinkhorn OT — finds globally optimal soft assignment
        log_C = torch.log((1.0 - sim).clamp(min=1e-6))
        log_T = self._sinkhorn_log(log_C, self.sinkhorn_eps,
                                   self.sinkhorn_iters)
        T = log_T.exp()                                              # (E, K, K)

        # Step 5: OT-weighted expected cosine similarity
        traj_sim = (T * sim).sum(dim=(1, 2))                         # (E,) ∈ [-1, 1]

        # Step 6: trust ∈ (0, 2)
        logit = self.trust_head(traj_sim.unsqueeze(-1)).squeeze(-1)
        return 1.0 + torch.tanh(logit)


# ═══════════════════════════════════════════════════════════════════════════════
# 8.  MessageTemporalEncoding  — enhanced per-head RoPE + Fourier + decay gate
#
#  Applied to live edge messages (E, dim) before aggregation at node i.
#
#  Four components, all applied together:
#
#  (a) TIMESTAMP NORMALISATION
#      Raw timestamps can be arbitrarily large absolute values (e.g. Unix time).
#      Large t values cause RoPE angles to wrap many times, losing information.
#      We apply learnable normalisation:  t_norm = (t - mu) / (sigma + ε)
#      where mu and sigma are EMA statistics updated during training.
#      This keeps angles in a well-behaved range regardless of the time scale.
#
#  (b) PER-HEAD RoPE
#      Message is split into t_heads segments. Each segment is rotated by
#      RoPE with a head-specific timescale sub-range. Some heads capture
#      fine-grained recency (hourly), others capture long-range patterns.
#
#  (c) MULTI-SCALE FOURIER (additive, shared)
#      Absolute temporal position signal via sin/cos of log-uniform frequencies.
#
#  (d) PER-HEAD TEMPORAL DECAY GATE
#      g_h = σ( -λ_h * |t_norm| + b_h )   ∈ (0, 1)
#      Each head has a learned decay rate λ_h. Fast heads (large λ) strongly
#      down-weight messages that are "old" relative to the mean timestamp.
#      Slow heads weight all messages more uniformly.
#      The gated message for head h: g_h * rope_h(msg_h) + (1-g_h) * msg_h
#      This is a soft interpolation between time-rotated and original content,
#      letting slow heads preserve content while fast heads inject recency.
# ═══════════════════════════════════════════════════════════════════════════════

class MessageTemporalEncoding(nn.Module):
    """
    Per-head multi-resolution RoPE + Fourier + decay gate for edge messages.

    Args:
        dim       : message dimension (even, divisible by t_heads)
        t_heads   : number of temporal resolution heads
        n_fourier : Fourier frequency bands (sin+cos each)
        ema_decay : momentum for running timestamp statistics
    """
    def __init__(self, dim: int, t_heads: int = 4, n_fourier: int = 16,
                 ema_decay: float = 0.99):
        super().__init__()
        assert dim % 2 == 0 and dim % t_heads == 0
        self.dim      = dim
        self.t_heads  = t_heads
        self.head_d   = dim // t_heads
        assert self.head_d % 2 == 0

        # ── (a) Learnable timestamp normalisation ─────────────────────────────
        # Running statistics for EMA normalisation of raw timestamps
        self.ema_decay = ema_decay
        self.register_buffer("t_mean", torch.zeros(1))
        self.register_buffer("t_var",  torch.ones(1))
        # Learnable affine rescale after normalisation
        self.t_scale = nn.Parameter(torch.ones(1))
        self.t_shift = nn.Parameter(torch.zeros(1))

        # ── (b) Per-head RoPE timescales ──────────────────────────────────────
        # Shorter range for messages: angles of normalised timestamps
        rope_params = []
        T_MIN, T_MAX = 0.0, 4.0   # timescales 1 → e^4 ≈ 55 (on normalised scale)
        for h in range(t_heads):
            lo = T_MIN + h       * (T_MAX - T_MIN) / t_heads
            hi = T_MIN + (h + 1) * (T_MAX - T_MIN) / t_heads
            rope_params.append(torch.linspace(lo, hi, self.head_d // 2))
        self.rope_log_ts = nn.Parameter(torch.stack(rope_params, dim=0))  # (H, hd//2)

        # ── (c) Multi-scale Fourier (shared, additive) ────────────────────────
        freqs = torch.logspace(-3, 1, n_fourier)
        self.register_buffer("fourier_freqs", freqs)
        self.fourier_proj = nn.Linear(2 * n_fourier, dim)

        # ── (d) Per-head temporal decay gate ──────────────────────────────────
        # λ_h: learned decay rate per head; fast heads init to larger values
        init_log_decay = torch.linspace(1.5, -0.5, t_heads)  # head 0 = fastest
        self.log_decay  = nn.Parameter(init_log_decay)        # (t_heads,)
        self.decay_bias = nn.Parameter(torch.ones(t_heads))   # (t_heads,)

    @torch.no_grad()
    def _update_stats(self, t: torch.Tensor):
        """EMA update of running mean and variance of timestamps."""
        mu  = t.float().mean()
        var = t.float().var().clamp(min=1e-4)
        self.t_mean = self.ema_decay * self.t_mean + (1 - self.ema_decay) * mu
        self.t_var  = self.ema_decay * self.t_var  + (1 - self.ema_decay) * var

    def _normalise_t(self, t: torch.Tensor) -> torch.Tensor:
        """Normalise timestamps to zero-mean, unit-variance, then rescale."""
        t_norm = (t.float() - self.t_mean) / (self.t_var.sqrt() + 1e-6)
        return self.t_scale * t_norm + self.t_shift      # (E,)

    def forward(self,
                msg: torch.Tensor,   # (E, dim)
                t:   torch.Tensor,   # (E,)   raw edge timestamps
                ) -> torch.Tensor:   # (E, dim)
        E = msg.size(0)

        # Update running stats during training
        if self.training:
            self._update_stats(t)

        # Normalise timestamps
        t_n = self._normalise_t(t)                                   # (E,)

        # ── (b) Per-head RoPE on normalised timestamps ────────────────────────
        rope_parts = []
        for h in range(self.t_heads):
            sl  = msg[:, h * self.head_d : (h + 1) * self.head_d]   # (E, hd)
            ts  = self.rope_log_ts[h].exp()                          # (hd//2,)
            ang = t_n.unsqueeze(-1) / ts                             # (E, hd//2)
            cos_a, sin_a = ang.cos(), ang.sin()
            m_even = sl[:, 0::2]
            m_odd  = sl[:, 1::2]
            rot = torch.stack(
                [m_even * cos_a - m_odd * sin_a,
                 m_even * sin_a + m_odd * cos_a], dim=-1
            ).reshape(E, self.head_d)
            rope_parts.append(rot)
        rope_out = torch.cat(rope_parts, dim=-1)                     # (E, dim)

        # ── (c) Multi-scale Fourier (additive, absolute time) ─────────────────
        phi    = t_n.unsqueeze(-1) * self.fourier_freqs              # (E, F)
        feat   = torch.cat([phi.sin(), phi.cos()], dim=-1)           # (E, 2F)
        fourier = self.fourier_proj(feat)                            # (E, dim)

        # ── (d) Per-head temporal decay gate ──────────────────────────────────
        # g_h = σ( -λ_h * |t_norm| + b_h )
        # Fast heads (large λ) strongly gate old messages toward identity.
        lam  = self.log_decay.exp()                                  # (t_heads,)
        gate = torch.sigmoid(
            -lam.unsqueeze(0) * t_n.abs().unsqueeze(-1)
            + self.decay_bias.unsqueeze(0)
        )                                                            # (E, t_heads)
        # Broadcast gate to full dim (each head covers head_d dims)
        gate_full = gate.repeat_interleave(self.head_d, dim=-1)      # (E, dim)
        # Gated blend: interpolate between rotated and original content
        gated_rope = gate_full * rope_out + (1 - gate_full) * msg    # (E, dim)

        return gated_rope + fourier                                   # (E, dim)


# ═══════════════════════════════════════════════════════════════════════════════
# 9.  GRN  — Gated Residual Network (from Temporal Fusion Transformer, Lim 2021)
#
#  A proven building block for temporal modelling. Replaces plain nn.Linear
#  for Q, K, V projections in TemporalTrustAttention.
#
#  GRN(x, c) = LayerNorm( x + W2 · ELU(W1 · [x, c] + b1) ⊙ σ(W_gate · [x, c]) )
#
#  Two important properties:
#    1. CONTEXT GATING: an optional context vector c (here: the node's GRU
#       memory) modulates the output via a sigmoid gate.  This means the
#       projection adapts to the node's historical state without needing
#       a full hypernetwork.
#    2. ELU NONLINEARITY: unlike GELU (smooth everywhere), ELU has exact
#       linearity for positive inputs and a negative saturation floor.
#       This is beneficial for temporal data where some features are truly
#       linear (trend) and others saturate (periodic amplitude).
#    3. SKIP CONNECTION: the residual allows the GRN to be identity-like
#       when no transformation is needed, which helps early training.
# ═══════════════════════════════════════════════════════════════════════════════


# ═══════════════════════════════════════════════════════════════════════════════
# 9.  TemporalMessageBank  — per-node circular buffer with cross-attention
#
#  Stores the last M aggregated messages per node.  After each aggregation,
#  cross-attention reads over this history:
#    Q = current aggregated message   (what I just received)
#    K/V = past M stored messages     (what I have received before)
#  Output: temporally-smoothed message blending current and historical signals.
#
#  Normal nodes: consistent messages → strong history alignment → stable repr.
#  Anomalous influence: divergence from history → dampened across time.
# ═══════════════════════════════════════════════════════════════════════════════

class TemporalMessageBank(nn.Module):
    """
    Per-node circular buffer of past aggregated messages with cross-attention.

    Args:
        num_nodes : total nodes in graph
        dim       : message / hidden dimension
        bank_size : M, number of past messages stored per node
    """
    def __init__(self, num_nodes: int, dim: int, bank_size: int = 8):
        super().__init__()
        self.bank_size = bank_size
        self.dim       = dim
        self.register_buffer("bank", torch.zeros(num_nodes, bank_size, dim))
        self.register_buffer("ptr",  torch.zeros(num_nodes, dtype=torch.long))

        self.q_proj   = nn.Linear(dim, dim)
        self.k_proj   = nn.Linear(dim, dim)
        self.v_proj   = nn.Linear(dim, dim)
        self.scale    = dim ** -0.5
        self.out_proj = nn.Linear(dim, dim)
        self.norm     = nn.LayerNorm(dim)

    @torch.no_grad()
    def write(self, idx: torch.Tensor, msg: torch.Tensor):
        """Write current aggregated message into the circular buffer."""
        for b in range(idx.size(0)):
            node = int(idx[b])
            p    = int(self.ptr[node]) % self.bank_size
            self.bank[node, p] = msg[b].detach()
            self.ptr[node] += 1

    def read_and_attend(self,
                        idx:     torch.Tensor,   # (N,)
                        cur_msg: torch.Tensor,   # (N, dim)
                        ) -> torch.Tensor:       # (N, dim)
        """Cross-attention: current message as Q, historical bank as K/V."""
        past = self.bank[idx]                                        # (N, M, dim)
        Q    = self.q_proj(cur_msg).unsqueeze(1)                     # (N, 1, dim)
        K    = self.k_proj(past)                                     # (N, M, dim)
        V    = self.v_proj(past)
        attn = torch.bmm(Q, K.transpose(1, 2)) * self.scale         # (N, 1, M)
        attn = F.softmax(attn, dim=-1)
        ctx  = torch.bmm(attn, V).squeeze(1)                        # (N, dim)
        return self.norm(cur_msg + self.out_proj(ctx))               # (N, dim)

    @torch.no_grad()
    def reset(self):
        self.bank.zero_()
        self.ptr.zero_()


class GRN(nn.Module):
    """
    Gated Residual Network projection with optional context conditioning.

    Args:
        in_dim  : input dimension
        out_dim : output dimension
        ctx_dim : context dimension (None = no context gating)
        dropout : dropout before residual add
    """
    def __init__(self, in_dim: int, out_dim: int,
                 ctx_dim: Optional[int] = None, dropout: float = 0.1):
        super().__init__()
        self.in_dim  = in_dim
        self.out_dim = out_dim
        ctx = ctx_dim or 0

        # Main transformation: [x, c] -> hidden -> out
        self.W1 = nn.Linear(in_dim + ctx, out_dim)
        self.W2 = nn.Linear(out_dim, out_dim)

        # Context gate: [x, c] -> gate ∈ (0,1) controlling residual weight
        self.W_gate = nn.Linear(in_dim + ctx, out_dim)

        # Skip connection projection (needed when in_dim ≠ out_dim)
        self.skip = (nn.Linear(in_dim, out_dim, bias=False)
                     if in_dim != out_dim else nn.Identity())

        self.norm    = nn.LayerNorm(out_dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self,
                x:   torch.Tensor,              # (B, in_dim)
                ctx: Optional[torch.Tensor] = None  # (B, ctx_dim)
                ) -> torch.Tensor:              # (B, out_dim)
        if ctx is not None:
            inp = torch.cat([x, ctx], dim=-1)   # (B, in_dim + ctx_dim)
        else:
            inp = x

        # Gated transformation
        h    = F.elu(self.W1(inp))              # (B, out_dim)
        h    = self.dropout(self.W2(h))         # (B, out_dim)
        gate = torch.sigmoid(self.W_gate(inp))  # (B, out_dim) ∈ (0,1)

        # Gated residual: gate blends transformed output with skip
        out = gate * h + (1 - gate) * self.skip(x)
        return self.norm(out)                   # (B, out_dim)


# ═══════════════════════════════════════════════════════════════════════════════
# 10.  TemporalTrustAttention  (one message-passing layer)
#
#  [1] GRN-PROJECTED Q / K / V  (context-conditioned via GRU memory)
#      Q  = GRN(edge_emb, ctx=mem_i)  interaction queries through node history
#      K  = GRN(x_j,      ctx=mem_j)  node j's key adapted to its history
#      V  = GRN(x_j,      ctx=mem_j)  value also history-conditioned
#
#  [2] RoPE APPLIED TO Q AND K BEFORE SCORING
#      This is the mathematically correct placement for RoPE.
#      Rotating Q and K by the edge timestamp means that the dot product
#      Q·K naturally captures relative temporal position:
#          (R_t Q)·(R_t K) = Q^T R_{0} K = Q^T K   (same time: unrotated)
#          (R_{t1} Q)·(R_{t2} K) = Q^T R_{t2-t1} K  (different times: rotated)
#      The score between Q and K reflects BOTH semantic compatibility AND
#      temporal proximity. Temporally distant Q-K pairs will have a
#      phase-shifted dot product, naturally reducing their score.
#      Per-head timescales are used (same as MessageTemporalEncoding).
#
#  [3] PER-HEAD BILINEAR SCORING  (replaces identity dot-product)
#      Standard dot-product: score = Q^T K  (uses identity metric)
#      Bilinear form:        score = Q^T W_h K  (each head learns its own metric)
#      W_h ∈ R^{hd×hd} is a learned per-head compatibility matrix.
#      This allows each head to discover a different notion of compatibility
#      between the edge query and the node key — e.g. one head may focus on
#      frequency features, another on structural role features.
#      To keep computation light, W_h is parameterised as a low-rank outer
#      product: W_h = U_h V_h^T where U_h, V_h ∈ R^{hd×r}, rank r << hd.
#
#  [4] SINKHORN TRUST  (K-view OT only, no h/deg)
#
#  [5] PER-HEAD MULTI-RESOLUTION MESSAGE TEMPORAL ENCODING  (with decay gate)
#
#  [6] TEMPORAL MESSAGE BANK
# ═══════════════════════════════════════════════════════════════════════════════

class TemporalTrustAttention(MessagePassing):
    def __init__(self, dim: int, heads: int = 8, dropout: float = 0.1,
                 window: int = 8, t2v_dim: int = 16,
                 num_nodes: int = 0, bank_size: int = 8,
                 t_heads: int = 4, bilinear_rank: int = 8):
        super().__init__(aggr="add", node_dim=0)
        assert dim % heads == 0
        self.dim        = dim
        self.heads      = heads
        self.head_dim   = dim // heads
        self.t_heads    = t_heads

        # ── [1] GRN projections (context-conditioned) ─────────────────────────
        self.q_grn  = GRN(dim, dim, ctx_dim=dim, dropout=dropout)
        self.k_grn  = GRN(dim, dim, ctx_dim=dim, dropout=dropout)
        self.v_grn  = GRN(dim, dim, ctx_dim=dim, dropout=dropout)
        self.o_proj = nn.Linear(dim, dim)

        # ── [2] Per-head RoPE on Q and K ─────────────────────────────────────
        # Timescales shared with message encoding for consistency
        # Each head gets its own sub-range of [0, 4] on normalised timestamps
        rope_params = []
        T_MIN, T_MAX = 0.0, 4.0
        hd = self.head_dim
        assert hd % 2 == 0
        for h in range(heads):
            lo = T_MIN + h       * (T_MAX - T_MIN) / heads
            hi = T_MIN + (h + 1) * (T_MAX - T_MIN) / heads
            rope_params.append(torch.linspace(lo, hi, hd // 2))
        self.qk_rope_log_ts = nn.Parameter(torch.stack(rope_params, dim=0))  # (H, hd//2)

        # Running timestamp stats for normalisation (shared with MessageTemporalEncoding)
        self.register_buffer("t_mean_qk", torch.zeros(1))
        self.register_buffer("t_var_qk",  torch.ones(1))
        self.t_scale_qk = nn.Parameter(torch.ones(1))
        self.t_shift_qk = nn.Parameter(torch.zeros(1))
        self.ema_decay = 0.99

        # ── [3] Per-head bilinear scoring (low-rank) ──────────────────────────
        # W_h = U_h @ V_h^T  where U_h, V_h ∈ R^{hd × r}
        # Parameterised as two (heads, hd, rank) tensors.
        # score_h = Q_h^T W_h K_h = (Q_h^T U_h)(V_h^T K_h) — factored efficiently.
        r = bilinear_rank
        self.bilin_U = nn.Parameter(torch.randn(heads, hd, r) * (r ** -0.5))
        self.bilin_V = nn.Parameter(torch.randn(heads, hd, r) * (r ** -0.5))
        self.scale   = r ** -0.5   # normalise by rank, not head_dim

        # ── [4] Sinkhorn trust (K-view only) ─────────────────────────────────
        self.trust = StructuralTrustModule(dim, window=window, t_heads=t_heads)

        # ── [5] Message temporal encoding (with decay gate) ───────────────────
        self.msg_time_enc = MessageTemporalEncoding(dim, t_heads=t_heads, n_fourier=16)

        # ── [6] Temporal message bank ─────────────────────────────────────────
        self.msg_bank = (TemporalMessageBank(num_nodes, dim, bank_size=bank_size)
                         if num_nodes > 0 else None)

        self.dropout = nn.Dropout(dropout)
        self.gate    = nn.Linear(dim, 1)
        self.norm    = nn.LayerNorm(dim)
        self.fusion  = nn.Sequential(
            nn.Linear(dim * 2, dim),
            nn.GELU(),
            nn.Linear(dim, dim),
        )
        self._trust_cache: Optional[torch.Tensor] = None

    # ──────────────────────────────────────────────────────────────────────────
    @torch.no_grad()
    def _update_t_stats(self, t: torch.Tensor):
        mu  = t.float().mean()
        var = t.float().var().clamp(min=1e-4)
        self.t_mean_qk = self.ema_decay * self.t_mean_qk + (1 - self.ema_decay) * mu
        self.t_var_qk  = self.ema_decay * self.t_var_qk  + (1 - self.ema_decay) * var

    def _normalise_t(self, t: torch.Tensor) -> torch.Tensor:
        t_n = (t.float() - self.t_mean_qk) / (self.t_var_qk.sqrt() + 1e-6)
        return self.t_scale_qk * t_n + self.t_shift_qk

    def _apply_rope_to_qk(self,
                           qk:  torch.Tensor,   # (E, heads, head_dim)
                           t_n: torch.Tensor,   # (E,)  normalised timestamps
                           ) -> torch.Tensor:   # (E, heads, head_dim)
        """
        Apply per-head RoPE to Q or K using normalised edge timestamps.
        Each head h rotates its head_dim vector by its own timescale band.
        """
        E = qk.size(0)
        parts = []
        for h in range(self.heads):
            sl  = qk[:, h, :]                                        # (E, hd)
            ts  = self.qk_rope_log_ts[h].exp()                       # (hd//2,)
            ang = t_n.unsqueeze(-1) / ts                             # (E, hd//2)
            cos_a, sin_a = ang.cos(), ang.sin()
            e_even = sl[:, 0::2]
            e_odd  = sl[:, 1::2]
            rot = torch.stack(
                [e_even * cos_a - e_odd * sin_a,
                 e_even * sin_a + e_odd * cos_a], dim=-1
            ).reshape(E, self.head_dim)
            parts.append(rot)
        return torch.stack(parts, dim=1)                             # (E, H, hd)

    # ──────────────────────────────────────────────────────────────────────────
    def forward(self,
                x:          torch.Tensor,   # (N, dim)
                edge_index: torch.Tensor,   # (2, E)
                edge_emb:   torch.Tensor,   # (E, dim)
                time_emb:   torch.Tensor,   # (E, dim) unused, kept for API
                edge_t:     torch.Tensor,   # (E,)  raw timestamps
                memory:     RecurrentMemory,
                evo_bank:   EvolutionBank,
                deg:        torch.Tensor,   # (N,)
                ) -> torch.Tensor:

        N    = x.size(0)
        nidx = torch.arange(N, device=x.device)
        src  = edge_index[0]

        # Update running timestamp statistics
        if self.training:
            self._update_t_stats(edge_t)
        t_n = self._normalise_t(edge_t)                             # (E,)

        mem_vals, _ = memory.read(nidx)
        embs_all, times_all = evo_bank.read(nidx)

        # GRN-projected Q/K/V
        mem_src = mem_vals[src]
        Q = self.q_grn(edge_emb, mem_src).view(-1, self.heads, self.head_dim)
        K = self.k_grn(x, mem_vals).view(N, self.heads, self.head_dim)
        V = self.v_grn(x, mem_vals).view(N, self.heads, self.head_dim)

        # Apply RoPE to Q (edge-level, already (E, H, hd))
        Q_rope = self._apply_rope_to_qk(Q, t_n)                     # (E, H, hd)
        # K will be indexed as K_j inside message() — apply RoPE there

        out = self.propagate(
            edge_index,
            x=x,
            Q_rope=Q_rope,
            K=K,
            V=V,
            mem_val=mem_vals,
            embs=embs_all,
            times=times_all,
            edge_t=edge_t,
            t_n=t_n,
            size=(N, N),
        )
        out = out.view(N, self.dim)

        # Temporal message bank
        if self.msg_bank is not None:
            out = self.msg_bank.read_and_attend(nidx, out)
            self.msg_bank.write(nidx, out)

        # Fuse with GRU memory + gated residual
        h   = memory.get_hidden(nidx)
        out = self.fusion(torch.cat([out, h], dim=-1))
        out = self.o_proj(out)
        g   = torch.sigmoid(self.gate(x))
        return self.norm(g * x + (1 - g) * out)

    # ──────────────────────────────────────────────────────────────────────────
    def message(self,
                x_i:      torch.Tensor,   # (E, dim)
                x_j:      torch.Tensor,
                Q_rope:   torch.Tensor,   # (E, H, hd)  Q already RoPE-encoded
                K_j:      torch.Tensor,   # (E, H, hd)  K from node j (raw)
                V_j:      torch.Tensor,   # (E, H, hd)
                mem_val_i: torch.Tensor,  # (E, dim)
                mem_val_j: torch.Tensor,
                embs_i:   torch.Tensor,   # (E, K, dim)
                embs_j:   torch.Tensor,
                times_i:  torch.Tensor,   # (E, K)
                times_j:  torch.Tensor,
                edge_t:   torch.Tensor,   # (E,)
                t_n:      torch.Tensor,   # (E,)  normalised timestamps
                index:    torch.Tensor,
                ) -> torch.Tensor:

        E = x_i.size(0)

        # ── [2] Apply RoPE to K_j using normalised timestamps ─────────────────
        # After rotation, Q_rope · K_j_rope reflects both semantic match AND
        # temporal proximity between the edge time and j's contribution time.
        K_j_rope = self._apply_rope_to_qk(K_j, t_n)                 # (E, H, hd)

        # ── [3] Per-head bilinear score: score_h = (Q^T U_h)(V_h^T K) ────────
        # Factored bilinear: avoids materialising the full hd×hd matrix.
        # Q^T U_h : (E, r)   and   V_h^T K : (E, r)
        # score_h = dot product of these two r-dim vectors.
        scores = torch.zeros(E, self.heads, device=x_i.device)
        for h in range(self.heads):
            q_h = Q_rope[:, h, :]                                    # (E, hd)
            k_h = K_j_rope[:, h, :]                                  # (E, hd)
            U_h = self.bilin_U[h]                                    # (hd, r)
            V_h = self.bilin_V[h]                                    # (hd, r)
            # (E, r) · (E, r) summed → (E,)
            scores[:, h] = ((q_h @ U_h) * (k_h @ V_h)).sum(-1) * self.scale

        # ── [4] Sinkhorn trust (K-view only) ─────────────────────────────────
        trust = self.trust(
            embs_i=embs_i, embs_j=embs_j,
            t_i=times_i,   t_j=times_j,
        )                                                            # (E,) ∈ (0,2)
        self._trust_cache = trust.detach()

        # Trust-scaled softmax
        scores = scores * trust.unsqueeze(-1)
        scores = softmax(scores, index)
        scores = self.dropout(scores)

        # ── [5] Value + multi-resolution temporal message encoding ────────────
        msg = (V_j * scores.unsqueeze(-1)).reshape(E, self.dim)
        msg = self.msg_time_enc(msg, edge_t)                         # (E, dim)
        return msg


# ═══════════════════════════════════════════════════════════════════════════════
# 11.  EnhancedTemporalGNN  — full model
# ═══════════════════════════════════════════════════════════════════════════════

class EnhancedTemporalGNN(nn.Module):
    """
    Full temporal GNN with K-view trajectory trust for representation learning.

    Expected batch fields:
        batch.x          (N, in_dim)    node features
        batch.edge_index (2, E)         directed edges [src, dst]
        batch.msg        (E, edge_dim)  edge / message features
        batch.t          (E,)           edge timestamps

    Returns:
        x  (N, hidden_dim)  node representations
    """
    def __init__(
        self,
        num_nodes:       int,
        in_dim:          int,
        edge_dim:        int,
        hidden_dim:      int,
        num_layers:      int   = 2,
        heads:           int   = 8,
        dropout:         float = 0.1,
        window:          int   = 8,
        t2v_dim:         int   = 16,
        bank_size:       int   = 8,
        bilinear_rank:   int   = 8,
        memory_momentum: float = 0.9,
    ):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.num_nodes  = num_nodes
        self.window     = window

        # Input projections
        self.input_proj = nn.Linear(in_dim, hidden_dim)
        self.edge_enc   = nn.Linear(edge_dim, hidden_dim)
        self.time_enc   = TimeEncoder(hidden_dim)

        # TransformerConv snapshot encoder: produces embeddings stored in EvolutionBank
        self.snapshot_gnn = SnapshotGNN(hidden_dim, heads=heads, dropout=dropout)

        # Persistent state
        self.memory   = RecurrentMemory(num_nodes, hidden_dim, momentum=memory_momentum)
        self.evo_bank = EvolutionBank(num_nodes, hidden_dim, window=window)

        # Main message-passing layers
        self.layers = nn.ModuleList([
            TemporalTrustAttention(
                hidden_dim, heads=heads, dropout=dropout,
                window=window, t2v_dim=t2v_dim,
                num_nodes=num_nodes, bank_size=bank_size,
                t_heads=4, bilinear_rank=bilinear_rank,
            )
            for _ in range(num_layers)
        ])

        self.final_norm = nn.LayerNorm(hidden_dim)
        self._init_weights()

    # --------------------------------------------------------------------------
    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)

    # --------------------------------------------------------------------------
    def forward(self, batch) -> torch.Tensor:
        N          = batch.x.size(0)
        device     = batch.x.device
        edge_index = batch.edge_index
        src, dst   = edge_index

        # ── Encode inputs ─────────────────────────────────────────────────────
        x        = self.input_proj(batch.x)                         # (N, hidden)
        edge_emb = self.edge_enc(batch.msg)                         # (E, hidden)

        # ── Node degrees ──────────────────────────────────────────────────────
        deg = degree(dst, num_nodes=N, dtype=torch.float).clamp(min=1.0)

        # ── Temporal encoding (for TimeEncoder / GRU context) ─────────────────
        node_last_t = torch.zeros(N, device=device)
        node_last_t.scatter_reduce_(
            0, src, batch.t.float(), reduce='amax', include_self=True
        )
        t_rel    = batch.t.float() - node_last_t[src]
        time_emb = self.time_enc(batch.t, t_rel)                    # (E, hidden)

        # ── Compute snapshot embedding for EvolutionBank ──────────────────────
        # TransformerConv operates directly on node features and edge_attr.
        # No time signal here — time is injected in TemporalPositionalEncoding.
        snapshot_emb = self.snapshot_gnn(x, edge_index, edge_emb)   # (N, hidden)

        # ── Message passing layers ─────────────────────────────────────────────
        # Pass raw edge timestamps (batch.t) separately for RoPE and decay gate
        for layer in self.layers:
            x = layer(x, edge_index, edge_emb, time_emb,
                      batch.t.float(),                               # edge_t
                      self.memory, self.evo_bank, deg)

        x = self.final_norm(x)

        # ── Update persistent state ───────────────────────────────────────────
        nidx      = torch.arange(N, device=device)
        current_t = batch.t.float().max().expand(N)

        # Write snapshot to EvolutionBank (the rich contextual embedding)
        self.evo_bank.write(nidx, snapshot_emb, current_t)
        # Update GRU memory
        self.memory.write(nidx, x)

        return x

    # --------------------------------------------------------------------------
    def get_trust_scores(self, layer_idx: int = -1) -> Optional[torch.Tensor]:
        """Cached trust scores (E,) in (0, 2) from last forward()."""
        return self.layers[layer_idx]._trust_cache

    @torch.no_grad()
    def reset_memory(self):
        self.memory.hidden.zero_()
        self.memory.variance.fill_(0.1)
        self.evo_bank.reset()
        for layer in self.layers:
            if layer.msg_bank is not None:
                layer.msg_bank.reset()


# ═══════════════════════════════════════════════════════════════════════════════
# Quick sanity check
# ═══════════════════════════════════════════════════════════════════════════════
if __name__ == "__main__":
    import types

    torch.manual_seed(42)

    N, E      = 50, 120
    in_dim    = 32
    edge_dim  = 16
    hidden    = 64
    num_nodes = 100
    K         = 8

    def make_batch(t_offset=0.0):
        return types.SimpleNamespace(
            x          = torch.randn(N, in_dim),
            edge_index = torch.randint(0, N, (2, E)),
            msg        = torch.randn(E, edge_dim),
            t          = torch.rand(E) * 1000 + t_offset,
        )

    model = EnhancedTemporalGNN(
        num_nodes=num_nodes, in_dim=in_dim, edge_dim=edge_dim,
        hidden_dim=hidden, num_layers=2, heads=4,
        window=K, t2v_dim=16, bank_size=8, bilinear_rank=8,
    )

    for step in range(3):
        emb   = model(make_batch(step * 1000))
        trust = model.get_trust_scores()
        print(f"[pass {step+1}] emb: {emb.shape} | "
              f"trust: [{trust.min():.4f}, {trust.max():.4f}] "
              f"mean={trust.mean():.4f}")
        assert trust.min() > 0.0 and trust.max() < 2.0

    print("\nAll checks passed. Trust is in (0, 2).")

[pass 1] emb: torch.Size([50, 64]) | trust: [1.1941, 1.1941] mean=1.1941
[pass 2] emb: torch.Size([50, 64]) | trust: [1.1617, 1.1941] mean=1.1767
[pass 3] emb: torch.Size([50, 64]) | trust: [1.1557, 1.1941] mean=1.1759

All checks passed. Trust is in (0, 2).


In [2]:
print(model)

EnhancedTemporalGNN(
  (input_proj): Linear(in_features=32, out_features=64, bias=True)
  (edge_enc): Linear(in_features=16, out_features=64, bias=True)
  (time_enc): TimeEncoder(
    (t2v_abs): Time2Vec()
    (t2v_rel): Time2Vec()
    (proj): Sequential(
      (0): Linear(in_features=64, out_features=64, bias=True)
      (1): GELU(approximate='none')
      (2): Linear(in_features=64, out_features=64, bias=True)
    )
  )
  (snapshot_gnn): SnapshotGNN(
    (conv1): TransformerConv(64, 16, heads=4)
    (norm1): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
    (conv2): TransformerConv(64, 16, heads=4)
    (norm2): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
    (out_proj): Linear(in_features=64, out_features=64, bias=True)
    (gate): Linear(in_features=64, out_features=1, bias=True)
  )
  (memory): RecurrentMemory(
    (gru): GRUCell(64, 64)
  )
  (evo_bank): EvolutionBank()
  (layers): ModuleList(
    (0-1): 2 x TemporalTrustAttention()
  )
  (final_norm): LayerNorm((6

In [ ]:
model = EnhancedTemporalGNN(
    num_nodes  = data.num_nodes,
    in_dim     = data.x.size(1),          # 1 for wikipedia
    edge_dim   = data.msg.size(1),         # 172 for wikipedia
    hidden_dim = args.hidden_channels,     # 64
    num_layers = args.nums_layers,         # 2
    heads      = args.heads,               # 8
    dropout    = args.dropout,             # 0.3
    window     = 8,
    memory_momentum = 0.9,
    t2v_dim=32,
    bilinear_rank=8
).to(device)

: 

### Test Model

In [6]:
import torch
import torch.nn as nn
import torch.nn.functional as F

from torch_geometric.datasets import JODIEDataset
from torch_geometric.nn import TransformerConv
from torch_geometric.nn.models.tgn import (
    TGNMemory,
    IdentityMessage,
    LastAggregator
)

In [17]:
import torch
import torch.nn as nn
import torch.nn.functional as F

from torch_geometric.datasets import JODIEDataset
from torch_geometric.nn import TransformerConv
from torch_geometric.nn.models.tgn import TGNMemory, IdentityMessage, LastAggregator

from sklearn.metrics import roc_auc_score

In [18]:
dataset = JODIEDataset(root="data", name="wikipedia")

data = dataset[0]

src = data.src
dst = data.dst
msg = data.msg
t = data.t

num_nodes = data.num_nodes
msg_dim = msg.shape[1]

In [19]:
num_events = src.shape[0]

train_size = int(num_events * 0.8)

train_idx = torch.arange(train_size)
test_idx = torch.arange(train_size, num_events)

In [20]:
class TemporalEncoder(nn.Module):

    def __init__(self, num_nodes, msg_dim, emb_dim=128):

        super().__init__()

        self.memory = TGNMemory(
            num_nodes=num_nodes,
            raw_msg_dim=msg_dim,
            memory_dim=emb_dim,
            time_dim=emb_dim,
            message_module=IdentityMessage(msg_dim, emb_dim, emb_dim),
            aggregator_module=LastAggregator()
        )

        self.gnn = TransformerConv(
            emb_dim,
            emb_dim,
            heads=2,
            dropout=0.1
        )

        self.mlp = nn.Sequential(
            nn.Linear(emb_dim * 2, emb_dim),
            nn.ReLU(),
            nn.Linear(emb_dim, emb_dim)
        )

    def forward(self, src, dst):

        node_ids = torch.arange(self.memory.num_nodes)

        z = self.memory(node_ids).float()

        edge_index = torch.stack([src, dst])

        z = self.gnn(z, edge_index)

        z = self.mlp(z)

        return z

In [21]:
class EdgeDecoder(nn.Module):

    def __init__(self, dim):

        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(dim * 2, dim),
            nn.ReLU(),
            nn.Linear(dim, 1)
        )

    def forward(self, zu, zv):

        x = torch.cat([zu, zv], dim=1)

        return torch.sigmoid(self.net(x))

In [22]:
class ProjectionHead(nn.Module):

    def __init__(self, dim):

        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(dim, dim),
            nn.ReLU(),
            nn.Linear(dim, dim)
        )

    def forward(self, z):

        return self.net(z)

In [23]:
class TCGAD(nn.Module):

    def __init__(self, num_nodes, msg_dim, emb_dim=128):

        super().__init__()

        self.encoder = TemporalEncoder(num_nodes, msg_dim, emb_dim)

        self.decoder = EdgeDecoder(emb_dim)

        self.projector = ProjectionHead(emb_dim)

    def forward(self, src, dst):

        z = self.encoder(src, dst)

        zu = z[src]
        zv = z[dst]

        prob = self.decoder(zu, zv)

        return z, zu, zv, prob

In [24]:
def contrastive_loss(z1, z2, tau=0.5):

    z1 = F.normalize(z1, dim=1)
    z2 = F.normalize(z2, dim=1)

    sim = torch.mm(z1, z2.t()) / tau

    labels = torch.arange(z1.size(0)).to(z1.device)

    return F.cross_entropy(sim, labels)

In [25]:
def test(model):

    model.eval()

    with torch.no_grad():

        test_src = src[test_idx]
        test_dst = dst[test_idx]

        z, zu, zv, prob = model(test_src, test_dst)

        pos_scores = prob.squeeze().cpu()

        # negative sampling
        neg_src = torch.randint(0, num_nodes, (len(test_src),))
        neg_dst = torch.randint(0, num_nodes, (len(test_dst),))

        _, zu_neg, zv_neg, prob_neg = model(neg_src, neg_dst)

        neg_scores = prob_neg.squeeze().cpu()

        scores = torch.cat([pos_scores, neg_scores])

        labels = torch.cat([
            torch.ones(len(pos_scores)),
            torch.zeros(len(neg_scores))
        ])

        auc = roc_auc_score(labels.numpy(), scores.numpy())

    return auc

In [26]:
model = TCGAD(num_nodes, msg_dim)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

for epoch in range(50):

    model.train()

    train_src = src[train_idx]
    train_dst = dst[train_idx]

    z, zu, zv, prob = model(train_src, train_dst)

    recon_loss = -torch.log(prob + 1e-8).mean()

    noise = torch.randn_like(z) * 0.01

    z2 = z + noise

    p1 = model.projector(z)
    p2 = model.projector(z2)

    cont_loss = contrastive_loss(p1, p2)

    loss = recon_loss + 0.5 * cont_loss

    optimizer.zero_grad()

    loss.backward()

    optimizer.step()

    auc = test(model)

    print(
        f"Epoch {epoch} | "
        f"Loss {loss.item():.4f} | "
        f"Test AUC {auc:.4f}"
    )

AttributeError: 'tuple' object has no attribute 'float'

In [28]:
import torch
import torch.nn as nn
import torch.nn.functional as F

from torch_geometric.datasets import JODIEDataset
from torch_geometric.nn import TransformerConv
from torch_geometric.nn.models.tgn import TGNMemory, IdentityMessage, LastAggregator

from sklearn.metrics import roc_auc_score

# -------------------------------
# Load dataset
# -------------------------------

dataset = JODIEDataset(root="data", name="wikipedia")
data = dataset[0]

src = data.src
dst = data.dst
msg = data.msg
t = data.t

num_nodes = data.num_nodes
msg_dim = msg.shape[1]

# train test split
num_events = src.shape[0]
train_size = int(num_events * 0.8)

train_idx = torch.arange(train_size)
test_idx = torch.arange(train_size, num_events)

# -------------------------------
# Temporal Encoder
# -------------------------------

class TemporalEncoder(nn.Module):

    def __init__(self, num_nodes, msg_dim, emb_dim=128):
        super().__init__()

        self.memory = TGNMemory(
            num_nodes=num_nodes,
            raw_msg_dim=msg_dim,
            memory_dim=emb_dim,
            time_dim=emb_dim,
            message_module=IdentityMessage(msg_dim, emb_dim, emb_dim),
            aggregator_module=LastAggregator()
        )

        self.gnn = TransformerConv(
            emb_dim,
            emb_dim,
            heads=2,
            dropout=0.1
        )

        self.mlp = nn.Sequential(
            nn.Linear(emb_dim*2, emb_dim),
            nn.ReLU(),
            nn.Linear(emb_dim, emb_dim)
        )

    def forward(self, src, dst):

        node_ids = torch.arange(self.memory.num_nodes)

        memory, last_update = self.memory(node_ids)

        z = memory.float()

        edge_index = torch.stack([src, dst])

        z = self.gnn(z, edge_index)

        z = self.mlp(z)

        return z

# -------------------------------
# Decoder
# -------------------------------

class EdgeDecoder(nn.Module):

    def __init__(self, dim):
        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(dim*2, dim),
            nn.ReLU(),
            nn.Linear(dim, 1)
        )

    def forward(self, zu, zv):

        x = torch.cat([zu, zv], dim=1)

        return torch.sigmoid(self.net(x))

# -------------------------------
# Contrastive head
# -------------------------------

class ProjectionHead(nn.Module):

    def __init__(self, dim):
        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(dim, dim),
            nn.ReLU(),
            nn.Linear(dim, dim)
        )

    def forward(self, z):
        return self.net(z)

# -------------------------------
# Full Model
# -------------------------------

class TCGAD(nn.Module):

    def __init__(self, num_nodes, msg_dim, emb_dim=128):
        super().__init__()

        self.encoder = TemporalEncoder(num_nodes, msg_dim, emb_dim)

        self.decoder = EdgeDecoder(emb_dim)

        self.projector = ProjectionHead(emb_dim)

    def forward(self, src, dst):

        z = self.encoder(src, dst)

        zu = z[src]
        zv = z[dst]

        prob = self.decoder(zu, zv)

        return z, zu, zv, prob

# -------------------------------
# Contrastive loss
# -------------------------------

def contrastive_loss(z1, z2, tau=0.5):

    z1 = F.normalize(z1, dim=1)
    z2 = F.normalize(z2, dim=1)

    sim = torch.mm(z1, z2.t()) / tau

    labels = torch.arange(z1.size(0))

    return F.cross_entropy(sim, labels)

# -------------------------------
# Test function (AUC)
# -------------------------------

def test(model):

    model.eval()

    with torch.no_grad():

        test_src = src[test_idx]
        test_dst = dst[test_idx]

        z, zu, zv, prob = model(test_src, test_dst)

        pos_scores = prob.squeeze()

        neg_src = torch.randint(0, num_nodes, (len(test_src),))
        neg_dst = torch.randint(0, num_nodes, (len(test_dst),))

        _, zu_neg, zv_neg, prob_neg = model(neg_src, neg_dst)

        neg_scores = prob_neg.squeeze()

        scores = torch.cat([pos_scores, neg_scores]).cpu()

        labels = torch.cat([
            torch.ones(len(pos_scores)),
            torch.zeros(len(neg_scores))
        ]).cpu()

        auc = roc_auc_score(labels.numpy(), scores.numpy())

    return auc

# -------------------------------
# Training
# -------------------------------

model = TCGAD(num_nodes, msg_dim)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

for epoch in range(30):

    model.train()

    train_src = src[train_idx]
    train_dst = dst[train_idx]

    z, zu, zv, prob = model(train_src, train_dst)

    recon_loss = -torch.log(prob + 1e-8).mean()

    noise = torch.randn_like(z) * 0.01
    z2 = z + noise

    p1 = model.projector(z)
    p2 = model.projector(z2)

    cont_loss = contrastive_loss(p1, p2)

    loss = recon_loss + 0.5 * cont_loss

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    auc = test(model)

    print(f"Epoch {epoch} | Loss {loss.item():.4f} | Test AUC {auc:.4f}")

Epoch 0 | Loss 5.2737 | Test AUC 0.0179
Epoch 1 | Loss 5.2446 | Test AUC 0.0165


RuntimeError: Trying to backward through the graph a second time (or directly access saved tensors after they have already been freed). Saved intermediate values of the graph are freed when you call .backward() or autograd.grad(). Specify retain_graph=True if you need to backward through the graph a second time or if you need to access saved tensors after calling backward.